In [ ]:
import os
os.environ.setdefault('TORCH_CUDA_ARCH_LIST', '7.5') # For Kaggle GPU compatibility
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import random
import cv2
from PIL import Image
from torch import amp  # For mixed precision training (new API)
import networkx as nx  # For graph operations

# Simplified GNN for graph processing
class PyramidalGNN(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(PyramidalGNN, self).__init__()
        self.lin = nn.Linear(in_channels, out_channels)
        self.gate = nn.Linear(in_channels * 2, out_channels)

    def forward(self, x, edge_index):
        # Simplified version - just apply linear transformation
        return self.lin(x)



# /kaggle/input/celeba-resized-6464/img_align_celeba/CelebA_Image_Cropped_64
# Configuration parameters
prune_threshold = 0.2  # Threshold for pruning low-value nodes
lambda_l1 = 100.0  # L1 reconstruction loss weight
n_critic = 10  # Number of discriminator updates per generator update (increased for better stability)

# Global prune_graph function using NetworkX
def prune_graph(model, graph, threshold=prune_threshold):
    # NetworkX pruning
    deg_cent = nx.degree_centrality(graph)
    low_value_nodes = [n for n, score in deg_cent.items() if score < threshold]
    graph.remove_nodes_from(low_value_nodes)
    return graph

# Try to import MTCNN for landmark detection
try:
    from mtcnn import MTCNN
    USE_MTCNN = False # Disable during training to avoid NumPy conversions
    print("MTCNN imported successfully for landmark detection (disabled for training).")
except ImportError:
    print("MTCNN not found. Landmark loss will be simplified.")
    USE_MTCNN = False
# ============================================================================
# CHECKPOINT CONFIGURATION - Resume Training Setup
# تنظیمات Checkpoint - برای ادامه آموزش
# ============================================================================

# 🔄 RESUME TRAINING: Set to True to continue from previous checkpoint
# ادامه آموزش: برای ادامه از checkpoint قبلی، True کنید
RESUME_TRAINING = True  # True: Resume from checkpoint | False: Start from scratch

# 📂 CHECKPOINT PATHS: Where to load/save checkpoints
# مسیرهای Checkpoint: محل بارگذاری/ذخیره checkpoint‌ها
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
CHECKPOINT_INTERVAL = 1  # Save checkpoint every N epochs (1 = every epoch)

# 📥 LOAD FROM KAGGLE DATASET: If you uploaded models as Kaggle dataset
# بارگذاری از Kaggle Dataset: اگر مدل‌ها را به عنوان dataset آپلود کرده‌اید
USE_KAGGLE_INPUT = True  # Set True if loading from Kaggle dataset input
KAGGLE_INPUT_DATASET = '/kaggle/input/image-inpainting-v1/pytorch/default/4'  # Update this path (e.g., '/kaggle/input/checkpoints')
# /kaggle/input/image-inpainting/pytorch/default/2
# /kaggle/input/image-inpainting-v1/pytorch/default/4
# /kaggle/input/image-inpainting-v1/pytorch/default/3
# /kaggle/input/image-inpainting/pytorch/default/4
# /kaggle/input/image-inpainting/pytorch/default/3
# 🗂️ LOAD FROM MODEL FILES: If you only have model state_dict files (*.pth)
# بارگذاری از فایل‌های مدل: اگر فقط فایل‌های state_dict مدل دارید
LOAD_FROM_MODELS = True  # Set True if loading from models folder (coarse_generator_best.pth, etc.)
MODELS_DIR = '/kaggle/working/models'  # Path to models folder
USE_BEST_MODELS = True  # True: Load *_best.pth | False: Load *_bottom_half.pth

# ⚠️ نکته: اگر از LOAD_FROM_MODELS استفاده کنید، فقط وزن‌های مدل بارگذاری می‌شوند
# optimizer/scheduler/training history بارگذاری نمی‌شوند و از اول مقداردهی می‌شوند

# ⚙️ TRAINING START POINT: Which epoch to start from
# نقطه شروع آموزش: از کدام epoch شروع شود
# Note: This will be automatically set when loading checkpoint
# توجه: این مقدار به صورت خودکار هنگام بارگذاری checkpoint تنظیم می‌شود

print("="*70)
print("🔧 CHECKPOINT CONFIGURATION")
print("="*70)
print(f"📌 Resume Training: {RESUME_TRAINING}")
print(f"📁 Checkpoint Directory: {CHECKPOINT_DIR}")
print(f"💾 Save Interval: Every {CHECKPOINT_INTERVAL} epoch(s)")
if USE_KAGGLE_INPUT:
    print(f"📥 Loading from Kaggle Dataset: {KAGGLE_INPUT_DATASET}")
if LOAD_FROM_MODELS:
    print(f"🗂️  Load from Models: {MODELS_DIR}")
    print(f"   Using: {'*_best.pth' if USE_BEST_MODELS else '*_bottom_half.pth'}")
print("="*70 + "\n")

# ============================================================================
# TRAINING CONFIGURATION - Initial configurations
# تنظیمات آموزش - پیکربندی اولیه
# ============================================================================

image_size = (64, 64)
batch_size = 128 # Further reduced for stability
eval_batch_size = 1
num_epochs = 10 # Increased for better convergence
learning_rate_g = 0.00005 # 5e-5 for Generator (reduced for more stable training)
learning_rate_d = 0.0002 # 2e-4 for Discriminator (reduced for better balance)
lambda_adv = 0.1 # Adversarial weight increased for better GAN balance
lambda_style = 0.05 # Reduced style weight
lambda_fm = 0.05 # Reduced feature matching weight
lambda_rec = 1.0 # Reduced reconstruction weight
lambda_identity = 0.005 # Reduced identity weight
lambda_landmark = 0.01 # Reduced landmark weight
lambda_perc = 0.1 # Reduced perceptual weight
lambda_vae = 0.01 # Further reduced VAE weight
lambda_struct = 0.05 # Reduced structural weight
lambda_d_reg = 1e-4 # Increased discriminator regularization
lambda_attention = 0.05 # Reduced attention loss weight
# NEW: Edge-Guided Inpainting Loss Weights (Fine-tuned for better quality)
lambda_edge = 0.7 # Edge loss weight for preserving nose/lips edges (increased from 0.5)
lambda_hf = 0.5 # High-frequency loss weight to reduce blurriness (increased from 0.3)
lambda_gradient = 0.6 # Gradient loss weight for sharp edges (increased from 0.4)
lambda_facial = 0.8 # Facial region loss weight for nose/lips focus (increased from 0.6)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# Data splitting configuration
def split_dataset(dataset_path, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    image_files = [f for f in os.listdir(dataset_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random.shuffle(image_files)
    total_files = len(image_files)
    train_end = int(total_files * train_ratio)
    val_end = train_end + int(total_files * val_ratio)
    train_files, val_files, test_files = image_files[:train_end], image_files[train_end:val_end], image_files[val_end:]
    print(f"Dataset split: Total={total_files}, Train={len(train_files)} ({len(train_files)/total_files*100:.1f}%), "
          f"Val={len(val_files)} ({len(val_files)/total_files*100:.1f}%), Test={len(test_files)} ({len(test_files)/total_files*100:.1f}%)")
    return train_files, val_files, test_files
# Custom Dataset
class CelebADataset(Dataset):
    def __init__(self, dataset_path, file_list, image_size):
        self.dataset_path = dataset_path
        self.file_list = file_list
        self.transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),  # Data augmentation: random flip
            transforms.RandomRotation(10),           # Data augmentation: slight rotation
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # -1 to 1
        ])
   
    def __len__(self):
        return len(self.file_list)
   
    def __getitem__(self, idx):
        img_path = os.path.join(self.dataset_path, self.file_list[idx])
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        return image
def create_split_datasets(dataset_path, train_files, val_files, test_files, image_size, batch_size):
    train_dataset = CelebADataset(dataset_path, train_files, image_size)
    val_dataset = CelebADataset(dataset_path, val_files, image_size)
    test_dataset = CelebADataset(dataset_path, test_files, image_size)
   
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader
# Enhanced ViT-based Preprocessor with Multi-layer Attention and Graph Pruning
class ViTPreprocessor(nn.Module):
    def __init__(self, dim=128, num_heads=8, patch_size=8, num_layers=3):
        super().__init__()
        self.patch_size = patch_size
        self.dim = dim
        self.num_heads = num_heads
        self.num_patches = (image_size[0] // patch_size) * (image_size[1] // patch_size)
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size, padding=0)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, dim))
       
        # Multi-layer attention blocks
        self.attention_blocks = nn.ModuleList([
            nn.ModuleDict({
                'attn': nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=0.1),
                'norm1': nn.LayerNorm(dim, eps=1e-6),
                'norm2': nn.LayerNorm(dim, eps=1e-6),
                'mlp': nn.Sequential(
                    nn.Linear(dim, dim * 4),
                    nn.GELU(),
                    nn.Dropout(0.1),
                    nn.Linear(dim * 4, dim),
                    nn.Dropout(0.1)
                )
            }) for _ in range(num_layers)
        ])
       
        self.final_norm = nn.LayerNorm(dim, eps=1e-6)
        self.gnn = PyramidalGNN(dim, dim)  # Add PyramidalGNN
   
    def build_graph(self, features):
        B, C, H, W = features.shape
        nodes = features.view(B, C, -1).permute(0, 2, 1)  # [B, num_patches, dim]
        
        # Use NetworkX for graph construction
        G = nx.Graph()
        for i in range(nodes.shape[1]):
            G.add_node(i, feat=nodes[0, i])
        for i in range(nodes.shape[1]):
            for j in range(i+1, nodes.shape[1]):
                sim = F.cosine_similarity(nodes[0, i:i+1], nodes[0, j:j+1])
                if sim > 0.5:
                    G.add_edge(i, j, weight=sim)
        return G
   
   
    def graph_enhanced_forward(self, x, graph):
        # Check if graph is empty after pruning
        node_list = list(graph.nodes)
        if not node_list:
            return x  # Return input features directly
        
        # Simple message passing with NetworkX
        for node in graph.nodes:
            neighbors = list(graph.neighbors(node))
            if neighbors:
                neighbor_feats = torch.stack([graph.nodes[n]['feat'] for n in neighbors])
                aggregated = torch.mean(neighbor_feats, dim=0)
                graph.nodes[node]['feat'] = (graph.nodes[node]['feat'] + aggregated) / 2
        
        # Final check before stacking
        final_nodes = list(graph.nodes)
        if not final_nodes:
            return x
        
        # Ensure all features have the same dtype
        node_features = [graph.nodes[n]['feat'] for n in sorted(final_nodes)]
        return torch.stack(node_features)
   
    def reconstruct_features(self, pruned_feats, pruned_indices, B, C, H, W):
        num_patches = H * W
        # Ensure dtype matches the input features
        full_feats = torch.zeros(B, num_patches, C, device=pruned_feats.device, dtype=pruned_feats.dtype)
        full_feats[:, pruned_indices] = pruned_feats
        return full_feats.reshape(B, H, W, C).permute(0, 3, 1, 2)
   
    def forward(self, x):
        embedded = self.patch_embed(x)  # [B, dim, H/p, W/p]
        B, C, H, W = embedded.shape
        flat = embedded.permute(0, 2, 3, 1).reshape(B, H * W, C)
        
        # Adaptive position embedding based on actual patch count
        num_patches = H * W
        if num_patches != self.num_patches:
            # Interpolate position embedding to match current patch count
            # Reshape to 3D for linear interpolation: [1, dim, num_patches]
            pos_embed_3d = self.pos_embed.permute(0, 2, 1)  # [1, dim, num_patches]
            pos_embed_adaptive = F.interpolate(
                pos_embed_3d,
                size=num_patches,
                mode='linear',
                align_corners=False
            ).permute(0, 2, 1)  # [1, num_patches, dim]
        else:
            pos_embed_adaptive = self.pos_embed
        
        flat = flat + pos_embed_adaptive
        
        # Build and prune graph
        graph = self.build_graph(embedded)
        pruned_graph = prune_graph(self, graph)
        
        # Get pruned features using NetworkX
        pruned_indices = list(pruned_graph.nodes)
        if pruned_indices:  # Check if not empty
            pruned_feats = torch.stack([pruned_graph.nodes[i]['feat'] for i in pruned_indices])
        else:
            pruned_feats = flat[0].clone()  # Use original flattened features for batch 0
            pruned_indices = list(range(flat.shape[1]))  # All indices
        
        # Enhanced processing on high-value nodes
        for block in self.attention_blocks:
            # Self-attention on pruned features
            attn_out, _ = block['attn'](pruned_feats, pruned_feats, pruned_feats)
            pruned_feats = block['norm1'](pruned_feats + attn_out)
            
            # MLP
            mlp_out = block['mlp'](pruned_feats)
            pruned_feats = block['norm2'](pruned_feats + mlp_out)
        
        # Graph-enhanced message passing
        enhanced_pruned = self.graph_enhanced_forward(pruned_feats, pruned_graph)
        
        # Reconstruct full features with low-value nodes getting minimal processing
        x = self.reconstruct_features(enhanced_pruned, pruned_indices, B, C, H, W)
        
        # Minimal processing for pruned (low-value) regions - e.g., average pooling
        full_x = self.final_norm(x.view(B, C, -1).permute(0, 2, 1)).permute(0, 2, 1).view(B, C, H, W)
        return full_x
# VAE Sampling Layer
class VAESampling(nn.Module):
    def __init__(self):
        super().__init__()
   
    def forward(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
# Enhanced VAE Encoder with Residual Connections and Graph Integration (No BatchNorm, Strided Conv)
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim
       
        # Enhanced encoder with strided convolutions (no batch norm)
        self.conv1 = nn.Conv2d(128, 64, kernel_size=4, stride=2, padding=1) # [B, 128, 8, 8] -> [B, 64, 4, 4]
        self.conv2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1) # [B, 64, 4, 4] -> [B, 128, 2, 2]
       
        # Residual block (no batch norm)
        self.res_conv = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
       
        self.flatten = nn.Flatten()
        # Use adaptive pooling to ensure consistent output size
        self.adaptive_pool = nn.AdaptiveAvgPool2d((2, 2))  # Always output 2x2
        self.fc_mu = nn.Linear(128 * 2 * 2, latent_dim) # 128 * 2 * 2 = 512
        self.fc_logvar = nn.Linear(128 * 2 * 2, latent_dim) # 128 * 2 * 2 = 512
   
    def forward(self, x):
        # Check input size and use adaptive approach for very small inputs
        B, C, H, W = x.shape
        
        # If input is too small for conv layers, use adaptive pooling directly
        if H < 4 or W < 4:
            # Use adaptive pooling to get to a reasonable size first
            x = F.adaptive_avg_pool2d(x, (8, 8))  # Resize to 8x8 minimum
            x = F.relu(self.conv1(x))  # 8x8 -> 4x4
            x = F.relu(self.conv2(x))  # 4x4 -> 2x2
        else:
            x = F.relu(self.conv1(x))
            x = F.relu(self.conv2(x))
       
        # Residual connection
        residual = x
        x = F.relu(self.res_conv(x))
        x = x + residual
       
        # Use adaptive pooling to ensure consistent size
        x = self.adaptive_pool(x)  # Always [B, 128, 2, 2]
        x = self.flatten(x)  # Always [B, 512]
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar
# Enhanced VAE Decoder with Residual Connections and Graph Integration (No BatchNorm)
class VAEDecoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim
       
        # Enhanced decoder with strided transposed convolutions (no batch norm)
        self.fc = nn.Linear(latent_dim, 8 * 8 * 128)
       
        # Residual block before upsampling (no batch norm)
        self.res_conv = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
       
        self.deconv1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.deconv3 = nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1)
   
    def forward(self, z):
        x = F.relu(self.fc(z))
        x = x.view(-1, 128, 8, 8)
       
        # Residual connection
        residual = x
        x = F.relu(self.res_conv(x))
        x = x + residual
       
        x = F.relu(self.deconv1(x)) # 8x8 -> 16x16
        x = F.relu(self.deconv2(x)) # 16x16 -> 32x32
        x = torch.tanh(self.deconv3(x)) # 32x32 -> 64x64
        return x
# Enhanced CoarseGenerator with Attention, Skip Connections, and Graph Pruning
# ============================================================================
# EDGE DETECTION AND EDGE GENERATOR
# ماژول‌های تشخیص و بازسازی لبه برای بهبود بازسازی بینی و لب‌ها
# ============================================================================

class EdgeDetector(nn.Module):
    """
    Edge detection using Sobel filters to extract edges from images
    استخراج لبه‌ها با استفاده از فیلترهای Sobel
    """
    def __init__(self):
        super().__init__()
        # Sobel filters for edge detection
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        
        self.register_buffer('sobel_x', sobel_x.repeat(3, 1, 1, 1))
        self.register_buffer('sobel_y', sobel_y.repeat(3, 1, 1, 1))
    
    def forward(self, x):
        # Apply Sobel filters to detect edges
        # x: (B, 3, H, W)
        edge_x = F.conv2d(x, self.sobel_x, padding=1, groups=3)
        edge_y = F.conv2d(x, self.sobel_y, padding=1, groups=3)
        
        # Compute edge magnitude
        edges = torch.sqrt(edge_x ** 2 + edge_y ** 2 + 1e-8)
        return edges

class EdgeGenerator(nn.Module):
    """
    Edge Generator for reconstructing edges of nose and lips
    Generator برای بازسازی لبه‌های بینی و لب‌ها
    """
    def __init__(self):
        super().__init__()
        
        # Encoder for edge feature extraction
        self.enc1 = nn.Sequential(
            nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1),  # Input: masked image + mask
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # Bottleneck with attention for edge features
        self.bottleneck = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # Attention mechanism
        self.attention = nn.MultiheadAttention(embed_dim=512, num_heads=8, dropout=0.1)
        self.attention_norm = nn.LayerNorm(512, eps=1e-6)
        
        # Decoder for edge reconstruction
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(512, 128, kernel_size=4, stride=2, padding=1),  # 256 + 256 from skip
            nn.ReLU(inplace=True)
        )
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(256, 64, kernel_size=4, stride=2, padding=1),   # 128 + 128 from skip
            nn.ReLU(inplace=True)
        )
        self.dec4 = nn.Sequential(
            nn.ConvTranspose2d(128, 3, kernel_size=4, stride=2, padding=1),    # 64 + 64 from skip
            nn.Sigmoid()  # Output edge map in [0, 1]
        )
    
    def forward(self, masked_image, mask):
        # Concatenate masked image and mask
        x = torch.cat([masked_image, mask], dim=1)
        
        # Encoder with skip connections
        enc1_out = self.enc1(x)       # 64x64 -> 32x32
        enc2_out = self.enc2(enc1_out) # 32x32 -> 16x16
        enc3_out = self.enc3(enc2_out) # 16x16 -> 8x8
        bottleneck = self.bottleneck(enc3_out)  # 8x8 -> 4x4
        
        # Apply attention to bottleneck features
        B, C, H, W = bottleneck.shape
        attn_input = bottleneck.permute(0, 2, 3, 1).reshape(B, H * W, C)
        attn_output, _ = self.attention(attn_input, attn_input, attn_input)
        enhanced_features = self.attention_norm(attn_input + attn_output)
        enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
        
        # Decoder with skip connections
        dec1_out = self.dec1(enhanced_features)              # 4x4 -> 8x8
        dec1_out = torch.cat([dec1_out, enc3_out], dim=1)    # Skip connection
        
        dec2_out = self.dec2(dec1_out)                       # 8x8 -> 16x16
        dec2_out = torch.cat([dec2_out, enc2_out], dim=1)    # Skip connection
        
        dec3_out = self.dec3(dec2_out)                       # 16x16 -> 32x32
        dec3_out = torch.cat([dec3_out, enc1_out], dim=1)    # Skip connection
        
        output = self.dec4(dec3_out)                         # 32x32 -> 64x64
        
        return output

class CoarseGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.preprocessor = ViTPreprocessor(dim=128, num_heads=8, patch_size=8, num_layers=3)
        self.encoder = VAEEncoder(latent_dim=128) # Increased latent dimension
        self.sampling = VAESampling()
        self.decoder = VAEDecoder(latent_dim=128)
       
        # Attention mechanism for better feature fusion
        self.attention = nn.MultiheadAttention(embed_dim=128, num_heads=8, dropout=0.1)
        self.attention_norm = nn.LayerNorm(128, eps=1e-6)
       
        # Skip connection processing
        self.skip_conv = nn.Conv2d(128, 128, kernel_size=1, padding=0)
   
    def forward(self, x, mask, edge_map=None):
        # If edge map is provided, concatenate with input for edge-guided inpainting
        if edge_map is not None:
            x = x + edge_map * 0.1  # Blend edge information with input
        
        # Preprocess with graph pruning
        original_features = self.preprocessor(x)
       
        # Apply attention to enhance features
        B, C, H, W = original_features.shape
        attn_input = original_features.permute(0, 2, 3, 1).reshape(B, H * W, C)
        attn_output, _ = self.attention(attn_input, attn_input, attn_input)
        enhanced_features = self.attention_norm(attn_input + attn_output)
        enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
       
        # Combine with skip connection
        enhanced_features = enhanced_features + self.skip_conv(original_features)
       
        mu, logvar = self.encoder(enhanced_features)
        z = self.sampling(mu, logvar)
        output = self.decoder(z)
        return output, mu, logvar
# Enhanced FineGenerator with U-Net Architecture, Attention, Edge-Guided, and Graph Cut Integration
class FineGenerator(nn.Module):
    def __init__(self):
        super().__init__()
       
        # Encoder with skip connections (edge-guided: 9 channels = 6 + 3 edge)
        self.enc1 = nn.Sequential(
            nn.Conv2d(9, 64, kernel_size=4, stride=2, padding=1),  # Added 3 channels for edge map
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
       
        # Bottleneck with attention
        self.bottleneck = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        )
       
        # Attention mechanism
        self.attention = nn.MultiheadAttention(embed_dim=512, num_heads=8, dropout=0.1)
        self.attention_norm = nn.LayerNorm(512, eps=1e-6)
       
        # Decoder with skip connections
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True)
        )
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(512, 128, kernel_size=4, stride=2, padding=1), # 256 + 256 from skip
            nn.ReLU(inplace=True)
        )
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(256, 64, kernel_size=4, stride=2, padding=1), # 128 + 128 from skip
            nn.ReLU(inplace=True)
        )
        self.dec4 = nn.Sequential(
            nn.ConvTranspose2d(128, 3, kernel_size=4, stride=2, padding=1), # 64 + 64 from skip
            nn.Tanh()
        )
   
    def forward(self, masked_images, coarse_output, masks, edge_map=None):
        # Ensure all inputs have the same spatial dimensions
        if masked_images.shape[2:] != coarse_output.shape[2:]:
            coarse_output = F.interpolate(coarse_output, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
        
        # If edge map is provided, concatenate it with input for edge-guided inpainting
        if edge_map is not None:
            if edge_map.shape[2:] != masked_images.shape[2:]:
                edge_map = F.interpolate(edge_map, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([masked_images, coarse_output, edge_map], dim=1)  # 3 + 3 + 3 = 9 channels
        else:
            # Fallback: use zeros for edge map if not provided
            edge_map_zeros = torch.zeros_like(masked_images)
            x = torch.cat([masked_images, coarse_output, edge_map_zeros], dim=1)
       
        # Encoder with skip connections
        enc1_out = self.enc1(x) # 64x64 -> 32x32
        enc2_out = self.enc2(enc1_out) # 32x32 -> 16x16
        enc3_out = self.enc3(enc2_out) # 16x16 -> 8x8
        bottleneck = self.bottleneck(enc3_out) # 8x8 -> 4x4
       
        # Apply attention to bottleneck features
        B, C, H, W = bottleneck.shape
        attn_input = bottleneck.permute(0, 2, 3, 1).reshape(B, H * W, C)
        attn_output, _ = self.attention(attn_input, attn_input, attn_input)
        enhanced_features = self.attention_norm(attn_input + attn_output)
        enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
       
        # Decoder with skip connections
        dec1_out = self.dec1(enhanced_features) # 4x4 -> 8x8
        dec1_out = torch.cat([dec1_out, enc3_out], dim=1) # Skip connection
       
        dec2_out = self.dec2(dec1_out) # 8x8 -> 16x16
        dec2_out = torch.cat([dec2_out, enc2_out], dim=1) # Skip connection
       
        dec3_out = self.dec3(dec2_out) # 16x16 -> 32x32
        dec3_out = torch.cat([dec3_out, enc1_out], dim=1) # Skip connection
       
        output = self.dec4(dec3_out) # 32x32 -> 64x64
       
        # Ensure output matches input size
        if output.shape[2:] != masked_images.shape[2:]:
            output = F.interpolate(output, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
       
        return output
# Stronger Discriminator with Spectral Normalization (No BatchNorm, More Layers)
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
       
        # Stronger discriminator with more layers and capacity (no batch norm)
        self.conv1 = nn.utils.spectral_norm(nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1))
        self.conv2 = nn.utils.spectral_norm(nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1))
        self.conv3 = nn.utils.spectral_norm(nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1))
        self.conv4 = nn.utils.spectral_norm(nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1))
        # Additional layers for stronger discriminator
        self.conv5 = nn.utils.spectral_norm(nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1))
        self.conv6 = nn.utils.spectral_norm(nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1))
       
        # Global average pooling for stability
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        # Deeper fully connected layers
        self.fc1 = nn.utils.spectral_norm(nn.Linear(512, 256))
        self.fc2 = nn.utils.spectral_norm(nn.Linear(256, 1))
       
        # Feature layers for feature matching loss
        self.feature_layers = [self.conv1, self.conv2, self.conv3, self.conv4, self.conv5, self.conv6]
   
    def forward(self, x, mask, return_features=False):
        input_concat = torch.cat([x, mask], dim=1)
        features = []
       
        # Main discriminator path with more layers
        x = F.leaky_relu(self.conv1(input_concat), 0.2, inplace=True)
        if return_features:
            features.append(x)
       
        x = F.leaky_relu(self.conv2(x), 0.2, inplace=True)
        if return_features:
            features.append(x)
       
        x = F.leaky_relu(self.conv3(x), 0.2, inplace=True)
        if return_features:
            features.append(x)
       
        x = F.leaky_relu(self.conv4(x), 0.2, inplace=True)
        if return_features:
            features.append(x)
       
        x = F.leaky_relu(self.conv5(x), 0.2, inplace=True)
        if return_features:
            features.append(x)
       
        x = F.leaky_relu(self.conv6(x), 0.2, inplace=True)
        if return_features:
            features.append(x)
       
        # Global average pooling for stability
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        
        # Deeper FC layers
        x = F.leaky_relu(self.fc1(x), 0.2, inplace=True)
        logits = self.fc2(x)
       
        if return_features:
            return logits, features
        return logits
# Generate adaptive mask
def generate_adaptive_mask(image):
    batch_size = image.shape[0]
    h, w = image_size
    mask = torch.ones(batch_size, 3, h, w, dtype=torch.float32, device=image.device)
    half_height = h // 2
    bottom_mask = torch.zeros(batch_size, 3, h - half_height, w, dtype=torch.float32, device=image.device)
    top_mask = torch.ones(batch_size, 3, half_height, w, dtype=torch.float32, device=image.device)
    mask = torch.cat([top_mask, bottom_mask], dim=2)
   
    if USE_MTCNN:
        detector = MTCNN()
        images_np = ((image * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
        landmarks = []
        for img in images_np:
            img_uint8 = img.astype(np.uint8)
            result = detector.detect_faces(img_uint8)
            landmarks.append(result[0]['keypoints'] if result else {})
       
        mask_np = mask.permute(0, 2, 3, 1).cpu().detach().numpy()
        for i, lm in enumerate(landmarks):
            if lm:
                for key in ['mouth_left', 'mouth_right']:
                    if key in lm:
                        x, y = lm[key]
                        y = min(max(y, half_height), h-1)
                        x = min(max(x, 0), w-1)
                        mask_np[i, y-5:y+5, x-5:x+5, :] = 0.0
        mask = torch.from_numpy(mask_np).permute(0, 3, 1, 2).to(device)
   
    return mask
# Loss functions
# Use new torchvision weights API to avoid deprecation warnings
try:
    vgg_weights = models.VGG16_Weights.DEFAULT
except AttributeError:
    # Fallback for very old torchvision
    vgg_weights = None
vgg = models.vgg16(weights=vgg_weights).features.to(device).eval()
loss_model = nn.Sequential(*[vgg[i] for i in range(16)]).to(device) # Up to block4_conv3
for param in loss_model.parameters():
    param.requires_grad = False

# Initialize Edge Detector for edge loss
edge_detector_global = EdgeDetector().to(device)
edge_detector_global.eval()
for param in edge_detector_global.parameters():
    param.requires_grad = False
def perceptual_loss(y_true, y_pred):
    y_true = F.interpolate(y_true, size=image_size, mode='bilinear', align_corners=False)
    y_pred = F.interpolate(y_pred, size=image_size, mode='bilinear', align_corners=False)
    y_true = y_true * 0.5 + 0.5 # Denormalize to [0, 1]
    y_pred = y_pred * 0.5 + 0.5
    true_features = loss_model(y_true)
    pred_features = loss_model(y_pred)
    return torch.mean((true_features - pred_features) ** 2)
def style_loss(y_true, y_pred):
    y_true = F.interpolate(y_true, size=image_size, mode='bilinear', align_corners=False)
    y_pred = F.interpolate(y_pred, size=image_size, mode='bilinear', align_corners=False)
    y_true = y_true * 0.5 + 0.5
    y_pred = y_pred * 0.5 + 0.5
    true_features = loss_model(y_true)
    pred_features = loss_model(y_pred)
   
    def gram_matrix(feat):
        B, C, H, W = feat.shape
        feat = feat.view(B, C, H * W)
        return torch.bmm(feat, feat.transpose(1, 2)) / (C * H * W)
   
    style_losses = [torch.mean((gram_matrix(t) - gram_matrix(p)) ** 2) for t, p in zip([true_features], [pred_features])]
    return torch.mean(torch.stack(style_losses))
def bottom_half_loss(y_true, y_pred, mask):
    bottom_region = (mask == 0).float()
    diff = (y_true - y_pred) ** 2 * bottom_region
    return diff.sum() / (bottom_region.sum() + 1e-8)
def feature_matching_loss(real_features, fake_features):
    losses = [torch.mean((r - f) ** 2) for r, f in zip(real_features, fake_features)]
    return torch.mean(torch.stack(losses))
def identity_loss(y_true, y_pred, mask):
    bottom_region = (mask == 0).float()
    y_true_face = y_true * bottom_region
    y_pred_face = y_pred * bottom_region
    return torch.mean(torch.abs(y_true_face - y_pred_face))
def landmark_guided_loss(y_true, y_pred, mask):
    if not USE_MTCNN:
        return bottom_half_loss(y_true, y_pred, mask)
    detector = MTCNN()
    bottom_region = (mask == 0).float()
    y_true_bottom = y_true * bottom_region
    y_pred_bottom = y_pred * bottom_region
    y_true_np = ((y_true_bottom * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
    y_pred_np = ((y_pred_bottom * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
    loss_tensor = torch.tensor(0.0, device=device, requires_grad=True)
    for true_img, pred_img in zip(y_true_np, y_pred_np):
        true_img_uint8 = true_img.astype(np.uint8)
        pred_img_uint8 = pred_img.astype(np.uint8)
        true_lm = detector.detect_faces(true_img_uint8)
        pred_lm = detector.detect_faces(pred_img_uint8)
        if true_lm and pred_lm:
            true_points = [true_lm[0]['keypoints'][k] for k in ['mouth_left', 'mouth_right']]
            pred_points = [pred_lm[0]['keypoints'][k] for k in ['mouth_left', 'mouth_right']]
            for t, p in zip(true_points, pred_points):
                diff_x = torch.tensor(t[0] - p[0], dtype=torch.float32, device=device)
                diff_y = torch.tensor(t[1] - p[1], dtype=torch.float32, device=device)
                loss_tensor = loss_tensor + (diff_x ** 2 + diff_y ** 2)
    return loss_tensor / y_true.shape[0]
def structural_ssim_loss(y_true, y_pred, mask):
    y_true = y_true * 0.5 + 0.5
    y_pred = y_pred * 0.5 + 0.5
    bottom_mask = (mask == 0).float()
    y_true_bottom = y_true * bottom_mask
    y_pred_bottom = y_pred * bottom_mask
    ssim_vals = []
    for i in range(y_true.shape[0]):
        true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
        pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
        ssim_val = ssim(true_img, pred_img, channel_axis=2, data_range=1.0, win_size=7)
        ssim_vals.append(ssim_val)
    return 1.0 - torch.mean(torch.tensor(ssim_vals, device=device))
# Advanced Loss Functions
def gradient_penalty_loss(discriminator, real_images, fake_images, masks, device):
    """Calculate gradient penalty for WGAN-GP"""
    batch_size = real_images.size(0)
    alpha = torch.rand(batch_size, 1, 1, 1).to(device)
    interpolated = alpha * real_images + (1 - alpha) * fake_images
   
    # Ensure interpolated tensor requires gradients
    interpolated = interpolated.detach().requires_grad_(True)
   
    disc_interpolated = discriminator(interpolated, masks, return_features=False)
   
    # Check if disc_interpolated requires gradients
    if not disc_interpolated.requires_grad:
        return torch.tensor(0.0, device=device, requires_grad=True)
   
    gradients = torch.autograd.grad(
        outputs=disc_interpolated,
        inputs=interpolated,
        grad_outputs=torch.ones_like(disc_interpolated).to(device),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
   
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty
def attention_loss(y_true, y_pred, mask):
    """Simplified attention-based loss focusing on important regions"""
    bottom_region = (mask == 0).float()
    diff = torch.abs(y_true - y_pred) * bottom_region
   
    # Create simple attention weights based on gradient magnitude
    true_grad_x = torch.abs(y_true[:, :, :, 1:] - y_true[:, :, :, :-1])
    true_grad_y = torch.abs(y_true[:, :, 1:, :] - y_true[:, :, :-1, :])
   
    # Pad gradients to match original size
    true_grad_x_padded = F.pad(true_grad_x, (0, 1, 0, 0), mode='constant', value=0)
    true_grad_y_padded = F.pad(true_grad_y, (0, 0, 0, 1), mode='constant', value=0)
   
    # Combine gradients and normalize
    attention_weights = (true_grad_x_padded + true_grad_y_padded).mean(dim=1, keepdim=True)
    attention_weights = attention_weights / (attention_weights.mean() + 1e-8) # Normalize
   
    # Apply attention weights
    weighted_loss = diff * attention_weights
    return weighted_loss.mean()

# ============================================================================
# NEW LOSS FUNCTIONS FOR EDGE-GUIDED INPAINTING
# توابع Loss جدید برای بهبود لبه‌ها و کاهش مات بودن
# ============================================================================

def edge_loss(y_true, y_pred, mask, edge_detector):
    """
    Edge loss to preserve edges in nose and lips regions
    Loss برای حفظ لبه‌های بینی و لب‌ها
    """
    # Extract edges from true and predicted images
    with torch.no_grad():
        edges_true = edge_detector(y_true * 0.5 + 0.5)  # Denormalize first
    edges_pred = edge_detector(y_pred * 0.5 + 0.5)
    
    # Focus on bottom half (masked region)
    bottom_region = (mask == 0).float()
    
    # Calculate edge reconstruction loss
    edge_diff = torch.abs(edges_true - edges_pred) * bottom_region
    
    return edge_diff.mean()

def high_frequency_loss(y_true, y_pred, mask):
    """
    High-frequency loss to reduce blurriness in bottom half
    Loss برای کاهش مات بودن در نیمه پایینی
    """
    # Apply Laplacian filter to detect high-frequency details
    laplacian_kernel = torch.tensor([
        [0, 1, 0],
        [1, -4, 1],
        [0, 1, 0]
    ], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(y_true.device)
    
    # Repeat for 3 channels
    laplacian_kernel = laplacian_kernel.repeat(3, 1, 1, 1)
    
    # Apply Laplacian to true and predicted images
    y_true_hf = F.conv2d(y_true, laplacian_kernel, padding=1, groups=3)
    y_pred_hf = F.conv2d(y_pred, laplacian_kernel, padding=1, groups=3)
    
    # Focus on bottom half
    bottom_region = (mask == 0).float()
    
    # Calculate high-frequency difference
    hf_diff = torch.abs(y_true_hf - y_pred_hf) * bottom_region
    
    return hf_diff.mean()

def gradient_loss(y_true, y_pred, mask):
    """
    Gradient loss to preserve sharp edges and textures
    Loss برای حفظ لبه‌های تیز و بافت‌ها
    """
    # Sobel filters for gradient calculation
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], 
                           dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(y_true.device)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], 
                           dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(y_true.device)
    
    # Repeat for 3 channels
    sobel_x = sobel_x.repeat(3, 1, 1, 1)
    sobel_y = sobel_y.repeat(3, 1, 1, 1)
    
    # Calculate gradients for true image
    grad_true_x = F.conv2d(y_true, sobel_x, padding=1, groups=3)
    grad_true_y = F.conv2d(y_true, sobel_y, padding=1, groups=3)
    grad_true = torch.sqrt(grad_true_x ** 2 + grad_true_y ** 2 + 1e-8)
    
    # Calculate gradients for predicted image
    grad_pred_x = F.conv2d(y_pred, sobel_x, padding=1, groups=3)
    grad_pred_y = F.conv2d(y_pred, sobel_y, padding=1, groups=3)
    grad_pred = torch.sqrt(grad_pred_x ** 2 + grad_pred_y ** 2 + 1e-8)
    
    # Focus on bottom half
    bottom_region = (mask == 0).float()
    
    # Calculate gradient difference
    grad_diff = torch.abs(grad_true - grad_pred) * bottom_region
    
    return grad_diff.mean()

def facial_region_loss(y_true, y_pred, mask):
    """
    Focused loss on nose and lips regions (center-bottom area)
    Loss متمرکز بر ناحیه بینی و لب‌ها
    """
    B, C, H, W = y_true.shape
    
    # Create a weight map focusing on center-bottom (nose and lips region)
    weight_map = torch.ones_like(mask)
    
    # Define nose and lips region (center-bottom)
    # Nose: roughly H/2 to 3H/4, center W/4 to 3W/4
    # Lips: roughly 3H/4 to H, center W/4 to 3W/4
    center_w_start = W // 4
    center_w_end = 3 * W // 4
    nose_h_start = H // 2
    lips_h_end = H
    
    # Increase weight for nose and lips region
    weight_map[:, :, nose_h_start:lips_h_end, center_w_start:center_w_end] = 3.0
    
    # Apply mask (only bottom half)
    bottom_region = (mask == 0).float()
    weight_map = weight_map * bottom_region
    
    # Calculate weighted L1 loss
    diff = torch.abs(y_true - y_pred) * weight_map
    
    return diff.sum() / (weight_map.sum() + 1e-8)

def generator_loss(disc_fake_logits, y_true, y_pred, disc_features_real, disc_features_fake, mask, mu, logvar):
    # Clamp logvar to prevent exp overflow
    logvar = torch.clamp(logvar, min=-10, max=10)
    
    # Calculate each loss with NaN protection
    adv_loss = -torch.mean(disc_fake_logits)
    adv_loss = torch.nan_to_num(adv_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    # L1 reconstruction loss (high weight for stable inpainting)
    bottom_region = (mask == 0).float()
    l1_loss = torch.mean(torch.abs(y_true - y_pred) * bottom_region)
    l1_loss = torch.nan_to_num(l1_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    rec_loss = bottom_half_loss(y_true, y_pred, mask)
    rec_loss = torch.nan_to_num(rec_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    perc_loss = perceptual_loss(y_true, y_pred)
    perc_loss = torch.nan_to_num(perc_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    sty_loss = style_loss(y_true, y_pred)
    sty_loss = torch.nan_to_num(sty_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    fm_loss = feature_matching_loss(disc_features_real, disc_features_fake)
    fm_loss = torch.nan_to_num(fm_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    id_loss = identity_loss(y_true, y_pred, mask)
    id_loss = torch.nan_to_num(id_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    lm_loss = landmark_guided_loss(y_true, y_pred, mask)
    lm_loss = torch.nan_to_num(lm_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    struct_loss = structural_ssim_loss(y_true, y_pred, mask)
    struct_loss = torch.nan_to_num(struct_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    att_loss = attention_loss(y_true, y_pred, mask)
    att_loss = torch.nan_to_num(att_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    # NEW: Edge-guided losses for better nose/lips reconstruction and sharpness
    edg_loss = edge_loss(y_true, y_pred, mask, edge_detector_global)
    edg_loss = torch.nan_to_num(edg_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    hf_loss = high_frequency_loss(y_true, y_pred, mask)
    hf_loss = torch.nan_to_num(hf_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    grad_loss = gradient_loss(y_true, y_pred, mask)
    grad_loss = torch.nan_to_num(grad_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    facial_loss = facial_region_loss(y_true, y_pred, mask)
    facial_loss = torch.nan_to_num(facial_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    # KL loss with clipping
    kl_loss = -0.5 * torch.mean(1 + logvar - mu**2 - torch.exp(logvar))
    kl_loss = torch.nan_to_num(kl_loss, nan=0.0, posinf=0.0, neginf=0.0)
    kl_loss = torch.clamp(kl_loss, min=0.0, max=10.0)  # Prevent extreme values
    
    # Calculate total loss with edge-guided losses for better quality
    total_loss = (lambda_adv * adv_loss + lambda_l1 * l1_loss + lambda_rec * rec_loss + lambda_perc * perc_loss +
                  lambda_style * sty_loss + lambda_fm * fm_loss + lambda_identity * id_loss +
                  lambda_landmark * lm_loss + lambda_struct * struct_loss + 
                  lambda_attention * att_loss + lambda_vae * kl_loss +
                  lambda_edge * edg_loss + lambda_hf * hf_loss + 
                  lambda_gradient * grad_loss + lambda_facial * facial_loss)
    
    # Final NaN check
    total_loss = torch.nan_to_num(total_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    return total_loss
def discriminator_loss(disc_real_logits, disc_fake_logits):
    # Clamp logits to prevent extreme values (tighter clamp to fix negative losses)
    disc_real_logits = torch.clamp(disc_real_logits, min=-5, max=5)
    disc_fake_logits = torch.clamp(disc_fake_logits, min=-5, max=5)
    
    real_loss = torch.mean(F.relu(1.0 - disc_real_logits))
    real_loss = torch.nan_to_num(real_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    fake_loss = torch.mean(F.relu(1.0 + disc_fake_logits))
    fake_loss = torch.nan_to_num(fake_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    reg = lambda_d_reg * (torch.mean(disc_real_logits**2) + torch.mean(disc_fake_logits**2))
    reg = torch.nan_to_num(reg, nan=0.0, posinf=0.0, neginf=0.0)
    
    total_loss = real_loss + fake_loss + reg
    total_loss = torch.nan_to_num(total_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
    return total_loss
# Evaluation functions
def calculate_psnr_bottom_half(y_true, y_pred, mask):
    y_true = y_true * 0.5 + 0.5
    y_pred = y_pred * 0.5 + 0.5
    bottom_mask = (mask == 0).float()
    y_true_bottom = y_true * bottom_mask
    y_pred_bottom = y_pred * bottom_mask
    psnr_vals = []
    for i in range(y_true.shape[0]):
        true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
        pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
        psnr_val = psnr(true_img, pred_img, data_range=1.0)
        psnr_vals.append(psnr_val)
    return torch.mean(torch.tensor(psnr_vals, device=device))
def calculate_ssim_bottom_half(y_true, y_pred, mask):
    y_true = y_true * 0.5 + 0.5
    y_pred = y_pred * 0.5 + 0.5
    bottom_mask = (mask == 0).float()
    y_true_bottom = y_true * bottom_mask
    y_pred_bottom = y_pred * bottom_mask
    ssim_vals = []
    for i in range(y_true.shape[0]):
        true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
        pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
        ssim_val = ssim(true_img, pred_img, channel_axis=2, data_range=1.0, win_size=7)
        ssim_vals.append(ssim_val)
    return torch.mean(torch.tensor(ssim_vals, device=device))
def calculate_mse_bottom_half(y_true, y_pred, mask):
    return bottom_half_loss(y_true, y_pred, mask)
# Training function with graph-aware processing and balanced D/G updates
def train_step(images, edge_generator, coarse_generator, fine_generator, discriminator, edge_optimizer, g_optimizer, d_optimizer, scaler, step_counter):
    images = images.to(device)
    masks = generate_adaptive_mask(images)
    masked_images = images * masks
   
    # Discriminator training (n_critic times more frequent)
    d_optimizer.zero_grad()
    
    # Train Discriminator multiple times
    d_loss_accumulated = 0.0
    for _ in range(n_critic):
        with amp.autocast('cuda'):
            # Generate fake images with multi-stage process: Edge -> Coarse -> Fine
            with torch.no_grad():
                # Stage 1: Generate edge map
                edge_map = edge_generator(masked_images, masks)
                
                # Stage 2: Coarse inpainting with edge guidance
                coarse_output, mu, logvar = multi_resolution_inpainting(masked_images, masks, coarse_generator, edge_map)
                if coarse_output.shape != images.shape:
                    coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
                
                # Stage 3: Fine inpainting with edge guidance
                fine_output = fine_generator(masked_images, coarse_output, masks, edge_map)
                if fine_output.shape != images.shape:
                    fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
                
                combined_input = images * masks + fine_output * (1.0 - masks)
            
            # Discriminator forward
            disc_real_logits = discriminator(images, masks, return_features=False)
            disc_fake_logits = discriminator(combined_input.detach(), masks, return_features=False)
            d_loss = discriminator_loss(disc_real_logits, disc_fake_logits)
        
        # Discriminator backward
        scaler.scale(d_loss).backward()
        d_loss_accumulated += d_loss.item()
    
    # Update discriminator after n_critic iterations
    scaler.unscale_(d_optimizer)
    torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
    scaler.step(d_optimizer)
    
    d_loss_val = d_loss_accumulated / n_critic
   
    # Generator training (once per n_critic discriminator updates)
    g_loss_val = 0.0
    edge_loss_val = 0.0
    if step_counter % n_critic == 0:
        # Train Edge Generator and Main Generators together
        edge_optimizer.zero_grad()
        g_optimizer.zero_grad()
        
        with amp.autocast('cuda'):
            # Stage 1: Generate edge map and compute edge reconstruction loss
            edge_map = edge_generator(masked_images, masks)
            
            # Extract true edges from original images
            with torch.no_grad():
                true_edges = edge_detector_global(images * 0.5 + 0.5)
            
            # Edge reconstruction loss (L1 loss on edge maps)
            bottom_region = (masks == 0).float()
            edge_recon_loss = torch.mean(torch.abs(true_edges - edge_map) * bottom_region)
            
            # Stage 2 & 3: Generator forward pass with edge guidance
            coarse_output, mu, logvar = multi_resolution_inpainting(masked_images, masks, coarse_generator, edge_map)
            if coarse_output.shape != images.shape:
                coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
            
            fine_output = fine_generator(masked_images, coarse_output, masks, edge_map)
            if fine_output.shape != images.shape:
                fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
            
            combined_input = images * masks + fine_output * (1.0 - masks)
            disc_real_logits, disc_features_real = discriminator(images, masks, return_features=True)
            disc_fake_logits, disc_features_fake = discriminator(combined_input, masks, return_features=True)
            g_loss = generator_loss(disc_fake_logits, images, fine_output, disc_features_real, disc_features_fake, masks, mu, logvar)
        
        # Combined backward pass (edge loss + generator loss)
        # This avoids inplace operation errors
        total_gen_loss = edge_recon_loss + g_loss
        
        scaler.scale(total_gen_loss).backward()
        
        # Unscale and clip gradients for both edge and main generators
        scaler.unscale_(edge_optimizer)
        scaler.unscale_(g_optimizer)
        torch.nn.utils.clip_grad_norm_(edge_generator.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(list(coarse_generator.parameters()) + list(fine_generator.parameters()), max_norm=1.0)
        
        # Update both optimizers
        scaler.step(edge_optimizer)
        scaler.step(g_optimizer)
        
        edge_loss_val = edge_recon_loss.item()
        g_loss_val = g_loss.item()
   
    # Update scaler after both steps
    scaler.update()
    
    # Check for NaN in losses
    if np.isnan(g_loss_val) or np.isinf(g_loss_val):
        print(f"⚠️ WARNING: G_Loss is {g_loss_val}, skipping this batch")
        g_loss_val = 0.0
        
    if np.isnan(d_loss_val) or np.isinf(d_loss_val):
        print(f"⚠️ WARNING: D_Loss is {d_loss_val}, skipping this batch")
        d_loss_val = 0.0
    
    if np.isnan(edge_loss_val) or np.isinf(edge_loss_val):
        print(f"⚠️ WARNING: Edge_Loss is {edge_loss_val}, skipping this batch")
        edge_loss_val = 0.0
   
    return g_loss_val, d_loss_val, edge_loss_val
# Add multi_resolution_inpainting function
def multi_resolution_inpainting(image, mask, model, edge_map=None, levels=3):
    outputs = []
    mus = []
    logvars = []
    for level in range(levels):
        scaled_image = F.interpolate(image, scale_factor=1 / (2 ** level), mode='bilinear')
        scaled_mask = F.interpolate(mask, scale_factor=1 / (2 ** level), mode='nearest')
        
        # Scale edge map if provided
        if edge_map is not None:
            scaled_edge_map = F.interpolate(edge_map, scale_factor=1 / (2 ** level), mode='bilinear')
            out, mu, logvar = model(scaled_image, scaled_mask, scaled_edge_map)
        else:
            out, mu, logvar = model(scaled_image, scaled_mask)  # Call full forward
        
        out_up = F.interpolate(out, size=image.shape[-2:], mode='bilinear')
        outputs.append(out_up)
        mus.append(mu)
        logvars.append(logvar)
    
    final_out = torch.mean(torch.stack(outputs), dim=0)
    final_mu = torch.mean(torch.stack(mus), dim=0)
    final_logvar = torch.mean(torch.stack(logvars), dim=0)
    return final_out, final_mu, final_logvar

# Add gradient checking function
def check_model_gradients(model, model_name="Model"):
    """Check if model has NaN or Inf gradients"""
    has_nan = False
    has_inf = False
    max_grad = 0.0
    
    for name, param in model.named_parameters():
        if param.grad is not None:
            if torch.isnan(param.grad).any():
                has_nan = True
                print(f"⚠️ NaN gradient in {model_name}.{name}")
            if torch.isinf(param.grad).any():
                has_inf = True
                print(f"⚠️ Inf gradient in {model_name}.{name}")
            max_grad = max(max_grad, param.grad.abs().max().item())
    
    return has_nan, has_inf, max_grad

# Add simple retrain function
def retrain(model, pruned_graph, optimizer):
    optimizer.zero_grad()
    # Dummy forward (adjust to use graph if needed)
    dummy_input = torch.randn(1, 3, 64, 64).to(device)
    dummy_mask = torch.randn(1, 3, 64, 64).to(device)
    output, _, _ = model(dummy_input, dummy_mask)
    loss = torch.mean(output)
    loss.backward()
    optimizer.step()
    return loss.item()

# Integrate into training loop
# In train_step function, after coarse_output:
# Add pruning and retraining every 10 epochs
# Load dataset

# celeba_dir = '/kaggle/input/celeba-resized-6464/img_align_celeba/CelebA_Image_Cropped_64'
celeba_dir = '/kaggle/input/celeba-6464/selected_celeba'

print(f"Loading dataset from {celeba_dir}...")
try:
    print(f"Dataset directory contents: {os.listdir(celeba_dir)[:5]}")
except Exception as e:
    print(f"Error accessing dataset directory: {e}")
    raise
# Split dataset
train_files, val_files, test_files = split_dataset(celeba_dir)
# Create datasets
train_loader, val_loader, test_loader = create_split_datasets(celeba_dir, train_files, val_files, test_files, image_size, batch_size)
# Build models (now with Edge Generator for multi-stage inpainting)
print("Building models...")
print("🔷 Multi-Stage Architecture: Edge → Coarse → Fine")
edge_generator = EdgeGenerator().to(device)
coarse_generator = CoarseGenerator().to(device)
fine_generator = FineGenerator().to(device)
discriminator = Discriminator().to(device)
print(f"Edge Generator: {sum(p.numel() for p in edge_generator.parameters()):,} parameters")
print(f"Coarse Generator: {sum(p.numel() for p in coarse_generator.parameters()):,} parameters")
print(f"Fine Generator: {sum(p.numel() for p in fine_generator.parameters()):,} parameters")
print(f"Discriminator: {sum(p.numel() for p in discriminator.parameters()):,} parameters")
# Test model dimensions
print("\nTesting model dimensions...")
test_input = torch.randn(1, 3, 64, 64).to(device)
test_mask = torch.randn(1, 3, 64, 64).to(device)
with torch.no_grad():
    coarse_output, mu, logvar = coarse_generator(test_input, test_mask)
    print(f"Coarse output shape: {coarse_output.shape}")
    fine_output = fine_generator(test_input, coarse_output, test_mask)
    print(f"Fine output shape: {fine_output.shape}")
    print(f"Expected shape: {test_input.shape}")
   
    # Test discriminator
    disc_logits, disc_features = discriminator(test_input, test_mask, return_features=True)
    print(f"Discriminator logits shape: {disc_logits.shape}")
    print(f"Discriminator features length: {len(disc_features)}")
   
    disc_logits_only = discriminator(test_input, test_mask, return_features=False)
    print(f"Discriminator logits only shape: {disc_logits_only.shape}")
   
    print("✓ Model dimensions are correct!")
# Enhanced Optimizers with Learning Rate Scheduling and beta1=0.5 for GAN stability
# NEW: Added Edge Generator optimizer
edge_optimizer = torch.optim.AdamW(
    edge_generator.parameters(),
    lr=learning_rate_g, betas=(0.5, 0.999), weight_decay=1e-4
)
g_optimizer = torch.optim.AdamW(
    list(coarse_generator.parameters()) + list(fine_generator.parameters()),
    lr=learning_rate_g, betas=(0.5, 0.999), weight_decay=1e-4
)
d_optimizer = torch.optim.AdamW(
    discriminator.parameters(),
    lr=learning_rate_d, betas=(0.5, 0.999), weight_decay=1e-4
)
print(f"Optimizer settings: G_LR={learning_rate_g}, D_LR={learning_rate_d}, beta1=0.5, n_critic={n_critic}")
print(f"✅ Edge Generator optimizer added for multi-stage training")
# Learning rate schedulers - ReduceLROnPlateau for adaptive learning based on validation loss
# Scheduler های Learning Rate - کاهش تطبیقی بر اساس validation loss
edge_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    edge_optimizer, mode='min', factor=0.5, patience=5, verbose=True, min_lr=learning_rate_g * 0.01
)
g_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    g_optimizer, mode='min', factor=0.5, patience=5, verbose=True, min_lr=learning_rate_g * 0.01
)
d_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    d_optimizer, mode='min', factor=0.5, patience=5, verbose=True, min_lr=learning_rate_d * 0.01
)
scaler = amp.GradScaler('cuda')
# Model health check before training
print("\n" + "="*60)
print("🔍 Model Health Check")
print("="*60)
test_batch = next(iter(train_loader)).to(device)
test_masks = generate_adaptive_mask(test_batch)
test_masked = test_batch * test_masks

print("Testing forward pass...")
with torch.no_grad():
    try:
        coarse_out, mu, logvar = coarse_generator(test_masked, test_masks)
        print(f"✓ Coarse Generator: output shape {coarse_out.shape}")
        print(f"  mu range: [{mu.min().item():.4f}, {mu.max().item():.4f}]")
        print(f"  logvar range: [{logvar.min().item():.4f}, {logvar.max().item():.4f}]")
        
        fine_out = fine_generator(test_masked, coarse_out, test_masks)
        print(f"✓ Fine Generator: output shape {fine_out.shape}")
        print(f"  output range: [{fine_out.min().item():.4f}, {fine_out.max().item():.4f}]")
        
        disc_out = discriminator(test_batch, test_masks)
        print(f"✓ Discriminator: logits shape {disc_out.shape}")
        print(f"  logits range: [{disc_out.min().item():.4f}, {disc_out.max().item():.4f}]")
        
        print("✅ All models are healthy!")
    except Exception as e:
        print(f"❌ Model health check failed: {e}")
        raise
print("="*60 + "\n")

# ============================================================================
# CHECKPOINT LOADING & TRAINING SETUP
# بارگذاری Checkpoint و راه‌اندازی آموزش
# ============================================================================

print("\n" + "="*70)
print("🚀 STARTING TRAINING SETUP")
print("="*70)

# Adjust paths if loading from Kaggle input dataset
if USE_KAGGLE_INPUT:
    checkpoint_load_dir = os.path.join(KAGGLE_INPUT_DATASET, 'checkpoints')
    models_load_dir = os.path.join(KAGGLE_INPUT_DATASET, 'models')
    print(f"📥 Loading from Kaggle dataset: {KAGGLE_INPUT_DATASET}")
else:
    checkpoint_load_dir = CHECKPOINT_DIR
    models_load_dir = MODELS_DIR
    print(f"📥 Loading from working directory")

# Create checkpoint save directory (always in /kaggle/working)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"💾 Checkpoints will be saved to: {CHECKPOINT_DIR}")
print(f"💾 Models will be saved to: {MODELS_DIR}")

# Initialize training variables
start_epoch = 0
best_val_loss = float('inf')
train_g_losses = []
train_d_losses = []
val_g_losses = []
val_d_losses = []
previous_val_g_loss = float('inf')  # For early stopping

# Try to load checkpoint/models if RESUME_TRAINING is True
checkpoint_loaded = False
models_loaded = False

if RESUME_TRAINING:
    # ========================================================================
    # Method 1: Try to load from MODEL FILES (state_dict only)
    # روش 1: بارگذاری از فایل‌های MODEL (فقط state_dict)
    # ========================================================================
    
    if LOAD_FROM_MODELS and not checkpoint_loaded:
        print(f"\n🔍 Method 1: Searching for model files...")
        print(f"📂 Models directory: {models_load_dir}")
        
        # Debug: Show what files actually exist
        if os.path.exists(models_load_dir):
            print(f"\n📄 Files found in directory:")
            actual_files = os.listdir(models_load_dir)
            if actual_files:
                for f in sorted(actual_files):
                    if f.endswith('.pth'):
                        file_path = os.path.join(models_load_dir, f)
                        size_mb = os.path.getsize(file_path) / (1024**2)
                        print(f"   • {f} ({size_mb:.2f} MB)")
            else:
                print(f"   (empty)")
        else:
            print(f"   ❌ Directory does not exist!")
        
        # Determine which models to load
        suffix = "_best.pth" if USE_BEST_MODELS else "_bottom_half.pth"
        
        model_files = {
            'coarse': os.path.join(models_load_dir, f'coarse_generator{suffix}'),
            'fine': os.path.join(models_load_dir, f'fine_generator{suffix}'),
            'disc': os.path.join(models_load_dir, f'discriminator{suffix}')
        }
        
        print(f"\n🔍 Looking for (USE_BEST_MODELS={USE_BEST_MODELS}):")
        for name, path in model_files.items():
            exists = "✅" if os.path.exists(path) else "❌"
            print(f"   {exists} {os.path.basename(path)}")
        
        # Check if all model files exist
        all_exist = all(os.path.exists(path) for path in model_files.values())
        
        if all_exist:
            try:
                print(f"📂 Found model files in: {models_load_dir}")
                print(f"   • coarse_generator{suffix}")
                print(f"   • fine_generator{suffix}")
                print(f"   • discriminator{suffix}")
                print(f"⏳ Loading model weights...")
                
                # Load model weights
                # Load model weights tolerantly in case of architecture drift
                coarse_sd = torch.load(model_files['coarse'], map_location=device)
                fine_sd = torch.load(model_files['fine'], map_location=device)
                disc_sd = torch.load(model_files['disc'], map_location=device)

                missing, unexpected = coarse_generator.load_state_dict(coarse_sd, strict=False)
                if missing or unexpected:
                    print("[coarse_generator] non-strict load:")
                    if missing:
                        print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
                    if unexpected:
                        print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

                missing, unexpected = fine_generator.load_state_dict(fine_sd, strict=False)
                if missing or unexpected:
                    print("[fine_generator] non-strict load:")
                    if missing:
                        print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
                    if unexpected:
                        print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

                missing, unexpected = discriminator.load_state_dict(disc_sd, strict=False)
                if missing or unexpected:
                    print("[discriminator] non-strict load:")
                    if missing:
                        print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
                    if unexpected:
                        print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
                
                models_loaded = True
                
                print("="*70)
                print("✅ MODEL WEIGHTS LOADED SUCCESSFULLY!")
                print("="*70)
                print(f"📊 Loading Information:")
                print(f"   • Model weights: ✅ Loaded")
                print(f"   • Optimizer states: ⚠️  Will be initialized (not in model files)")
                print(f"   • Scheduler states: ⚠️  Will be initialized (not in model files)")
                print(f"   • Training history: ⚠️  Starting fresh (not in model files)")
                print(f"   • Starting epoch: 1 (training continues with loaded weights)")
                print(f"\n⚠️  NOTE: Model files only contain weights, not optimizer/scheduler.")
                print(f"   Training will continue with these weights but fresh optimizer.")
                print("="*70)
                
            except Exception as e:
                print(f"⚠️  Failed to load model files")
                print(f"   Error: {str(e)}")
                models_loaded = False
        else:
            print(f"⚠️  Not all model files found in {models_load_dir}")
            for name, path in model_files.items():
                exists = "✓" if os.path.exists(path) else "✗"
                print(f"   {exists} {os.path.basename(path)}")
    
    # ========================================================================
    # Method 2: Try to load from CHECKPOINT FILES (full checkpoint)
    # روش 2: بارگذاری از فایل‌های CHECKPOINT (checkpoint کامل)
    # ========================================================================
    
    if not checkpoint_loaded and not models_loaded:
        print(f"\n🔍 Method 2: Searching for checkpoint files...")
        
        # Try different checkpoint sources
        checkpoint_paths_to_try = [
            os.path.join(checkpoint_load_dir, 'latest_checkpoint.pth'),
            os.path.join(checkpoint_load_dir, f'checkpoint_epoch_{num_epochs}.pth'),
            os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth'),  # Fallback to working dir
        ]
        
        for checkpoint_path in checkpoint_paths_to_try:
            if os.path.exists(checkpoint_path):
                try:
                    print(f"📂 Found checkpoint: {checkpoint_path}")
                    print(f"⏳ Loading checkpoint...")
                    
                    checkpoint = torch.load(checkpoint_path, map_location=device)
                    
                    # Load model states tolerantly (including edge_generator if available)
                    if 'edge_generator' in checkpoint:
                        missing, unexpected = edge_generator.load_state_dict(checkpoint['edge_generator'], strict=False)
                        if missing or unexpected:
                            print("[edge_generator ckpt] non-strict load:")
                            if missing:
                                print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
                            if unexpected:
                                print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
                    else:
                        print("[edge_generator] not found in checkpoint - using random initialization")
                    
                    missing, unexpected = coarse_generator.load_state_dict(checkpoint['coarse_generator'], strict=False)
                    if missing or unexpected:
                        print("[coarse_generator ckpt] non-strict load:")
                        if missing:
                            print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
                        if unexpected:
                            print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

                    missing, unexpected = fine_generator.load_state_dict(checkpoint['fine_generator'], strict=False)
                    if missing or unexpected:
                        print("[fine_generator ckpt] non-strict load:")
                        if missing:
                            print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
                        if unexpected:
                            print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

                    missing, unexpected = discriminator.load_state_dict(checkpoint['discriminator'], strict=False)
                    if missing or unexpected:
                        print("[discriminator ckpt] non-strict load:")
                        if missing:
                            print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
                        if unexpected:
                            print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
                    
                    # Load optimizer states
                    g_optimizer.load_state_dict(checkpoint['g_optimizer'])
                    d_optimizer.load_state_dict(checkpoint['d_optimizer'])
                    
                    # Load scheduler states
                    g_scheduler.load_state_dict(checkpoint['g_scheduler'])
                    d_scheduler.load_state_dict(checkpoint['d_scheduler'])
                    
                    # Load training progress
                    start_epoch = checkpoint['epoch'] + 1
                    best_val_loss = checkpoint.get('best_val_loss', float('inf'))
                    train_g_losses = checkpoint.get('train_g_losses', [])
                    train_d_losses = checkpoint.get('train_d_losses', [])
                    val_g_losses = checkpoint.get('val_g_losses', [])
                    val_d_losses = checkpoint.get('val_d_losses', [])
                    
                    checkpoint_loaded = True
                    
                    print("="*70)
                    print("✅ FULL CHECKPOINT LOADED SUCCESSFULLY!")
                    print("="*70)
                    print(f"📊 Resume Information:")
                    print(f"   • Model weights: ✅ Loaded")
                    print(f"   • Optimizer states: ✅ Loaded")
                    print(f"   • Scheduler states: ✅ Loaded")
                    print(f"   • Starting from epoch: {start_epoch + 1}")
                    print(f"   • Best validation loss: {best_val_loss:.6f}")
                    print(f"   • Training history: {len(train_g_losses)} epochs")
                    print(f"   • Remaining epochs: {num_epochs - start_epoch}")
                    print("="*70)
                    
                    break  # Successfully loaded, exit loop
                    
                except Exception as e:
                    print(f"⚠️  Failed to load checkpoint from {checkpoint_path}")
                    print(f"   Error: {str(e)}")
                    print(f"   Trying next checkpoint source...")
                    continue

# ========================================================================
# If nothing loaded, start fresh
# اگر چیزی بارگذاری نشد، از اول شروع کن
# ========================================================================

if not checkpoint_loaded and not models_loaded:
    if RESUME_TRAINING:
        print("\n" + "="*70)
        print("⚠️  NO CHECKPOINT OR MODELS FOUND - Starting from scratch")
        print("="*70)
        print("📝 Locations searched:")
        print(f"\n   Models folder: {models_load_dir}")
        if os.path.exists(models_load_dir):
            files = os.listdir(models_load_dir)
            if files:
                print(f"   Found {len(files)} file(s):")
                for f in files[:5]:  # Show first 5
                    print(f"      • {f}")
            else:
                print(f"      (empty)")
        else:
            print(f"      (not found)")
        
        print(f"\n   Checkpoints folder: {checkpoint_load_dir}")
        if os.path.exists(checkpoint_load_dir):
            files = [f for f in os.listdir(checkpoint_load_dir) if f.endswith('.pth')]
            if files:
                print(f"   Found {len(files)} checkpoint(s):")
                for f in files[:5]:
                    print(f"      • {f}")
            else:
                print(f"      (empty)")
        else:
            print(f"      (not found)")
        
        print("\n💡 To resume training in next run:")
        print("   1. Make sure model/checkpoint files exist")
        print("   2. Set LOAD_FROM_MODELS = True for model files")
        print("   3. If using Kaggle dataset, set USE_KAGGLE_INPUT = True")
        print("   4. Update MODELS_DIR or KAGGLE_INPUT_DATASET path")
        print("="*70)
    else:
        print("\n" + "="*70)
        print("🆕 STARTING FRESH TRAINING")
        print("="*70)
        print("   RESUME_TRAINING is set to False")
        print("   Training will start from epoch 1")
        print("="*70)

print(f"\n🎯 Training will run from epoch {start_epoch + 1} to {num_epochs}")
print(f"💾 Checkpoints will be saved every {CHECKPOINT_INTERVAL} epoch(s)")
print("="*70 + "\n")

for epoch in range(start_epoch, num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    edge_generator.train()
    coarse_generator.train()
    fine_generator.train()
    discriminator.train()
    epoch_g_loss = epoch_d_loss = epoch_edge_loss = num_batches = 0
   
    for step, images in enumerate(train_loader):
        # Pass step counter for balanced D/G updates (NEW: added edge_generator and edge_optimizer)
        g_loss_batch, d_loss_batch, edge_loss_batch = train_step(images, edge_generator, coarse_generator, fine_generator, discriminator, edge_optimizer, g_optimizer, d_optimizer, scaler, step)
        epoch_g_loss += g_loss_batch
        epoch_edge_loss += edge_loss_batch
        epoch_d_loss += d_loss_batch
        num_batches += 1
        
        # Enhanced monitoring
        if step % 50 == 0:
            current_g_lr = g_optimizer.param_groups[0]['lr']
            current_d_lr = d_optimizer.param_groups[0]['lr']
            current_edge_lr = edge_optimizer.param_groups[0]['lr']
            g_or_d = "E+G+D" if step % n_critic == 0 else "D only"
            print(f" Step {step:3d} [{g_or_d}]: Edge_Loss: {edge_loss_batch:.4f}, G_Loss: {g_loss_batch:.4f}, D_Loss: {d_loss_batch:.4f} | LR: E={current_edge_lr:.6f}, G={current_g_lr:.6f}, D={current_d_lr:.6f}")
            
        # Emergency stop if losses explode
        if np.isnan(g_loss_batch) and np.isnan(d_loss_batch):
            print(f"⚠️ CRITICAL: Both losses are NaN at step {step}. Stopping epoch early.")
            break
   
    avg_g_loss = epoch_g_loss / num_batches if num_batches > 0 else 0
    avg_d_loss = epoch_d_loss / num_batches if num_batches > 0 else 0
    avg_edge_loss = epoch_edge_loss / num_batches if num_batches > 0 else 0
    train_g_losses.append(avg_g_loss)
    train_d_losses.append(avg_d_loss)
    print(f"Epoch {epoch+1} Training - Edge_Loss: {avg_edge_loss:.4f}, G_Loss: {avg_g_loss:.4f}, D_Loss: {avg_d_loss:.4f}")
   
    # Enable MTCNN for validation
    if 'MTCNN' in globals():
        USE_MTCNN = True
   
    if epoch % 10 == 0:
        # Get sample batch for graph creation
        sample_images = next(iter(train_loader)).to(device)
        sample_masks = generate_adaptive_mask(sample_images)
        sample_masked = sample_images * sample_masks
        
        # Get features from preprocessor
        features = coarse_generator.preprocessor(sample_masked)
        
        # Build graph
        sample_graph = coarse_generator.preprocessor.build_graph(features)
        
        pruned_graph = prune_graph(coarse_generator, sample_graph)
        
        # Retrain
        for _ in range(5):
            retrain_loss = retrain(coarse_generator, pruned_graph, g_optimizer)
   
    edge_generator.eval()
    coarse_generator.eval()
    fine_generator.eval()
    discriminator.eval()
    val_g_loss = val_d_loss = val_batches = 0
    with torch.no_grad():
        for images in val_loader:
            images = images.to(device)
            masks = generate_adaptive_mask(images)
            masked_images = images * masks
            
            # Multi-stage: Edge -> Coarse -> Fine
            edge_map = edge_generator(masked_images, masks)
            coarse_output, mu, logvar = coarse_generator(masked_images, masks, edge_map)
            # Debug: Check shapes
            if coarse_output.shape != images.shape:
                coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
           
            fine_output = fine_generator(masked_images, coarse_output, masks, edge_map)
            # Debug: Check shapes
            if fine_output.shape != images.shape:
                fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
           
            combined_input = images * masks + fine_output * (1.0 - masks)
           
            disc_real_logits, disc_features_real = discriminator(images, masks, return_features=True)
            disc_fake_logits, disc_features_fake = discriminator(combined_input, masks, return_features=True)
           
            g_loss = generator_loss(disc_fake_logits, images, fine_output, disc_features_real, disc_features_fake, masks, mu, logvar)
            d_loss = discriminator_loss(disc_real_logits, disc_fake_logits)
           
            val_g_loss += g_loss.item()
            val_d_loss += d_loss.item()
            val_batches += 1
   
    avg_val_g_loss = val_g_loss / val_batches if val_batches > 0 else 0
    avg_val_d_loss = val_d_loss / val_batches if val_batches > 0 else 0
    val_g_losses.append(avg_val_g_loss)
    val_d_losses.append(avg_val_d_loss)
    print(f"Epoch {epoch+1} Validation - G_Loss: {avg_val_g_loss:.4f}, D_Loss: {avg_val_d_loss:.4f}")
   
    # Update learning rates based on validation loss (ReduceLROnPlateau)
    # به‌روزرسانی learning rate بر اساس validation loss
    edge_scheduler.step(avg_val_g_loss)
    g_scheduler.step(avg_val_g_loss)
    d_scheduler.step(avg_val_d_loss)
    current_g_lr = g_optimizer.param_groups[0]['lr']
    current_d_lr = d_optimizer.param_groups[0]['lr']
    current_edge_lr = edge_optimizer.param_groups[0]['lr']
    print(f"📊 Current LR - Edge: {current_edge_lr:.6f}, G: {current_g_lr:.6f}, D: {current_d_lr:.6f}")
   
    if avg_val_g_loss < best_val_loss:
        best_val_loss = avg_val_g_loss
        print(f" New best validation loss! Saving best models...")
        os.makedirs('/kaggle/working/models', exist_ok=True)
        torch.save(edge_generator.state_dict(), "/kaggle/working/models/edge_generator_best.pth")
        torch.save(coarse_generator.state_dict(), "/kaggle/working/models/coarse_generator_best.pth")
        torch.save(fine_generator.state_dict(), "/kaggle/working/models/fine_generator_best.pth")
        torch.save(discriminator.state_dict(), "/kaggle/working/models/discriminator_best.pth")

    # Early stopping check
    if avg_val_g_loss >= previous_val_g_loss:
        print(f"⚠️ Early stopping triggered! Val loss not improved: {avg_val_g_loss:.4f} vs {previous_val_g_loss:.4f}")
        break
    previous_val_g_loss = avg_val_g_loss
   
    # Save checkpoint only when training is completely finished
    if (epoch + 1) == num_epochs:
        print(f"\n{'='*70}")
        print(f"💾 SAVING FINAL CHECKPOINT - Epoch {epoch+1}/{num_epochs}")
        print(f"{'='*70}")
        
        checkpoint_file = os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch+1}.pth')
        latest_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth')
        
        checkpoint = {
            'epoch': epoch,
            'edge_generator': edge_generator.state_dict(),
            'coarse_generator': coarse_generator.state_dict(),
            'fine_generator': fine_generator.state_dict(),
            'discriminator': discriminator.state_dict(),
            'edge_optimizer': edge_optimizer.state_dict(),
            'g_optimizer': g_optimizer.state_dict(),
            'd_optimizer': d_optimizer.state_dict(),
            'edge_scheduler': edge_scheduler.state_dict(),
            'g_scheduler': g_scheduler.state_dict(),
            'd_scheduler': d_scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'train_g_losses': train_g_losses,
            'train_d_losses': train_d_losses,
            'val_g_losses': val_g_losses,
            'val_d_losses': val_d_losses,
        }
        
        # Save numbered checkpoint
        torch.save(checkpoint, checkpoint_file)
        checkpoint_size = os.path.getsize(checkpoint_file) / (1024**2)  # MB
        print(f"✅ Saved: {checkpoint_file}")
        print(f"   Size: {checkpoint_size:.2f} MB")
        
        # Also save as latest checkpoint
        torch.save(checkpoint, latest_checkpoint_path)
        print(f"✅ Saved: {latest_checkpoint_path}")
        
        print(f"\n📊 Checkpoint Info:")
        print(f"   • Completed epochs: {epoch + 1}")
        print(f"   • Best val loss: {best_val_loss:.6f}")
        print(f"   • Current G loss: {avg_g_loss:.6f}")
        print(f"   • Current D loss: {avg_d_loss:.6f}")
        
        if (epoch + 1) < num_epochs:
            print(f"\n🔄 To resume from this checkpoint in next run:")
            print(f"   1. Download '{CHECKPOINT_DIR}' folder from Kaggle Output")
            print(f"   2. Upload as Kaggle Dataset (or keep in working directory)")
            print(f"   3. In code, set: RESUME_TRAINING = True")
            print(f"   4. If using dataset, set: USE_KAGGLE_INPUT = True")
            print(f"   5. Update: KAGGLE_INPUT_DATASET = '/kaggle/input/your-dataset-name'")
        else:
            print(f"\n🎉 TRAINING COMPLETED!")
            print(f"   All {num_epochs} epochs finished successfully!")
        
        print(f"{'='*70}\n")
   
    # Plot training progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        plt.figure(figsize=(15, 5))
       
        plt.subplot(1, 3, 1)
        plt.plot(range(1, len(train_g_losses) + 1), train_g_losses, label='Generator Loss', color='blue')
        plt.plot(range(1, len(val_g_losses) + 1), val_g_losses, label='Val Generator Loss', color='lightblue')
        plt.title('Generator Loss Progress')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
       
        plt.subplot(1, 3, 2)
        plt.plot(range(1, len(train_d_losses) + 1), train_d_losses, label='Discriminator Loss', color='red')
        plt.plot(range(1, len(val_d_losses) + 1), val_d_losses, label='Val Discriminator Loss', color='lightcoral')
        plt.title('Discriminator Loss Progress')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
       
        plt.subplot(1, 3, 3)
        plt.plot(range(1, len(train_g_losses) + 1), train_g_losses, label='G Loss', color='blue')
        plt.plot(range(1, len(train_d_losses) + 1), train_d_losses, label='D Loss', color='red')
        plt.title('Training Loss Comparison')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
       
        plt.tight_layout()
        plt.savefig(f'/kaggle/working/training_progress_epoch_{epoch+1}.png', dpi=150, bbox_inches='tight')
        plt.show()
# Save final models
print("Saving final models...")
os.makedirs('/kaggle/working/models', exist_ok=True)
torch.save(edge_generator.state_dict(), "/kaggle/working/models/edge_generator_bottom_half.pth")
torch.save(coarse_generator.state_dict(), "/kaggle/working/models/coarse_generator_bottom_half.pth")
torch.save(fine_generator.state_dict(), "/kaggle/working/models/fine_generator_bottom_half.pth")
torch.save(discriminator.state_dict(), "/kaggle/working/models/discriminator_bottom_half.pth")
# Enable MTCNN for evaluation
if 'MTCNN' in globals():
    USE_MTCNN = True
# Evaluate on test set
print("Starting evaluation on test set...")
os.makedirs('/kaggle/working/results', exist_ok=True)
edge_generator.eval()
coarse_generator.eval()
fine_generator.eval()
discriminator.eval()
psnr_values, ssim_values, mse_values, identity_values, landmark_values = [], [], [], [], []
num_samples = 10 # Increased for better evaluation
plt.figure(figsize=(20, 10))
with torch.no_grad():
    for i, images in enumerate(test_loader):
        if i >= num_samples:
            break
        images = images.to(device)
        masks = generate_adaptive_mask(images)
        masked_images = images * masks
        
        # Multi-stage: Edge -> Coarse -> Fine
        edge_map = edge_generator(masked_images, masks)
        coarse_output, mu, logvar = coarse_generator(masked_images, masks, edge_map)
        # Debug: Check shapes
        if coarse_output.shape != images.shape:
            coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
       
        fine_output = fine_generator(masked_images, coarse_output, masks, edge_map)
        # Debug: Check shapes
        if fine_output.shape != images.shape:
            fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
       
        reconstructed = images * masks + fine_output * (1.0 - masks)
       
        psnr_val = calculate_psnr_bottom_half(images, reconstructed, masks).item()
        ssim_val = calculate_ssim_bottom_half(images, reconstructed, masks).item()
        mse_val = calculate_mse_bottom_half(images, reconstructed, masks).item()
        identity_val = identity_loss(images, reconstructed, masks).item()
        landmark_val = landmark_guided_loss(images, reconstructed, masks).item()
       
        psnr_values.append(psnr_val)
        ssim_values.append(ssim_val)
        mse_values.append(mse_val)
        identity_values.append(identity_val)
        landmark_values.append(landmark_val)
       
        plt.subplot(4, num_samples, i + 1)
        plt.title("Original Image")
        plt.imshow(images[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
        plt.axis("off")
       
        plt.subplot(4, num_samples, num_samples + i + 1)
        plt.title("Top Half (Input)")
        plt.imshow(masked_images[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
        plt.axis("off")
       
        plt.subplot(4, num_samples, 2*num_samples + i + 1)
        plt.title("Reconstructed Bottom Half")
        plt.imshow(reconstructed[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
        plt.axis("off")
       
        diff = torch.abs(images - reconstructed) * (1 - masks)
        plt.subplot(4, num_samples, 3*num_samples + i + 1)
        plt.title("Difference Map")
        plt.imshow(diff[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5, cmap='hot')
        plt.axis("off")
       
        print(f"Sample {i+1}: PSNR: {psnr_val:.4f}, SSIM: {ssim_val:.4f}, MSE: {mse_val:.4f}, Identity: {identity_val:.4f}, Landmark: {landmark_val:.4f}")
plt.tight_layout()
plt.savefig('/kaggle/working/results/bottom_half_reconstruction_results.png', dpi=150, bbox_inches='tight')
plt.show()
# Plot metrics
plt.figure(figsize=(20, 10))
plt.subplot(2, 3, 1)
plt.plot(range(1, num_samples + 1), psnr_values, marker='o', linewidth=2, markersize=8)
plt.title("PSNR (Bottom Half)", fontsize=14)
plt.xlabel("Sample", fontsize=12)
plt.ylabel("Value", fontsize=12)
plt.grid(True, alpha=0.3)
plt.subplot(2, 3, 2)
plt.plot(range(1, num_samples + 1), ssim_values, marker='s', linewidth=2, markersize=8, color='orange')
plt.title("SSIM (Bottom Half)", fontsize=14)
plt.xlabel("Sample", fontsize=12)
plt.ylabel("Value", fontsize=12)
plt.grid(True, alpha=0.3)
plt.subplot(2, 3, 3)
plt.plot(range(1, num_samples + 1), mse_values, marker='^', linewidth=2, markersize=8, color='green')
plt.title("MSE (Bottom Half)", fontsize=14)
plt.xlabel("Sample", fontsize=12)
plt.ylabel("Value", fontsize=12)
plt.grid(True, alpha=0.3)
plt.subplot(2, 3, 4)
plt.plot(range(1, num_samples + 1), identity_values, marker='d', linewidth=2, markersize=8, color='purple')
plt.title("Identity Loss", fontsize=14)
plt.xlabel("Sample", fontsize=12)
plt.ylabel("Value", fontsize=12)
plt.grid(True, alpha=0.3)
plt.subplot(2, 3, 5)
plt.plot(range(1, num_samples + 1), landmark_values, marker='*', linewidth=2, markersize=8, color='brown')
plt.title("Landmark Loss", fontsize=14)
plt.xlabel("Sample", fontsize=12)
plt.ylabel("Value", fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/results/bottom_half_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
# Print and save evaluation results
print("\n" + "="*60)
print("Enhanced Evaluation Metrics (Bottom Half) - Test Set:")
print("="*60)
print(f"Average PSNR: {np.mean(psnr_values):.4f} dB (±{np.std(psnr_values):.4f})")
print(f"Average SSIM: {np.mean(ssim_values):.4f} (±{np.std(ssim_values):.4f})")
print(f"Average MSE: {np.mean(mse_values):.4f} (±{np.std(mse_values):.4f})")
print(f"Average Identity Loss: {np.mean(identity_values):.4f} (±{np.std(identity_values):.4f})")
print(f"Average Landmark Loss: {np.mean(landmark_values):.4f} (±{np.std(landmark_values):.4f})")
print("="*60)
print(f"Best PSNR: {np.max(psnr_values):.4f} dB")
print(f"Best SSIM: {np.max(ssim_values):.4f}")
print(f"Worst PSNR: {np.min(psnr_values):.4f} dB")
print(f"Worst SSIM: {np.min(ssim_values):.4f}")
print("="*60)
with open('/kaggle/working/results/enhanced_bottom_half_metrics.txt', 'w', encoding='utf-8') as f:
    f.write("Enhanced Bottom Half Face Reconstruction Evaluation Results - Test Set\n")
    f.write("="*60 + "\n")
    f.write(f"Average PSNR: {np.mean(psnr_values):.4f} dB (±{np.std(psnr_values):.4f})\n")
    f.write(f"Average SSIM: {np.mean(ssim_values):.4f} (±{np.std(ssim_values):.4f})\n")
    f.write(f"Average MSE: {np.mean(mse_values):.4f} (±{np.std(mse_values):.4f})\n")
    f.write(f"Average Identity Loss: {np.mean(identity_values):.4f} (±{np.std(identity_values):.4f})\n")
    f.write(f"Average Landmark Loss: {np.mean(landmark_values):.4f} (±{np.std(landmark_values):.4f})\n")
    f.write("="*60 + "\n")
    f.write(f"Best PSNR: {np.max(psnr_values):.4f} dB\n")
    f.write(f"Best SSIM: {np.max(ssim_values):.4f}\n")
    f.write(f"Worst PSNR: {np.min(psnr_values):.4f} dB\n")
    f.write(f"Worst SSIM: {np.min(ssim_values):.4f}\n")
    f.write("="*60 + "\n")
    for i, (psnr_val, ssim_val, mse_val, id_val, lm_val) in enumerate(zip(psnr_values, ssim_values, mse_values, identity_values, landmark_values)):
        f.write(f"Sample {i+1}: PSNR={psnr_val:.4f}, SSIM={ssim_val:.4f}, MSE={mse_val:.4f}, Identity={id_val:.4f}, Landmark={lm_val:.4f}\n")
print(f"\nTraining and evaluation complete!")
print(f"Models saved in '/kaggle/working/models/' directory.")
print(f"Evaluation results saved in '/kaggle/working/results/' directory.")

In [ ]:
# import os
# os.environ.setdefault('TORCH_CUDA_ARCH_LIST', '7.5') # For Kaggle GPU compatibility
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torchvision.models as models
# import torchvision.transforms as transforms
# from torch.utils.data import Dataset, DataLoader
# import matplotlib.pyplot as plt
# import numpy as np
# from skimage.metrics import peak_signal_noise_ratio as psnr
# from skimage.metrics import structural_similarity as ssim
# import random
# import cv2
# from PIL import Image
# from torch import amp  # For mixed precision training (new API)
# import networkx as nx  # For graph operations

# # Simplified GNN for graph processing
# class PyramidalGNN(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super(PyramidalGNN, self).__init__()
#         self.lin = nn.Linear(in_channels, out_channels)
#         self.gate = nn.Linear(in_channels * 2, out_channels)

#     def forward(self, x, edge_index):
#         # Simplified version - just apply linear transformation
#         return self.lin(x)



# # /kaggle/input/celeba-resized-6464/img_align_celeba/CelebA_Image_Cropped_64
# # Configuration parameters
# prune_threshold = 0.2  # Threshold for pruning low-value nodes
# lambda_l1 = 100.0  # L1 reconstruction loss weight
# n_critic = 5  # Number of discriminator updates per generator update

# # Global prune_graph function using NetworkX
# def prune_graph(model, graph, threshold=prune_threshold):
#     # NetworkX pruning
#     deg_cent = nx.degree_centrality(graph)
#     low_value_nodes = [n for n, score in deg_cent.items() if score < threshold]
#     graph.remove_nodes_from(low_value_nodes)
#     return graph

# # Try to import MTCNN for landmark detection
# try:
#     from mtcnn import MTCNN
#     USE_MTCNN = False # Disable during training to avoid NumPy conversions
#     print("MTCNN imported successfully for landmark detection (disabled for training).")
# except ImportError:
#     print("MTCNN not found. Landmark loss will be simplified.")
#     USE_MTCNN = False
# # ============================================================================
# # CHECKPOINT CONFIGURATION - Resume Training Setup
# # تنظیمات Checkpoint - برای ادامه آموزش
# # ============================================================================

# # 🔄 RESUME TRAINING: Set to True to continue from previous checkpoint
# # ادامه آموزش: برای ادامه از checkpoint قبلی، True کنید
# RESUME_TRAINING = True  # True: Resume from checkpoint | False: Start from scratch

# # 📂 CHECKPOINT PATHS: Where to load/save checkpoints
# # مسیرهای Checkpoint: محل بارگذاری/ذخیره checkpoint‌ها
# CHECKPOINT_DIR = '/kaggle/working/checkpoints'
# CHECKPOINT_INTERVAL = 1  # Save checkpoint every N epochs (1 = every epoch)

# # 📥 LOAD FROM KAGGLE DATASET: If you uploaded models as Kaggle dataset
# # بارگذاری از Kaggle Dataset: اگر مدل‌ها را به عنوان dataset آپلود کرده‌اید
# USE_KAGGLE_INPUT = True  # Set True if loading from Kaggle dataset input
# KAGGLE_INPUT_DATASET = '/kaggle/input/image-inpainting-v1/pytorch/default/3'  # Update this path (e.g., '/kaggle/input/checkpoints')
# # /kaggle/input/image-inpainting/pytorch/default/2
# # /kaggle/input/image-inpainting-v1/pytorch/default/3
# # /kaggle/input/image-inpainting/pytorch/default/4
# # /kaggle/input/image-inpainting/pytorch/default/3
# # 🗂️ LOAD FROM MODEL FILES: If you only have model state_dict files (*.pth)
# # بارگذاری از فایل‌های مدل: اگر فقط فایل‌های state_dict مدل دارید
# LOAD_FROM_MODELS = True  # Set True if loading from models folder (coarse_generator_best.pth, etc.)
# MODELS_DIR = '/kaggle/working/models'  # Path to models folder
# USE_BEST_MODELS = True  # True: Load *_best.pth | False: Load *_bottom_half.pth

# # ⚠️ نکته: اگر از LOAD_FROM_MODELS استفاده کنید، فقط وزن‌های مدل بارگذاری می‌شوند
# # optimizer/scheduler/training history بارگذاری نمی‌شوند و از اول مقداردهی می‌شوند

# # ⚙️ TRAINING START POINT: Which epoch to start from
# # نقطه شروع آموزش: از کدام epoch شروع شود
# # Note: This will be automatically set when loading checkpoint
# # توجه: این مقدار به صورت خودکار هنگام بارگذاری checkpoint تنظیم می‌شود

# print("="*70)
# print("🔧 CHECKPOINT CONFIGURATION")
# print("="*70)
# print(f"📌 Resume Training: {RESUME_TRAINING}")
# print(f"📁 Checkpoint Directory: {CHECKPOINT_DIR}")
# print(f"💾 Save Interval: Every {CHECKPOINT_INTERVAL} epoch(s)")
# if USE_KAGGLE_INPUT:
#     print(f"📥 Loading from Kaggle Dataset: {KAGGLE_INPUT_DATASET}")
# if LOAD_FROM_MODELS:
#     print(f"🗂️  Load from Models: {MODELS_DIR}")
#     print(f"   Using: {'*_best.pth' if USE_BEST_MODELS else '*_bottom_half.pth'}")
# print("="*70 + "\n")

# # ============================================================================
# # TRAINING CONFIGURATION - Initial configurations
# # تنظیمات آموزش - پیکربندی اولیه
# # ============================================================================

# image_size = (64, 64)
# batch_size = 128 # Further reduced for stability
# eval_batch_size = 1
# num_epochs = 20 # Increased for better convergence
# learning_rate_g = 0.00005 # 5e-5 for Generator (reduced for more stable training)
# learning_rate_d = 0.0002 # 2e-4 for Discriminator (reduced for better balance)
# lambda_adv = 0.05 # Further reduced adversarial weight
# lambda_style = 0.05 # Reduced style weight
# lambda_fm = 0.05 # Reduced feature matching weight
# lambda_rec = 1.0 # Reduced reconstruction weight
# lambda_identity = 0.005 # Reduced identity weight
# lambda_landmark = 0.01 # Reduced landmark weight
# lambda_perc = 0.1 # Reduced perceptual weight
# lambda_vae = 0.01 # Further reduced VAE weight
# lambda_struct = 0.05 # Reduced structural weight
# lambda_d_reg = 1e-4 # Increased discriminator regularization
# lambda_attention = 0.05 # Reduced attention loss weight
# # NEW: Edge-Guided Inpainting Loss Weights (Fine-tuned for better quality)
# lambda_edge = 0.7 # Edge loss weight for preserving nose/lips edges (increased from 0.5)
# lambda_hf = 0.5 # High-frequency loss weight to reduce blurriness (increased from 0.3)
# lambda_gradient = 0.6 # Gradient loss weight for sharp edges (increased from 0.4)
# lambda_facial = 0.8 # Facial region loss weight for nose/lips focus (increased from 0.6)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")
# # Data splitting configuration
# def split_dataset(dataset_path, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     image_files = [f for f in os.listdir(dataset_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
#     random.shuffle(image_files)
#     total_files = len(image_files)
#     train_end = int(total_files * train_ratio)
#     val_end = train_end + int(total_files * val_ratio)
#     train_files, val_files, test_files = image_files[:train_end], image_files[train_end:val_end], image_files[val_end:]
#     print(f"Dataset split: Total={total_files}, Train={len(train_files)} ({len(train_files)/total_files*100:.1f}%), "
#           f"Val={len(val_files)} ({len(val_files)/total_files*100:.1f}%), Test={len(test_files)} ({len(test_files)/total_files*100:.1f}%)")
#     return train_files, val_files, test_files
# # Custom Dataset
# class CelebADataset(Dataset):
#     def __init__(self, dataset_path, file_list, image_size):
#         self.dataset_path = dataset_path
#         self.file_list = file_list
#         self.transform = transforms.Compose([
#             transforms.Resize(image_size),
#             transforms.ToTensor(),
#             transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # -1 to 1
#         ])
   
#     def __len__(self):
#         return len(self.file_list)
   
#     def __getitem__(self, idx):
#         img_path = os.path.join(self.dataset_path, self.file_list[idx])
#         image = Image.open(img_path).convert('RGB')
#         image = self.transform(image)
#         return image
# def create_split_datasets(dataset_path, train_files, val_files, test_files, image_size, batch_size):
#     train_dataset = CelebADataset(dataset_path, train_files, image_size)
#     val_dataset = CelebADataset(dataset_path, val_files, image_size)
#     test_dataset = CelebADataset(dataset_path, test_files, image_size)
   
#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
#     val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
#     test_loader = DataLoader(test_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=2, pin_memory=True)
#     return train_loader, val_loader, test_loader
# # Enhanced ViT-based Preprocessor with Multi-layer Attention and Graph Pruning
# class ViTPreprocessor(nn.Module):
#     def __init__(self, dim=128, num_heads=8, patch_size=8, num_layers=3):
#         super().__init__()
#         self.patch_size = patch_size
#         self.dim = dim
#         self.num_heads = num_heads
#         self.num_patches = (image_size[0] // patch_size) * (image_size[1] // patch_size)
#         self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size, padding=0)
#         self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, dim))
       
#         # Multi-layer attention blocks
#         self.attention_blocks = nn.ModuleList([
#             nn.ModuleDict({
#                 'attn': nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=0.1),
#                 'norm1': nn.LayerNorm(dim, eps=1e-6),
#                 'norm2': nn.LayerNorm(dim, eps=1e-6),
#                 'mlp': nn.Sequential(
#                     nn.Linear(dim, dim * 4),
#                     nn.GELU(),
#                     nn.Dropout(0.1),
#                     nn.Linear(dim * 4, dim),
#                     nn.Dropout(0.1)
#                 )
#             }) for _ in range(num_layers)
#         ])
       
#         self.final_norm = nn.LayerNorm(dim, eps=1e-6)
#         self.gnn = PyramidalGNN(dim, dim)  # Add PyramidalGNN
   
#     def build_graph(self, features):
#         B, C, H, W = features.shape
#         nodes = features.view(B, C, -1).permute(0, 2, 1)  # [B, num_patches, dim]
        
#         # Use NetworkX for graph construction
#         G = nx.Graph()
#         for i in range(nodes.shape[1]):
#             G.add_node(i, feat=nodes[0, i])
#         for i in range(nodes.shape[1]):
#             for j in range(i+1, nodes.shape[1]):
#                 sim = F.cosine_similarity(nodes[0, i:i+1], nodes[0, j:j+1])
#                 if sim > 0.5:
#                     G.add_edge(i, j, weight=sim)
#         return G
   
   
#     def graph_enhanced_forward(self, x, graph):
#         # Check if graph is empty after pruning
#         node_list = list(graph.nodes)
#         if not node_list:
#             return x  # Return input features directly
        
#         # Simple message passing with NetworkX
#         for node in graph.nodes:
#             neighbors = list(graph.neighbors(node))
#             if neighbors:
#                 neighbor_feats = torch.stack([graph.nodes[n]['feat'] for n in neighbors])
#                 aggregated = torch.mean(neighbor_feats, dim=0)
#                 graph.nodes[node]['feat'] = (graph.nodes[node]['feat'] + aggregated) / 2
        
#         # Final check before stacking
#         final_nodes = list(graph.nodes)
#         if not final_nodes:
#             return x
        
#         # Ensure all features have the same dtype
#         node_features = [graph.nodes[n]['feat'] for n in sorted(final_nodes)]
#         return torch.stack(node_features)
   
#     def reconstruct_features(self, pruned_feats, pruned_indices, B, C, H, W):
#         num_patches = H * W
#         # Ensure dtype matches the input features
#         full_feats = torch.zeros(B, num_patches, C, device=pruned_feats.device, dtype=pruned_feats.dtype)
#         full_feats[:, pruned_indices] = pruned_feats
#         return full_feats.reshape(B, H, W, C).permute(0, 3, 1, 2)
   
#     def forward(self, x):
#         embedded = self.patch_embed(x)  # [B, dim, H/p, W/p]
#         B, C, H, W = embedded.shape
#         flat = embedded.permute(0, 2, 3, 1).reshape(B, H * W, C)
        
#         # Adaptive position embedding based on actual patch count
#         num_patches = H * W
#         if num_patches != self.num_patches:
#             # Interpolate position embedding to match current patch count
#             # Reshape to 3D for linear interpolation: [1, dim, num_patches]
#             pos_embed_3d = self.pos_embed.permute(0, 2, 1)  # [1, dim, num_patches]
#             pos_embed_adaptive = F.interpolate(
#                 pos_embed_3d,
#                 size=num_patches,
#                 mode='linear',
#                 align_corners=False
#             ).permute(0, 2, 1)  # [1, num_patches, dim]
#         else:
#             pos_embed_adaptive = self.pos_embed
        
#         flat = flat + pos_embed_adaptive
        
#         # Build and prune graph
#         graph = self.build_graph(embedded)
#         pruned_graph = prune_graph(self, graph)
        
#         # Get pruned features using NetworkX
#         pruned_indices = list(pruned_graph.nodes)
#         if pruned_indices:  # Check if not empty
#             pruned_feats = torch.stack([pruned_graph.nodes[i]['feat'] for i in pruned_indices])
#         else:
#             pruned_feats = flat[0].clone()  # Use original flattened features for batch 0
#             pruned_indices = list(range(flat.shape[1]))  # All indices
        
#         # Enhanced processing on high-value nodes
#         for block in self.attention_blocks:
#             # Self-attention on pruned features
#             attn_out, _ = block['attn'](pruned_feats, pruned_feats, pruned_feats)
#             pruned_feats = block['norm1'](pruned_feats + attn_out)
            
#             # MLP
#             mlp_out = block['mlp'](pruned_feats)
#             pruned_feats = block['norm2'](pruned_feats + mlp_out)
        
#         # Graph-enhanced message passing
#         enhanced_pruned = self.graph_enhanced_forward(pruned_feats, pruned_graph)
        
#         # Reconstruct full features with low-value nodes getting minimal processing
#         x = self.reconstruct_features(enhanced_pruned, pruned_indices, B, C, H, W)
        
#         # Minimal processing for pruned (low-value) regions - e.g., average pooling
#         full_x = self.final_norm(x.view(B, C, -1).permute(0, 2, 1)).permute(0, 2, 1).view(B, C, H, W)
#         return full_x
# # VAE Sampling Layer
# class VAESampling(nn.Module):
#     def __init__(self):
#         super().__init__()
   
#     def forward(self, mu, logvar):
#         std = torch.exp(0.5 * logvar)
#         eps = torch.randn_like(std)
#         return mu + eps * std
# # Enhanced VAE Encoder with Residual Connections and Graph Integration (No BatchNorm, Strided Conv)
# class VAEEncoder(nn.Module):
#     def __init__(self, latent_dim=128):
#         super().__init__()
#         self.latent_dim = latent_dim
       
#         # Enhanced encoder with strided convolutions (no batch norm)
#         self.conv1 = nn.Conv2d(128, 64, kernel_size=4, stride=2, padding=1) # [B, 128, 8, 8] -> [B, 64, 4, 4]
#         self.conv2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1) # [B, 64, 4, 4] -> [B, 128, 2, 2]
       
#         # Residual block (no batch norm)
#         self.res_conv = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
       
#         self.flatten = nn.Flatten()
#         # Use adaptive pooling to ensure consistent output size
#         self.adaptive_pool = nn.AdaptiveAvgPool2d((2, 2))  # Always output 2x2
#         self.fc_mu = nn.Linear(128 * 2 * 2, latent_dim) # 128 * 2 * 2 = 512
#         self.fc_logvar = nn.Linear(128 * 2 * 2, latent_dim) # 128 * 2 * 2 = 512
   
#     def forward(self, x):
#         # Check input size and use adaptive approach for very small inputs
#         B, C, H, W = x.shape
        
#         # If input is too small for conv layers, use adaptive pooling directly
#         if H < 4 or W < 4:
#             # Use adaptive pooling to get to a reasonable size first
#             x = F.adaptive_avg_pool2d(x, (8, 8))  # Resize to 8x8 minimum
#             x = F.relu(self.conv1(x))  # 8x8 -> 4x4
#             x = F.relu(self.conv2(x))  # 4x4 -> 2x2
#         else:
#             x = F.relu(self.conv1(x))
#             x = F.relu(self.conv2(x))
       
#         # Residual connection
#         residual = x
#         x = F.relu(self.res_conv(x))
#         x = x + residual
       
#         # Use adaptive pooling to ensure consistent size
#         x = self.adaptive_pool(x)  # Always [B, 128, 2, 2]
#         x = self.flatten(x)  # Always [B, 512]
#         mu = self.fc_mu(x)
#         logvar = self.fc_logvar(x)
#         return mu, logvar
# # Enhanced VAE Decoder with Residual Connections and Graph Integration (No BatchNorm)
# class VAEDecoder(nn.Module):
#     def __init__(self, latent_dim=128):
#         super().__init__()
#         self.latent_dim = latent_dim
       
#         # Enhanced decoder with strided transposed convolutions (no batch norm)
#         self.fc = nn.Linear(latent_dim, 8 * 8 * 128)
       
#         # Residual block before upsampling (no batch norm)
#         self.res_conv = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
       
#         self.deconv1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
#         self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
#         self.deconv3 = nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1)
   
#     def forward(self, z):
#         x = F.relu(self.fc(z))
#         x = x.view(-1, 128, 8, 8)
       
#         # Residual connection
#         residual = x
#         x = F.relu(self.res_conv(x))
#         x = x + residual
       
#         x = F.relu(self.deconv1(x)) # 8x8 -> 16x16
#         x = F.relu(self.deconv2(x)) # 16x16 -> 32x32
#         x = torch.tanh(self.deconv3(x)) # 32x32 -> 64x64
#         return x
# # Enhanced CoarseGenerator with Attention, Skip Connections, and Graph Pruning
# # ============================================================================
# # EDGE DETECTION AND EDGE GENERATOR
# # ماژول‌های تشخیص و بازسازی لبه برای بهبود بازسازی بینی و لب‌ها
# # ============================================================================

# class EdgeDetector(nn.Module):
#     """
#     Edge detection using Sobel filters to extract edges from images
#     استخراج لبه‌ها با استفاده از فیلترهای Sobel
#     """
#     def __init__(self):
#         super().__init__()
#         # Sobel filters for edge detection
#         sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
#         sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        
#         self.register_buffer('sobel_x', sobel_x.repeat(3, 1, 1, 1))
#         self.register_buffer('sobel_y', sobel_y.repeat(3, 1, 1, 1))
    
#     def forward(self, x):
#         # Apply Sobel filters to detect edges
#         # x: (B, 3, H, W)
#         edge_x = F.conv2d(x, self.sobel_x, padding=1, groups=3)
#         edge_y = F.conv2d(x, self.sobel_y, padding=1, groups=3)
        
#         # Compute edge magnitude
#         edges = torch.sqrt(edge_x ** 2 + edge_y ** 2 + 1e-8)
#         return edges

# class EdgeGenerator(nn.Module):
#     """
#     Edge Generator for reconstructing edges of nose and lips
#     Generator برای بازسازی لبه‌های بینی و لب‌ها
#     """
#     def __init__(self):
#         super().__init__()
        
#         # Encoder for edge feature extraction
#         self.enc1 = nn.Sequential(
#             nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1),  # Input: masked image + mask
#             nn.LeakyReLU(0.2, inplace=True)
#         )
#         self.enc2 = nn.Sequential(
#             nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
#         self.enc3 = nn.Sequential(
#             nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
        
#         # Bottleneck with attention for edge features
#         self.bottleneck = nn.Sequential(
#             nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
        
#         # Attention mechanism
#         self.attention = nn.MultiheadAttention(embed_dim=512, num_heads=8, dropout=0.1)
#         self.attention_norm = nn.LayerNorm(512, eps=1e-6)
        
#         # Decoder for edge reconstruction
#         self.dec1 = nn.Sequential(
#             nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
#             nn.ReLU(inplace=True)
#         )
#         self.dec2 = nn.Sequential(
#             nn.ConvTranspose2d(512, 128, kernel_size=4, stride=2, padding=1),  # 256 + 256 from skip
#             nn.ReLU(inplace=True)
#         )
#         self.dec3 = nn.Sequential(
#             nn.ConvTranspose2d(256, 64, kernel_size=4, stride=2, padding=1),   # 128 + 128 from skip
#             nn.ReLU(inplace=True)
#         )
#         self.dec4 = nn.Sequential(
#             nn.ConvTranspose2d(128, 3, kernel_size=4, stride=2, padding=1),    # 64 + 64 from skip
#             nn.Sigmoid()  # Output edge map in [0, 1]
#         )
    
#     def forward(self, masked_image, mask):
#         # Concatenate masked image and mask
#         x = torch.cat([masked_image, mask], dim=1)
        
#         # Encoder with skip connections
#         enc1_out = self.enc1(x)       # 64x64 -> 32x32
#         enc2_out = self.enc2(enc1_out) # 32x32 -> 16x16
#         enc3_out = self.enc3(enc2_out) # 16x16 -> 8x8
#         bottleneck = self.bottleneck(enc3_out)  # 8x8 -> 4x4
        
#         # Apply attention to bottleneck features
#         B, C, H, W = bottleneck.shape
#         attn_input = bottleneck.permute(0, 2, 3, 1).reshape(B, H * W, C)
#         attn_output, _ = self.attention(attn_input, attn_input, attn_input)
#         enhanced_features = self.attention_norm(attn_input + attn_output)
#         enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
        
#         # Decoder with skip connections
#         dec1_out = self.dec1(enhanced_features)              # 4x4 -> 8x8
#         dec1_out = torch.cat([dec1_out, enc3_out], dim=1)    # Skip connection
        
#         dec2_out = self.dec2(dec1_out)                       # 8x8 -> 16x16
#         dec2_out = torch.cat([dec2_out, enc2_out], dim=1)    # Skip connection
        
#         dec3_out = self.dec3(dec2_out)                       # 16x16 -> 32x32
#         dec3_out = torch.cat([dec3_out, enc1_out], dim=1)    # Skip connection
        
#         output = self.dec4(dec3_out)                         # 32x32 -> 64x64
        
#         return output

# class CoarseGenerator(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.preprocessor = ViTPreprocessor(dim=128, num_heads=8, patch_size=8, num_layers=3)
#         self.encoder = VAEEncoder(latent_dim=128) # Increased latent dimension
#         self.sampling = VAESampling()
#         self.decoder = VAEDecoder(latent_dim=128)
       
#         # Attention mechanism for better feature fusion
#         self.attention = nn.MultiheadAttention(embed_dim=128, num_heads=8, dropout=0.1)
#         self.attention_norm = nn.LayerNorm(128, eps=1e-6)
       
#         # Skip connection processing
#         self.skip_conv = nn.Conv2d(128, 128, kernel_size=1, padding=0)
   
#     def forward(self, x, mask, edge_map=None):
#         # If edge map is provided, concatenate with input for edge-guided inpainting
#         if edge_map is not None:
#             x = x + edge_map * 0.1  # Blend edge information with input
        
#         # Preprocess with graph pruning
#         original_features = self.preprocessor(x)
       
#         # Apply attention to enhance features
#         B, C, H, W = original_features.shape
#         attn_input = original_features.permute(0, 2, 3, 1).reshape(B, H * W, C)
#         attn_output, _ = self.attention(attn_input, attn_input, attn_input)
#         enhanced_features = self.attention_norm(attn_input + attn_output)
#         enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
       
#         # Combine with skip connection
#         enhanced_features = enhanced_features + self.skip_conv(original_features)
       
#         mu, logvar = self.encoder(enhanced_features)
#         z = self.sampling(mu, logvar)
#         output = self.decoder(z)
#         return output, mu, logvar
# # Enhanced FineGenerator with U-Net Architecture, Attention, Edge-Guided, and Graph Cut Integration
# class FineGenerator(nn.Module):
#     def __init__(self):
#         super().__init__()
       
#         # Encoder with skip connections (edge-guided: 9 channels = 6 + 3 edge)
#         self.enc1 = nn.Sequential(
#             nn.Conv2d(9, 64, kernel_size=4, stride=2, padding=1),  # Added 3 channels for edge map
#             nn.LeakyReLU(0.2, inplace=True)
#         )
#         self.enc2 = nn.Sequential(
#             nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
#         self.enc3 = nn.Sequential(
#             nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
       
#         # Bottleneck with attention
#         self.bottleneck = nn.Sequential(
#             nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
       
#         # Attention mechanism
#         self.attention = nn.MultiheadAttention(embed_dim=512, num_heads=8, dropout=0.1)
#         self.attention_norm = nn.LayerNorm(512, eps=1e-6)
       
#         # Decoder with skip connections
#         self.dec1 = nn.Sequential(
#             nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
#             nn.ReLU(inplace=True)
#         )
#         self.dec2 = nn.Sequential(
#             nn.ConvTranspose2d(512, 128, kernel_size=4, stride=2, padding=1), # 256 + 256 from skip
#             nn.ReLU(inplace=True)
#         )
#         self.dec3 = nn.Sequential(
#             nn.ConvTranspose2d(256, 64, kernel_size=4, stride=2, padding=1), # 128 + 128 from skip
#             nn.ReLU(inplace=True)
#         )
#         self.dec4 = nn.Sequential(
#             nn.ConvTranspose2d(128, 3, kernel_size=4, stride=2, padding=1), # 64 + 64 from skip
#             nn.Tanh()
#         )
   
#     def forward(self, masked_images, coarse_output, masks, edge_map=None):
#         # Ensure all inputs have the same spatial dimensions
#         if masked_images.shape[2:] != coarse_output.shape[2:]:
#             coarse_output = F.interpolate(coarse_output, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
        
#         # If edge map is provided, concatenate it with input for edge-guided inpainting
#         if edge_map is not None:
#             if edge_map.shape[2:] != masked_images.shape[2:]:
#                 edge_map = F.interpolate(edge_map, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
#             x = torch.cat([masked_images, coarse_output, edge_map], dim=1)  # 3 + 3 + 3 = 9 channels
#         else:
#             # Fallback: use zeros for edge map if not provided
#             edge_map_zeros = torch.zeros_like(masked_images)
#             x = torch.cat([masked_images, coarse_output, edge_map_zeros], dim=1)
       
#         # Encoder with skip connections
#         enc1_out = self.enc1(x) # 64x64 -> 32x32
#         enc2_out = self.enc2(enc1_out) # 32x32 -> 16x16
#         enc3_out = self.enc3(enc2_out) # 16x16 -> 8x8
#         bottleneck = self.bottleneck(enc3_out) # 8x8 -> 4x4
       
#         # Apply attention to bottleneck features
#         B, C, H, W = bottleneck.shape
#         attn_input = bottleneck.permute(0, 2, 3, 1).reshape(B, H * W, C)
#         attn_output, _ = self.attention(attn_input, attn_input, attn_input)
#         enhanced_features = self.attention_norm(attn_input + attn_output)
#         enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
       
#         # Decoder with skip connections
#         dec1_out = self.dec1(enhanced_features) # 4x4 -> 8x8
#         dec1_out = torch.cat([dec1_out, enc3_out], dim=1) # Skip connection
       
#         dec2_out = self.dec2(dec1_out) # 8x8 -> 16x16
#         dec2_out = torch.cat([dec2_out, enc2_out], dim=1) # Skip connection
       
#         dec3_out = self.dec3(dec2_out) # 16x16 -> 32x32
#         dec3_out = torch.cat([dec3_out, enc1_out], dim=1) # Skip connection
       
#         output = self.dec4(dec3_out) # 32x32 -> 64x64
       
#         # Ensure output matches input size
#         if output.shape[2:] != masked_images.shape[2:]:
#             output = F.interpolate(output, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
       
#         return output
# # Stronger Discriminator with Spectral Normalization (No BatchNorm, More Layers)
# class Discriminator(nn.Module):
#     def __init__(self):
#         super().__init__()
       
#         # Stronger discriminator with more layers and capacity (no batch norm)
#         self.conv1 = nn.utils.spectral_norm(nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1))
#         self.conv2 = nn.utils.spectral_norm(nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1))
#         self.conv3 = nn.utils.spectral_norm(nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1))
#         self.conv4 = nn.utils.spectral_norm(nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1))
#         # Additional layers for stronger discriminator
#         self.conv5 = nn.utils.spectral_norm(nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1))
#         self.conv6 = nn.utils.spectral_norm(nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1))
       
#         # Global average pooling for stability
#         self.global_pool = nn.AdaptiveAvgPool2d(1)
#         # Deeper fully connected layers
#         self.fc1 = nn.utils.spectral_norm(nn.Linear(512, 256))
#         self.fc2 = nn.utils.spectral_norm(nn.Linear(256, 1))
       
#         # Feature layers for feature matching loss
#         self.feature_layers = [self.conv1, self.conv2, self.conv3, self.conv4, self.conv5, self.conv6]
   
#     def forward(self, x, mask, return_features=False):
#         input_concat = torch.cat([x, mask], dim=1)
#         features = []
       
#         # Main discriminator path with more layers
#         x = F.leaky_relu(self.conv1(input_concat), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv2(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv3(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv4(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv5(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv6(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         # Global average pooling for stability
#         x = self.global_pool(x)
#         x = x.view(x.size(0), -1)
        
#         # Deeper FC layers
#         x = F.leaky_relu(self.fc1(x), 0.2, inplace=True)
#         logits = self.fc2(x)
       
#         if return_features:
#             return logits, features
#         return logits
# # Generate adaptive mask
# def generate_adaptive_mask(image):
#     batch_size = image.shape[0]
#     h, w = image_size
#     mask = torch.ones(batch_size, 3, h, w, dtype=torch.float32, device=image.device)
#     half_height = h // 2
#     bottom_mask = torch.zeros(batch_size, 3, h - half_height, w, dtype=torch.float32, device=image.device)
#     top_mask = torch.ones(batch_size, 3, half_height, w, dtype=torch.float32, device=image.device)
#     mask = torch.cat([top_mask, bottom_mask], dim=2)
   
#     if USE_MTCNN:
#         detector = MTCNN()
#         images_np = ((image * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
#         landmarks = []
#         for img in images_np:
#             img_uint8 = img.astype(np.uint8)
#             result = detector.detect_faces(img_uint8)
#             landmarks.append(result[0]['keypoints'] if result else {})
       
#         mask_np = mask.permute(0, 2, 3, 1).cpu().detach().numpy()
#         for i, lm in enumerate(landmarks):
#             if lm:
#                 for key in ['mouth_left', 'mouth_right']:
#                     if key in lm:
#                         x, y = lm[key]
#                         y = min(max(y, half_height), h-1)
#                         x = min(max(x, 0), w-1)
#                         mask_np[i, y-5:y+5, x-5:x+5, :] = 0.0
#         mask = torch.from_numpy(mask_np).permute(0, 3, 1, 2).to(device)
   
#     return mask
# # Loss functions
# # Use new torchvision weights API to avoid deprecation warnings
# try:
#     vgg_weights = models.VGG16_Weights.DEFAULT
# except AttributeError:
#     # Fallback for very old torchvision
#     vgg_weights = None
# vgg = models.vgg16(weights=vgg_weights).features.to(device).eval()
# loss_model = nn.Sequential(*[vgg[i] for i in range(16)]).to(device) # Up to block4_conv3
# for param in loss_model.parameters():
#     param.requires_grad = False

# # Initialize Edge Detector for edge loss
# edge_detector_global = EdgeDetector().to(device)
# edge_detector_global.eval()
# for param in edge_detector_global.parameters():
#     param.requires_grad = False
# def perceptual_loss(y_true, y_pred):
#     y_true = F.interpolate(y_true, size=image_size, mode='bilinear', align_corners=False)
#     y_pred = F.interpolate(y_pred, size=image_size, mode='bilinear', align_corners=False)
#     y_true = y_true * 0.5 + 0.5 # Denormalize to [0, 1]
#     y_pred = y_pred * 0.5 + 0.5
#     true_features = loss_model(y_true)
#     pred_features = loss_model(y_pred)
#     return torch.mean((true_features - pred_features) ** 2)
# def style_loss(y_true, y_pred):
#     y_true = F.interpolate(y_true, size=image_size, mode='bilinear', align_corners=False)
#     y_pred = F.interpolate(y_pred, size=image_size, mode='bilinear', align_corners=False)
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     true_features = loss_model(y_true)
#     pred_features = loss_model(y_pred)
   
#     def gram_matrix(feat):
#         B, C, H, W = feat.shape
#         feat = feat.view(B, C, H * W)
#         return torch.bmm(feat, feat.transpose(1, 2)) / (C * H * W)
   
#     style_losses = [torch.mean((gram_matrix(t) - gram_matrix(p)) ** 2) for t, p in zip([true_features], [pred_features])]
#     return torch.mean(torch.stack(style_losses))
# def bottom_half_loss(y_true, y_pred, mask):
#     bottom_region = (mask == 0).float()
#     diff = (y_true - y_pred) ** 2 * bottom_region
#     return diff.sum() / (bottom_region.sum() + 1e-8)
# def feature_matching_loss(real_features, fake_features):
#     losses = [torch.mean((r - f) ** 2) for r, f in zip(real_features, fake_features)]
#     return torch.mean(torch.stack(losses))
# def identity_loss(y_true, y_pred, mask):
#     bottom_region = (mask == 0).float()
#     y_true_face = y_true * bottom_region
#     y_pred_face = y_pred * bottom_region
#     return torch.mean(torch.abs(y_true_face - y_pred_face))
# def landmark_guided_loss(y_true, y_pred, mask):
#     if not USE_MTCNN:
#         return bottom_half_loss(y_true, y_pred, mask)
#     detector = MTCNN()
#     bottom_region = (mask == 0).float()
#     y_true_bottom = y_true * bottom_region
#     y_pred_bottom = y_pred * bottom_region
#     y_true_np = ((y_true_bottom * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
#     y_pred_np = ((y_pred_bottom * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
#     loss_tensor = torch.tensor(0.0, device=device, requires_grad=True)
#     for true_img, pred_img in zip(y_true_np, y_pred_np):
#         true_img_uint8 = true_img.astype(np.uint8)
#         pred_img_uint8 = pred_img.astype(np.uint8)
#         true_lm = detector.detect_faces(true_img_uint8)
#         pred_lm = detector.detect_faces(pred_img_uint8)
#         if true_lm and pred_lm:
#             true_points = [true_lm[0]['keypoints'][k] for k in ['mouth_left', 'mouth_right']]
#             pred_points = [pred_lm[0]['keypoints'][k] for k in ['mouth_left', 'mouth_right']]
#             for t, p in zip(true_points, pred_points):
#                 diff_x = torch.tensor(t[0] - p[0], dtype=torch.float32, device=device)
#                 diff_y = torch.tensor(t[1] - p[1], dtype=torch.float32, device=device)
#                 loss_tensor = loss_tensor + (diff_x ** 2 + diff_y ** 2)
#     return loss_tensor / y_true.shape[0]
# def structural_ssim_loss(y_true, y_pred, mask):
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     bottom_mask = (mask == 0).float()
#     y_true_bottom = y_true * bottom_mask
#     y_pred_bottom = y_pred * bottom_mask
#     ssim_vals = []
#     for i in range(y_true.shape[0]):
#         true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         ssim_val = ssim(true_img, pred_img, channel_axis=2, data_range=1.0, win_size=7)
#         ssim_vals.append(ssim_val)
#     return 1.0 - torch.mean(torch.tensor(ssim_vals, device=device))
# # Advanced Loss Functions
# def gradient_penalty_loss(discriminator, real_images, fake_images, masks, device):
#     """Calculate gradient penalty for WGAN-GP"""
#     batch_size = real_images.size(0)
#     alpha = torch.rand(batch_size, 1, 1, 1).to(device)
#     interpolated = alpha * real_images + (1 - alpha) * fake_images
   
#     # Ensure interpolated tensor requires gradients
#     interpolated = interpolated.detach().requires_grad_(True)
   
#     disc_interpolated = discriminator(interpolated, masks, return_features=False)
   
#     # Check if disc_interpolated requires gradients
#     if not disc_interpolated.requires_grad:
#         return torch.tensor(0.0, device=device, requires_grad=True)
   
#     gradients = torch.autograd.grad(
#         outputs=disc_interpolated,
#         inputs=interpolated,
#         grad_outputs=torch.ones_like(disc_interpolated).to(device),
#         create_graph=True,
#         retain_graph=True,
#         only_inputs=True
#     )[0]
   
#     gradients = gradients.view(gradients.size(0), -1)
#     gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
#     return gradient_penalty
# def attention_loss(y_true, y_pred, mask):
#     """Simplified attention-based loss focusing on important regions"""
#     bottom_region = (mask == 0).float()
#     diff = torch.abs(y_true - y_pred) * bottom_region
   
#     # Create simple attention weights based on gradient magnitude
#     true_grad_x = torch.abs(y_true[:, :, :, 1:] - y_true[:, :, :, :-1])
#     true_grad_y = torch.abs(y_true[:, :, 1:, :] - y_true[:, :, :-1, :])
   
#     # Pad gradients to match original size
#     true_grad_x_padded = F.pad(true_grad_x, (0, 1, 0, 0), mode='constant', value=0)
#     true_grad_y_padded = F.pad(true_grad_y, (0, 0, 0, 1), mode='constant', value=0)
   
#     # Combine gradients and normalize
#     attention_weights = (true_grad_x_padded + true_grad_y_padded).mean(dim=1, keepdim=True)
#     attention_weights = attention_weights / (attention_weights.mean() + 1e-8) # Normalize
   
#     # Apply attention weights
#     weighted_loss = diff * attention_weights
#     return weighted_loss.mean()

# # ============================================================================
# # NEW LOSS FUNCTIONS FOR EDGE-GUIDED INPAINTING
# # توابع Loss جدید برای بهبود لبه‌ها و کاهش مات بودن
# # ============================================================================

# def edge_loss(y_true, y_pred, mask, edge_detector):
#     """
#     Edge loss to preserve edges in nose and lips regions
#     Loss برای حفظ لبه‌های بینی و لب‌ها
#     """
#     # Extract edges from true and predicted images
#     with torch.no_grad():
#         edges_true = edge_detector(y_true * 0.5 + 0.5)  # Denormalize first
#     edges_pred = edge_detector(y_pred * 0.5 + 0.5)
    
#     # Focus on bottom half (masked region)
#     bottom_region = (mask == 0).float()
    
#     # Calculate edge reconstruction loss
#     edge_diff = torch.abs(edges_true - edges_pred) * bottom_region
    
#     return edge_diff.mean()

# def high_frequency_loss(y_true, y_pred, mask):
#     """
#     High-frequency loss to reduce blurriness in bottom half
#     Loss برای کاهش مات بودن در نیمه پایینی
#     """
#     # Apply Laplacian filter to detect high-frequency details
#     laplacian_kernel = torch.tensor([
#         [0, 1, 0],
#         [1, -4, 1],
#         [0, 1, 0]
#     ], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(y_true.device)
    
#     # Repeat for 3 channels
#     laplacian_kernel = laplacian_kernel.repeat(3, 1, 1, 1)
    
#     # Apply Laplacian to true and predicted images
#     y_true_hf = F.conv2d(y_true, laplacian_kernel, padding=1, groups=3)
#     y_pred_hf = F.conv2d(y_pred, laplacian_kernel, padding=1, groups=3)
    
#     # Focus on bottom half
#     bottom_region = (mask == 0).float()
    
#     # Calculate high-frequency difference
#     hf_diff = torch.abs(y_true_hf - y_pred_hf) * bottom_region
    
#     return hf_diff.mean()

# def gradient_loss(y_true, y_pred, mask):
#     """
#     Gradient loss to preserve sharp edges and textures
#     Loss برای حفظ لبه‌های تیز و بافت‌ها
#     """
#     # Sobel filters for gradient calculation
#     sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], 
#                            dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(y_true.device)
#     sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], 
#                            dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(y_true.device)
    
#     # Repeat for 3 channels
#     sobel_x = sobel_x.repeat(3, 1, 1, 1)
#     sobel_y = sobel_y.repeat(3, 1, 1, 1)
    
#     # Calculate gradients for true image
#     grad_true_x = F.conv2d(y_true, sobel_x, padding=1, groups=3)
#     grad_true_y = F.conv2d(y_true, sobel_y, padding=1, groups=3)
#     grad_true = torch.sqrt(grad_true_x ** 2 + grad_true_y ** 2 + 1e-8)
    
#     # Calculate gradients for predicted image
#     grad_pred_x = F.conv2d(y_pred, sobel_x, padding=1, groups=3)
#     grad_pred_y = F.conv2d(y_pred, sobel_y, padding=1, groups=3)
#     grad_pred = torch.sqrt(grad_pred_x ** 2 + grad_pred_y ** 2 + 1e-8)
    
#     # Focus on bottom half
#     bottom_region = (mask == 0).float()
    
#     # Calculate gradient difference
#     grad_diff = torch.abs(grad_true - grad_pred) * bottom_region
    
#     return grad_diff.mean()

# def facial_region_loss(y_true, y_pred, mask):
#     """
#     Focused loss on nose and lips regions (center-bottom area)
#     Loss متمرکز بر ناحیه بینی و لب‌ها
#     """
#     B, C, H, W = y_true.shape
    
#     # Create a weight map focusing on center-bottom (nose and lips region)
#     weight_map = torch.ones_like(mask)
    
#     # Define nose and lips region (center-bottom)
#     # Nose: roughly H/2 to 3H/4, center W/4 to 3W/4
#     # Lips: roughly 3H/4 to H, center W/4 to 3W/4
#     center_w_start = W // 4
#     center_w_end = 3 * W // 4
#     nose_h_start = H // 2
#     lips_h_end = H
    
#     # Increase weight for nose and lips region
#     weight_map[:, :, nose_h_start:lips_h_end, center_w_start:center_w_end] = 3.0
    
#     # Apply mask (only bottom half)
#     bottom_region = (mask == 0).float()
#     weight_map = weight_map * bottom_region
    
#     # Calculate weighted L1 loss
#     diff = torch.abs(y_true - y_pred) * weight_map
    
#     return diff.sum() / (weight_map.sum() + 1e-8)

# def generator_loss(disc_fake_logits, y_true, y_pred, disc_features_real, disc_features_fake, mask, mu, logvar):
#     # Clamp logvar to prevent exp overflow
#     logvar = torch.clamp(logvar, min=-10, max=10)
    
#     # Calculate each loss with NaN protection
#     adv_loss = -torch.mean(disc_fake_logits)
#     adv_loss = torch.nan_to_num(adv_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     # L1 reconstruction loss (high weight for stable inpainting)
#     bottom_region = (mask == 0).float()
#     l1_loss = torch.mean(torch.abs(y_true - y_pred) * bottom_region)
#     l1_loss = torch.nan_to_num(l1_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     rec_loss = bottom_half_loss(y_true, y_pred, mask)
#     rec_loss = torch.nan_to_num(rec_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     perc_loss = perceptual_loss(y_true, y_pred)
#     perc_loss = torch.nan_to_num(perc_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     sty_loss = style_loss(y_true, y_pred)
#     sty_loss = torch.nan_to_num(sty_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     fm_loss = feature_matching_loss(disc_features_real, disc_features_fake)
#     fm_loss = torch.nan_to_num(fm_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     id_loss = identity_loss(y_true, y_pred, mask)
#     id_loss = torch.nan_to_num(id_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     lm_loss = landmark_guided_loss(y_true, y_pred, mask)
#     lm_loss = torch.nan_to_num(lm_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     struct_loss = structural_ssim_loss(y_true, y_pred, mask)
#     struct_loss = torch.nan_to_num(struct_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     att_loss = attention_loss(y_true, y_pred, mask)
#     att_loss = torch.nan_to_num(att_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     # NEW: Edge-guided losses for better nose/lips reconstruction and sharpness
#     edg_loss = edge_loss(y_true, y_pred, mask, edge_detector_global)
#     edg_loss = torch.nan_to_num(edg_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     hf_loss = high_frequency_loss(y_true, y_pred, mask)
#     hf_loss = torch.nan_to_num(hf_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     grad_loss = gradient_loss(y_true, y_pred, mask)
#     grad_loss = torch.nan_to_num(grad_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     facial_loss = facial_region_loss(y_true, y_pred, mask)
#     facial_loss = torch.nan_to_num(facial_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     # KL loss with clipping
#     kl_loss = -0.5 * torch.mean(1 + logvar - mu**2 - torch.exp(logvar))
#     kl_loss = torch.nan_to_num(kl_loss, nan=0.0, posinf=0.0, neginf=0.0)
#     kl_loss = torch.clamp(kl_loss, min=0.0, max=10.0)  # Prevent extreme values
    
#     # Calculate total loss with edge-guided losses for better quality
#     total_loss = (lambda_adv * adv_loss + lambda_l1 * l1_loss + lambda_rec * rec_loss + lambda_perc * perc_loss +
#                   lambda_style * sty_loss + lambda_fm * fm_loss + lambda_identity * id_loss +
#                   lambda_landmark * lm_loss + lambda_struct * struct_loss + 
#                   lambda_attention * att_loss + lambda_vae * kl_loss +
#                   lambda_edge * edg_loss + lambda_hf * hf_loss + 
#                   lambda_gradient * grad_loss + lambda_facial * facial_loss)
    
#     # Final NaN check
#     total_loss = torch.nan_to_num(total_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     return total_loss
# def discriminator_loss(disc_real_logits, disc_fake_logits):
#     # Clamp logits to prevent extreme values
#     disc_real_logits = torch.clamp(disc_real_logits, min=-10, max=10)
#     disc_fake_logits = torch.clamp(disc_fake_logits, min=-10, max=10)
    
#     real_loss = torch.mean(F.relu(1.0 - disc_real_logits))
#     real_loss = torch.nan_to_num(real_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     fake_loss = torch.mean(F.relu(1.0 + disc_fake_logits))
#     fake_loss = torch.nan_to_num(fake_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     reg = lambda_d_reg * (torch.mean(disc_real_logits**2) + torch.mean(disc_fake_logits**2))
#     reg = torch.nan_to_num(reg, nan=0.0, posinf=0.0, neginf=0.0)
    
#     total_loss = real_loss + fake_loss + reg
#     total_loss = torch.nan_to_num(total_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     return total_loss
# # Evaluation functions
# def calculate_psnr_bottom_half(y_true, y_pred, mask):
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     bottom_mask = (mask == 0).float()
#     y_true_bottom = y_true * bottom_mask
#     y_pred_bottom = y_pred * bottom_mask
#     psnr_vals = []
#     for i in range(y_true.shape[0]):
#         true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         psnr_val = psnr(true_img, pred_img, data_range=1.0)
#         psnr_vals.append(psnr_val)
#     return torch.mean(torch.tensor(psnr_vals, device=device))
# def calculate_ssim_bottom_half(y_true, y_pred, mask):
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     bottom_mask = (mask == 0).float()
#     y_true_bottom = y_true * bottom_mask
#     y_pred_bottom = y_pred * bottom_mask
#     ssim_vals = []
#     for i in range(y_true.shape[0]):
#         true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         ssim_val = ssim(true_img, pred_img, channel_axis=2, data_range=1.0, win_size=7)
#         ssim_vals.append(ssim_val)
#     return torch.mean(torch.tensor(ssim_vals, device=device))
# def calculate_mse_bottom_half(y_true, y_pred, mask):
#     return bottom_half_loss(y_true, y_pred, mask)
# # Training function with graph-aware processing and balanced D/G updates
# def train_step(images, edge_generator, coarse_generator, fine_generator, discriminator, edge_optimizer, g_optimizer, d_optimizer, scaler, step_counter):
#     images = images.to(device)
#     masks = generate_adaptive_mask(images)
#     masked_images = images * masks
   
#     # Discriminator training (n_critic times more frequent)
#     d_optimizer.zero_grad()
    
#     # Train Discriminator multiple times
#     d_loss_accumulated = 0.0
#     for _ in range(n_critic):
#         with amp.autocast('cuda'):
#             # Generate fake images with multi-stage process: Edge -> Coarse -> Fine
#             with torch.no_grad():
#                 # Stage 1: Generate edge map
#                 edge_map = edge_generator(masked_images, masks)
                
#                 # Stage 2: Coarse inpainting with edge guidance
#                 coarse_output, mu, logvar = multi_resolution_inpainting(masked_images, masks, coarse_generator, edge_map)
#                 if coarse_output.shape != images.shape:
#                     coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
                
#                 # Stage 3: Fine inpainting with edge guidance
#                 fine_output = fine_generator(masked_images, coarse_output, masks, edge_map)
#                 if fine_output.shape != images.shape:
#                     fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
                
#                 combined_input = images * masks + fine_output * (1.0 - masks)
            
#             # Discriminator forward
#             disc_real_logits = discriminator(images, masks, return_features=False)
#             disc_fake_logits = discriminator(combined_input.detach(), masks, return_features=False)
#             d_loss = discriminator_loss(disc_real_logits, disc_fake_logits)
        
#         # Discriminator backward
#         scaler.scale(d_loss).backward()
#         d_loss_accumulated += d_loss.item()
    
#     # Update discriminator after n_critic iterations
#     scaler.unscale_(d_optimizer)
#     torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
#     scaler.step(d_optimizer)
    
#     d_loss_val = d_loss_accumulated / n_critic
   
#     # Generator training (once per n_critic discriminator updates)
#     g_loss_val = 0.0
#     edge_loss_val = 0.0
#     if step_counter % n_critic == 0:
#         # Train Edge Generator and Main Generators together
#         edge_optimizer.zero_grad()
#         g_optimizer.zero_grad()
        
#         with amp.autocast('cuda'):
#             # Stage 1: Generate edge map and compute edge reconstruction loss
#             edge_map = edge_generator(masked_images, masks)
            
#             # Extract true edges from original images
#             with torch.no_grad():
#                 true_edges = edge_detector_global(images * 0.5 + 0.5)
            
#             # Edge reconstruction loss (L1 loss on edge maps)
#             bottom_region = (masks == 0).float()
#             edge_recon_loss = torch.mean(torch.abs(true_edges - edge_map) * bottom_region)
            
#             # Stage 2 & 3: Generator forward pass with edge guidance
#             coarse_output, mu, logvar = multi_resolution_inpainting(masked_images, masks, coarse_generator, edge_map)
#             if coarse_output.shape != images.shape:
#                 coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
            
#             fine_output = fine_generator(masked_images, coarse_output, masks, edge_map)
#             if fine_output.shape != images.shape:
#                 fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
            
#             combined_input = images * masks + fine_output * (1.0 - masks)
#             disc_real_logits, disc_features_real = discriminator(images, masks, return_features=True)
#             disc_fake_logits, disc_features_fake = discriminator(combined_input, masks, return_features=True)
#             g_loss = generator_loss(disc_fake_logits, images, fine_output, disc_features_real, disc_features_fake, masks, mu, logvar)
        
#         # Combined backward pass (edge loss + generator loss)
#         # This avoids inplace operation errors
#         total_gen_loss = edge_recon_loss + g_loss
        
#         scaler.scale(total_gen_loss).backward()
        
#         # Unscale and clip gradients for both edge and main generators
#         scaler.unscale_(edge_optimizer)
#         scaler.unscale_(g_optimizer)
#         torch.nn.utils.clip_grad_norm_(edge_generator.parameters(), max_norm=1.0)
#         torch.nn.utils.clip_grad_norm_(list(coarse_generator.parameters()) + list(fine_generator.parameters()), max_norm=1.0)
        
#         # Update both optimizers
#         scaler.step(edge_optimizer)
#         scaler.step(g_optimizer)
        
#         edge_loss_val = edge_recon_loss.item()
#         g_loss_val = g_loss.item()
   
#     # Update scaler after both steps
#     scaler.update()
    
#     # Check for NaN in losses
#     if np.isnan(g_loss_val) or np.isinf(g_loss_val):
#         print(f"⚠️ WARNING: G_Loss is {g_loss_val}, skipping this batch")
#         g_loss_val = 0.0
        
#     if np.isnan(d_loss_val) or np.isinf(d_loss_val):
#         print(f"⚠️ WARNING: D_Loss is {d_loss_val}, skipping this batch")
#         d_loss_val = 0.0
    
#     if np.isnan(edge_loss_val) or np.isinf(edge_loss_val):
#         print(f"⚠️ WARNING: Edge_Loss is {edge_loss_val}, skipping this batch")
#         edge_loss_val = 0.0
   
#     return g_loss_val, d_loss_val, edge_loss_val
# # Add multi_resolution_inpainting function
# def multi_resolution_inpainting(image, mask, model, edge_map=None, levels=3):
#     outputs = []
#     mus = []
#     logvars = []
#     for level in range(levels):
#         scaled_image = F.interpolate(image, scale_factor=1 / (2 ** level), mode='bilinear')
#         scaled_mask = F.interpolate(mask, scale_factor=1 / (2 ** level), mode='nearest')
        
#         # Scale edge map if provided
#         if edge_map is not None:
#             scaled_edge_map = F.interpolate(edge_map, scale_factor=1 / (2 ** level), mode='bilinear')
#             out, mu, logvar = model(scaled_image, scaled_mask, scaled_edge_map)
#         else:
#             out, mu, logvar = model(scaled_image, scaled_mask)  # Call full forward
        
#         out_up = F.interpolate(out, size=image.shape[-2:], mode='bilinear')
#         outputs.append(out_up)
#         mus.append(mu)
#         logvars.append(logvar)
    
#     final_out = torch.mean(torch.stack(outputs), dim=0)
#     final_mu = torch.mean(torch.stack(mus), dim=0)
#     final_logvar = torch.mean(torch.stack(logvars), dim=0)
#     return final_out, final_mu, final_logvar

# # Add gradient checking function
# def check_model_gradients(model, model_name="Model"):
#     """Check if model has NaN or Inf gradients"""
#     has_nan = False
#     has_inf = False
#     max_grad = 0.0
    
#     for name, param in model.named_parameters():
#         if param.grad is not None:
#             if torch.isnan(param.grad).any():
#                 has_nan = True
#                 print(f"⚠️ NaN gradient in {model_name}.{name}")
#             if torch.isinf(param.grad).any():
#                 has_inf = True
#                 print(f"⚠️ Inf gradient in {model_name}.{name}")
#             max_grad = max(max_grad, param.grad.abs().max().item())
    
#     return has_nan, has_inf, max_grad

# # Add simple retrain function
# def retrain(model, pruned_graph, optimizer):
#     optimizer.zero_grad()
#     # Dummy forward (adjust to use graph if needed)
#     dummy_input = torch.randn(1, 3, 64, 64).to(device)
#     dummy_mask = torch.randn(1, 3, 64, 64).to(device)
#     output, _, _ = model(dummy_input, dummy_mask)
#     loss = torch.mean(output)
#     loss.backward()
#     optimizer.step()
#     return loss.item()

# # Integrate into training loop
# # In train_step function, after coarse_output:
# # Add pruning and retraining every 10 epochs
# # Load dataset

# # celeba_dir = '/kaggle/input/celeba-resized-6464/img_align_celeba/CelebA_Image_Cropped_64'
# celeba_dir = '/kaggle/input/celeba-6464/selected_celeba'

# print(f"Loading dataset from {celeba_dir}...")
# try:
#     print(f"Dataset directory contents: {os.listdir(celeba_dir)[:5]}")
# except Exception as e:
#     print(f"Error accessing dataset directory: {e}")
#     raise
# # Split dataset
# train_files, val_files, test_files = split_dataset(celeba_dir)
# # Create datasets
# train_loader, val_loader, test_loader = create_split_datasets(celeba_dir, train_files, val_files, test_files, image_size, batch_size)
# # Build models (now with Edge Generator for multi-stage inpainting)
# print("Building models...")
# print("🔷 Multi-Stage Architecture: Edge → Coarse → Fine")
# edge_generator = EdgeGenerator().to(device)
# coarse_generator = CoarseGenerator().to(device)
# fine_generator = FineGenerator().to(device)
# discriminator = Discriminator().to(device)
# print(f"Edge Generator: {sum(p.numel() for p in edge_generator.parameters()):,} parameters")
# print(f"Coarse Generator: {sum(p.numel() for p in coarse_generator.parameters()):,} parameters")
# print(f"Fine Generator: {sum(p.numel() for p in fine_generator.parameters()):,} parameters")
# print(f"Discriminator: {sum(p.numel() for p in discriminator.parameters()):,} parameters")
# # Test model dimensions
# print("\nTesting model dimensions...")
# test_input = torch.randn(1, 3, 64, 64).to(device)
# test_mask = torch.randn(1, 3, 64, 64).to(device)
# with torch.no_grad():
#     coarse_output, mu, logvar = coarse_generator(test_input, test_mask)
#     print(f"Coarse output shape: {coarse_output.shape}")
#     fine_output = fine_generator(test_input, coarse_output, test_mask)
#     print(f"Fine output shape: {fine_output.shape}")
#     print(f"Expected shape: {test_input.shape}")
   
#     # Test discriminator
#     disc_logits, disc_features = discriminator(test_input, test_mask, return_features=True)
#     print(f"Discriminator logits shape: {disc_logits.shape}")
#     print(f"Discriminator features length: {len(disc_features)}")
   
#     disc_logits_only = discriminator(test_input, test_mask, return_features=False)
#     print(f"Discriminator logits only shape: {disc_logits_only.shape}")
   
#     print("✓ Model dimensions are correct!")
# # Enhanced Optimizers with Learning Rate Scheduling and beta1=0.5 for GAN stability
# # NEW: Added Edge Generator optimizer
# edge_optimizer = torch.optim.AdamW(
#     edge_generator.parameters(),
#     lr=learning_rate_g, betas=(0.5, 0.999), weight_decay=1e-4
# )
# g_optimizer = torch.optim.AdamW(
#     list(coarse_generator.parameters()) + list(fine_generator.parameters()),
#     lr=learning_rate_g, betas=(0.5, 0.999), weight_decay=1e-4
# )
# d_optimizer = torch.optim.AdamW(
#     discriminator.parameters(),
#     lr=learning_rate_d, betas=(0.5, 0.999), weight_decay=1e-4
# )
# print(f"Optimizer settings: G_LR={learning_rate_g}, D_LR={learning_rate_d}, beta1=0.5, n_critic={n_critic}")
# print(f"✅ Edge Generator optimizer added for multi-stage training")
# # Learning rate schedulers - ReduceLROnPlateau for adaptive learning based on validation loss
# # Scheduler های Learning Rate - کاهش تطبیقی بر اساس validation loss
# edge_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     edge_optimizer, mode='min', factor=0.5, patience=3, verbose=True, min_lr=learning_rate_g * 0.01
# )
# g_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     g_optimizer, mode='min', factor=0.5, patience=3, verbose=True, min_lr=learning_rate_g * 0.01
# )
# d_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     d_optimizer, mode='min', factor=0.5, patience=3, verbose=True, min_lr=learning_rate_d * 0.01
# )
# scaler = amp.GradScaler('cuda')
# # Model health check before training
# print("\n" + "="*60)
# print("🔍 Model Health Check")
# print("="*60)
# test_batch = next(iter(train_loader)).to(device)
# test_masks = generate_adaptive_mask(test_batch)
# test_masked = test_batch * test_masks

# print("Testing forward pass...")
# with torch.no_grad():
#     try:
#         coarse_out, mu, logvar = coarse_generator(test_masked, test_masks)
#         print(f"✓ Coarse Generator: output shape {coarse_out.shape}")
#         print(f"  mu range: [{mu.min().item():.4f}, {mu.max().item():.4f}]")
#         print(f"  logvar range: [{logvar.min().item():.4f}, {logvar.max().item():.4f}]")
        
#         fine_out = fine_generator(test_masked, coarse_out, test_masks)
#         print(f"✓ Fine Generator: output shape {fine_out.shape}")
#         print(f"  output range: [{fine_out.min().item():.4f}, {fine_out.max().item():.4f}]")
        
#         disc_out = discriminator(test_batch, test_masks)
#         print(f"✓ Discriminator: logits shape {disc_out.shape}")
#         print(f"  logits range: [{disc_out.min().item():.4f}, {disc_out.max().item():.4f}]")
        
#         print("✅ All models are healthy!")
#     except Exception as e:
#         print(f"❌ Model health check failed: {e}")
#         raise
# print("="*60 + "\n")

# # ============================================================================
# # CHECKPOINT LOADING & TRAINING SETUP
# # بارگذاری Checkpoint و راه‌اندازی آموزش
# # ============================================================================

# print("\n" + "="*70)
# print("🚀 STARTING TRAINING SETUP")
# print("="*70)

# # Adjust paths if loading from Kaggle input dataset
# if USE_KAGGLE_INPUT:
#     checkpoint_load_dir = os.path.join(KAGGLE_INPUT_DATASET, 'checkpoints')
#     models_load_dir = os.path.join(KAGGLE_INPUT_DATASET, 'models')
#     print(f"📥 Loading from Kaggle dataset: {KAGGLE_INPUT_DATASET}")
# else:
#     checkpoint_load_dir = CHECKPOINT_DIR
#     models_load_dir = MODELS_DIR
#     print(f"📥 Loading from working directory")

# # Create checkpoint save directory (always in /kaggle/working)
# os.makedirs(CHECKPOINT_DIR, exist_ok=True)
# os.makedirs(MODELS_DIR, exist_ok=True)
# print(f"💾 Checkpoints will be saved to: {CHECKPOINT_DIR}")
# print(f"💾 Models will be saved to: {MODELS_DIR}")

# # Initialize training variables
# start_epoch = 0
# best_val_loss = float('inf')
# train_g_losses = []
# train_d_losses = []
# val_g_losses = []
# val_d_losses = []

# # Try to load checkpoint/models if RESUME_TRAINING is True
# checkpoint_loaded = False
# models_loaded = False

# if RESUME_TRAINING:
#     # ========================================================================
#     # Method 1: Try to load from MODEL FILES (state_dict only)
#     # روش 1: بارگذاری از فایل‌های MODEL (فقط state_dict)
#     # ========================================================================
    
#     if LOAD_FROM_MODELS and not checkpoint_loaded:
#         print(f"\n🔍 Method 1: Searching for model files...")
#         print(f"📂 Models directory: {models_load_dir}")
        
#         # Debug: Show what files actually exist
#         if os.path.exists(models_load_dir):
#             print(f"\n📄 Files found in directory:")
#             actual_files = os.listdir(models_load_dir)
#             if actual_files:
#                 for f in sorted(actual_files):
#                     if f.endswith('.pth'):
#                         file_path = os.path.join(models_load_dir, f)
#                         size_mb = os.path.getsize(file_path) / (1024**2)
#                         print(f"   • {f} ({size_mb:.2f} MB)")
#             else:
#                 print(f"   (empty)")
#         else:
#             print(f"   ❌ Directory does not exist!")
        
#         # Determine which models to load
#         suffix = "_best.pth" if USE_BEST_MODELS else "_bottom_half.pth"
        
#         model_files = {
#             'coarse': os.path.join(models_load_dir, f'coarse_generator{suffix}'),
#             'fine': os.path.join(models_load_dir, f'fine_generator{suffix}'),
#             'disc': os.path.join(models_load_dir, f'discriminator{suffix}')
#         }
        
#         print(f"\n🔍 Looking for (USE_BEST_MODELS={USE_BEST_MODELS}):")
#         for name, path in model_files.items():
#             exists = "✅" if os.path.exists(path) else "❌"
#             print(f"   {exists} {os.path.basename(path)}")
        
#         # Check if all model files exist
#         all_exist = all(os.path.exists(path) for path in model_files.values())
        
#         if all_exist:
#             try:
#                 print(f"📂 Found model files in: {models_load_dir}")
#                 print(f"   • coarse_generator{suffix}")
#                 print(f"   • fine_generator{suffix}")
#                 print(f"   • discriminator{suffix}")
#                 print(f"⏳ Loading model weights...")
                
#                 # Load model weights
#                 # Load model weights tolerantly in case of architecture drift
#                 coarse_sd = torch.load(model_files['coarse'], map_location=device)
#                 fine_sd = torch.load(model_files['fine'], map_location=device)
#                 disc_sd = torch.load(model_files['disc'], map_location=device)

#                 missing, unexpected = coarse_generator.load_state_dict(coarse_sd, strict=False)
#                 if missing or unexpected:
#                     print("[coarse_generator] non-strict load:")
#                     if missing:
#                         print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                     if unexpected:
#                         print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                 missing, unexpected = fine_generator.load_state_dict(fine_sd, strict=False)
#                 if missing or unexpected:
#                     print("[fine_generator] non-strict load:")
#                     if missing:
#                         print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                     if unexpected:
#                         print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                 missing, unexpected = discriminator.load_state_dict(disc_sd, strict=False)
#                 if missing or unexpected:
#                     print("[discriminator] non-strict load:")
#                     if missing:
#                         print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                     if unexpected:
#                         print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
                
#                 models_loaded = True
                
#                 print("="*70)
#                 print("✅ MODEL WEIGHTS LOADED SUCCESSFULLY!")
#                 print("="*70)
#                 print(f"📊 Loading Information:")
#                 print(f"   • Model weights: ✅ Loaded")
#                 print(f"   • Optimizer states: ⚠️  Will be initialized (not in model files)")
#                 print(f"   • Scheduler states: ⚠️  Will be initialized (not in model files)")
#                 print(f"   • Training history: ⚠️  Starting fresh (not in model files)")
#                 print(f"   • Starting epoch: 1 (training continues with loaded weights)")
#                 print(f"\n⚠️  NOTE: Model files only contain weights, not optimizer/scheduler.")
#                 print(f"   Training will continue with these weights but fresh optimizer.")
#                 print("="*70)
                
#             except Exception as e:
#                 print(f"⚠️  Failed to load model files")
#                 print(f"   Error: {str(e)}")
#                 models_loaded = False
#         else:
#             print(f"⚠️  Not all model files found in {models_load_dir}")
#             for name, path in model_files.items():
#                 exists = "✓" if os.path.exists(path) else "✗"
#                 print(f"   {exists} {os.path.basename(path)}")
    
#     # ========================================================================
#     # Method 2: Try to load from CHECKPOINT FILES (full checkpoint)
#     # روش 2: بارگذاری از فایل‌های CHECKPOINT (checkpoint کامل)
#     # ========================================================================
    
#     if not checkpoint_loaded and not models_loaded:
#         print(f"\n🔍 Method 2: Searching for checkpoint files...")
        
#         # Try different checkpoint sources
#         checkpoint_paths_to_try = [
#             os.path.join(checkpoint_load_dir, 'latest_checkpoint.pth'),
#             os.path.join(checkpoint_load_dir, f'checkpoint_epoch_{num_epochs}.pth'),
#             os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth'),  # Fallback to working dir
#         ]
        
#         for checkpoint_path in checkpoint_paths_to_try:
#             if os.path.exists(checkpoint_path):
#                 try:
#                     print(f"📂 Found checkpoint: {checkpoint_path}")
#                     print(f"⏳ Loading checkpoint...")
                    
#                     checkpoint = torch.load(checkpoint_path, map_location=device)
                    
#                     # Load model states tolerantly (including edge_generator if available)
#                     if 'edge_generator' in checkpoint:
#                         missing, unexpected = edge_generator.load_state_dict(checkpoint['edge_generator'], strict=False)
#                         if missing or unexpected:
#                             print("[edge_generator ckpt] non-strict load:")
#                             if missing:
#                                 print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                             if unexpected:
#                                 print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
#                     else:
#                         print("[edge_generator] not found in checkpoint - using random initialization")
                    
#                     missing, unexpected = coarse_generator.load_state_dict(checkpoint['coarse_generator'], strict=False)
#                     if missing or unexpected:
#                         print("[coarse_generator ckpt] non-strict load:")
#                         if missing:
#                             print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                         if unexpected:
#                             print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                     missing, unexpected = fine_generator.load_state_dict(checkpoint['fine_generator'], strict=False)
#                     if missing or unexpected:
#                         print("[fine_generator ckpt] non-strict load:")
#                         if missing:
#                             print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                         if unexpected:
#                             print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                     missing, unexpected = discriminator.load_state_dict(checkpoint['discriminator'], strict=False)
#                     if missing or unexpected:
#                         print("[discriminator ckpt] non-strict load:")
#                         if missing:
#                             print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                         if unexpected:
#                             print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
                    
#                     # Load optimizer states
#                     g_optimizer.load_state_dict(checkpoint['g_optimizer'])
#                     d_optimizer.load_state_dict(checkpoint['d_optimizer'])
                    
#                     # Load scheduler states
#                     g_scheduler.load_state_dict(checkpoint['g_scheduler'])
#                     d_scheduler.load_state_dict(checkpoint['d_scheduler'])
                    
#                     # Load training progress
#                     start_epoch = checkpoint['epoch'] + 1
#                     best_val_loss = checkpoint.get('best_val_loss', float('inf'))
#                     train_g_losses = checkpoint.get('train_g_losses', [])
#                     train_d_losses = checkpoint.get('train_d_losses', [])
#                     val_g_losses = checkpoint.get('val_g_losses', [])
#                     val_d_losses = checkpoint.get('val_d_losses', [])
                    
#                     checkpoint_loaded = True
                    
#                     print("="*70)
#                     print("✅ FULL CHECKPOINT LOADED SUCCESSFULLY!")
#                     print("="*70)
#                     print(f"📊 Resume Information:")
#                     print(f"   • Model weights: ✅ Loaded")
#                     print(f"   • Optimizer states: ✅ Loaded")
#                     print(f"   • Scheduler states: ✅ Loaded")
#                     print(f"   • Starting from epoch: {start_epoch + 1}")
#                     print(f"   • Best validation loss: {best_val_loss:.6f}")
#                     print(f"   • Training history: {len(train_g_losses)} epochs")
#                     print(f"   • Remaining epochs: {num_epochs - start_epoch}")
#                     print("="*70)
                    
#                     break  # Successfully loaded, exit loop
                    
#                 except Exception as e:
#                     print(f"⚠️  Failed to load checkpoint from {checkpoint_path}")
#                     print(f"   Error: {str(e)}")
#                     print(f"   Trying next checkpoint source...")
#                     continue

# # ========================================================================
# # If nothing loaded, start fresh
# # اگر چیزی بارگذاری نشد، از اول شروع کن
# # ========================================================================

# if not checkpoint_loaded and not models_loaded:
#     if RESUME_TRAINING:
#         print("\n" + "="*70)
#         print("⚠️  NO CHECKPOINT OR MODELS FOUND - Starting from scratch")
#         print("="*70)
#         print("📝 Locations searched:")
#         print(f"\n   Models folder: {models_load_dir}")
#         if os.path.exists(models_load_dir):
#             files = os.listdir(models_load_dir)
#             if files:
#                 print(f"   Found {len(files)} file(s):")
#                 for f in files[:5]:  # Show first 5
#                     print(f"      • {f}")
#             else:
#                 print(f"      (empty)")
#         else:
#             print(f"      (not found)")
        
#         print(f"\n   Checkpoints folder: {checkpoint_load_dir}")
#         if os.path.exists(checkpoint_load_dir):
#             files = [f for f in os.listdir(checkpoint_load_dir) if f.endswith('.pth')]
#             if files:
#                 print(f"   Found {len(files)} checkpoint(s):")
#                 for f in files[:5]:
#                     print(f"      • {f}")
#             else:
#                 print(f"      (empty)")
#         else:
#             print(f"      (not found)")
        
#         print("\n💡 To resume training in next run:")
#         print("   1. Make sure model/checkpoint files exist")
#         print("   2. Set LOAD_FROM_MODELS = True for model files")
#         print("   3. If using Kaggle dataset, set USE_KAGGLE_INPUT = True")
#         print("   4. Update MODELS_DIR or KAGGLE_INPUT_DATASET path")
#         print("="*70)
#     else:
#         print("\n" + "="*70)
#         print("🆕 STARTING FRESH TRAINING")
#         print("="*70)
#         print("   RESUME_TRAINING is set to False")
#         print("   Training will start from epoch 1")
#         print("="*70)

# print(f"\n🎯 Training will run from epoch {start_epoch + 1} to {num_epochs}")
# print(f"💾 Checkpoints will be saved every {CHECKPOINT_INTERVAL} epoch(s)")
# print("="*70 + "\n")

# for epoch in range(start_epoch, num_epochs):
#     print(f"Epoch {epoch+1}/{num_epochs}")
#     edge_generator.train()
#     coarse_generator.train()
#     fine_generator.train()
#     discriminator.train()
#     epoch_g_loss = epoch_d_loss = epoch_edge_loss = num_batches = 0
   
#     for step, images in enumerate(train_loader):
#         # Pass step counter for balanced D/G updates (NEW: added edge_generator and edge_optimizer)
#         g_loss_batch, d_loss_batch, edge_loss_batch = train_step(images, edge_generator, coarse_generator, fine_generator, discriminator, edge_optimizer, g_optimizer, d_optimizer, scaler, step)
#         epoch_g_loss += g_loss_batch
#         epoch_edge_loss += edge_loss_batch
#         epoch_d_loss += d_loss_batch
#         num_batches += 1
        
#         # Enhanced monitoring
#         if step % 50 == 0:
#             current_g_lr = g_optimizer.param_groups[0]['lr']
#             current_d_lr = d_optimizer.param_groups[0]['lr']
#             current_edge_lr = edge_optimizer.param_groups[0]['lr']
#             g_or_d = "E+G+D" if step % n_critic == 0 else "D only"
#             print(f" Step {step:3d} [{g_or_d}]: Edge_Loss: {edge_loss_batch:.4f}, G_Loss: {g_loss_batch:.4f}, D_Loss: {d_loss_batch:.4f} | LR: E={current_edge_lr:.6f}, G={current_g_lr:.6f}, D={current_d_lr:.6f}")
            
#         # Emergency stop if losses explode
#         if np.isnan(g_loss_batch) and np.isnan(d_loss_batch):
#             print(f"⚠️ CRITICAL: Both losses are NaN at step {step}. Stopping epoch early.")
#             break
   
#     avg_g_loss = epoch_g_loss / num_batches if num_batches > 0 else 0
#     avg_d_loss = epoch_d_loss / num_batches if num_batches > 0 else 0
#     avg_edge_loss = epoch_edge_loss / num_batches if num_batches > 0 else 0
#     train_g_losses.append(avg_g_loss)
#     train_d_losses.append(avg_d_loss)
#     print(f"Epoch {epoch+1} Training - Edge_Loss: {avg_edge_loss:.4f}, G_Loss: {avg_g_loss:.4f}, D_Loss: {avg_d_loss:.4f}")
   
#     # Enable MTCNN for validation
#     if 'MTCNN' in globals():
#         USE_MTCNN = True
   
#     if epoch % 10 == 0:
#         # Get sample batch for graph creation
#         sample_images = next(iter(train_loader)).to(device)
#         sample_masks = generate_adaptive_mask(sample_images)
#         sample_masked = sample_images * sample_masks
        
#         # Get features from preprocessor
#         features = coarse_generator.preprocessor(sample_masked)
        
#         # Build graph
#         sample_graph = coarse_generator.preprocessor.build_graph(features)
        
#         pruned_graph = prune_graph(coarse_generator, sample_graph)
        
#         # Retrain
#         for _ in range(5):
#             retrain_loss = retrain(coarse_generator, pruned_graph, g_optimizer)
   
#     edge_generator.eval()
#     coarse_generator.eval()
#     fine_generator.eval()
#     discriminator.eval()
#     val_g_loss = val_d_loss = val_batches = 0
#     with torch.no_grad():
#         for images in val_loader:
#             images = images.to(device)
#             masks = generate_adaptive_mask(images)
#             masked_images = images * masks
            
#             # Multi-stage: Edge -> Coarse -> Fine
#             edge_map = edge_generator(masked_images, masks)
#             coarse_output, mu, logvar = coarse_generator(masked_images, masks, edge_map)
#             # Debug: Check shapes
#             if coarse_output.shape != images.shape:
#                 coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
           
#             fine_output = fine_generator(masked_images, coarse_output, masks, edge_map)
#             # Debug: Check shapes
#             if fine_output.shape != images.shape:
#                 fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
           
#             combined_input = images * masks + fine_output * (1.0 - masks)
           
#             disc_real_logits, disc_features_real = discriminator(images, masks, return_features=True)
#             disc_fake_logits, disc_features_fake = discriminator(combined_input, masks, return_features=True)
           
#             g_loss = generator_loss(disc_fake_logits, images, fine_output, disc_features_real, disc_features_fake, masks, mu, logvar)
#             d_loss = discriminator_loss(disc_real_logits, disc_fake_logits)
           
#             val_g_loss += g_loss.item()
#             val_d_loss += d_loss.item()
#             val_batches += 1
   
#     avg_val_g_loss = val_g_loss / val_batches if val_batches > 0 else 0
#     avg_val_d_loss = val_d_loss / val_batches if val_batches > 0 else 0
#     val_g_losses.append(avg_val_g_loss)
#     val_d_losses.append(avg_val_d_loss)
#     print(f"Epoch {epoch+1} Validation - G_Loss: {avg_val_g_loss:.4f}, D_Loss: {avg_val_d_loss:.4f}")
   
#     # Update learning rates based on validation loss (ReduceLROnPlateau)
#     # به‌روزرسانی learning rate بر اساس validation loss
#     edge_scheduler.step(avg_val_g_loss)
#     g_scheduler.step(avg_val_g_loss)
#     d_scheduler.step(avg_val_d_loss)
#     current_g_lr = g_optimizer.param_groups[0]['lr']
#     current_d_lr = d_optimizer.param_groups[0]['lr']
#     current_edge_lr = edge_optimizer.param_groups[0]['lr']
#     print(f"📊 Current LR - Edge: {current_edge_lr:.6f}, G: {current_g_lr:.6f}, D: {current_d_lr:.6f}")
   
#     if avg_val_g_loss < best_val_loss:
#         best_val_loss = avg_val_g_loss
#         print(f" New best validation loss! Saving best models...")
#         os.makedirs('/kaggle/working/models', exist_ok=True)
#         torch.save(edge_generator.state_dict(), "/kaggle/working/models/edge_generator_best.pth")
#         torch.save(coarse_generator.state_dict(), "/kaggle/working/models/coarse_generator_best.pth")
#         torch.save(fine_generator.state_dict(), "/kaggle/working/models/fine_generator_best.pth")
#         torch.save(discriminator.state_dict(), "/kaggle/working/models/discriminator_best.pth")
   
#     # Save checkpoint only when training is completely finished
#     if (epoch + 1) == num_epochs:
#         print(f"\n{'='*70}")
#         print(f"💾 SAVING FINAL CHECKPOINT - Epoch {epoch+1}/{num_epochs}")
#         print(f"{'='*70}")
        
#         checkpoint_file = os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch+1}.pth')
#         latest_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth')
        
#         checkpoint = {
#             'epoch': epoch,
#             'edge_generator': edge_generator.state_dict(),
#             'coarse_generator': coarse_generator.state_dict(),
#             'fine_generator': fine_generator.state_dict(),
#             'discriminator': discriminator.state_dict(),
#             'edge_optimizer': edge_optimizer.state_dict(),
#             'g_optimizer': g_optimizer.state_dict(),
#             'd_optimizer': d_optimizer.state_dict(),
#             'edge_scheduler': edge_scheduler.state_dict(),
#             'g_scheduler': g_scheduler.state_dict(),
#             'd_scheduler': d_scheduler.state_dict(),
#             'best_val_loss': best_val_loss,
#             'train_g_losses': train_g_losses,
#             'train_d_losses': train_d_losses,
#             'val_g_losses': val_g_losses,
#             'val_d_losses': val_d_losses,
#         }
        
#         # Save numbered checkpoint
#         torch.save(checkpoint, checkpoint_file)
#         checkpoint_size = os.path.getsize(checkpoint_file) / (1024**2)  # MB
#         print(f"✅ Saved: {checkpoint_file}")
#         print(f"   Size: {checkpoint_size:.2f} MB")
        
#         # Also save as latest checkpoint
#         torch.save(checkpoint, latest_checkpoint_path)
#         print(f"✅ Saved: {latest_checkpoint_path}")
        
#         print(f"\n📊 Checkpoint Info:")
#         print(f"   • Completed epochs: {epoch + 1}")
#         print(f"   • Best val loss: {best_val_loss:.6f}")
#         print(f"   • Current G loss: {avg_g_loss:.6f}")
#         print(f"   • Current D loss: {avg_d_loss:.6f}")
        
#         if (epoch + 1) < num_epochs:
#             print(f"\n🔄 To resume from this checkpoint in next run:")
#             print(f"   1. Download '{CHECKPOINT_DIR}' folder from Kaggle Output")
#             print(f"   2. Upload as Kaggle Dataset (or keep in working directory)")
#             print(f"   3. In code, set: RESUME_TRAINING = True")
#             print(f"   4. If using dataset, set: USE_KAGGLE_INPUT = True")
#             print(f"   5. Update: KAGGLE_INPUT_DATASET = '/kaggle/input/your-dataset-name'")
#         else:
#             print(f"\n🎉 TRAINING COMPLETED!")
#             print(f"   All {num_epochs} epochs finished successfully!")
        
#         print(f"{'='*70}\n")
   
#     # Plot training progress every 5 epochs
#     if (epoch + 1) % 5 == 0:
#         plt.figure(figsize=(15, 5))
       
#         plt.subplot(1, 3, 1)
#         plt.plot(range(1, len(train_g_losses) + 1), train_g_losses, label='Generator Loss', color='blue')
#         plt.plot(range(1, len(val_g_losses) + 1), val_g_losses, label='Val Generator Loss', color='lightblue')
#         plt.title('Generator Loss Progress')
#         plt.xlabel('Epoch')
#         plt.ylabel('Loss')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
       
#         plt.subplot(1, 3, 2)
#         plt.plot(range(1, len(train_d_losses) + 1), train_d_losses, label='Discriminator Loss', color='red')
#         plt.plot(range(1, len(val_d_losses) + 1), val_d_losses, label='Val Discriminator Loss', color='lightcoral')
#         plt.title('Discriminator Loss Progress')
#         plt.xlabel('Epoch')
#         plt.ylabel('Loss')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
       
#         plt.subplot(1, 3, 3)
#         plt.plot(range(1, len(train_g_losses) + 1), train_g_losses, label='G Loss', color='blue')
#         plt.plot(range(1, len(train_d_losses) + 1), train_d_losses, label='D Loss', color='red')
#         plt.title('Training Loss Comparison')
#         plt.xlabel('Epoch')
#         plt.ylabel('Loss')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
       
#         plt.tight_layout()
#         plt.savefig(f'/kaggle/working/training_progress_epoch_{epoch+1}.png', dpi=150, bbox_inches='tight')
#         plt.show()
# # Save final models
# print("Saving final models...")
# os.makedirs('/kaggle/working/models', exist_ok=True)
# torch.save(edge_generator.state_dict(), "/kaggle/working/models/edge_generator_bottom_half.pth")
# torch.save(coarse_generator.state_dict(), "/kaggle/working/models/coarse_generator_bottom_half.pth")
# torch.save(fine_generator.state_dict(), "/kaggle/working/models/fine_generator_bottom_half.pth")
# torch.save(discriminator.state_dict(), "/kaggle/working/models/discriminator_bottom_half.pth")
# # Enable MTCNN for evaluation
# if 'MTCNN' in globals():
#     USE_MTCNN = True
# # Evaluate on test set
# print("Starting evaluation on test set...")
# os.makedirs('/kaggle/working/results', exist_ok=True)
# edge_generator.eval()
# coarse_generator.eval()
# fine_generator.eval()
# discriminator.eval()
# psnr_values, ssim_values, mse_values, identity_values, landmark_values = [], [], [], [], []
# num_samples = 10 # Increased for better evaluation
# plt.figure(figsize=(20, 10))
# with torch.no_grad():
#     for i, images in enumerate(test_loader):
#         if i >= num_samples:
#             break
#         images = images.to(device)
#         masks = generate_adaptive_mask(images)
#         masked_images = images * masks
        
#         # Multi-stage: Edge -> Coarse -> Fine
#         edge_map = edge_generator(masked_images, masks)
#         coarse_output, mu, logvar = coarse_generator(masked_images, masks, edge_map)
#         # Debug: Check shapes
#         if coarse_output.shape != images.shape:
#             coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
       
#         fine_output = fine_generator(masked_images, coarse_output, masks, edge_map)
#         # Debug: Check shapes
#         if fine_output.shape != images.shape:
#             fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
       
#         reconstructed = images * masks + fine_output * (1.0 - masks)
       
#         psnr_val = calculate_psnr_bottom_half(images, reconstructed, masks).item()
#         ssim_val = calculate_ssim_bottom_half(images, reconstructed, masks).item()
#         mse_val = calculate_mse_bottom_half(images, reconstructed, masks).item()
#         identity_val = identity_loss(images, reconstructed, masks).item()
#         landmark_val = landmark_guided_loss(images, reconstructed, masks).item()
       
#         psnr_values.append(psnr_val)
#         ssim_values.append(ssim_val)
#         mse_values.append(mse_val)
#         identity_values.append(identity_val)
#         landmark_values.append(landmark_val)
       
#         plt.subplot(4, num_samples, i + 1)
#         plt.title("Original Image")
#         plt.imshow(images[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#         plt.axis("off")
       
#         plt.subplot(4, num_samples, num_samples + i + 1)
#         plt.title("Top Half (Input)")
#         plt.imshow(masked_images[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#         plt.axis("off")
       
#         plt.subplot(4, num_samples, 2*num_samples + i + 1)
#         plt.title("Reconstructed Bottom Half")
#         plt.imshow(reconstructed[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#         plt.axis("off")
       
#         diff = torch.abs(images - reconstructed) * (1 - masks)
#         plt.subplot(4, num_samples, 3*num_samples + i + 1)
#         plt.title("Difference Map")
#         plt.imshow(diff[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5, cmap='hot')
#         plt.axis("off")
       
#         print(f"Sample {i+1}: PSNR: {psnr_val:.4f}, SSIM: {ssim_val:.4f}, MSE: {mse_val:.4f}, Identity: {identity_val:.4f}, Landmark: {landmark_val:.4f}")
# plt.tight_layout()
# plt.savefig('/kaggle/working/results/bottom_half_reconstruction_results.png', dpi=150, bbox_inches='tight')
# plt.show()
# # Plot metrics
# plt.figure(figsize=(20, 10))
# plt.subplot(2, 3, 1)
# plt.plot(range(1, num_samples + 1), psnr_values, marker='o', linewidth=2, markersize=8)
# plt.title("PSNR (Bottom Half)", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 2)
# plt.plot(range(1, num_samples + 1), ssim_values, marker='s', linewidth=2, markersize=8, color='orange')
# plt.title("SSIM (Bottom Half)", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 3)
# plt.plot(range(1, num_samples + 1), mse_values, marker='^', linewidth=2, markersize=8, color='green')
# plt.title("MSE (Bottom Half)", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 4)
# plt.plot(range(1, num_samples + 1), identity_values, marker='d', linewidth=2, markersize=8, color='purple')
# plt.title("Identity Loss", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 5)
# plt.plot(range(1, num_samples + 1), landmark_values, marker='*', linewidth=2, markersize=8, color='brown')
# plt.title("Landmark Loss", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.savefig('/kaggle/working/results/bottom_half_metrics.png', dpi=150, bbox_inches='tight')
# plt.show()
# # Print and save evaluation results
# print("\n" + "="*60)
# print("Enhanced Evaluation Metrics (Bottom Half) - Test Set:")
# print("="*60)
# print(f"Average PSNR: {np.mean(psnr_values):.4f} dB (±{np.std(psnr_values):.4f})")
# print(f"Average SSIM: {np.mean(ssim_values):.4f} (±{np.std(ssim_values):.4f})")
# print(f"Average MSE: {np.mean(mse_values):.4f} (±{np.std(mse_values):.4f})")
# print(f"Average Identity Loss: {np.mean(identity_values):.4f} (±{np.std(identity_values):.4f})")
# print(f"Average Landmark Loss: {np.mean(landmark_values):.4f} (±{np.std(landmark_values):.4f})")
# print("="*60)
# print(f"Best PSNR: {np.max(psnr_values):.4f} dB")
# print(f"Best SSIM: {np.max(ssim_values):.4f}")
# print(f"Worst PSNR: {np.min(psnr_values):.4f} dB")
# print(f"Worst SSIM: {np.min(ssim_values):.4f}")
# print("="*60)
# with open('/kaggle/working/results/enhanced_bottom_half_metrics.txt', 'w', encoding='utf-8') as f:
#     f.write("Enhanced Bottom Half Face Reconstruction Evaluation Results - Test Set\n")
#     f.write("="*60 + "\n")
#     f.write(f"Average PSNR: {np.mean(psnr_values):.4f} dB (±{np.std(psnr_values):.4f})\n")
#     f.write(f"Average SSIM: {np.mean(ssim_values):.4f} (±{np.std(ssim_values):.4f})\n")
#     f.write(f"Average MSE: {np.mean(mse_values):.4f} (±{np.std(mse_values):.4f})\n")
#     f.write(f"Average Identity Loss: {np.mean(identity_values):.4f} (±{np.std(identity_values):.4f})\n")
#     f.write(f"Average Landmark Loss: {np.mean(landmark_values):.4f} (±{np.std(landmark_values):.4f})\n")
#     f.write("="*60 + "\n")
#     f.write(f"Best PSNR: {np.max(psnr_values):.4f} dB\n")
#     f.write(f"Best SSIM: {np.max(ssim_values):.4f}\n")
#     f.write(f"Worst PSNR: {np.min(psnr_values):.4f} dB\n")
#     f.write(f"Worst SSIM: {np.min(ssim_values):.4f}\n")
#     f.write("="*60 + "\n")
#     for i, (psnr_val, ssim_val, mse_val, id_val, lm_val) in enumerate(zip(psnr_values, ssim_values, mse_values, identity_values, landmark_values)):
#         f.write(f"Sample {i+1}: PSNR={psnr_val:.4f}, SSIM={ssim_val:.4f}, MSE={mse_val:.4f}, Identity={id_val:.4f}, Landmark={lm_val:.4f}\n")
# print(f"\nTraining and evaluation complete!")
# print(f"Models saved in '/kaggle/working/models/' directory.")
# print(f"Evaluation results saved in '/kaggle/working/results/' directory.")

In [ ]:
# import os
# os.environ.setdefault('TORCH_CUDA_ARCH_LIST', '7.5') # For Kaggle GPU compatibility
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torchvision.models as models
# import torchvision.transforms as transforms
# from torch.utils.data import Dataset, DataLoader
# import matplotlib.pyplot as plt
# import numpy as np
# from skimage.metrics import peak_signal_noise_ratio as psnr
# from skimage.metrics import structural_similarity as ssim
# import random
# import cv2
# from PIL import Image
# from torch import amp  # For mixed precision training (new API)
# import networkx as nx  # For graph operations

# # Simplified GNN for graph processing
# class PyramidalGNN(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super(PyramidalGNN, self).__init__()
#         self.lin = nn.Linear(in_channels, out_channels)
#         self.gate = nn.Linear(in_channels * 2, out_channels)

#     def forward(self, x, edge_index):
#         # Simplified version - just apply linear transformation
#         return self.lin(x)



# # /kaggle/input/celeba-resized-6464/img_align_celeba/CelebA_Image_Cropped_64
# # Configuration parameters
# prune_threshold = 0.2  # Threshold for pruning low-value nodes
# lambda_l1 = 100.0  # L1 reconstruction loss weight
# n_critic = 5  # Number of discriminator updates per generator update

# # Global prune_graph function using NetworkX
# def prune_graph(model, graph, threshold=prune_threshold):
#     # NetworkX pruning
#     deg_cent = nx.degree_centrality(graph)
#     low_value_nodes = [n for n, score in deg_cent.items() if score < threshold]
#     graph.remove_nodes_from(low_value_nodes)
#     return graph

# # Try to import MTCNN for landmark detection
# try:
#     from mtcnn import MTCNN
#     USE_MTCNN = False # Disable during training to avoid NumPy conversions
#     print("MTCNN imported successfully for landmark detection (disabled for training).")
# except ImportError:
#     print("MTCNN not found. Landmark loss will be simplified.")
#     USE_MTCNN = False
# # ============================================================================
# # CHECKPOINT CONFIGURATION - Resume Training Setup
# # تنظیمات Checkpoint - برای ادامه آموزش
# # ============================================================================

# # 🔄 RESUME TRAINING: Set to True to continue from previous checkpoint
# # ادامه آموزش: برای ادامه از checkpoint قبلی، True کنید
# RESUME_TRAINING = True  # True: Resume from checkpoint | False: Start from scratch

# # 📂 CHECKPOINT PATHS: Where to load/save checkpoints
# # مسیرهای Checkpoint: محل بارگذاری/ذخیره checkpoint‌ها
# CHECKPOINT_DIR = '/kaggle/working/checkpoints'
# CHECKPOINT_INTERVAL = 1  # Save checkpoint every N epochs (1 = every epoch)

# # 📥 LOAD FROM KAGGLE DATASET: If you uploaded models as Kaggle dataset
# # بارگذاری از Kaggle Dataset: اگر مدل‌ها را به عنوان dataset آپلود کرده‌اید
# USE_KAGGLE_INPUT = True  # Set True if loading from Kaggle dataset input
# KAGGLE_INPUT_DATASET = '/kaggle/input/image-inpainting-v1/pytorch/default/1'  # Update this path (e.g., '/kaggle/input/checkpoints')
# # /kaggle/input/image-inpainting/pytorch/default/2
# # /kaggle/input/image-inpainting/pytorch/default/4
# # /kaggle/input/image-inpainting/pytorch/default/3
# # 🗂️ LOAD FROM MODEL FILES: If you only have model state_dict files (*.pth)
# # بارگذاری از فایل‌های مدل: اگر فقط فایل‌های state_dict مدل دارید
# LOAD_FROM_MODELS = True  # Set True if loading from models folder (coarse_generator_best.pth, etc.)
# MODELS_DIR = '/kaggle/working/models'  # Path to models folder
# USE_BEST_MODELS = True  # True: Load *_best.pth | False: Load *_bottom_half.pth

# # ⚠️ نکته: اگر از LOAD_FROM_MODELS استفاده کنید، فقط وزن‌های مدل بارگذاری می‌شوند
# # optimizer/scheduler/training history بارگذاری نمی‌شوند و از اول مقداردهی می‌شوند

# # ⚙️ TRAINING START POINT: Which epoch to start from
# # نقطه شروع آموزش: از کدام epoch شروع شود
# # Note: This will be automatically set when loading checkpoint
# # توجه: این مقدار به صورت خودکار هنگام بارگذاری checkpoint تنظیم می‌شود

# print("="*70)
# print("🔧 CHECKPOINT CONFIGURATION")
# print("="*70)
# print(f"📌 Resume Training: {RESUME_TRAINING}")
# print(f"📁 Checkpoint Directory: {CHECKPOINT_DIR}")
# print(f"💾 Save Interval: Every {CHECKPOINT_INTERVAL} epoch(s)")
# if USE_KAGGLE_INPUT:
#     print(f"📥 Loading from Kaggle Dataset: {KAGGLE_INPUT_DATASET}")
# if LOAD_FROM_MODELS:
#     print(f"🗂️  Load from Models: {MODELS_DIR}")
#     print(f"   Using: {'*_best.pth' if USE_BEST_MODELS else '*_bottom_half.pth'}")
# print("="*70 + "\n")

# # ============================================================================
# # TRAINING CONFIGURATION - Initial configurations
# # تنظیمات آموزش - پیکربندی اولیه
# # ============================================================================

# image_size = (64, 64)
# batch_size = 128 # Further reduced for stability
# eval_batch_size = 1
# num_epochs = 20 # Increased for better convergence
# learning_rate_g = 0.0001 # 1e-4 for Generator
# learning_rate_d = 0.0004 # 4e-4 for Discriminator (higher to strengthen D)
# lambda_adv = 0.05 # Further reduced adversarial weight
# lambda_style = 0.05 # Reduced style weight
# lambda_fm = 0.05 # Reduced feature matching weight
# lambda_rec = 1.0 # Reduced reconstruction weight
# lambda_identity = 0.005 # Reduced identity weight
# lambda_landmark = 0.01 # Reduced landmark weight
# lambda_perc = 0.1 # Reduced perceptual weight
# lambda_vae = 0.01 # Further reduced VAE weight
# lambda_struct = 0.05 # Reduced structural weight
# lambda_d_reg = 1e-4 # Increased discriminator regularization
# lambda_attention = 0.05 # Reduced attention loss weight
# # NEW: Edge-Guided Inpainting Loss Weights
# lambda_edge = 0.5 # Edge loss weight for preserving nose/lips edges
# lambda_hf = 0.3 # High-frequency loss weight to reduce blurriness
# lambda_gradient = 0.4 # Gradient loss weight for sharp edges
# lambda_facial = 0.6 # Facial region loss weight for nose/lips focus
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")
# # Data splitting configuration
# def split_dataset(dataset_path, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     image_files = [f for f in os.listdir(dataset_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
#     random.shuffle(image_files)
#     total_files = len(image_files)
#     train_end = int(total_files * train_ratio)
#     val_end = train_end + int(total_files * val_ratio)
#     train_files, val_files, test_files = image_files[:train_end], image_files[train_end:val_end], image_files[val_end:]
#     print(f"Dataset split: Total={total_files}, Train={len(train_files)} ({len(train_files)/total_files*100:.1f}%), "
#           f"Val={len(val_files)} ({len(val_files)/total_files*100:.1f}%), Test={len(test_files)} ({len(test_files)/total_files*100:.1f}%)")
#     return train_files, val_files, test_files
# # Custom Dataset
# class CelebADataset(Dataset):
#     def __init__(self, dataset_path, file_list, image_size):
#         self.dataset_path = dataset_path
#         self.file_list = file_list
#         self.transform = transforms.Compose([
#             transforms.Resize(image_size),
#             transforms.ToTensor(),
#             transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # -1 to 1
#         ])
   
#     def __len__(self):
#         return len(self.file_list)
   
#     def __getitem__(self, idx):
#         img_path = os.path.join(self.dataset_path, self.file_list[idx])
#         image = Image.open(img_path).convert('RGB')
#         image = self.transform(image)
#         return image
# def create_split_datasets(dataset_path, train_files, val_files, test_files, image_size, batch_size):
#     train_dataset = CelebADataset(dataset_path, train_files, image_size)
#     val_dataset = CelebADataset(dataset_path, val_files, image_size)
#     test_dataset = CelebADataset(dataset_path, test_files, image_size)
   
#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
#     val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
#     test_loader = DataLoader(test_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=2, pin_memory=True)
#     return train_loader, val_loader, test_loader
# # Enhanced ViT-based Preprocessor with Multi-layer Attention and Graph Pruning
# class ViTPreprocessor(nn.Module):
#     def __init__(self, dim=128, num_heads=8, patch_size=8, num_layers=3):
#         super().__init__()
#         self.patch_size = patch_size
#         self.dim = dim
#         self.num_heads = num_heads
#         self.num_patches = (image_size[0] // patch_size) * (image_size[1] // patch_size)
#         self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size, padding=0)
#         self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, dim))
       
#         # Multi-layer attention blocks
#         self.attention_blocks = nn.ModuleList([
#             nn.ModuleDict({
#                 'attn': nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=0.1),
#                 'norm1': nn.LayerNorm(dim, eps=1e-6),
#                 'norm2': nn.LayerNorm(dim, eps=1e-6),
#                 'mlp': nn.Sequential(
#                     nn.Linear(dim, dim * 4),
#                     nn.GELU(),
#                     nn.Dropout(0.1),
#                     nn.Linear(dim * 4, dim),
#                     nn.Dropout(0.1)
#                 )
#             }) for _ in range(num_layers)
#         ])
       
#         self.final_norm = nn.LayerNorm(dim, eps=1e-6)
#         self.gnn = PyramidalGNN(dim, dim)  # Add PyramidalGNN
   
#     def build_graph(self, features):
#         B, C, H, W = features.shape
#         nodes = features.view(B, C, -1).permute(0, 2, 1)  # [B, num_patches, dim]
        
#         # Use NetworkX for graph construction
#         G = nx.Graph()
#         for i in range(nodes.shape[1]):
#             G.add_node(i, feat=nodes[0, i])
#         for i in range(nodes.shape[1]):
#             for j in range(i+1, nodes.shape[1]):
#                 sim = F.cosine_similarity(nodes[0, i:i+1], nodes[0, j:j+1])
#                 if sim > 0.5:
#                     G.add_edge(i, j, weight=sim)
#         return G
   
   
#     def graph_enhanced_forward(self, x, graph):
#         # Check if graph is empty after pruning
#         node_list = list(graph.nodes)
#         if not node_list:
#             return x  # Return input features directly
        
#         # Simple message passing with NetworkX
#         for node in graph.nodes:
#             neighbors = list(graph.neighbors(node))
#             if neighbors:
#                 neighbor_feats = torch.stack([graph.nodes[n]['feat'] for n in neighbors])
#                 aggregated = torch.mean(neighbor_feats, dim=0)
#                 graph.nodes[node]['feat'] = (graph.nodes[node]['feat'] + aggregated) / 2
        
#         # Final check before stacking
#         final_nodes = list(graph.nodes)
#         if not final_nodes:
#             return x
        
#         # Ensure all features have the same dtype
#         node_features = [graph.nodes[n]['feat'] for n in sorted(final_nodes)]
#         return torch.stack(node_features)
   
#     def reconstruct_features(self, pruned_feats, pruned_indices, B, C, H, W):
#         num_patches = H * W
#         # Ensure dtype matches the input features
#         full_feats = torch.zeros(B, num_patches, C, device=pruned_feats.device, dtype=pruned_feats.dtype)
#         full_feats[:, pruned_indices] = pruned_feats
#         return full_feats.reshape(B, H, W, C).permute(0, 3, 1, 2)
   
#     def forward(self, x):
#         embedded = self.patch_embed(x)  # [B, dim, H/p, W/p]
#         B, C, H, W = embedded.shape
#         flat = embedded.permute(0, 2, 3, 1).reshape(B, H * W, C)
        
#         # Adaptive position embedding based on actual patch count
#         num_patches = H * W
#         if num_patches != self.num_patches:
#             # Interpolate position embedding to match current patch count
#             # Reshape to 3D for linear interpolation: [1, dim, num_patches]
#             pos_embed_3d = self.pos_embed.permute(0, 2, 1)  # [1, dim, num_patches]
#             pos_embed_adaptive = F.interpolate(
#                 pos_embed_3d,
#                 size=num_patches,
#                 mode='linear',
#                 align_corners=False
#             ).permute(0, 2, 1)  # [1, num_patches, dim]
#         else:
#             pos_embed_adaptive = self.pos_embed
        
#         flat = flat + pos_embed_adaptive
        
#         # Build and prune graph
#         graph = self.build_graph(embedded)
#         pruned_graph = prune_graph(self, graph)
        
#         # Get pruned features using NetworkX
#         pruned_indices = list(pruned_graph.nodes)
#         if pruned_indices:  # Check if not empty
#             pruned_feats = torch.stack([pruned_graph.nodes[i]['feat'] for i in pruned_indices])
#         else:
#             pruned_feats = flat[0].clone()  # Use original flattened features for batch 0
#             pruned_indices = list(range(flat.shape[1]))  # All indices
        
#         # Enhanced processing on high-value nodes
#         for block in self.attention_blocks:
#             # Self-attention on pruned features
#             attn_out, _ = block['attn'](pruned_feats, pruned_feats, pruned_feats)
#             pruned_feats = block['norm1'](pruned_feats + attn_out)
            
#             # MLP
#             mlp_out = block['mlp'](pruned_feats)
#             pruned_feats = block['norm2'](pruned_feats + mlp_out)
        
#         # Graph-enhanced message passing
#         enhanced_pruned = self.graph_enhanced_forward(pruned_feats, pruned_graph)
        
#         # Reconstruct full features with low-value nodes getting minimal processing
#         x = self.reconstruct_features(enhanced_pruned, pruned_indices, B, C, H, W)
        
#         # Minimal processing for pruned (low-value) regions - e.g., average pooling
#         full_x = self.final_norm(x.view(B, C, -1).permute(0, 2, 1)).permute(0, 2, 1).view(B, C, H, W)
#         return full_x
# # VAE Sampling Layer
# class VAESampling(nn.Module):
#     def __init__(self):
#         super().__init__()
   
#     def forward(self, mu, logvar):
#         std = torch.exp(0.5 * logvar)
#         eps = torch.randn_like(std)
#         return mu + eps * std
# # Enhanced VAE Encoder with Residual Connections and Graph Integration (No BatchNorm, Strided Conv)
# class VAEEncoder(nn.Module):
#     def __init__(self, latent_dim=128):
#         super().__init__()
#         self.latent_dim = latent_dim
       
#         # Enhanced encoder with strided convolutions (no batch norm)
#         self.conv1 = nn.Conv2d(128, 64, kernel_size=4, stride=2, padding=1) # [B, 128, 8, 8] -> [B, 64, 4, 4]
#         self.conv2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1) # [B, 64, 4, 4] -> [B, 128, 2, 2]
       
#         # Residual block (no batch norm)
#         self.res_conv = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
       
#         self.flatten = nn.Flatten()
#         # Use adaptive pooling to ensure consistent output size
#         self.adaptive_pool = nn.AdaptiveAvgPool2d((2, 2))  # Always output 2x2
#         self.fc_mu = nn.Linear(128 * 2 * 2, latent_dim) # 128 * 2 * 2 = 512
#         self.fc_logvar = nn.Linear(128 * 2 * 2, latent_dim) # 128 * 2 * 2 = 512
   
#     def forward(self, x):
#         # Check input size and use adaptive approach for very small inputs
#         B, C, H, W = x.shape
        
#         # If input is too small for conv layers, use adaptive pooling directly
#         if H < 4 or W < 4:
#             # Use adaptive pooling to get to a reasonable size first
#             x = F.adaptive_avg_pool2d(x, (8, 8))  # Resize to 8x8 minimum
#             x = F.relu(self.conv1(x))  # 8x8 -> 4x4
#             x = F.relu(self.conv2(x))  # 4x4 -> 2x2
#         else:
#             x = F.relu(self.conv1(x))
#             x = F.relu(self.conv2(x))
       
#         # Residual connection
#         residual = x
#         x = F.relu(self.res_conv(x))
#         x = x + residual
       
#         # Use adaptive pooling to ensure consistent size
#         x = self.adaptive_pool(x)  # Always [B, 128, 2, 2]
#         x = self.flatten(x)  # Always [B, 512]
#         mu = self.fc_mu(x)
#         logvar = self.fc_logvar(x)
#         return mu, logvar
# # Enhanced VAE Decoder with Residual Connections and Graph Integration (No BatchNorm)
# class VAEDecoder(nn.Module):
#     def __init__(self, latent_dim=128):
#         super().__init__()
#         self.latent_dim = latent_dim
       
#         # Enhanced decoder with strided transposed convolutions (no batch norm)
#         self.fc = nn.Linear(latent_dim, 8 * 8 * 128)
       
#         # Residual block before upsampling (no batch norm)
#         self.res_conv = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
       
#         self.deconv1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
#         self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
#         self.deconv3 = nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1)
   
#     def forward(self, z):
#         x = F.relu(self.fc(z))
#         x = x.view(-1, 128, 8, 8)
       
#         # Residual connection
#         residual = x
#         x = F.relu(self.res_conv(x))
#         x = x + residual
       
#         x = F.relu(self.deconv1(x)) # 8x8 -> 16x16
#         x = F.relu(self.deconv2(x)) # 16x16 -> 32x32
#         x = torch.tanh(self.deconv3(x)) # 32x32 -> 64x64
#         return x
# # Enhanced CoarseGenerator with Attention, Skip Connections, and Graph Pruning
# # ============================================================================
# # EDGE DETECTION AND EDGE GENERATOR
# # ماژول‌های تشخیص و بازسازی لبه برای بهبود بازسازی بینی و لب‌ها
# # ============================================================================

# class EdgeDetector(nn.Module):
#     """
#     Edge detection using Sobel filters to extract edges from images
#     استخراج لبه‌ها با استفاده از فیلترهای Sobel
#     """
#     def __init__(self):
#         super().__init__()
#         # Sobel filters for edge detection
#         sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
#         sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        
#         self.register_buffer('sobel_x', sobel_x.repeat(3, 1, 1, 1))
#         self.register_buffer('sobel_y', sobel_y.repeat(3, 1, 1, 1))
    
#     def forward(self, x):
#         # Apply Sobel filters to detect edges
#         # x: (B, 3, H, W)
#         edge_x = F.conv2d(x, self.sobel_x, padding=1, groups=3)
#         edge_y = F.conv2d(x, self.sobel_y, padding=1, groups=3)
        
#         # Compute edge magnitude
#         edges = torch.sqrt(edge_x ** 2 + edge_y ** 2 + 1e-8)
#         return edges

# class EdgeGenerator(nn.Module):
#     """
#     Edge Generator for reconstructing edges of nose and lips
#     Generator برای بازسازی لبه‌های بینی و لب‌ها
#     """
#     def __init__(self):
#         super().__init__()
        
#         # Encoder for edge feature extraction
#         self.enc1 = nn.Sequential(
#             nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1),  # Input: masked image + mask
#             nn.LeakyReLU(0.2, inplace=True)
#         )
#         self.enc2 = nn.Sequential(
#             nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
#         self.enc3 = nn.Sequential(
#             nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
        
#         # Bottleneck with attention for edge features
#         self.bottleneck = nn.Sequential(
#             nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
        
#         # Attention mechanism
#         self.attention = nn.MultiheadAttention(embed_dim=512, num_heads=8, dropout=0.1)
#         self.attention_norm = nn.LayerNorm(512, eps=1e-6)
        
#         # Decoder for edge reconstruction
#         self.dec1 = nn.Sequential(
#             nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
#             nn.ReLU(inplace=True)
#         )
#         self.dec2 = nn.Sequential(
#             nn.ConvTranspose2d(512, 128, kernel_size=4, stride=2, padding=1),  # 256 + 256 from skip
#             nn.ReLU(inplace=True)
#         )
#         self.dec3 = nn.Sequential(
#             nn.ConvTranspose2d(256, 64, kernel_size=4, stride=2, padding=1),   # 128 + 128 from skip
#             nn.ReLU(inplace=True)
#         )
#         self.dec4 = nn.Sequential(
#             nn.ConvTranspose2d(128, 3, kernel_size=4, stride=2, padding=1),    # 64 + 64 from skip
#             nn.Sigmoid()  # Output edge map in [0, 1]
#         )
    
#     def forward(self, masked_image, mask):
#         # Concatenate masked image and mask
#         x = torch.cat([masked_image, mask], dim=1)
        
#         # Encoder with skip connections
#         enc1_out = self.enc1(x)       # 64x64 -> 32x32
#         enc2_out = self.enc2(enc1_out) # 32x32 -> 16x16
#         enc3_out = self.enc3(enc2_out) # 16x16 -> 8x8
#         bottleneck = self.bottleneck(enc3_out)  # 8x8 -> 4x4
        
#         # Apply attention to bottleneck features
#         B, C, H, W = bottleneck.shape
#         attn_input = bottleneck.permute(0, 2, 3, 1).reshape(B, H * W, C)
#         attn_output, _ = self.attention(attn_input, attn_input, attn_input)
#         enhanced_features = self.attention_norm(attn_input + attn_output)
#         enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
        
#         # Decoder with skip connections
#         dec1_out = self.dec1(enhanced_features)              # 4x4 -> 8x8
#         dec1_out = torch.cat([dec1_out, enc3_out], dim=1)    # Skip connection
        
#         dec2_out = self.dec2(dec1_out)                       # 8x8 -> 16x16
#         dec2_out = torch.cat([dec2_out, enc2_out], dim=1)    # Skip connection
        
#         dec3_out = self.dec3(dec2_out)                       # 16x16 -> 32x32
#         dec3_out = torch.cat([dec3_out, enc1_out], dim=1)    # Skip connection
        
#         output = self.dec4(dec3_out)                         # 32x32 -> 64x64
        
#         return output

# class CoarseGenerator(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.preprocessor = ViTPreprocessor(dim=128, num_heads=8, patch_size=8, num_layers=3)
#         self.encoder = VAEEncoder(latent_dim=128) # Increased latent dimension
#         self.sampling = VAESampling()
#         self.decoder = VAEDecoder(latent_dim=128)
       
#         # Attention mechanism for better feature fusion
#         self.attention = nn.MultiheadAttention(embed_dim=128, num_heads=8, dropout=0.1)
#         self.attention_norm = nn.LayerNorm(128, eps=1e-6)
       
#         # Skip connection processing
#         self.skip_conv = nn.Conv2d(128, 128, kernel_size=1, padding=0)
   
#     def forward(self, x, mask, edge_map=None):
#         # If edge map is provided, concatenate with input for edge-guided inpainting
#         if edge_map is not None:
#             x = x + edge_map * 0.1  # Blend edge information with input
        
#         # Preprocess with graph pruning
#         original_features = self.preprocessor(x)
       
#         # Apply attention to enhance features
#         B, C, H, W = original_features.shape
#         attn_input = original_features.permute(0, 2, 3, 1).reshape(B, H * W, C)
#         attn_output, _ = self.attention(attn_input, attn_input, attn_input)
#         enhanced_features = self.attention_norm(attn_input + attn_output)
#         enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
       
#         # Combine with skip connection
#         enhanced_features = enhanced_features + self.skip_conv(original_features)
       
#         mu, logvar = self.encoder(enhanced_features)
#         z = self.sampling(mu, logvar)
#         output = self.decoder(z)
#         return output, mu, logvar
# # Enhanced FineGenerator with U-Net Architecture, Attention, Edge-Guided, and Graph Cut Integration
# class FineGenerator(nn.Module):
#     def __init__(self):
#         super().__init__()
       
#         # Encoder with skip connections (edge-guided: 9 channels = 6 + 3 edge)
#         self.enc1 = nn.Sequential(
#             nn.Conv2d(9, 64, kernel_size=4, stride=2, padding=1),  # Added 3 channels for edge map
#             nn.LeakyReLU(0.2, inplace=True)
#         )
#         self.enc2 = nn.Sequential(
#             nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
#         self.enc3 = nn.Sequential(
#             nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
       
#         # Bottleneck with attention
#         self.bottleneck = nn.Sequential(
#             nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
       
#         # Attention mechanism
#         self.attention = nn.MultiheadAttention(embed_dim=512, num_heads=8, dropout=0.1)
#         self.attention_norm = nn.LayerNorm(512, eps=1e-6)
       
#         # Decoder with skip connections
#         self.dec1 = nn.Sequential(
#             nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
#             nn.ReLU(inplace=True)
#         )
#         self.dec2 = nn.Sequential(
#             nn.ConvTranspose2d(512, 128, kernel_size=4, stride=2, padding=1), # 256 + 256 from skip
#             nn.ReLU(inplace=True)
#         )
#         self.dec3 = nn.Sequential(
#             nn.ConvTranspose2d(256, 64, kernel_size=4, stride=2, padding=1), # 128 + 128 from skip
#             nn.ReLU(inplace=True)
#         )
#         self.dec4 = nn.Sequential(
#             nn.ConvTranspose2d(128, 3, kernel_size=4, stride=2, padding=1), # 64 + 64 from skip
#             nn.Tanh()
#         )
   
#     def forward(self, masked_images, coarse_output, masks, edge_map=None):
#         # Ensure all inputs have the same spatial dimensions
#         if masked_images.shape[2:] != coarse_output.shape[2:]:
#             coarse_output = F.interpolate(coarse_output, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
        
#         # If edge map is provided, concatenate it with input for edge-guided inpainting
#         if edge_map is not None:
#             if edge_map.shape[2:] != masked_images.shape[2:]:
#                 edge_map = F.interpolate(edge_map, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
#             x = torch.cat([masked_images, coarse_output, edge_map], dim=1)  # 3 + 3 + 3 = 9 channels
#         else:
#             # Fallback: use zeros for edge map if not provided
#             edge_map_zeros = torch.zeros_like(masked_images)
#             x = torch.cat([masked_images, coarse_output, edge_map_zeros], dim=1)
       
#         # Encoder with skip connections
#         enc1_out = self.enc1(x) # 64x64 -> 32x32
#         enc2_out = self.enc2(enc1_out) # 32x32 -> 16x16
#         enc3_out = self.enc3(enc2_out) # 16x16 -> 8x8
#         bottleneck = self.bottleneck(enc3_out) # 8x8 -> 4x4
       
#         # Apply attention to bottleneck features
#         B, C, H, W = bottleneck.shape
#         attn_input = bottleneck.permute(0, 2, 3, 1).reshape(B, H * W, C)
#         attn_output, _ = self.attention(attn_input, attn_input, attn_input)
#         enhanced_features = self.attention_norm(attn_input + attn_output)
#         enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
       
#         # Decoder with skip connections
#         dec1_out = self.dec1(enhanced_features) # 4x4 -> 8x8
#         dec1_out = torch.cat([dec1_out, enc3_out], dim=1) # Skip connection
       
#         dec2_out = self.dec2(dec1_out) # 8x8 -> 16x16
#         dec2_out = torch.cat([dec2_out, enc2_out], dim=1) # Skip connection
       
#         dec3_out = self.dec3(dec2_out) # 16x16 -> 32x32
#         dec3_out = torch.cat([dec3_out, enc1_out], dim=1) # Skip connection
       
#         output = self.dec4(dec3_out) # 32x32 -> 64x64
       
#         # Ensure output matches input size
#         if output.shape[2:] != masked_images.shape[2:]:
#             output = F.interpolate(output, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
       
#         return output
# # Stronger Discriminator with Spectral Normalization (No BatchNorm, More Layers)
# class Discriminator(nn.Module):
#     def __init__(self):
#         super().__init__()
       
#         # Stronger discriminator with more layers and capacity (no batch norm)
#         self.conv1 = nn.utils.spectral_norm(nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1))
#         self.conv2 = nn.utils.spectral_norm(nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1))
#         self.conv3 = nn.utils.spectral_norm(nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1))
#         self.conv4 = nn.utils.spectral_norm(nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1))
#         # Additional layers for stronger discriminator
#         self.conv5 = nn.utils.spectral_norm(nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1))
#         self.conv6 = nn.utils.spectral_norm(nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1))
       
#         # Global average pooling for stability
#         self.global_pool = nn.AdaptiveAvgPool2d(1)
#         # Deeper fully connected layers
#         self.fc1 = nn.utils.spectral_norm(nn.Linear(512, 256))
#         self.fc2 = nn.utils.spectral_norm(nn.Linear(256, 1))
       
#         # Feature layers for feature matching loss
#         self.feature_layers = [self.conv1, self.conv2, self.conv3, self.conv4, self.conv5, self.conv6]
   
#     def forward(self, x, mask, return_features=False):
#         input_concat = torch.cat([x, mask], dim=1)
#         features = []
       
#         # Main discriminator path with more layers
#         x = F.leaky_relu(self.conv1(input_concat), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv2(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv3(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv4(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv5(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv6(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         # Global average pooling for stability
#         x = self.global_pool(x)
#         x = x.view(x.size(0), -1)
        
#         # Deeper FC layers
#         x = F.leaky_relu(self.fc1(x), 0.2, inplace=True)
#         logits = self.fc2(x)
       
#         if return_features:
#             return logits, features
#         return logits
# # Generate adaptive mask
# def generate_adaptive_mask(image):
#     batch_size = image.shape[0]
#     h, w = image_size
#     mask = torch.ones(batch_size, 3, h, w, dtype=torch.float32, device=image.device)
#     half_height = h // 2
#     bottom_mask = torch.zeros(batch_size, 3, h - half_height, w, dtype=torch.float32, device=image.device)
#     top_mask = torch.ones(batch_size, 3, half_height, w, dtype=torch.float32, device=image.device)
#     mask = torch.cat([top_mask, bottom_mask], dim=2)
   
#     if USE_MTCNN:
#         detector = MTCNN()
#         images_np = ((image * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
#         landmarks = []
#         for img in images_np:
#             img_uint8 = img.astype(np.uint8)
#             result = detector.detect_faces(img_uint8)
#             landmarks.append(result[0]['keypoints'] if result else {})
       
#         mask_np = mask.permute(0, 2, 3, 1).cpu().detach().numpy()
#         for i, lm in enumerate(landmarks):
#             if lm:
#                 for key in ['mouth_left', 'mouth_right']:
#                     if key in lm:
#                         x, y = lm[key]
#                         y = min(max(y, half_height), h-1)
#                         x = min(max(x, 0), w-1)
#                         mask_np[i, y-5:y+5, x-5:x+5, :] = 0.0
#         mask = torch.from_numpy(mask_np).permute(0, 3, 1, 2).to(device)
   
#     return mask
# # Loss functions
# # Use new torchvision weights API to avoid deprecation warnings
# try:
#     vgg_weights = models.VGG16_Weights.DEFAULT
# except AttributeError:
#     # Fallback for very old torchvision
#     vgg_weights = None
# vgg = models.vgg16(weights=vgg_weights).features.to(device).eval()
# loss_model = nn.Sequential(*[vgg[i] for i in range(16)]).to(device) # Up to block4_conv3
# for param in loss_model.parameters():
#     param.requires_grad = False

# # Initialize Edge Detector for edge loss
# edge_detector_global = EdgeDetector().to(device)
# edge_detector_global.eval()
# for param in edge_detector_global.parameters():
#     param.requires_grad = False
# def perceptual_loss(y_true, y_pred):
#     y_true = F.interpolate(y_true, size=image_size, mode='bilinear', align_corners=False)
#     y_pred = F.interpolate(y_pred, size=image_size, mode='bilinear', align_corners=False)
#     y_true = y_true * 0.5 + 0.5 # Denormalize to [0, 1]
#     y_pred = y_pred * 0.5 + 0.5
#     true_features = loss_model(y_true)
#     pred_features = loss_model(y_pred)
#     return torch.mean((true_features - pred_features) ** 2)
# def style_loss(y_true, y_pred):
#     y_true = F.interpolate(y_true, size=image_size, mode='bilinear', align_corners=False)
#     y_pred = F.interpolate(y_pred, size=image_size, mode='bilinear', align_corners=False)
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     true_features = loss_model(y_true)
#     pred_features = loss_model(y_pred)
   
#     def gram_matrix(feat):
#         B, C, H, W = feat.shape
#         feat = feat.view(B, C, H * W)
#         return torch.bmm(feat, feat.transpose(1, 2)) / (C * H * W)
   
#     style_losses = [torch.mean((gram_matrix(t) - gram_matrix(p)) ** 2) for t, p in zip([true_features], [pred_features])]
#     return torch.mean(torch.stack(style_losses))
# def bottom_half_loss(y_true, y_pred, mask):
#     bottom_region = (mask == 0).float()
#     diff = (y_true - y_pred) ** 2 * bottom_region
#     return diff.sum() / (bottom_region.sum() + 1e-8)
# def feature_matching_loss(real_features, fake_features):
#     losses = [torch.mean((r - f) ** 2) for r, f in zip(real_features, fake_features)]
#     return torch.mean(torch.stack(losses))
# def identity_loss(y_true, y_pred, mask):
#     bottom_region = (mask == 0).float()
#     y_true_face = y_true * bottom_region
#     y_pred_face = y_pred * bottom_region
#     return torch.mean(torch.abs(y_true_face - y_pred_face))
# def landmark_guided_loss(y_true, y_pred, mask):
#     if not USE_MTCNN:
#         return bottom_half_loss(y_true, y_pred, mask)
#     detector = MTCNN()
#     bottom_region = (mask == 0).float()
#     y_true_bottom = y_true * bottom_region
#     y_pred_bottom = y_pred * bottom_region
#     y_true_np = ((y_true_bottom * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
#     y_pred_np = ((y_pred_bottom * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
#     loss_tensor = torch.tensor(0.0, device=device, requires_grad=True)
#     for true_img, pred_img in zip(y_true_np, y_pred_np):
#         true_img_uint8 = true_img.astype(np.uint8)
#         pred_img_uint8 = pred_img.astype(np.uint8)
#         true_lm = detector.detect_faces(true_img_uint8)
#         pred_lm = detector.detect_faces(pred_img_uint8)
#         if true_lm and pred_lm:
#             true_points = [true_lm[0]['keypoints'][k] for k in ['mouth_left', 'mouth_right']]
#             pred_points = [pred_lm[0]['keypoints'][k] for k in ['mouth_left', 'mouth_right']]
#             for t, p in zip(true_points, pred_points):
#                 diff_x = torch.tensor(t[0] - p[0], dtype=torch.float32, device=device)
#                 diff_y = torch.tensor(t[1] - p[1], dtype=torch.float32, device=device)
#                 loss_tensor = loss_tensor + (diff_x ** 2 + diff_y ** 2)
#     return loss_tensor / y_true.shape[0]
# def structural_ssim_loss(y_true, y_pred, mask):
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     bottom_mask = (mask == 0).float()
#     y_true_bottom = y_true * bottom_mask
#     y_pred_bottom = y_pred * bottom_mask
#     ssim_vals = []
#     for i in range(y_true.shape[0]):
#         true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         ssim_val = ssim(true_img, pred_img, channel_axis=2, data_range=1.0, win_size=7)
#         ssim_vals.append(ssim_val)
#     return 1.0 - torch.mean(torch.tensor(ssim_vals, device=device))
# # Advanced Loss Functions
# def gradient_penalty_loss(discriminator, real_images, fake_images, masks, device):
#     """Calculate gradient penalty for WGAN-GP"""
#     batch_size = real_images.size(0)
#     alpha = torch.rand(batch_size, 1, 1, 1).to(device)
#     interpolated = alpha * real_images + (1 - alpha) * fake_images
   
#     # Ensure interpolated tensor requires gradients
#     interpolated = interpolated.detach().requires_grad_(True)
   
#     disc_interpolated = discriminator(interpolated, masks, return_features=False)
   
#     # Check if disc_interpolated requires gradients
#     if not disc_interpolated.requires_grad:
#         return torch.tensor(0.0, device=device, requires_grad=True)
   
#     gradients = torch.autograd.grad(
#         outputs=disc_interpolated,
#         inputs=interpolated,
#         grad_outputs=torch.ones_like(disc_interpolated).to(device),
#         create_graph=True,
#         retain_graph=True,
#         only_inputs=True
#     )[0]
   
#     gradients = gradients.view(gradients.size(0), -1)
#     gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
#     return gradient_penalty
# def attention_loss(y_true, y_pred, mask):
#     """Simplified attention-based loss focusing on important regions"""
#     bottom_region = (mask == 0).float()
#     diff = torch.abs(y_true - y_pred) * bottom_region
   
#     # Create simple attention weights based on gradient magnitude
#     true_grad_x = torch.abs(y_true[:, :, :, 1:] - y_true[:, :, :, :-1])
#     true_grad_y = torch.abs(y_true[:, :, 1:, :] - y_true[:, :, :-1, :])
   
#     # Pad gradients to match original size
#     true_grad_x_padded = F.pad(true_grad_x, (0, 1, 0, 0), mode='constant', value=0)
#     true_grad_y_padded = F.pad(true_grad_y, (0, 0, 0, 1), mode='constant', value=0)
   
#     # Combine gradients and normalize
#     attention_weights = (true_grad_x_padded + true_grad_y_padded).mean(dim=1, keepdim=True)
#     attention_weights = attention_weights / (attention_weights.mean() + 1e-8) # Normalize
   
#     # Apply attention weights
#     weighted_loss = diff * attention_weights
#     return weighted_loss.mean()

# # ============================================================================
# # NEW LOSS FUNCTIONS FOR EDGE-GUIDED INPAINTING
# # توابع Loss جدید برای بهبود لبه‌ها و کاهش مات بودن
# # ============================================================================

# def edge_loss(y_true, y_pred, mask, edge_detector):
#     """
#     Edge loss to preserve edges in nose and lips regions
#     Loss برای حفظ لبه‌های بینی و لب‌ها
#     """
#     # Extract edges from true and predicted images
#     with torch.no_grad():
#         edges_true = edge_detector(y_true * 0.5 + 0.5)  # Denormalize first
#     edges_pred = edge_detector(y_pred * 0.5 + 0.5)
    
#     # Focus on bottom half (masked region)
#     bottom_region = (mask == 0).float()
    
#     # Calculate edge reconstruction loss
#     edge_diff = torch.abs(edges_true - edges_pred) * bottom_region
    
#     return edge_diff.mean()

# def high_frequency_loss(y_true, y_pred, mask):
#     """
#     High-frequency loss to reduce blurriness in bottom half
#     Loss برای کاهش مات بودن در نیمه پایینی
#     """
#     # Apply Laplacian filter to detect high-frequency details
#     laplacian_kernel = torch.tensor([
#         [0, 1, 0],
#         [1, -4, 1],
#         [0, 1, 0]
#     ], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(y_true.device)
    
#     # Repeat for 3 channels
#     laplacian_kernel = laplacian_kernel.repeat(3, 1, 1, 1)
    
#     # Apply Laplacian to true and predicted images
#     y_true_hf = F.conv2d(y_true, laplacian_kernel, padding=1, groups=3)
#     y_pred_hf = F.conv2d(y_pred, laplacian_kernel, padding=1, groups=3)
    
#     # Focus on bottom half
#     bottom_region = (mask == 0).float()
    
#     # Calculate high-frequency difference
#     hf_diff = torch.abs(y_true_hf - y_pred_hf) * bottom_region
    
#     return hf_diff.mean()

# def gradient_loss(y_true, y_pred, mask):
#     """
#     Gradient loss to preserve sharp edges and textures
#     Loss برای حفظ لبه‌های تیز و بافت‌ها
#     """
#     # Sobel filters for gradient calculation
#     sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], 
#                            dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(y_true.device)
#     sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], 
#                            dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(y_true.device)
    
#     # Repeat for 3 channels
#     sobel_x = sobel_x.repeat(3, 1, 1, 1)
#     sobel_y = sobel_y.repeat(3, 1, 1, 1)
    
#     # Calculate gradients for true image
#     grad_true_x = F.conv2d(y_true, sobel_x, padding=1, groups=3)
#     grad_true_y = F.conv2d(y_true, sobel_y, padding=1, groups=3)
#     grad_true = torch.sqrt(grad_true_x ** 2 + grad_true_y ** 2 + 1e-8)
    
#     # Calculate gradients for predicted image
#     grad_pred_x = F.conv2d(y_pred, sobel_x, padding=1, groups=3)
#     grad_pred_y = F.conv2d(y_pred, sobel_y, padding=1, groups=3)
#     grad_pred = torch.sqrt(grad_pred_x ** 2 + grad_pred_y ** 2 + 1e-8)
    
#     # Focus on bottom half
#     bottom_region = (mask == 0).float()
    
#     # Calculate gradient difference
#     grad_diff = torch.abs(grad_true - grad_pred) * bottom_region
    
#     return grad_diff.mean()

# def facial_region_loss(y_true, y_pred, mask):
#     """
#     Focused loss on nose and lips regions (center-bottom area)
#     Loss متمرکز بر ناحیه بینی و لب‌ها
#     """
#     B, C, H, W = y_true.shape
    
#     # Create a weight map focusing on center-bottom (nose and lips region)
#     weight_map = torch.ones_like(mask)
    
#     # Define nose and lips region (center-bottom)
#     # Nose: roughly H/2 to 3H/4, center W/4 to 3W/4
#     # Lips: roughly 3H/4 to H, center W/4 to 3W/4
#     center_w_start = W // 4
#     center_w_end = 3 * W // 4
#     nose_h_start = H // 2
#     lips_h_end = H
    
#     # Increase weight for nose and lips region
#     weight_map[:, :, nose_h_start:lips_h_end, center_w_start:center_w_end] = 3.0
    
#     # Apply mask (only bottom half)
#     bottom_region = (mask == 0).float()
#     weight_map = weight_map * bottom_region
    
#     # Calculate weighted L1 loss
#     diff = torch.abs(y_true - y_pred) * weight_map
    
#     return diff.sum() / (weight_map.sum() + 1e-8)

# def generator_loss(disc_fake_logits, y_true, y_pred, disc_features_real, disc_features_fake, mask, mu, logvar):
#     # Clamp logvar to prevent exp overflow
#     logvar = torch.clamp(logvar, min=-10, max=10)
    
#     # Calculate each loss with NaN protection
#     adv_loss = -torch.mean(disc_fake_logits)
#     adv_loss = torch.nan_to_num(adv_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     # L1 reconstruction loss (high weight for stable inpainting)
#     bottom_region = (mask == 0).float()
#     l1_loss = torch.mean(torch.abs(y_true - y_pred) * bottom_region)
#     l1_loss = torch.nan_to_num(l1_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     rec_loss = bottom_half_loss(y_true, y_pred, mask)
#     rec_loss = torch.nan_to_num(rec_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     perc_loss = perceptual_loss(y_true, y_pred)
#     perc_loss = torch.nan_to_num(perc_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     sty_loss = style_loss(y_true, y_pred)
#     sty_loss = torch.nan_to_num(sty_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     fm_loss = feature_matching_loss(disc_features_real, disc_features_fake)
#     fm_loss = torch.nan_to_num(fm_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     id_loss = identity_loss(y_true, y_pred, mask)
#     id_loss = torch.nan_to_num(id_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     lm_loss = landmark_guided_loss(y_true, y_pred, mask)
#     lm_loss = torch.nan_to_num(lm_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     struct_loss = structural_ssim_loss(y_true, y_pred, mask)
#     struct_loss = torch.nan_to_num(struct_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     att_loss = attention_loss(y_true, y_pred, mask)
#     att_loss = torch.nan_to_num(att_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     # NEW: Edge-guided losses for better nose/lips reconstruction and sharpness
#     edg_loss = edge_loss(y_true, y_pred, mask, edge_detector_global)
#     edg_loss = torch.nan_to_num(edg_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     hf_loss = high_frequency_loss(y_true, y_pred, mask)
#     hf_loss = torch.nan_to_num(hf_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     grad_loss = gradient_loss(y_true, y_pred, mask)
#     grad_loss = torch.nan_to_num(grad_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     facial_loss = facial_region_loss(y_true, y_pred, mask)
#     facial_loss = torch.nan_to_num(facial_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     # KL loss with clipping
#     kl_loss = -0.5 * torch.mean(1 + logvar - mu**2 - torch.exp(logvar))
#     kl_loss = torch.nan_to_num(kl_loss, nan=0.0, posinf=0.0, neginf=0.0)
#     kl_loss = torch.clamp(kl_loss, min=0.0, max=10.0)  # Prevent extreme values
    
#     # Calculate total loss with edge-guided losses for better quality
#     total_loss = (lambda_adv * adv_loss + lambda_l1 * l1_loss + lambda_rec * rec_loss + lambda_perc * perc_loss +
#                   lambda_style * sty_loss + lambda_fm * fm_loss + lambda_identity * id_loss +
#                   lambda_landmark * lm_loss + lambda_struct * struct_loss + 
#                   lambda_attention * att_loss + lambda_vae * kl_loss +
#                   lambda_edge * edg_loss + lambda_hf * hf_loss + 
#                   lambda_gradient * grad_loss + lambda_facial * facial_loss)
    
#     # Final NaN check
#     total_loss = torch.nan_to_num(total_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     return total_loss
# def discriminator_loss(disc_real_logits, disc_fake_logits):
#     # Clamp logits to prevent extreme values
#     disc_real_logits = torch.clamp(disc_real_logits, min=-10, max=10)
#     disc_fake_logits = torch.clamp(disc_fake_logits, min=-10, max=10)
    
#     real_loss = torch.mean(F.relu(1.0 - disc_real_logits))
#     real_loss = torch.nan_to_num(real_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     fake_loss = torch.mean(F.relu(1.0 + disc_fake_logits))
#     fake_loss = torch.nan_to_num(fake_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     reg = lambda_d_reg * (torch.mean(disc_real_logits**2) + torch.mean(disc_fake_logits**2))
#     reg = torch.nan_to_num(reg, nan=0.0, posinf=0.0, neginf=0.0)
    
#     total_loss = real_loss + fake_loss + reg
#     total_loss = torch.nan_to_num(total_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     return total_loss
# # Evaluation functions
# def calculate_psnr_bottom_half(y_true, y_pred, mask):
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     bottom_mask = (mask == 0).float()
#     y_true_bottom = y_true * bottom_mask
#     y_pred_bottom = y_pred * bottom_mask
#     psnr_vals = []
#     for i in range(y_true.shape[0]):
#         true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         psnr_val = psnr(true_img, pred_img, data_range=1.0)
#         psnr_vals.append(psnr_val)
#     return torch.mean(torch.tensor(psnr_vals, device=device))
# def calculate_ssim_bottom_half(y_true, y_pred, mask):
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     bottom_mask = (mask == 0).float()
#     y_true_bottom = y_true * bottom_mask
#     y_pred_bottom = y_pred * bottom_mask
#     ssim_vals = []
#     for i in range(y_true.shape[0]):
#         true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         ssim_val = ssim(true_img, pred_img, channel_axis=2, data_range=1.0, win_size=7)
#         ssim_vals.append(ssim_val)
#     return torch.mean(torch.tensor(ssim_vals, device=device))
# def calculate_mse_bottom_half(y_true, y_pred, mask):
#     return bottom_half_loss(y_true, y_pred, mask)
# # Training function with graph-aware processing and balanced D/G updates
# def train_step(images, edge_generator, coarse_generator, fine_generator, discriminator, edge_optimizer, g_optimizer, d_optimizer, scaler, step_counter):
#     images = images.to(device)
#     masks = generate_adaptive_mask(images)
#     masked_images = images * masks
   
#     # Discriminator training (n_critic times more frequent)
#     d_optimizer.zero_grad()
    
#     # Train Discriminator multiple times
#     d_loss_accumulated = 0.0
#     for _ in range(n_critic):
#         with amp.autocast('cuda'):
#             # Generate fake images with multi-stage process: Edge -> Coarse -> Fine
#             with torch.no_grad():
#                 # Stage 1: Generate edge map
#                 edge_map = edge_generator(masked_images, masks)
                
#                 # Stage 2: Coarse inpainting with edge guidance
#                 coarse_output, mu, logvar = multi_resolution_inpainting(masked_images, masks, coarse_generator, edge_map)
#                 if coarse_output.shape != images.shape:
#                     coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
                
#                 # Stage 3: Fine inpainting with edge guidance
#                 fine_output = fine_generator(masked_images, coarse_output, masks, edge_map)
#                 if fine_output.shape != images.shape:
#                     fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
                
#                 combined_input = images * masks + fine_output * (1.0 - masks)
            
#             # Discriminator forward
#             disc_real_logits = discriminator(images, masks, return_features=False)
#             disc_fake_logits = discriminator(combined_input.detach(), masks, return_features=False)
#             d_loss = discriminator_loss(disc_real_logits, disc_fake_logits)
        
#         # Discriminator backward
#         scaler.scale(d_loss).backward()
#         d_loss_accumulated += d_loss.item()
    
#     # Update discriminator after n_critic iterations
#     scaler.unscale_(d_optimizer)
#     torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
#     scaler.step(d_optimizer)
    
#     d_loss_val = d_loss_accumulated / n_critic
   
#     # Generator training (once per n_critic discriminator updates)
#     g_loss_val = 0.0
#     edge_loss_val = 0.0
#     if step_counter % n_critic == 0:
#         # Train Edge Generator and Main Generators together
#         edge_optimizer.zero_grad()
#         g_optimizer.zero_grad()
        
#         with amp.autocast('cuda'):
#             # Stage 1: Generate edge map and compute edge reconstruction loss
#             edge_map = edge_generator(masked_images, masks)
            
#             # Extract true edges from original images
#             with torch.no_grad():
#                 true_edges = edge_detector_global(images * 0.5 + 0.5)
            
#             # Edge reconstruction loss (L1 loss on edge maps)
#             bottom_region = (masks == 0).float()
#             edge_recon_loss = torch.mean(torch.abs(true_edges - edge_map) * bottom_region)
            
#             # Stage 2 & 3: Generator forward pass with edge guidance
#             coarse_output, mu, logvar = multi_resolution_inpainting(masked_images, masks, coarse_generator, edge_map)
#             if coarse_output.shape != images.shape:
#                 coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
            
#             fine_output = fine_generator(masked_images, coarse_output, masks, edge_map)
#             if fine_output.shape != images.shape:
#                 fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
            
#             combined_input = images * masks + fine_output * (1.0 - masks)
#             disc_real_logits, disc_features_real = discriminator(images, masks, return_features=True)
#             disc_fake_logits, disc_features_fake = discriminator(combined_input, masks, return_features=True)
#             g_loss = generator_loss(disc_fake_logits, images, fine_output, disc_features_real, disc_features_fake, masks, mu, logvar)
        
#         # Combined backward pass (edge loss + generator loss)
#         # This avoids inplace operation errors
#         total_gen_loss = edge_recon_loss + g_loss
        
#         scaler.scale(total_gen_loss).backward()
        
#         # Unscale and clip gradients for both edge and main generators
#         scaler.unscale_(edge_optimizer)
#         scaler.unscale_(g_optimizer)
#         torch.nn.utils.clip_grad_norm_(edge_generator.parameters(), max_norm=1.0)
#         torch.nn.utils.clip_grad_norm_(list(coarse_generator.parameters()) + list(fine_generator.parameters()), max_norm=1.0)
        
#         # Update both optimizers
#         scaler.step(edge_optimizer)
#         scaler.step(g_optimizer)
        
#         edge_loss_val = edge_recon_loss.item()
#         g_loss_val = g_loss.item()
   
#     # Update scaler after both steps
#     scaler.update()
    
#     # Check for NaN in losses
#     if np.isnan(g_loss_val) or np.isinf(g_loss_val):
#         print(f"⚠️ WARNING: G_Loss is {g_loss_val}, skipping this batch")
#         g_loss_val = 0.0
        
#     if np.isnan(d_loss_val) or np.isinf(d_loss_val):
#         print(f"⚠️ WARNING: D_Loss is {d_loss_val}, skipping this batch")
#         d_loss_val = 0.0
    
#     if np.isnan(edge_loss_val) or np.isinf(edge_loss_val):
#         print(f"⚠️ WARNING: Edge_Loss is {edge_loss_val}, skipping this batch")
#         edge_loss_val = 0.0
   
#     return g_loss_val, d_loss_val, edge_loss_val
# # Add multi_resolution_inpainting function
# def multi_resolution_inpainting(image, mask, model, edge_map=None, levels=3):
#     outputs = []
#     mus = []
#     logvars = []
#     for level in range(levels):
#         scaled_image = F.interpolate(image, scale_factor=1 / (2 ** level), mode='bilinear')
#         scaled_mask = F.interpolate(mask, scale_factor=1 / (2 ** level), mode='nearest')
        
#         # Scale edge map if provided
#         if edge_map is not None:
#             scaled_edge_map = F.interpolate(edge_map, scale_factor=1 / (2 ** level), mode='bilinear')
#             out, mu, logvar = model(scaled_image, scaled_mask, scaled_edge_map)
#         else:
#             out, mu, logvar = model(scaled_image, scaled_mask)  # Call full forward
        
#         out_up = F.interpolate(out, size=image.shape[-2:], mode='bilinear')
#         outputs.append(out_up)
#         mus.append(mu)
#         logvars.append(logvar)
    
#     final_out = torch.mean(torch.stack(outputs), dim=0)
#     final_mu = torch.mean(torch.stack(mus), dim=0)
#     final_logvar = torch.mean(torch.stack(logvars), dim=0)
#     return final_out, final_mu, final_logvar

# # Add gradient checking function
# def check_model_gradients(model, model_name="Model"):
#     """Check if model has NaN or Inf gradients"""
#     has_nan = False
#     has_inf = False
#     max_grad = 0.0
    
#     for name, param in model.named_parameters():
#         if param.grad is not None:
#             if torch.isnan(param.grad).any():
#                 has_nan = True
#                 print(f"⚠️ NaN gradient in {model_name}.{name}")
#             if torch.isinf(param.grad).any():
#                 has_inf = True
#                 print(f"⚠️ Inf gradient in {model_name}.{name}")
#             max_grad = max(max_grad, param.grad.abs().max().item())
    
#     return has_nan, has_inf, max_grad

# # Add simple retrain function
# def retrain(model, pruned_graph, optimizer):
#     optimizer.zero_grad()
#     # Dummy forward (adjust to use graph if needed)
#     dummy_input = torch.randn(1, 3, 64, 64).to(device)
#     dummy_mask = torch.randn(1, 3, 64, 64).to(device)
#     output, _, _ = model(dummy_input, dummy_mask)
#     loss = torch.mean(output)
#     loss.backward()
#     optimizer.step()
#     return loss.item()

# # Integrate into training loop
# # In train_step function, after coarse_output:
# # Add pruning and retraining every 10 epochs
# # Load dataset

# # celeba_dir = '/kaggle/input/celeba-resized-6464/img_align_celeba/CelebA_Image_Cropped_64'
# celeba_dir = '/kaggle/input/celeba-6464/selected_celeba'

# print(f"Loading dataset from {celeba_dir}...")
# try:
#     print(f"Dataset directory contents: {os.listdir(celeba_dir)[:5]}")
# except Exception as e:
#     print(f"Error accessing dataset directory: {e}")
#     raise
# # Split dataset
# train_files, val_files, test_files = split_dataset(celeba_dir)
# # Create datasets
# train_loader, val_loader, test_loader = create_split_datasets(celeba_dir, train_files, val_files, test_files, image_size, batch_size)
# # Build models (now with Edge Generator for multi-stage inpainting)
# print("Building models...")
# print("🔷 Multi-Stage Architecture: Edge → Coarse → Fine")
# edge_generator = EdgeGenerator().to(device)
# coarse_generator = CoarseGenerator().to(device)
# fine_generator = FineGenerator().to(device)
# discriminator = Discriminator().to(device)
# print(f"Edge Generator: {sum(p.numel() for p in edge_generator.parameters()):,} parameters")
# print(f"Coarse Generator: {sum(p.numel() for p in coarse_generator.parameters()):,} parameters")
# print(f"Fine Generator: {sum(p.numel() for p in fine_generator.parameters()):,} parameters")
# print(f"Discriminator: {sum(p.numel() for p in discriminator.parameters()):,} parameters")
# # Test model dimensions
# print("\nTesting model dimensions...")
# test_input = torch.randn(1, 3, 64, 64).to(device)
# test_mask = torch.randn(1, 3, 64, 64).to(device)
# with torch.no_grad():
#     coarse_output, mu, logvar = coarse_generator(test_input, test_mask)
#     print(f"Coarse output shape: {coarse_output.shape}")
#     fine_output = fine_generator(test_input, coarse_output, test_mask)
#     print(f"Fine output shape: {fine_output.shape}")
#     print(f"Expected shape: {test_input.shape}")
   
#     # Test discriminator
#     disc_logits, disc_features = discriminator(test_input, test_mask, return_features=True)
#     print(f"Discriminator logits shape: {disc_logits.shape}")
#     print(f"Discriminator features length: {len(disc_features)}")
   
#     disc_logits_only = discriminator(test_input, test_mask, return_features=False)
#     print(f"Discriminator logits only shape: {disc_logits_only.shape}")
   
#     print("✓ Model dimensions are correct!")
# # Enhanced Optimizers with Learning Rate Scheduling and beta1=0.5 for GAN stability
# # NEW: Added Edge Generator optimizer
# edge_optimizer = torch.optim.AdamW(
#     edge_generator.parameters(),
#     lr=learning_rate_g, betas=(0.5, 0.999), weight_decay=1e-4
# )
# g_optimizer = torch.optim.AdamW(
#     list(coarse_generator.parameters()) + list(fine_generator.parameters()),
#     lr=learning_rate_g, betas=(0.5, 0.999), weight_decay=1e-4
# )
# d_optimizer = torch.optim.AdamW(
#     discriminator.parameters(),
#     lr=learning_rate_d, betas=(0.5, 0.999), weight_decay=1e-4
# )
# print(f"Optimizer settings: G_LR={learning_rate_g}, D_LR={learning_rate_d}, beta1=0.5, n_critic={n_critic}")
# print(f"✅ Edge Generator optimizer added for multi-stage training")
# # Learning rate schedulers (NEW: added for Edge Generator)
# edge_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     edge_optimizer, T_0=10, T_mult=2, eta_min=learning_rate_g * 0.01
# )
# g_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     g_optimizer, T_0=10, T_mult=2, eta_min=learning_rate_g * 0.01
# )
# d_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     d_optimizer, T_0=10, T_mult=2, eta_min=learning_rate_d * 0.01
# )
# scaler = amp.GradScaler('cuda')
# # Model health check before training
# print("\n" + "="*60)
# print("🔍 Model Health Check")
# print("="*60)
# test_batch = next(iter(train_loader)).to(device)
# test_masks = generate_adaptive_mask(test_batch)
# test_masked = test_batch * test_masks

# print("Testing forward pass...")
# with torch.no_grad():
#     try:
#         coarse_out, mu, logvar = coarse_generator(test_masked, test_masks)
#         print(f"✓ Coarse Generator: output shape {coarse_out.shape}")
#         print(f"  mu range: [{mu.min().item():.4f}, {mu.max().item():.4f}]")
#         print(f"  logvar range: [{logvar.min().item():.4f}, {logvar.max().item():.4f}]")
        
#         fine_out = fine_generator(test_masked, coarse_out, test_masks)
#         print(f"✓ Fine Generator: output shape {fine_out.shape}")
#         print(f"  output range: [{fine_out.min().item():.4f}, {fine_out.max().item():.4f}]")
        
#         disc_out = discriminator(test_batch, test_masks)
#         print(f"✓ Discriminator: logits shape {disc_out.shape}")
#         print(f"  logits range: [{disc_out.min().item():.4f}, {disc_out.max().item():.4f}]")
        
#         print("✅ All models are healthy!")
#     except Exception as e:
#         print(f"❌ Model health check failed: {e}")
#         raise
# print("="*60 + "\n")

# # ============================================================================
# # CHECKPOINT LOADING & TRAINING SETUP
# # بارگذاری Checkpoint و راه‌اندازی آموزش
# # ============================================================================

# print("\n" + "="*70)
# print("🚀 STARTING TRAINING SETUP")
# print("="*70)

# # Adjust paths if loading from Kaggle input dataset
# if USE_KAGGLE_INPUT:
#     checkpoint_load_dir = os.path.join(KAGGLE_INPUT_DATASET, 'checkpoints')
#     models_load_dir = os.path.join(KAGGLE_INPUT_DATASET, 'models')
#     print(f"📥 Loading from Kaggle dataset: {KAGGLE_INPUT_DATASET}")
# else:
#     checkpoint_load_dir = CHECKPOINT_DIR
#     models_load_dir = MODELS_DIR
#     print(f"📥 Loading from working directory")

# # Create checkpoint save directory (always in /kaggle/working)
# os.makedirs(CHECKPOINT_DIR, exist_ok=True)
# os.makedirs(MODELS_DIR, exist_ok=True)
# print(f"💾 Checkpoints will be saved to: {CHECKPOINT_DIR}")
# print(f"💾 Models will be saved to: {MODELS_DIR}")

# # Initialize training variables
# start_epoch = 0
# best_val_loss = float('inf')
# train_g_losses = []
# train_d_losses = []
# val_g_losses = []
# val_d_losses = []

# # Try to load checkpoint/models if RESUME_TRAINING is True
# checkpoint_loaded = False
# models_loaded = False

# if RESUME_TRAINING:
#     # ========================================================================
#     # Method 1: Try to load from MODEL FILES (state_dict only)
#     # روش 1: بارگذاری از فایل‌های MODEL (فقط state_dict)
#     # ========================================================================
    
#     if LOAD_FROM_MODELS and not checkpoint_loaded:
#         print(f"\n🔍 Method 1: Searching for model files...")
#         print(f"📂 Models directory: {models_load_dir}")
        
#         # Debug: Show what files actually exist
#         if os.path.exists(models_load_dir):
#             print(f"\n📄 Files found in directory:")
#             actual_files = os.listdir(models_load_dir)
#             if actual_files:
#                 for f in sorted(actual_files):
#                     if f.endswith('.pth'):
#                         file_path = os.path.join(models_load_dir, f)
#                         size_mb = os.path.getsize(file_path) / (1024**2)
#                         print(f"   • {f} ({size_mb:.2f} MB)")
#             else:
#                 print(f"   (empty)")
#         else:
#             print(f"   ❌ Directory does not exist!")
        
#         # Determine which models to load
#         suffix = "_best.pth" if USE_BEST_MODELS else "_bottom_half.pth"
        
#         model_files = {
#             'coarse': os.path.join(models_load_dir, f'coarse_generator{suffix}'),
#             'fine': os.path.join(models_load_dir, f'fine_generator{suffix}'),
#             'disc': os.path.join(models_load_dir, f'discriminator{suffix}')
#         }
        
#         print(f"\n🔍 Looking for (USE_BEST_MODELS={USE_BEST_MODELS}):")
#         for name, path in model_files.items():
#             exists = "✅" if os.path.exists(path) else "❌"
#             print(f"   {exists} {os.path.basename(path)}")
        
#         # Check if all model files exist
#         all_exist = all(os.path.exists(path) for path in model_files.values())
        
#         if all_exist:
#             try:
#                 print(f"📂 Found model files in: {models_load_dir}")
#                 print(f"   • coarse_generator{suffix}")
#                 print(f"   • fine_generator{suffix}")
#                 print(f"   • discriminator{suffix}")
#                 print(f"⏳ Loading model weights...")
                
#                 # Load model weights
#                 # Load model weights tolerantly in case of architecture drift
#                 coarse_sd = torch.load(model_files['coarse'], map_location=device)
#                 fine_sd = torch.load(model_files['fine'], map_location=device)
#                 disc_sd = torch.load(model_files['disc'], map_location=device)

#                 missing, unexpected = coarse_generator.load_state_dict(coarse_sd, strict=False)
#                 if missing or unexpected:
#                     print("[coarse_generator] non-strict load:")
#                     if missing:
#                         print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                     if unexpected:
#                         print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                 missing, unexpected = fine_generator.load_state_dict(fine_sd, strict=False)
#                 if missing or unexpected:
#                     print("[fine_generator] non-strict load:")
#                     if missing:
#                         print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                     if unexpected:
#                         print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                 missing, unexpected = discriminator.load_state_dict(disc_sd, strict=False)
#                 if missing or unexpected:
#                     print("[discriminator] non-strict load:")
#                     if missing:
#                         print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                     if unexpected:
#                         print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
                
#                 models_loaded = True
                
#                 print("="*70)
#                 print("✅ MODEL WEIGHTS LOADED SUCCESSFULLY!")
#                 print("="*70)
#                 print(f"📊 Loading Information:")
#                 print(f"   • Model weights: ✅ Loaded")
#                 print(f"   • Optimizer states: ⚠️  Will be initialized (not in model files)")
#                 print(f"   • Scheduler states: ⚠️  Will be initialized (not in model files)")
#                 print(f"   • Training history: ⚠️  Starting fresh (not in model files)")
#                 print(f"   • Starting epoch: 1 (training continues with loaded weights)")
#                 print(f"\n⚠️  NOTE: Model files only contain weights, not optimizer/scheduler.")
#                 print(f"   Training will continue with these weights but fresh optimizer.")
#                 print("="*70)
                
#             except Exception as e:
#                 print(f"⚠️  Failed to load model files")
#                 print(f"   Error: {str(e)}")
#                 models_loaded = False
#         else:
#             print(f"⚠️  Not all model files found in {models_load_dir}")
#             for name, path in model_files.items():
#                 exists = "✓" if os.path.exists(path) else "✗"
#                 print(f"   {exists} {os.path.basename(path)}")
    
#     # ========================================================================
#     # Method 2: Try to load from CHECKPOINT FILES (full checkpoint)
#     # روش 2: بارگذاری از فایل‌های CHECKPOINT (checkpoint کامل)
#     # ========================================================================
    
#     if not checkpoint_loaded and not models_loaded:
#         print(f"\n🔍 Method 2: Searching for checkpoint files...")
        
#         # Try different checkpoint sources
#         checkpoint_paths_to_try = [
#             os.path.join(checkpoint_load_dir, 'latest_checkpoint.pth'),
#             os.path.join(checkpoint_load_dir, f'checkpoint_epoch_{num_epochs}.pth'),
#             os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth'),  # Fallback to working dir
#         ]
        
#         for checkpoint_path in checkpoint_paths_to_try:
#             if os.path.exists(checkpoint_path):
#                 try:
#                     print(f"📂 Found checkpoint: {checkpoint_path}")
#                     print(f"⏳ Loading checkpoint...")
                    
#                     checkpoint = torch.load(checkpoint_path, map_location=device)
                    
#                     # Load model states tolerantly (including edge_generator if available)
#                     if 'edge_generator' in checkpoint:
#                         missing, unexpected = edge_generator.load_state_dict(checkpoint['edge_generator'], strict=False)
#                         if missing or unexpected:
#                             print("[edge_generator ckpt] non-strict load:")
#                             if missing:
#                                 print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                             if unexpected:
#                                 print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
#                     else:
#                         print("[edge_generator] not found in checkpoint - using random initialization")
                    
#                     missing, unexpected = coarse_generator.load_state_dict(checkpoint['coarse_generator'], strict=False)
#                     if missing or unexpected:
#                         print("[coarse_generator ckpt] non-strict load:")
#                         if missing:
#                             print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                         if unexpected:
#                             print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                     missing, unexpected = fine_generator.load_state_dict(checkpoint['fine_generator'], strict=False)
#                     if missing or unexpected:
#                         print("[fine_generator ckpt] non-strict load:")
#                         if missing:
#                             print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                         if unexpected:
#                             print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                     missing, unexpected = discriminator.load_state_dict(checkpoint['discriminator'], strict=False)
#                     if missing or unexpected:
#                         print("[discriminator ckpt] non-strict load:")
#                         if missing:
#                             print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                         if unexpected:
#                             print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
                    
#                     # Load optimizer states
#                     g_optimizer.load_state_dict(checkpoint['g_optimizer'])
#                     d_optimizer.load_state_dict(checkpoint['d_optimizer'])
                    
#                     # Load scheduler states
#                     g_scheduler.load_state_dict(checkpoint['g_scheduler'])
#                     d_scheduler.load_state_dict(checkpoint['d_scheduler'])
                    
#                     # Load training progress
#                     start_epoch = checkpoint['epoch'] + 1
#                     best_val_loss = checkpoint.get('best_val_loss', float('inf'))
#                     train_g_losses = checkpoint.get('train_g_losses', [])
#                     train_d_losses = checkpoint.get('train_d_losses', [])
#                     val_g_losses = checkpoint.get('val_g_losses', [])
#                     val_d_losses = checkpoint.get('val_d_losses', [])
                    
#                     checkpoint_loaded = True
                    
#                     print("="*70)
#                     print("✅ FULL CHECKPOINT LOADED SUCCESSFULLY!")
#                     print("="*70)
#                     print(f"📊 Resume Information:")
#                     print(f"   • Model weights: ✅ Loaded")
#                     print(f"   • Optimizer states: ✅ Loaded")
#                     print(f"   • Scheduler states: ✅ Loaded")
#                     print(f"   • Starting from epoch: {start_epoch + 1}")
#                     print(f"   • Best validation loss: {best_val_loss:.6f}")
#                     print(f"   • Training history: {len(train_g_losses)} epochs")
#                     print(f"   • Remaining epochs: {num_epochs - start_epoch}")
#                     print("="*70)
                    
#                     break  # Successfully loaded, exit loop
                    
#                 except Exception as e:
#                     print(f"⚠️  Failed to load checkpoint from {checkpoint_path}")
#                     print(f"   Error: {str(e)}")
#                     print(f"   Trying next checkpoint source...")
#                     continue

# # ========================================================================
# # If nothing loaded, start fresh
# # اگر چیزی بارگذاری نشد، از اول شروع کن
# # ========================================================================

# if not checkpoint_loaded and not models_loaded:
#     if RESUME_TRAINING:
#         print("\n" + "="*70)
#         print("⚠️  NO CHECKPOINT OR MODELS FOUND - Starting from scratch")
#         print("="*70)
#         print("📝 Locations searched:")
#         print(f"\n   Models folder: {models_load_dir}")
#         if os.path.exists(models_load_dir):
#             files = os.listdir(models_load_dir)
#             if files:
#                 print(f"   Found {len(files)} file(s):")
#                 for f in files[:5]:  # Show first 5
#                     print(f"      • {f}")
#             else:
#                 print(f"      (empty)")
#         else:
#             print(f"      (not found)")
        
#         print(f"\n   Checkpoints folder: {checkpoint_load_dir}")
#         if os.path.exists(checkpoint_load_dir):
#             files = [f for f in os.listdir(checkpoint_load_dir) if f.endswith('.pth')]
#             if files:
#                 print(f"   Found {len(files)} checkpoint(s):")
#                 for f in files[:5]:
#                     print(f"      • {f}")
#             else:
#                 print(f"      (empty)")
#         else:
#             print(f"      (not found)")
        
#         print("\n💡 To resume training in next run:")
#         print("   1. Make sure model/checkpoint files exist")
#         print("   2. Set LOAD_FROM_MODELS = True for model files")
#         print("   3. If using Kaggle dataset, set USE_KAGGLE_INPUT = True")
#         print("   4. Update MODELS_DIR or KAGGLE_INPUT_DATASET path")
#         print("="*70)
#     else:
#         print("\n" + "="*70)
#         print("🆕 STARTING FRESH TRAINING")
#         print("="*70)
#         print("   RESUME_TRAINING is set to False")
#         print("   Training will start from epoch 1")
#         print("="*70)

# print(f"\n🎯 Training will run from epoch {start_epoch + 1} to {num_epochs}")
# print(f"💾 Checkpoints will be saved every {CHECKPOINT_INTERVAL} epoch(s)")
# print("="*70 + "\n")

# for epoch in range(start_epoch, num_epochs):
#     print(f"Epoch {epoch+1}/{num_epochs}")
#     edge_generator.train()
#     coarse_generator.train()
#     fine_generator.train()
#     discriminator.train()
#     epoch_g_loss = epoch_d_loss = epoch_edge_loss = num_batches = 0
   
#     for step, images in enumerate(train_loader):
#         # Pass step counter for balanced D/G updates (NEW: added edge_generator and edge_optimizer)
#         g_loss_batch, d_loss_batch, edge_loss_batch = train_step(images, edge_generator, coarse_generator, fine_generator, discriminator, edge_optimizer, g_optimizer, d_optimizer, scaler, step)
#         epoch_g_loss += g_loss_batch
#         epoch_edge_loss += edge_loss_batch
#         epoch_d_loss += d_loss_batch
#         num_batches += 1
        
#         # Enhanced monitoring
#         if step % 50 == 0:
#             current_g_lr = g_optimizer.param_groups[0]['lr']
#             current_d_lr = d_optimizer.param_groups[0]['lr']
#             current_edge_lr = edge_optimizer.param_groups[0]['lr']
#             g_or_d = "E+G+D" if step % n_critic == 0 else "D only"
#             print(f" Step {step:3d} [{g_or_d}]: Edge_Loss: {edge_loss_batch:.4f}, G_Loss: {g_loss_batch:.4f}, D_Loss: {d_loss_batch:.4f} | LR: E={current_edge_lr:.6f}, G={current_g_lr:.6f}, D={current_d_lr:.6f}")
            
#         # Emergency stop if losses explode
#         if np.isnan(g_loss_batch) and np.isnan(d_loss_batch):
#             print(f"⚠️ CRITICAL: Both losses are NaN at step {step}. Stopping epoch early.")
#             break
   
#     avg_g_loss = epoch_g_loss / num_batches if num_batches > 0 else 0
#     avg_d_loss = epoch_d_loss / num_batches if num_batches > 0 else 0
#     avg_edge_loss = epoch_edge_loss / num_batches if num_batches > 0 else 0
#     train_g_losses.append(avg_g_loss)
#     train_d_losses.append(avg_d_loss)
#     print(f"Epoch {epoch+1} Training - Edge_Loss: {avg_edge_loss:.4f}, G_Loss: {avg_g_loss:.4f}, D_Loss: {avg_d_loss:.4f}")
   
#     # Update learning rates (NEW: added edge_scheduler)
#     edge_scheduler.step()
#     g_scheduler.step()
#     d_scheduler.step()
#     current_g_lr = g_optimizer.param_groups[0]['lr']
#     current_d_lr = d_optimizer.param_groups[0]['lr']
#     print(f" Current LR - G: {current_g_lr:.6f}, D: {current_d_lr:.6f}")
   
#     # Enable MTCNN for validation
#     if 'MTCNN' in globals():
#         USE_MTCNN = True
   
#     if epoch % 10 == 0:
#         # Get sample batch for graph creation
#         sample_images = next(iter(train_loader)).to(device)
#         sample_masks = generate_adaptive_mask(sample_images)
#         sample_masked = sample_images * sample_masks
        
#         # Get features from preprocessor
#         features = coarse_generator.preprocessor(sample_masked)
        
#         # Build graph
#         sample_graph = coarse_generator.preprocessor.build_graph(features)
        
#         pruned_graph = prune_graph(coarse_generator, sample_graph)
        
#         # Retrain
#         for _ in range(5):
#             retrain_loss = retrain(coarse_generator, pruned_graph, g_optimizer)
   
#     edge_generator.eval()
#     coarse_generator.eval()
#     fine_generator.eval()
#     discriminator.eval()
#     val_g_loss = val_d_loss = val_batches = 0
#     with torch.no_grad():
#         for images in val_loader:
#             images = images.to(device)
#             masks = generate_adaptive_mask(images)
#             masked_images = images * masks
            
#             # Multi-stage: Edge -> Coarse -> Fine
#             edge_map = edge_generator(masked_images, masks)
#             coarse_output, mu, logvar = coarse_generator(masked_images, masks, edge_map)
#             # Debug: Check shapes
#             if coarse_output.shape != images.shape:
#                 coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
           
#             fine_output = fine_generator(masked_images, coarse_output, masks, edge_map)
#             # Debug: Check shapes
#             if fine_output.shape != images.shape:
#                 fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
           
#             combined_input = images * masks + fine_output * (1.0 - masks)
           
#             disc_real_logits, disc_features_real = discriminator(images, masks, return_features=True)
#             disc_fake_logits, disc_features_fake = discriminator(combined_input, masks, return_features=True)
           
#             g_loss = generator_loss(disc_fake_logits, images, fine_output, disc_features_real, disc_features_fake, masks, mu, logvar)
#             d_loss = discriminator_loss(disc_real_logits, disc_fake_logits)
           
#             val_g_loss += g_loss.item()
#             val_d_loss += d_loss.item()
#             val_batches += 1
   
#     avg_val_g_loss = val_g_loss / val_batches if val_batches > 0 else 0
#     avg_val_d_loss = val_d_loss / val_batches if val_batches > 0 else 0
#     val_g_losses.append(avg_val_g_loss)
#     val_d_losses.append(avg_val_d_loss)
#     print(f"Epoch {epoch+1} Validation - G_Loss: {avg_val_g_loss:.4f}, D_Loss: {avg_val_d_loss:.4f}")
   
#     if avg_val_g_loss < best_val_loss:
#         best_val_loss = avg_val_g_loss
#         print(f" New best validation loss! Saving best models...")
#         os.makedirs('/kaggle/working/models', exist_ok=True)
#         torch.save(edge_generator.state_dict(), "/kaggle/working/models/edge_generator_best.pth")
#         torch.save(coarse_generator.state_dict(), "/kaggle/working/models/coarse_generator_best.pth")
#         torch.save(fine_generator.state_dict(), "/kaggle/working/models/fine_generator_best.pth")
#         torch.save(discriminator.state_dict(), "/kaggle/working/models/discriminator_best.pth")
   
#     # Save checkpoint every CHECKPOINT_INTERVAL epochs
#     if (epoch + 1) % CHECKPOINT_INTERVAL == 0 or (epoch + 1) == num_epochs:
#         print(f"\n{'='*70}")
#         print(f"💾 SAVING CHECKPOINT - Epoch {epoch+1}/{num_epochs}")
#         print(f"{'='*70}")
        
#         checkpoint_file = os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch+1}.pth')
#         latest_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth')
        
#         checkpoint = {
#             'epoch': epoch,
#             'edge_generator': edge_generator.state_dict(),
#             'coarse_generator': coarse_generator.state_dict(),
#             'fine_generator': fine_generator.state_dict(),
#             'discriminator': discriminator.state_dict(),
#             'edge_optimizer': edge_optimizer.state_dict(),
#             'g_optimizer': g_optimizer.state_dict(),
#             'd_optimizer': d_optimizer.state_dict(),
#             'edge_scheduler': edge_scheduler.state_dict(),
#             'g_scheduler': g_scheduler.state_dict(),
#             'd_scheduler': d_scheduler.state_dict(),
#             'best_val_loss': best_val_loss,
#             'train_g_losses': train_g_losses,
#             'train_d_losses': train_d_losses,
#             'val_g_losses': val_g_losses,
#             'val_d_losses': val_d_losses,
#         }
        
#         # Save numbered checkpoint
#         torch.save(checkpoint, checkpoint_file)
#         checkpoint_size = os.path.getsize(checkpoint_file) / (1024**2)  # MB
#         print(f"✅ Saved: {checkpoint_file}")
#         print(f"   Size: {checkpoint_size:.2f} MB")
        
#         # Also save as latest checkpoint
#         torch.save(checkpoint, latest_checkpoint_path)
#         print(f"✅ Saved: {latest_checkpoint_path}")
        
#         print(f"\n📊 Checkpoint Info:")
#         print(f"   • Completed epochs: {epoch + 1}")
#         print(f"   • Best val loss: {best_val_loss:.6f}")
#         print(f"   • Current G loss: {avg_g_loss:.6f}")
#         print(f"   • Current D loss: {avg_d_loss:.6f}")
        
#         if (epoch + 1) < num_epochs:
#             print(f"\n🔄 To resume from this checkpoint in next run:")
#             print(f"   1. Download '{CHECKPOINT_DIR}' folder from Kaggle Output")
#             print(f"   2. Upload as Kaggle Dataset (or keep in working directory)")
#             print(f"   3. In code, set: RESUME_TRAINING = True")
#             print(f"   4. If using dataset, set: USE_KAGGLE_INPUT = True")
#             print(f"   5. Update: KAGGLE_INPUT_DATASET = '/kaggle/input/your-dataset-name'")
#         else:
#             print(f"\n🎉 TRAINING COMPLETED!")
#             print(f"   All {num_epochs} epochs finished successfully!")
        
#         print(f"{'='*70}\n")
   
#     # Plot training progress every 5 epochs
#     if (epoch + 1) % 5 == 0:
#         plt.figure(figsize=(15, 5))
       
#         plt.subplot(1, 3, 1)
#         plt.plot(range(1, len(train_g_losses) + 1), train_g_losses, label='Generator Loss', color='blue')
#         plt.plot(range(1, len(val_g_losses) + 1), val_g_losses, label='Val Generator Loss', color='lightblue')
#         plt.title('Generator Loss Progress')
#         plt.xlabel('Epoch')
#         plt.ylabel('Loss')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
       
#         plt.subplot(1, 3, 2)
#         plt.plot(range(1, len(train_d_losses) + 1), train_d_losses, label='Discriminator Loss', color='red')
#         plt.plot(range(1, len(val_d_losses) + 1), val_d_losses, label='Val Discriminator Loss', color='lightcoral')
#         plt.title('Discriminator Loss Progress')
#         plt.xlabel('Epoch')
#         plt.ylabel('Loss')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
       
#         plt.subplot(1, 3, 3)
#         plt.plot(range(1, len(train_g_losses) + 1), train_g_losses, label='G Loss', color='blue')
#         plt.plot(range(1, len(train_d_losses) + 1), train_d_losses, label='D Loss', color='red')
#         plt.title('Training Loss Comparison')
#         plt.xlabel('Epoch')
#         plt.ylabel('Loss')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
       
#         plt.tight_layout()
#         plt.savefig(f'/kaggle/working/training_progress_epoch_{epoch+1}.png', dpi=150, bbox_inches='tight')
#         plt.show()
# # Save final models
# print("Saving final models...")
# os.makedirs('/kaggle/working/models', exist_ok=True)
# torch.save(edge_generator.state_dict(), "/kaggle/working/models/edge_generator_bottom_half.pth")
# torch.save(coarse_generator.state_dict(), "/kaggle/working/models/coarse_generator_bottom_half.pth")
# torch.save(fine_generator.state_dict(), "/kaggle/working/models/fine_generator_bottom_half.pth")
# torch.save(discriminator.state_dict(), "/kaggle/working/models/discriminator_bottom_half.pth")
# # Enable MTCNN for evaluation
# if 'MTCNN' in globals():
#     USE_MTCNN = True
# # Evaluate on test set
# print("Starting evaluation on test set...")
# os.makedirs('/kaggle/working/results', exist_ok=True)
# edge_generator.eval()
# coarse_generator.eval()
# fine_generator.eval()
# discriminator.eval()
# psnr_values, ssim_values, mse_values, identity_values, landmark_values = [], [], [], [], []
# num_samples = 10 # Increased for better evaluation
# plt.figure(figsize=(20, 10))
# with torch.no_grad():
#     for i, images in enumerate(test_loader):
#         if i >= num_samples:
#             break
#         images = images.to(device)
#         masks = generate_adaptive_mask(images)
#         masked_images = images * masks
        
#         # Multi-stage: Edge -> Coarse -> Fine
#         edge_map = edge_generator(masked_images, masks)
#         coarse_output, mu, logvar = coarse_generator(masked_images, masks, edge_map)
#         # Debug: Check shapes
#         if coarse_output.shape != images.shape:
#             coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
       
#         fine_output = fine_generator(masked_images, coarse_output, masks, edge_map)
#         # Debug: Check shapes
#         if fine_output.shape != images.shape:
#             fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
       
#         reconstructed = images * masks + fine_output * (1.0 - masks)
       
#         psnr_val = calculate_psnr_bottom_half(images, reconstructed, masks).item()
#         ssim_val = calculate_ssim_bottom_half(images, reconstructed, masks).item()
#         mse_val = calculate_mse_bottom_half(images, reconstructed, masks).item()
#         identity_val = identity_loss(images, reconstructed, masks).item()
#         landmark_val = landmark_guided_loss(images, reconstructed, masks).item()
       
#         psnr_values.append(psnr_val)
#         ssim_values.append(ssim_val)
#         mse_values.append(mse_val)
#         identity_values.append(identity_val)
#         landmark_values.append(landmark_val)
       
#         plt.subplot(4, num_samples, i + 1)
#         plt.title("Original Image")
#         plt.imshow(images[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#         plt.axis("off")
       
#         plt.subplot(4, num_samples, num_samples + i + 1)
#         plt.title("Top Half (Input)")
#         plt.imshow(masked_images[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#         plt.axis("off")
       
#         plt.subplot(4, num_samples, 2*num_samples + i + 1)
#         plt.title("Reconstructed Bottom Half")
#         plt.imshow(reconstructed[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#         plt.axis("off")
       
#         diff = torch.abs(images - reconstructed) * (1 - masks)
#         plt.subplot(4, num_samples, 3*num_samples + i + 1)
#         plt.title("Difference Map")
#         plt.imshow(diff[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5, cmap='hot')
#         plt.axis("off")
       
#         print(f"Sample {i+1}: PSNR: {psnr_val:.4f}, SSIM: {ssim_val:.4f}, MSE: {mse_val:.4f}, Identity: {identity_val:.4f}, Landmark: {landmark_val:.4f}")
# plt.tight_layout()
# plt.savefig('/kaggle/working/results/bottom_half_reconstruction_results.png', dpi=150, bbox_inches='tight')
# plt.show()
# # Plot metrics
# plt.figure(figsize=(20, 10))
# plt.subplot(2, 3, 1)
# plt.plot(range(1, num_samples + 1), psnr_values, marker='o', linewidth=2, markersize=8)
# plt.title("PSNR (Bottom Half)", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 2)
# plt.plot(range(1, num_samples + 1), ssim_values, marker='s', linewidth=2, markersize=8, color='orange')
# plt.title("SSIM (Bottom Half)", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 3)
# plt.plot(range(1, num_samples + 1), mse_values, marker='^', linewidth=2, markersize=8, color='green')
# plt.title("MSE (Bottom Half)", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 4)
# plt.plot(range(1, num_samples + 1), identity_values, marker='d', linewidth=2, markersize=8, color='purple')
# plt.title("Identity Loss", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 5)
# plt.plot(range(1, num_samples + 1), landmark_values, marker='*', linewidth=2, markersize=8, color='brown')
# plt.title("Landmark Loss", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.savefig('/kaggle/working/results/bottom_half_metrics.png', dpi=150, bbox_inches='tight')
# plt.show()
# # Print and save evaluation results
# print("\n" + "="*60)
# print("Enhanced Evaluation Metrics (Bottom Half) - Test Set:")
# print("="*60)
# print(f"Average PSNR: {np.mean(psnr_values):.4f} dB (±{np.std(psnr_values):.4f})")
# print(f"Average SSIM: {np.mean(ssim_values):.4f} (±{np.std(ssim_values):.4f})")
# print(f"Average MSE: {np.mean(mse_values):.4f} (±{np.std(mse_values):.4f})")
# print(f"Average Identity Loss: {np.mean(identity_values):.4f} (±{np.std(identity_values):.4f})")
# print(f"Average Landmark Loss: {np.mean(landmark_values):.4f} (±{np.std(landmark_values):.4f})")
# print("="*60)
# print(f"Best PSNR: {np.max(psnr_values):.4f} dB")
# print(f"Best SSIM: {np.max(ssim_values):.4f}")
# print(f"Worst PSNR: {np.min(psnr_values):.4f} dB")
# print(f"Worst SSIM: {np.min(ssim_values):.4f}")
# print("="*60)
# with open('/kaggle/working/results/enhanced_bottom_half_metrics.txt', 'w', encoding='utf-8') as f:
#     f.write("Enhanced Bottom Half Face Reconstruction Evaluation Results - Test Set\n")
#     f.write("="*60 + "\n")
#     f.write(f"Average PSNR: {np.mean(psnr_values):.4f} dB (±{np.std(psnr_values):.4f})\n")
#     f.write(f"Average SSIM: {np.mean(ssim_values):.4f} (±{np.std(ssim_values):.4f})\n")
#     f.write(f"Average MSE: {np.mean(mse_values):.4f} (±{np.std(mse_values):.4f})\n")
#     f.write(f"Average Identity Loss: {np.mean(identity_values):.4f} (±{np.std(identity_values):.4f})\n")
#     f.write(f"Average Landmark Loss: {np.mean(landmark_values):.4f} (±{np.std(landmark_values):.4f})\n")
#     f.write("="*60 + "\n")
#     f.write(f"Best PSNR: {np.max(psnr_values):.4f} dB\n")
#     f.write(f"Best SSIM: {np.max(ssim_values):.4f}\n")
#     f.write(f"Worst PSNR: {np.min(psnr_values):.4f} dB\n")
#     f.write(f"Worst SSIM: {np.min(ssim_values):.4f}\n")
#     f.write("="*60 + "\n")
#     for i, (psnr_val, ssim_val, mse_val, id_val, lm_val) in enumerate(zip(psnr_values, ssim_values, mse_values, identity_values, landmark_values)):
#         f.write(f"Sample {i+1}: PSNR={psnr_val:.4f}, SSIM={ssim_val:.4f}, MSE={mse_val:.4f}, Identity={id_val:.4f}, Landmark={lm_val:.4f}\n")
# print(f"\nTraining and evaluation complete!")
# print(f"Models saved in '/kaggle/working/models/' directory.")
# print(f"Evaluation results saved in '/kaggle/working/results/' directory.")

In [ ]:
# import os
# os.environ.setdefault('TORCH_CUDA_ARCH_LIST', '7.5') # For Kaggle GPU compatibility
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torchvision.models as models
# import torchvision.transforms as transforms
# from torch.utils.data import Dataset, DataLoader
# import matplotlib.pyplot as plt
# import numpy as np
# from skimage.metrics import peak_signal_noise_ratio as psnr
# from skimage.metrics import structural_similarity as ssim
# import random
# import cv2
# from PIL import Image
# from torch import amp  # For mixed precision training (new API)
# import networkx as nx  # For graph operations

# # Simplified GNN for graph processing
# class PyramidalGNN(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super(PyramidalGNN, self).__init__()
#         self.lin = nn.Linear(in_channels, out_channels)
#         self.gate = nn.Linear(in_channels * 2, out_channels)

#     def forward(self, x, edge_index):
#         # Simplified version - just apply linear transformation
#         return self.lin(x)



# # /kaggle/input/celeba-resized-6464/img_align_celeba/CelebA_Image_Cropped_64
# # Configuration parameters
# prune_threshold = 0.2  # Threshold for pruning low-value nodes
# lambda_l1 = 100.0  # L1 reconstruction loss weight
# n_critic = 5  # Number of discriminator updates per generator update

# # Global prune_graph function using NetworkX
# def prune_graph(model, graph, threshold=prune_threshold):
#     # NetworkX pruning
#     deg_cent = nx.degree_centrality(graph)
#     low_value_nodes = [n for n, score in deg_cent.items() if score < threshold]
#     graph.remove_nodes_from(low_value_nodes)
#     return graph

# # Try to import MTCNN for landmark detection
# try:
#     from mtcnn import MTCNN
#     USE_MTCNN = False # Disable during training to avoid NumPy conversions
#     print("MTCNN imported successfully for landmark detection (disabled for training).")
# except ImportError:
#     print("MTCNN not found. Landmark loss will be simplified.")
#     USE_MTCNN = False
# # ============================================================================
# # CHECKPOINT CONFIGURATION - Resume Training Setup
# # تنظیمات Checkpoint - برای ادامه آموزش
# # ============================================================================

# # 🔄 RESUME TRAINING: Set to True to continue from previous checkpoint
# # ادامه آموزش: برای ادامه از checkpoint قبلی، True کنید
# RESUME_TRAINING = True  # True: Resume from checkpoint | False: Start from scratch

# # 📂 CHECKPOINT PATHS: Where to load/save checkpoints
# # مسیرهای Checkpoint: محل بارگذاری/ذخیره checkpoint‌ها
# CHECKPOINT_DIR = '/kaggle/working/checkpoints'
# CHECKPOINT_INTERVAL = 1  # Save checkpoint every N epochs (1 = every epoch)

# # 📥 LOAD FROM KAGGLE DATASET: If you uploaded models as Kaggle dataset
# # بارگذاری از Kaggle Dataset: اگر مدل‌ها را به عنوان dataset آپلود کرده‌اید
# USE_KAGGLE_INPUT = True  # Set True if loading from Kaggle dataset input
# KAGGLE_INPUT_DATASET = '/kaggle/input/image-inpainting/pytorch/default/4'  # Update this path (e.g., '/kaggle/input/checkpoints')
# # /kaggle/input/image-inpainting/pytorch/default/2
# # /kaggle/input/image-inpainting/pytorch/default/4
# # /kaggle/input/image-inpainting/pytorch/default/3
# # 🗂️ LOAD FROM MODEL FILES: If you only have model state_dict files (*.pth)
# # بارگذاری از فایل‌های مدل: اگر فقط فایل‌های state_dict مدل دارید
# LOAD_FROM_MODELS = True  # Set True if loading from models folder (coarse_generator_best.pth, etc.)
# MODELS_DIR = '/kaggle/working/models'  # Path to models folder
# USE_BEST_MODELS = True  # True: Load *_best.pth | False: Load *_bottom_half.pth

# # ⚠️ نکته: اگر از LOAD_FROM_MODELS استفاده کنید، فقط وزن‌های مدل بارگذاری می‌شوند
# # optimizer/scheduler/training history بارگذاری نمی‌شوند و از اول مقداردهی می‌شوند

# # ⚙️ TRAINING START POINT: Which epoch to start from
# # نقطه شروع آموزش: از کدام epoch شروع شود
# # Note: This will be automatically set when loading checkpoint
# # توجه: این مقدار به صورت خودکار هنگام بارگذاری checkpoint تنظیم می‌شود

# print("="*70)
# print("🔧 CHECKPOINT CONFIGURATION")
# print("="*70)
# print(f"📌 Resume Training: {RESUME_TRAINING}")
# print(f"📁 Checkpoint Directory: {CHECKPOINT_DIR}")
# print(f"💾 Save Interval: Every {CHECKPOINT_INTERVAL} epoch(s)")
# if USE_KAGGLE_INPUT:
#     print(f"📥 Loading from Kaggle Dataset: {KAGGLE_INPUT_DATASET}")
# if LOAD_FROM_MODELS:
#     print(f"🗂️  Load from Models: {MODELS_DIR}")
#     print(f"   Using: {'*_best.pth' if USE_BEST_MODELS else '*_bottom_half.pth'}")
# print("="*70 + "\n")

# # ============================================================================
# # TRAINING CONFIGURATION - Initial configurations
# # تنظیمات آموزش - پیکربندی اولیه
# # ============================================================================

# image_size = (64, 64)
# batch_size = 128 # Further reduced for stability
# eval_batch_size = 1
# num_epochs = 10 # Increased for better convergence
# learning_rate_g = 0.0001 # 1e-4 for Generator
# learning_rate_d = 0.0004 # 4e-4 for Discriminator (higher to strengthen D)
# lambda_adv = 0.05 # Further reduced adversarial weight
# lambda_style = 0.05 # Reduced style weight
# lambda_fm = 0.05 # Reduced feature matching weight
# lambda_rec = 1.0 # Reduced reconstruction weight
# lambda_identity = 0.005 # Reduced identity weight
# lambda_landmark = 0.01 # Reduced landmark weight
# lambda_perc = 0.1 # Reduced perceptual weight
# lambda_vae = 0.01 # Further reduced VAE weight
# lambda_struct = 0.05 # Reduced structural weight
# lambda_d_reg = 1e-4 # Increased discriminator regularization
# lambda_attention = 0.05 # Reduced attention loss weight
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")
# # Data splitting configuration
# def split_dataset(dataset_path, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     image_files = [f for f in os.listdir(dataset_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
#     random.shuffle(image_files)
#     total_files = len(image_files)
#     train_end = int(total_files * train_ratio)
#     val_end = train_end + int(total_files * val_ratio)
#     train_files, val_files, test_files = image_files[:train_end], image_files[train_end:val_end], image_files[val_end:]
#     print(f"Dataset split: Total={total_files}, Train={len(train_files)} ({len(train_files)/total_files*100:.1f}%), "
#           f"Val={len(val_files)} ({len(val_files)/total_files*100:.1f}%), Test={len(test_files)} ({len(test_files)/total_files*100:.1f}%)")
#     return train_files, val_files, test_files
# # Custom Dataset
# class CelebADataset(Dataset):
#     def __init__(self, dataset_path, file_list, image_size):
#         self.dataset_path = dataset_path
#         self.file_list = file_list
#         self.transform = transforms.Compose([
#             transforms.Resize(image_size),
#             transforms.ToTensor(),
#             transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # -1 to 1
#         ])
   
#     def __len__(self):
#         return len(self.file_list)
   
#     def __getitem__(self, idx):
#         img_path = os.path.join(self.dataset_path, self.file_list[idx])
#         image = Image.open(img_path).convert('RGB')
#         image = self.transform(image)
#         return image
# def create_split_datasets(dataset_path, train_files, val_files, test_files, image_size, batch_size):
#     train_dataset = CelebADataset(dataset_path, train_files, image_size)
#     val_dataset = CelebADataset(dataset_path, val_files, image_size)
#     test_dataset = CelebADataset(dataset_path, test_files, image_size)
   
#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
#     val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
#     test_loader = DataLoader(test_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=2, pin_memory=True)
#     return train_loader, val_loader, test_loader
# # Enhanced ViT-based Preprocessor with Multi-layer Attention and Graph Pruning
# class ViTPreprocessor(nn.Module):
#     def __init__(self, dim=128, num_heads=8, patch_size=8, num_layers=3):
#         super().__init__()
#         self.patch_size = patch_size
#         self.dim = dim
#         self.num_heads = num_heads
#         self.num_patches = (image_size[0] // patch_size) * (image_size[1] // patch_size)
#         self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size, padding=0)
#         self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, dim))
       
#         # Multi-layer attention blocks
#         self.attention_blocks = nn.ModuleList([
#             nn.ModuleDict({
#                 'attn': nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=0.1),
#                 'norm1': nn.LayerNorm(dim, eps=1e-6),
#                 'norm2': nn.LayerNorm(dim, eps=1e-6),
#                 'mlp': nn.Sequential(
#                     nn.Linear(dim, dim * 4),
#                     nn.GELU(),
#                     nn.Dropout(0.1),
#                     nn.Linear(dim * 4, dim),
#                     nn.Dropout(0.1)
#                 )
#             }) for _ in range(num_layers)
#         ])
       
#         self.final_norm = nn.LayerNorm(dim, eps=1e-6)
#         self.gnn = PyramidalGNN(dim, dim)  # Add PyramidalGNN
   
#     def build_graph(self, features):
#         B, C, H, W = features.shape
#         nodes = features.view(B, C, -1).permute(0, 2, 1)  # [B, num_patches, dim]
        
#         # Use NetworkX for graph construction
#         G = nx.Graph()
#         for i in range(nodes.shape[1]):
#             G.add_node(i, feat=nodes[0, i])
#         for i in range(nodes.shape[1]):
#             for j in range(i+1, nodes.shape[1]):
#                 sim = F.cosine_similarity(nodes[0, i:i+1], nodes[0, j:j+1])
#                 if sim > 0.5:
#                     G.add_edge(i, j, weight=sim)
#         return G
   
   
#     def graph_enhanced_forward(self, x, graph):
#         # Check if graph is empty after pruning
#         node_list = list(graph.nodes)
#         if not node_list:
#             return x  # Return input features directly
        
#         # Simple message passing with NetworkX
#         for node in graph.nodes:
#             neighbors = list(graph.neighbors(node))
#             if neighbors:
#                 neighbor_feats = torch.stack([graph.nodes[n]['feat'] for n in neighbors])
#                 aggregated = torch.mean(neighbor_feats, dim=0)
#                 graph.nodes[node]['feat'] = (graph.nodes[node]['feat'] + aggregated) / 2
        
#         # Final check before stacking
#         final_nodes = list(graph.nodes)
#         if not final_nodes:
#             return x
        
#         # Ensure all features have the same dtype
#         node_features = [graph.nodes[n]['feat'] for n in sorted(final_nodes)]
#         return torch.stack(node_features)
   
#     def reconstruct_features(self, pruned_feats, pruned_indices, B, C, H, W):
#         num_patches = H * W
#         # Ensure dtype matches the input features
#         full_feats = torch.zeros(B, num_patches, C, device=pruned_feats.device, dtype=pruned_feats.dtype)
#         full_feats[:, pruned_indices] = pruned_feats
#         return full_feats.reshape(B, H, W, C).permute(0, 3, 1, 2)
   
#     def forward(self, x):
#         embedded = self.patch_embed(x)  # [B, dim, H/p, W/p]
#         B, C, H, W = embedded.shape
#         flat = embedded.permute(0, 2, 3, 1).reshape(B, H * W, C)
        
#         # Adaptive position embedding based on actual patch count
#         num_patches = H * W
#         if num_patches != self.num_patches:
#             # Interpolate position embedding to match current patch count
#             # Reshape to 3D for linear interpolation: [1, dim, num_patches]
#             pos_embed_3d = self.pos_embed.permute(0, 2, 1)  # [1, dim, num_patches]
#             pos_embed_adaptive = F.interpolate(
#                 pos_embed_3d,
#                 size=num_patches,
#                 mode='linear',
#                 align_corners=False
#             ).permute(0, 2, 1)  # [1, num_patches, dim]
#         else:
#             pos_embed_adaptive = self.pos_embed
        
#         flat = flat + pos_embed_adaptive
        
#         # Build and prune graph
#         graph = self.build_graph(embedded)
#         pruned_graph = prune_graph(self, graph)
        
#         # Get pruned features using NetworkX
#         pruned_indices = list(pruned_graph.nodes)
#         if pruned_indices:  # Check if not empty
#             pruned_feats = torch.stack([pruned_graph.nodes[i]['feat'] for i in pruned_indices])
#         else:
#             pruned_feats = flat[0].clone()  # Use original flattened features for batch 0
#             pruned_indices = list(range(flat.shape[1]))  # All indices
        
#         # Enhanced processing on high-value nodes
#         for block in self.attention_blocks:
#             # Self-attention on pruned features
#             attn_out, _ = block['attn'](pruned_feats, pruned_feats, pruned_feats)
#             pruned_feats = block['norm1'](pruned_feats + attn_out)
            
#             # MLP
#             mlp_out = block['mlp'](pruned_feats)
#             pruned_feats = block['norm2'](pruned_feats + mlp_out)
        
#         # Graph-enhanced message passing
#         enhanced_pruned = self.graph_enhanced_forward(pruned_feats, pruned_graph)
        
#         # Reconstruct full features with low-value nodes getting minimal processing
#         x = self.reconstruct_features(enhanced_pruned, pruned_indices, B, C, H, W)
        
#         # Minimal processing for pruned (low-value) regions - e.g., average pooling
#         full_x = self.final_norm(x.view(B, C, -1).permute(0, 2, 1)).permute(0, 2, 1).view(B, C, H, W)
#         return full_x
# # VAE Sampling Layer
# class VAESampling(nn.Module):
#     def __init__(self):
#         super().__init__()
   
#     def forward(self, mu, logvar):
#         std = torch.exp(0.5 * logvar)
#         eps = torch.randn_like(std)
#         return mu + eps * std
# # Enhanced VAE Encoder with Residual Connections and Graph Integration (No BatchNorm, Strided Conv)
# class VAEEncoder(nn.Module):
#     def __init__(self, latent_dim=128):
#         super().__init__()
#         self.latent_dim = latent_dim
       
#         # Enhanced encoder with strided convolutions (no batch norm)
#         self.conv1 = nn.Conv2d(128, 64, kernel_size=4, stride=2, padding=1) # [B, 128, 8, 8] -> [B, 64, 4, 4]
#         self.conv2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1) # [B, 64, 4, 4] -> [B, 128, 2, 2]
       
#         # Residual block (no batch norm)
#         self.res_conv = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
       
#         self.flatten = nn.Flatten()
#         # Use adaptive pooling to ensure consistent output size
#         self.adaptive_pool = nn.AdaptiveAvgPool2d((2, 2))  # Always output 2x2
#         self.fc_mu = nn.Linear(128 * 2 * 2, latent_dim) # 128 * 2 * 2 = 512
#         self.fc_logvar = nn.Linear(128 * 2 * 2, latent_dim) # 128 * 2 * 2 = 512
   
#     def forward(self, x):
#         # Check input size and use adaptive approach for very small inputs
#         B, C, H, W = x.shape
        
#         # If input is too small for conv layers, use adaptive pooling directly
#         if H < 4 or W < 4:
#             # Use adaptive pooling to get to a reasonable size first
#             x = F.adaptive_avg_pool2d(x, (8, 8))  # Resize to 8x8 minimum
#             x = F.relu(self.conv1(x))  # 8x8 -> 4x4
#             x = F.relu(self.conv2(x))  # 4x4 -> 2x2
#         else:
#             x = F.relu(self.conv1(x))
#             x = F.relu(self.conv2(x))
       
#         # Residual connection
#         residual = x
#         x = F.relu(self.res_conv(x))
#         x = x + residual
       
#         # Use adaptive pooling to ensure consistent size
#         x = self.adaptive_pool(x)  # Always [B, 128, 2, 2]
#         x = self.flatten(x)  # Always [B, 512]
#         mu = self.fc_mu(x)
#         logvar = self.fc_logvar(x)
#         return mu, logvar
# # Enhanced VAE Decoder with Residual Connections and Graph Integration (No BatchNorm)
# class VAEDecoder(nn.Module):
#     def __init__(self, latent_dim=128):
#         super().__init__()
#         self.latent_dim = latent_dim
       
#         # Enhanced decoder with strided transposed convolutions (no batch norm)
#         self.fc = nn.Linear(latent_dim, 8 * 8 * 128)
       
#         # Residual block before upsampling (no batch norm)
#         self.res_conv = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
       
#         self.deconv1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
#         self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
#         self.deconv3 = nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1)
   
#     def forward(self, z):
#         x = F.relu(self.fc(z))
#         x = x.view(-1, 128, 8, 8)
       
#         # Residual connection
#         residual = x
#         x = F.relu(self.res_conv(x))
#         x = x + residual
       
#         x = F.relu(self.deconv1(x)) # 8x8 -> 16x16
#         x = F.relu(self.deconv2(x)) # 16x16 -> 32x32
#         x = torch.tanh(self.deconv3(x)) # 32x32 -> 64x64
#         return x
# # Enhanced CoarseGenerator with Attention, Skip Connections, and Graph Pruning
# class CoarseGenerator(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.preprocessor = ViTPreprocessor(dim=128, num_heads=8, patch_size=8, num_layers=3)
#         self.encoder = VAEEncoder(latent_dim=128) # Increased latent dimension
#         self.sampling = VAESampling()
#         self.decoder = VAEDecoder(latent_dim=128)
       
#         # Attention mechanism for better feature fusion
#         self.attention = nn.MultiheadAttention(embed_dim=128, num_heads=8, dropout=0.1)
#         self.attention_norm = nn.LayerNorm(128, eps=1e-6)
       
#         # Skip connection processing
#         self.skip_conv = nn.Conv2d(128, 128, kernel_size=1, padding=0)
   
#     def forward(self, x, mask):
#         # Preprocess with graph pruning
#         original_features = self.preprocessor(x)
       
#         # Apply attention to enhance features
#         B, C, H, W = original_features.shape
#         attn_input = original_features.permute(0, 2, 3, 1).reshape(B, H * W, C)
#         attn_output, _ = self.attention(attn_input, attn_input, attn_input)
#         enhanced_features = self.attention_norm(attn_input + attn_output)
#         enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
       
#         # Combine with skip connection
#         enhanced_features = enhanced_features + self.skip_conv(original_features)
       
#         mu, logvar = self.encoder(enhanced_features)
#         z = self.sampling(mu, logvar)
#         output = self.decoder(z)
#         return output, mu, logvar
# # Enhanced FineGenerator with U-Net Architecture, Attention, and Graph Cut Integration (No BatchNorm)
# class FineGenerator(nn.Module):
#     def __init__(self):
#         super().__init__()
       
#         # Encoder with skip connections (no batch norm, strided conv)
#         self.enc1 = nn.Sequential(
#             nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
#         self.enc2 = nn.Sequential(
#             nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
#         self.enc3 = nn.Sequential(
#             nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
       
#         # Bottleneck with attention (no batch norm)
#         self.bottleneck = nn.Sequential(
#             nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
       
#         # Attention mechanism
#         self.attention = nn.MultiheadAttention(embed_dim=512, num_heads=8, dropout=0.1)
#         self.attention_norm = nn.LayerNorm(512, eps=1e-6)
       
#         # Decoder with skip connections (no batch norm, strided transposed conv)
#         self.dec1 = nn.Sequential(
#             nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
#             nn.ReLU(inplace=True)
#         )
#         self.dec2 = nn.Sequential(
#             nn.ConvTranspose2d(512, 128, kernel_size=4, stride=2, padding=1), # 256 + 256 from skip
#             nn.ReLU(inplace=True)
#         )
#         self.dec3 = nn.Sequential(
#             nn.ConvTranspose2d(256, 64, kernel_size=4, stride=2, padding=1), # 128 + 128 from skip
#             nn.ReLU(inplace=True)
#         )
#         self.dec4 = nn.Sequential(
#             nn.ConvTranspose2d(128, 3, kernel_size=4, stride=2, padding=1), # 64 + 64 from skip
#             nn.Tanh()
#         )
   
#     def forward(self, masked_images, coarse_output, masks):
#         # Ensure all inputs have the same spatial dimensions
#         if masked_images.shape[2:] != coarse_output.shape[2:]:
#             coarse_output = F.interpolate(coarse_output, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
       
#         x = torch.cat([masked_images, coarse_output], dim=1)
       
#         # Encoder with skip connections
#         enc1_out = self.enc1(x) # 64x64 -> 32x32
#         enc2_out = self.enc2(enc1_out) # 32x32 -> 16x16
#         enc3_out = self.enc3(enc2_out) # 16x16 -> 8x8
#         bottleneck = self.bottleneck(enc3_out) # 8x8 -> 4x4
       
#         # Apply attention to bottleneck features
#         B, C, H, W = bottleneck.shape
#         attn_input = bottleneck.permute(0, 2, 3, 1).reshape(B, H * W, C)
#         attn_output, _ = self.attention(attn_input, attn_input, attn_input)
#         enhanced_features = self.attention_norm(attn_input + attn_output)
#         enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
       
#         # Decoder with skip connections
#         dec1_out = self.dec1(enhanced_features) # 4x4 -> 8x8
#         dec1_out = torch.cat([dec1_out, enc3_out], dim=1) # Skip connection
       
#         dec2_out = self.dec2(dec1_out) # 8x8 -> 16x16
#         dec2_out = torch.cat([dec2_out, enc2_out], dim=1) # Skip connection
       
#         dec3_out = self.dec3(dec2_out) # 16x16 -> 32x32
#         dec3_out = torch.cat([dec3_out, enc1_out], dim=1) # Skip connection
       
#         output = self.dec4(dec3_out) # 32x32 -> 64x64
       
#         # Ensure output matches input size
#         if output.shape[2:] != masked_images.shape[2:]:
#             output = F.interpolate(output, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
       
#         return output
# # Stronger Discriminator with Spectral Normalization (No BatchNorm, More Layers)
# class Discriminator(nn.Module):
#     def __init__(self):
#         super().__init__()
       
#         # Stronger discriminator with more layers and capacity (no batch norm)
#         self.conv1 = nn.utils.spectral_norm(nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1))
#         self.conv2 = nn.utils.spectral_norm(nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1))
#         self.conv3 = nn.utils.spectral_norm(nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1))
#         self.conv4 = nn.utils.spectral_norm(nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1))
#         # Additional layers for stronger discriminator
#         self.conv5 = nn.utils.spectral_norm(nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1))
#         self.conv6 = nn.utils.spectral_norm(nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1))
       
#         # Global average pooling for stability
#         self.global_pool = nn.AdaptiveAvgPool2d(1)
#         # Deeper fully connected layers
#         self.fc1 = nn.utils.spectral_norm(nn.Linear(512, 256))
#         self.fc2 = nn.utils.spectral_norm(nn.Linear(256, 1))
       
#         # Feature layers for feature matching loss
#         self.feature_layers = [self.conv1, self.conv2, self.conv3, self.conv4, self.conv5, self.conv6]
   
#     def forward(self, x, mask, return_features=False):
#         input_concat = torch.cat([x, mask], dim=1)
#         features = []
       
#         # Main discriminator path with more layers
#         x = F.leaky_relu(self.conv1(input_concat), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv2(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv3(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv4(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv5(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv6(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         # Global average pooling for stability
#         x = self.global_pool(x)
#         x = x.view(x.size(0), -1)
        
#         # Deeper FC layers
#         x = F.leaky_relu(self.fc1(x), 0.2, inplace=True)
#         logits = self.fc2(x)
       
#         if return_features:
#             return logits, features
#         return logits
# # Generate adaptive mask
# def generate_adaptive_mask(image):
#     batch_size = image.shape[0]
#     h, w = image_size
#     mask = torch.ones(batch_size, 3, h, w, dtype=torch.float32, device=image.device)
#     half_height = h // 2
#     bottom_mask = torch.zeros(batch_size, 3, h - half_height, w, dtype=torch.float32, device=image.device)
#     top_mask = torch.ones(batch_size, 3, half_height, w, dtype=torch.float32, device=image.device)
#     mask = torch.cat([top_mask, bottom_mask], dim=2)
   
#     if USE_MTCNN:
#         detector = MTCNN()
#         images_np = ((image * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
#         landmarks = []
#         for img in images_np:
#             img_uint8 = img.astype(np.uint8)
#             result = detector.detect_faces(img_uint8)
#             landmarks.append(result[0]['keypoints'] if result else {})
       
#         mask_np = mask.permute(0, 2, 3, 1).cpu().detach().numpy()
#         for i, lm in enumerate(landmarks):
#             if lm:
#                 for key in ['mouth_left', 'mouth_right']:
#                     if key in lm:
#                         x, y = lm[key]
#                         y = min(max(y, half_height), h-1)
#                         x = min(max(x, 0), w-1)
#                         mask_np[i, y-5:y+5, x-5:x+5, :] = 0.0
#         mask = torch.from_numpy(mask_np).permute(0, 3, 1, 2).to(device)
   
#     return mask
# # Loss functions
# # Use new torchvision weights API to avoid deprecation warnings
# try:
#     vgg_weights = models.VGG16_Weights.DEFAULT
# except AttributeError:
#     # Fallback for very old torchvision
#     vgg_weights = None
# vgg = models.vgg16(weights=vgg_weights).features.to(device).eval()
# loss_model = nn.Sequential(*[vgg[i] for i in range(16)]).to(device) # Up to block4_conv3
# for param in loss_model.parameters():
#     param.requires_grad = False
# def perceptual_loss(y_true, y_pred):
#     y_true = F.interpolate(y_true, size=image_size, mode='bilinear', align_corners=False)
#     y_pred = F.interpolate(y_pred, size=image_size, mode='bilinear', align_corners=False)
#     y_true = y_true * 0.5 + 0.5 # Denormalize to [0, 1]
#     y_pred = y_pred * 0.5 + 0.5
#     true_features = loss_model(y_true)
#     pred_features = loss_model(y_pred)
#     return torch.mean((true_features - pred_features) ** 2)
# def style_loss(y_true, y_pred):
#     y_true = F.interpolate(y_true, size=image_size, mode='bilinear', align_corners=False)
#     y_pred = F.interpolate(y_pred, size=image_size, mode='bilinear', align_corners=False)
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     true_features = loss_model(y_true)
#     pred_features = loss_model(y_pred)
   
#     def gram_matrix(feat):
#         B, C, H, W = feat.shape
#         feat = feat.view(B, C, H * W)
#         return torch.bmm(feat, feat.transpose(1, 2)) / (C * H * W)
   
#     style_losses = [torch.mean((gram_matrix(t) - gram_matrix(p)) ** 2) for t, p in zip([true_features], [pred_features])]
#     return torch.mean(torch.stack(style_losses))
# def bottom_half_loss(y_true, y_pred, mask):
#     bottom_region = (mask == 0).float()
#     diff = (y_true - y_pred) ** 2 * bottom_region
#     return diff.sum() / (bottom_region.sum() + 1e-8)
# def feature_matching_loss(real_features, fake_features):
#     losses = [torch.mean((r - f) ** 2) for r, f in zip(real_features, fake_features)]
#     return torch.mean(torch.stack(losses))
# def identity_loss(y_true, y_pred, mask):
#     bottom_region = (mask == 0).float()
#     y_true_face = y_true * bottom_region
#     y_pred_face = y_pred * bottom_region
#     return torch.mean(torch.abs(y_true_face - y_pred_face))
# def landmark_guided_loss(y_true, y_pred, mask):
#     if not USE_MTCNN:
#         return bottom_half_loss(y_true, y_pred, mask)
#     detector = MTCNN()
#     bottom_region = (mask == 0).float()
#     y_true_bottom = y_true * bottom_region
#     y_pred_bottom = y_pred * bottom_region
#     y_true_np = ((y_true_bottom * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
#     y_pred_np = ((y_pred_bottom * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
#     loss_tensor = torch.tensor(0.0, device=device, requires_grad=True)
#     for true_img, pred_img in zip(y_true_np, y_pred_np):
#         true_img_uint8 = true_img.astype(np.uint8)
#         pred_img_uint8 = pred_img.astype(np.uint8)
#         true_lm = detector.detect_faces(true_img_uint8)
#         pred_lm = detector.detect_faces(pred_img_uint8)
#         if true_lm and pred_lm:
#             true_points = [true_lm[0]['keypoints'][k] for k in ['mouth_left', 'mouth_right']]
#             pred_points = [pred_lm[0]['keypoints'][k] for k in ['mouth_left', 'mouth_right']]
#             for t, p in zip(true_points, pred_points):
#                 diff_x = torch.tensor(t[0] - p[0], dtype=torch.float32, device=device)
#                 diff_y = torch.tensor(t[1] - p[1], dtype=torch.float32, device=device)
#                 loss_tensor = loss_tensor + (diff_x ** 2 + diff_y ** 2)
#     return loss_tensor / y_true.shape[0]
# def structural_ssim_loss(y_true, y_pred, mask):
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     bottom_mask = (mask == 0).float()
#     y_true_bottom = y_true * bottom_mask
#     y_pred_bottom = y_pred * bottom_mask
#     ssim_vals = []
#     for i in range(y_true.shape[0]):
#         true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         ssim_val = ssim(true_img, pred_img, channel_axis=2, data_range=1.0, win_size=7)
#         ssim_vals.append(ssim_val)
#     return 1.0 - torch.mean(torch.tensor(ssim_vals, device=device))
# # Advanced Loss Functions
# def gradient_penalty_loss(discriminator, real_images, fake_images, masks, device):
#     """Calculate gradient penalty for WGAN-GP"""
#     batch_size = real_images.size(0)
#     alpha = torch.rand(batch_size, 1, 1, 1).to(device)
#     interpolated = alpha * real_images + (1 - alpha) * fake_images
   
#     # Ensure interpolated tensor requires gradients
#     interpolated = interpolated.detach().requires_grad_(True)
   
#     disc_interpolated = discriminator(interpolated, masks, return_features=False)
   
#     # Check if disc_interpolated requires gradients
#     if not disc_interpolated.requires_grad:
#         return torch.tensor(0.0, device=device, requires_grad=True)
   
#     gradients = torch.autograd.grad(
#         outputs=disc_interpolated,
#         inputs=interpolated,
#         grad_outputs=torch.ones_like(disc_interpolated).to(device),
#         create_graph=True,
#         retain_graph=True,
#         only_inputs=True
#     )[0]
   
#     gradients = gradients.view(gradients.size(0), -1)
#     gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
#     return gradient_penalty
# def attention_loss(y_true, y_pred, mask):
#     """Simplified attention-based loss focusing on important regions"""
#     bottom_region = (mask == 0).float()
#     diff = torch.abs(y_true - y_pred) * bottom_region
   
#     # Create simple attention weights based on gradient magnitude
#     true_grad_x = torch.abs(y_true[:, :, :, 1:] - y_true[:, :, :, :-1])
#     true_grad_y = torch.abs(y_true[:, :, 1:, :] - y_true[:, :, :-1, :])
   
#     # Pad gradients to match original size
#     true_grad_x_padded = F.pad(true_grad_x, (0, 1, 0, 0), mode='constant', value=0)
#     true_grad_y_padded = F.pad(true_grad_y, (0, 0, 0, 1), mode='constant', value=0)
   
#     # Combine gradients and normalize
#     attention_weights = (true_grad_x_padded + true_grad_y_padded).mean(dim=1, keepdim=True)
#     attention_weights = attention_weights / (attention_weights.mean() + 1e-8) # Normalize
   
#     # Apply attention weights
#     weighted_loss = diff * attention_weights
#     return weighted_loss.mean()
# def generator_loss(disc_fake_logits, y_true, y_pred, disc_features_real, disc_features_fake, mask, mu, logvar):
#     # Clamp logvar to prevent exp overflow
#     logvar = torch.clamp(logvar, min=-10, max=10)
    
#     # Calculate each loss with NaN protection
#     adv_loss = -torch.mean(disc_fake_logits)
#     adv_loss = torch.nan_to_num(adv_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     # L1 reconstruction loss (high weight for stable inpainting)
#     bottom_region = (mask == 0).float()
#     l1_loss = torch.mean(torch.abs(y_true - y_pred) * bottom_region)
#     l1_loss = torch.nan_to_num(l1_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     rec_loss = bottom_half_loss(y_true, y_pred, mask)
#     rec_loss = torch.nan_to_num(rec_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     perc_loss = perceptual_loss(y_true, y_pred)
#     perc_loss = torch.nan_to_num(perc_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     sty_loss = style_loss(y_true, y_pred)
#     sty_loss = torch.nan_to_num(sty_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     fm_loss = feature_matching_loss(disc_features_real, disc_features_fake)
#     fm_loss = torch.nan_to_num(fm_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     id_loss = identity_loss(y_true, y_pred, mask)
#     id_loss = torch.nan_to_num(id_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     lm_loss = landmark_guided_loss(y_true, y_pred, mask)
#     lm_loss = torch.nan_to_num(lm_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     struct_loss = structural_ssim_loss(y_true, y_pred, mask)
#     struct_loss = torch.nan_to_num(struct_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     att_loss = attention_loss(y_true, y_pred, mask)
#     att_loss = torch.nan_to_num(att_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     # KL loss with clipping
#     kl_loss = -0.5 * torch.mean(1 + logvar - mu**2 - torch.exp(logvar))
#     kl_loss = torch.nan_to_num(kl_loss, nan=0.0, posinf=0.0, neginf=0.0)
#     kl_loss = torch.clamp(kl_loss, min=0.0, max=10.0)  # Prevent extreme values
    
#     # Calculate total loss with L1 loss (high weight for inpainting)
#     total_loss = (lambda_adv * adv_loss + lambda_l1 * l1_loss + lambda_rec * rec_loss + lambda_perc * perc_loss +
#                   lambda_style * sty_loss + lambda_fm * fm_loss + lambda_identity * id_loss +
#                   lambda_landmark * lm_loss + lambda_struct * struct_loss + 
#                   lambda_attention * att_loss + lambda_vae * kl_loss)
    
#     # Final NaN check
#     total_loss = torch.nan_to_num(total_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     return total_loss
# def discriminator_loss(disc_real_logits, disc_fake_logits):
#     # Clamp logits to prevent extreme values
#     disc_real_logits = torch.clamp(disc_real_logits, min=-10, max=10)
#     disc_fake_logits = torch.clamp(disc_fake_logits, min=-10, max=10)
    
#     real_loss = torch.mean(F.relu(1.0 - disc_real_logits))
#     real_loss = torch.nan_to_num(real_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     fake_loss = torch.mean(F.relu(1.0 + disc_fake_logits))
#     fake_loss = torch.nan_to_num(fake_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     reg = lambda_d_reg * (torch.mean(disc_real_logits**2) + torch.mean(disc_fake_logits**2))
#     reg = torch.nan_to_num(reg, nan=0.0, posinf=0.0, neginf=0.0)
    
#     total_loss = real_loss + fake_loss + reg
#     total_loss = torch.nan_to_num(total_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     return total_loss
# # Evaluation functions
# def calculate_psnr_bottom_half(y_true, y_pred, mask):
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     bottom_mask = (mask == 0).float()
#     y_true_bottom = y_true * bottom_mask
#     y_pred_bottom = y_pred * bottom_mask
#     psnr_vals = []
#     for i in range(y_true.shape[0]):
#         true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         psnr_val = psnr(true_img, pred_img, data_range=1.0)
#         psnr_vals.append(psnr_val)
#     return torch.mean(torch.tensor(psnr_vals, device=device))
# def calculate_ssim_bottom_half(y_true, y_pred, mask):
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     bottom_mask = (mask == 0).float()
#     y_true_bottom = y_true * bottom_mask
#     y_pred_bottom = y_pred * bottom_mask
#     ssim_vals = []
#     for i in range(y_true.shape[0]):
#         true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         ssim_val = ssim(true_img, pred_img, channel_axis=2, data_range=1.0, win_size=7)
#         ssim_vals.append(ssim_val)
#     return torch.mean(torch.tensor(ssim_vals, device=device))
# def calculate_mse_bottom_half(y_true, y_pred, mask):
#     return bottom_half_loss(y_true, y_pred, mask)
# # Training function with graph-aware processing and balanced D/G updates
# def train_step(images, coarse_generator, fine_generator, discriminator, g_optimizer, d_optimizer, scaler, step_counter):
#     images = images.to(device)
#     masks = generate_adaptive_mask(images)
#     masked_images = images * masks
   
#     # Discriminator training (n_critic times more frequent)
#     d_optimizer.zero_grad()
    
#     # Train Discriminator multiple times
#     d_loss_accumulated = 0.0
#     for _ in range(n_critic):
#         with amp.autocast('cuda'):
#             # Generate fake images
#             with torch.no_grad():
#                 coarse_output, mu, logvar = multi_resolution_inpainting(masked_images, masks, coarse_generator)
#                 if coarse_output.shape != images.shape:
#                     coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
                
#                 fine_output = fine_generator(masked_images, coarse_output, masks)
#                 if fine_output.shape != images.shape:
#                     fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
                
#                 combined_input = images * masks + fine_output * (1.0 - masks)
            
#             # Discriminator forward
#             disc_real_logits = discriminator(images, masks, return_features=False)
#             disc_fake_logits = discriminator(combined_input.detach(), masks, return_features=False)
#             d_loss = discriminator_loss(disc_real_logits, disc_fake_logits)
        
#         # Discriminator backward
#         scaler.scale(d_loss).backward()
#         d_loss_accumulated += d_loss.item()
    
#     # Update discriminator after n_critic iterations
#     scaler.unscale_(d_optimizer)
#     torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
#     scaler.step(d_optimizer)
    
#     d_loss_val = d_loss_accumulated / n_critic
   
#     # Generator training (once per n_critic discriminator updates)
#     g_loss_val = 0.0
#     if step_counter % n_critic == 0:
#         g_optimizer.zero_grad()
        
#         with amp.autocast('cuda'):
#             # Generator forward pass and loss
#             coarse_output, mu, logvar = multi_resolution_inpainting(masked_images, masks, coarse_generator)
#             if coarse_output.shape != images.shape:
#                 coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
            
#             fine_output = fine_generator(masked_images, coarse_output, masks)
#             if fine_output.shape != images.shape:
#                 fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
            
#             combined_input = images * masks + fine_output * (1.0 - masks)
#             disc_real_logits, disc_features_real = discriminator(images, masks, return_features=True)
#             disc_fake_logits, disc_features_fake = discriminator(combined_input, masks, return_features=True)
#             g_loss = generator_loss(disc_fake_logits, images, fine_output, disc_features_real, disc_features_fake, masks, mu, logvar)
        
#         # Generator backward pass with gradient clipping
#         scaler.scale(g_loss).backward()
#         scaler.unscale_(g_optimizer)
#         torch.nn.utils.clip_grad_norm_(list(coarse_generator.parameters()) + list(fine_generator.parameters()), max_norm=1.0)
#         scaler.step(g_optimizer)
        
#         g_loss_val = g_loss.item()
   
#     # Update scaler after both steps
#     scaler.update()
    
#     # Check for NaN in losses
#     if np.isnan(g_loss_val) or np.isinf(g_loss_val):
#         print(f"⚠️ WARNING: G_Loss is {g_loss_val}, skipping this batch")
#         g_loss_val = 0.0
        
#     if np.isnan(d_loss_val) or np.isinf(d_loss_val):
#         print(f"⚠️ WARNING: D_Loss is {d_loss_val}, skipping this batch")
#         d_loss_val = 0.0
   
#     return g_loss_val, d_loss_val
# # Add multi_resolution_inpainting function
# def multi_resolution_inpainting(image, mask, model, levels=3):
#     outputs = []
#     mus = []
#     logvars = []
#     for level in range(levels):
#         scaled_image = F.interpolate(image, scale_factor=1 / (2 ** level), mode='bilinear')
#         scaled_mask = F.interpolate(mask, scale_factor=1 / (2 ** level), mode='nearest')
        
#         out, mu, logvar = model(scaled_image, scaled_mask)  # Call full forward
        
#         out_up = F.interpolate(out, size=image.shape[-2:], mode='bilinear')
#         outputs.append(out_up)
#         mus.append(mu)
#         logvars.append(logvar)
    
#     final_out = torch.mean(torch.stack(outputs), dim=0)
#     final_mu = torch.mean(torch.stack(mus), dim=0)
#     final_logvar = torch.mean(torch.stack(logvars), dim=0)
#     return final_out, final_mu, final_logvar

# # Add gradient checking function
# def check_model_gradients(model, model_name="Model"):
#     """Check if model has NaN or Inf gradients"""
#     has_nan = False
#     has_inf = False
#     max_grad = 0.0
    
#     for name, param in model.named_parameters():
#         if param.grad is not None:
#             if torch.isnan(param.grad).any():
#                 has_nan = True
#                 print(f"⚠️ NaN gradient in {model_name}.{name}")
#             if torch.isinf(param.grad).any():
#                 has_inf = True
#                 print(f"⚠️ Inf gradient in {model_name}.{name}")
#             max_grad = max(max_grad, param.grad.abs().max().item())
    
#     return has_nan, has_inf, max_grad

# # Add simple retrain function
# def retrain(model, pruned_graph, optimizer):
#     optimizer.zero_grad()
#     # Dummy forward (adjust to use graph if needed)
#     dummy_input = torch.randn(1, 3, 64, 64).to(device)
#     dummy_mask = torch.randn(1, 3, 64, 64).to(device)
#     output, _, _ = model(dummy_input, dummy_mask)
#     loss = torch.mean(output)
#     loss.backward()
#     optimizer.step()
#     return loss.item()

# # Integrate into training loop
# # In train_step function, after coarse_output:
# # Add pruning and retraining every 10 epochs
# # Load dataset

# # celeba_dir = '/kaggle/input/celeba-resized-6464/img_align_celeba/CelebA_Image_Cropped_64'
# celeba_dir = '/kaggle/input/celeba-6464/selected_celeba'

# print(f"Loading dataset from {celeba_dir}...")
# try:
#     print(f"Dataset directory contents: {os.listdir(celeba_dir)[:5]}")
# except Exception as e:
#     print(f"Error accessing dataset directory: {e}")
#     raise
# # Split dataset
# train_files, val_files, test_files = split_dataset(celeba_dir)
# # Create datasets
# train_loader, val_loader, test_loader = create_split_datasets(celeba_dir, train_files, val_files, test_files, image_size, batch_size)
# # Build models
# print("Building models...")
# coarse_generator = CoarseGenerator().to(device)
# fine_generator = FineGenerator().to(device)
# discriminator = Discriminator().to(device)
# print(f"Coarse Generator: {sum(p.numel() for p in coarse_generator.parameters()):,} parameters")
# print(f"Fine Generator: {sum(p.numel() for p in fine_generator.parameters()):,} parameters")
# print(f"Discriminator: {sum(p.numel() for p in discriminator.parameters()):,} parameters")
# # Test model dimensions
# print("\nTesting model dimensions...")
# test_input = torch.randn(1, 3, 64, 64).to(device)
# test_mask = torch.randn(1, 3, 64, 64).to(device)
# with torch.no_grad():
#     coarse_output, mu, logvar = coarse_generator(test_input, test_mask)
#     print(f"Coarse output shape: {coarse_output.shape}")
#     fine_output = fine_generator(test_input, coarse_output, test_mask)
#     print(f"Fine output shape: {fine_output.shape}")
#     print(f"Expected shape: {test_input.shape}")
   
#     # Test discriminator
#     disc_logits, disc_features = discriminator(test_input, test_mask, return_features=True)
#     print(f"Discriminator logits shape: {disc_logits.shape}")
#     print(f"Discriminator features length: {len(disc_features)}")
   
#     disc_logits_only = discriminator(test_input, test_mask, return_features=False)
#     print(f"Discriminator logits only shape: {disc_logits_only.shape}")
   
#     print("✓ Model dimensions are correct!")
# # Enhanced Optimizers with Learning Rate Scheduling and beta1=0.5 for GAN stability
# g_optimizer = torch.optim.AdamW(
#     list(coarse_generator.parameters()) + list(fine_generator.parameters()),
#     lr=learning_rate_g, betas=(0.5, 0.999), weight_decay=1e-4
# )
# d_optimizer = torch.optim.AdamW(
#     discriminator.parameters(),
#     lr=learning_rate_d, betas=(0.5, 0.999), weight_decay=1e-4
# )
# print(f"Optimizer settings: G_LR={learning_rate_g}, D_LR={learning_rate_d}, beta1=0.5, n_critic={n_critic}")
# # Learning rate schedulers
# g_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     g_optimizer, T_0=10, T_mult=2, eta_min=learning_rate_g * 0.01
# )
# d_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     d_optimizer, T_0=10, T_mult=2, eta_min=learning_rate_d * 0.01
# )
# scaler = amp.GradScaler('cuda')
# # Model health check before training
# print("\n" + "="*60)
# print("🔍 Model Health Check")
# print("="*60)
# test_batch = next(iter(train_loader)).to(device)
# test_masks = generate_adaptive_mask(test_batch)
# test_masked = test_batch * test_masks

# print("Testing forward pass...")
# with torch.no_grad():
#     try:
#         coarse_out, mu, logvar = coarse_generator(test_masked, test_masks)
#         print(f"✓ Coarse Generator: output shape {coarse_out.shape}")
#         print(f"  mu range: [{mu.min().item():.4f}, {mu.max().item():.4f}]")
#         print(f"  logvar range: [{logvar.min().item():.4f}, {logvar.max().item():.4f}]")
        
#         fine_out = fine_generator(test_masked, coarse_out, test_masks)
#         print(f"✓ Fine Generator: output shape {fine_out.shape}")
#         print(f"  output range: [{fine_out.min().item():.4f}, {fine_out.max().item():.4f}]")
        
#         disc_out = discriminator(test_batch, test_masks)
#         print(f"✓ Discriminator: logits shape {disc_out.shape}")
#         print(f"  logits range: [{disc_out.min().item():.4f}, {disc_out.max().item():.4f}]")
        
#         print("✅ All models are healthy!")
#     except Exception as e:
#         print(f"❌ Model health check failed: {e}")
#         raise
# print("="*60 + "\n")

# # ============================================================================
# # CHECKPOINT LOADING & TRAINING SETUP
# # بارگذاری Checkpoint و راه‌اندازی آموزش
# # ============================================================================

# print("\n" + "="*70)
# print("🚀 STARTING TRAINING SETUP")
# print("="*70)

# # Adjust paths if loading from Kaggle input dataset
# if USE_KAGGLE_INPUT:
#     checkpoint_load_dir = os.path.join(KAGGLE_INPUT_DATASET, 'checkpoints')
#     models_load_dir = os.path.join(KAGGLE_INPUT_DATASET, 'models')
#     print(f"📥 Loading from Kaggle dataset: {KAGGLE_INPUT_DATASET}")
# else:
#     checkpoint_load_dir = CHECKPOINT_DIR
#     models_load_dir = MODELS_DIR
#     print(f"📥 Loading from working directory")

# # Create checkpoint save directory (always in /kaggle/working)
# os.makedirs(CHECKPOINT_DIR, exist_ok=True)
# os.makedirs(MODELS_DIR, exist_ok=True)
# print(f"💾 Checkpoints will be saved to: {CHECKPOINT_DIR}")
# print(f"💾 Models will be saved to: {MODELS_DIR}")

# # Initialize training variables
# start_epoch = 0
# best_val_loss = float('inf')
# train_g_losses = []
# train_d_losses = []
# val_g_losses = []
# val_d_losses = []

# # Try to load checkpoint/models if RESUME_TRAINING is True
# checkpoint_loaded = False
# models_loaded = False

# if RESUME_TRAINING:
#     # ========================================================================
#     # Method 1: Try to load from MODEL FILES (state_dict only)
#     # روش 1: بارگذاری از فایل‌های MODEL (فقط state_dict)
#     # ========================================================================
    
#     if LOAD_FROM_MODELS and not checkpoint_loaded:
#         print(f"\n🔍 Method 1: Searching for model files...")
#         print(f"📂 Models directory: {models_load_dir}")
        
#         # Debug: Show what files actually exist
#         if os.path.exists(models_load_dir):
#             print(f"\n📄 Files found in directory:")
#             actual_files = os.listdir(models_load_dir)
#             if actual_files:
#                 for f in sorted(actual_files):
#                     if f.endswith('.pth'):
#                         file_path = os.path.join(models_load_dir, f)
#                         size_mb = os.path.getsize(file_path) / (1024**2)
#                         print(f"   • {f} ({size_mb:.2f} MB)")
#             else:
#                 print(f"   (empty)")
#         else:
#             print(f"   ❌ Directory does not exist!")
        
#         # Determine which models to load
#         suffix = "_best.pth" if USE_BEST_MODELS else "_bottom_half.pth"
        
#         model_files = {
#             'coarse': os.path.join(models_load_dir, f'coarse_generator{suffix}'),
#             'fine': os.path.join(models_load_dir, f'fine_generator{suffix}'),
#             'disc': os.path.join(models_load_dir, f'discriminator{suffix}')
#         }
        
#         print(f"\n🔍 Looking for (USE_BEST_MODELS={USE_BEST_MODELS}):")
#         for name, path in model_files.items():
#             exists = "✅" if os.path.exists(path) else "❌"
#             print(f"   {exists} {os.path.basename(path)}")
        
#         # Check if all model files exist
#         all_exist = all(os.path.exists(path) for path in model_files.values())
        
#         if all_exist:
#             try:
#                 print(f"📂 Found model files in: {models_load_dir}")
#                 print(f"   • coarse_generator{suffix}")
#                 print(f"   • fine_generator{suffix}")
#                 print(f"   • discriminator{suffix}")
#                 print(f"⏳ Loading model weights...")
                
#                 # Load model weights
#                 # Load model weights tolerantly in case of architecture drift
#                 coarse_sd = torch.load(model_files['coarse'], map_location=device)
#                 fine_sd = torch.load(model_files['fine'], map_location=device)
#                 disc_sd = torch.load(model_files['disc'], map_location=device)

#                 missing, unexpected = coarse_generator.load_state_dict(coarse_sd, strict=False)
#                 if missing or unexpected:
#                     print("[coarse_generator] non-strict load:")
#                     if missing:
#                         print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                     if unexpected:
#                         print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                 missing, unexpected = fine_generator.load_state_dict(fine_sd, strict=False)
#                 if missing or unexpected:
#                     print("[fine_generator] non-strict load:")
#                     if missing:
#                         print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                     if unexpected:
#                         print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                 missing, unexpected = discriminator.load_state_dict(disc_sd, strict=False)
#                 if missing or unexpected:
#                     print("[discriminator] non-strict load:")
#                     if missing:
#                         print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                     if unexpected:
#                         print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
                
#                 models_loaded = True
                
#                 print("="*70)
#                 print("✅ MODEL WEIGHTS LOADED SUCCESSFULLY!")
#                 print("="*70)
#                 print(f"📊 Loading Information:")
#                 print(f"   • Model weights: ✅ Loaded")
#                 print(f"   • Optimizer states: ⚠️  Will be initialized (not in model files)")
#                 print(f"   • Scheduler states: ⚠️  Will be initialized (not in model files)")
#                 print(f"   • Training history: ⚠️  Starting fresh (not in model files)")
#                 print(f"   • Starting epoch: 1 (training continues with loaded weights)")
#                 print(f"\n⚠️  NOTE: Model files only contain weights, not optimizer/scheduler.")
#                 print(f"   Training will continue with these weights but fresh optimizer.")
#                 print("="*70)
                
#             except Exception as e:
#                 print(f"⚠️  Failed to load model files")
#                 print(f"   Error: {str(e)}")
#                 models_loaded = False
#         else:
#             print(f"⚠️  Not all model files found in {models_load_dir}")
#             for name, path in model_files.items():
#                 exists = "✓" if os.path.exists(path) else "✗"
#                 print(f"   {exists} {os.path.basename(path)}")
    
#     # ========================================================================
#     # Method 2: Try to load from CHECKPOINT FILES (full checkpoint)
#     # روش 2: بارگذاری از فایل‌های CHECKPOINT (checkpoint کامل)
#     # ========================================================================
    
#     if not checkpoint_loaded and not models_loaded:
#         print(f"\n🔍 Method 2: Searching for checkpoint files...")
        
#         # Try different checkpoint sources
#         checkpoint_paths_to_try = [
#             os.path.join(checkpoint_load_dir, 'latest_checkpoint.pth'),
#             os.path.join(checkpoint_load_dir, f'checkpoint_epoch_{num_epochs}.pth'),
#             os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth'),  # Fallback to working dir
#         ]
        
#         for checkpoint_path in checkpoint_paths_to_try:
#             if os.path.exists(checkpoint_path):
#                 try:
#                     print(f"📂 Found checkpoint: {checkpoint_path}")
#                     print(f"⏳ Loading checkpoint...")
                    
#                     checkpoint = torch.load(checkpoint_path, map_location=device)
                    
#                     # Load model states tolerantly
#                     missing, unexpected = coarse_generator.load_state_dict(checkpoint['coarse_generator'], strict=False)
#                     if missing or unexpected:
#                         print("[coarse_generator ckpt] non-strict load:")
#                         if missing:
#                             print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                         if unexpected:
#                             print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                     missing, unexpected = fine_generator.load_state_dict(checkpoint['fine_generator'], strict=False)
#                     if missing or unexpected:
#                         print("[fine_generator ckpt] non-strict load:")
#                         if missing:
#                             print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                         if unexpected:
#                             print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                     missing, unexpected = discriminator.load_state_dict(checkpoint['discriminator'], strict=False)
#                     if missing or unexpected:
#                         print("[discriminator ckpt] non-strict load:")
#                         if missing:
#                             print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                         if unexpected:
#                             print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
                    
#                     # Load optimizer states
#                     g_optimizer.load_state_dict(checkpoint['g_optimizer'])
#                     d_optimizer.load_state_dict(checkpoint['d_optimizer'])
                    
#                     # Load scheduler states
#                     g_scheduler.load_state_dict(checkpoint['g_scheduler'])
#                     d_scheduler.load_state_dict(checkpoint['d_scheduler'])
                    
#                     # Load training progress
#                     start_epoch = checkpoint['epoch'] + 1
#                     best_val_loss = checkpoint.get('best_val_loss', float('inf'))
#                     train_g_losses = checkpoint.get('train_g_losses', [])
#                     train_d_losses = checkpoint.get('train_d_losses', [])
#                     val_g_losses = checkpoint.get('val_g_losses', [])
#                     val_d_losses = checkpoint.get('val_d_losses', [])
                    
#                     checkpoint_loaded = True
                    
#                     print("="*70)
#                     print("✅ FULL CHECKPOINT LOADED SUCCESSFULLY!")
#                     print("="*70)
#                     print(f"📊 Resume Information:")
#                     print(f"   • Model weights: ✅ Loaded")
#                     print(f"   • Optimizer states: ✅ Loaded")
#                     print(f"   • Scheduler states: ✅ Loaded")
#                     print(f"   • Starting from epoch: {start_epoch + 1}")
#                     print(f"   • Best validation loss: {best_val_loss:.6f}")
#                     print(f"   • Training history: {len(train_g_losses)} epochs")
#                     print(f"   • Remaining epochs: {num_epochs - start_epoch}")
#                     print("="*70)
                    
#                     break  # Successfully loaded, exit loop
                    
#                 except Exception as e:
#                     print(f"⚠️  Failed to load checkpoint from {checkpoint_path}")
#                     print(f"   Error: {str(e)}")
#                     print(f"   Trying next checkpoint source...")
#                     continue

# # ========================================================================
# # If nothing loaded, start fresh
# # اگر چیزی بارگذاری نشد، از اول شروع کن
# # ========================================================================

# if not checkpoint_loaded and not models_loaded:
#     if RESUME_TRAINING:
#         print("\n" + "="*70)
#         print("⚠️  NO CHECKPOINT OR MODELS FOUND - Starting from scratch")
#         print("="*70)
#         print("📝 Locations searched:")
#         print(f"\n   Models folder: {models_load_dir}")
#         if os.path.exists(models_load_dir):
#             files = os.listdir(models_load_dir)
#             if files:
#                 print(f"   Found {len(files)} file(s):")
#                 for f in files[:5]:  # Show first 5
#                     print(f"      • {f}")
#             else:
#                 print(f"      (empty)")
#         else:
#             print(f"      (not found)")
        
#         print(f"\n   Checkpoints folder: {checkpoint_load_dir}")
#         if os.path.exists(checkpoint_load_dir):
#             files = [f for f in os.listdir(checkpoint_load_dir) if f.endswith('.pth')]
#             if files:
#                 print(f"   Found {len(files)} checkpoint(s):")
#                 for f in files[:5]:
#                     print(f"      • {f}")
#             else:
#                 print(f"      (empty)")
#         else:
#             print(f"      (not found)")
        
#         print("\n💡 To resume training in next run:")
#         print("   1. Make sure model/checkpoint files exist")
#         print("   2. Set LOAD_FROM_MODELS = True for model files")
#         print("   3. If using Kaggle dataset, set USE_KAGGLE_INPUT = True")
#         print("   4. Update MODELS_DIR or KAGGLE_INPUT_DATASET path")
#         print("="*70)
#     else:
#         print("\n" + "="*70)
#         print("🆕 STARTING FRESH TRAINING")
#         print("="*70)
#         print("   RESUME_TRAINING is set to False")
#         print("   Training will start from epoch 1")
#         print("="*70)

# print(f"\n🎯 Training will run from epoch {start_epoch + 1} to {num_epochs}")
# print(f"💾 Checkpoints will be saved every {CHECKPOINT_INTERVAL} epoch(s)")
# print("="*70 + "\n")

# for epoch in range(start_epoch, num_epochs):
#     print(f"Epoch {epoch+1}/{num_epochs}")
#     coarse_generator.train()
#     fine_generator.train()
#     discriminator.train()
#     epoch_g_loss = epoch_d_loss = num_batches = 0
   
#     for step, images in enumerate(train_loader):
#         # Pass step counter for balanced D/G updates
#         g_loss, d_loss = train_step(images, coarse_generator, fine_generator, discriminator, g_optimizer, d_optimizer, scaler, step)
#         epoch_g_loss += g_loss
#         epoch_d_loss += d_loss
#         num_batches += 1
        
#         # Enhanced monitoring
#         if step % 50 == 0:
#             current_g_lr = g_optimizer.param_groups[0]['lr']
#             current_d_lr = d_optimizer.param_groups[0]['lr']
#             g_or_d = "G+D" if step % n_critic == 0 else "D only"
#             print(f" Step {step:3d} [{g_or_d}]: G_Loss: {g_loss:.4f}, D_Loss: {d_loss:.4f} | LR: G={current_g_lr:.6f}, D={current_d_lr:.6f}")
            
#         # Emergency stop if losses explode
#         if np.isnan(g_loss) and np.isnan(d_loss):
#             print(f"⚠️ CRITICAL: Both losses are NaN at step {step}. Stopping epoch early.")
#             break
   
#     avg_g_loss = epoch_g_loss / num_batches if num_batches > 0 else 0
#     avg_d_loss = epoch_d_loss / num_batches if num_batches > 0 else 0
#     train_g_losses.append(avg_g_loss)
#     train_d_losses.append(avg_d_loss)
#     print(f"Epoch {epoch+1} Training - G_Loss: {avg_g_loss:.4f}, D_Loss: {avg_d_loss:.4f}")
   
#     # Update learning rates
#     g_scheduler.step()
#     d_scheduler.step()
#     current_g_lr = g_optimizer.param_groups[0]['lr']
#     current_d_lr = d_optimizer.param_groups[0]['lr']
#     print(f" Current LR - G: {current_g_lr:.6f}, D: {current_d_lr:.6f}")
   
#     # Enable MTCNN for validation
#     if 'MTCNN' in globals():
#         USE_MTCNN = True
   
#     if epoch % 10 == 0:
#         # Get sample batch for graph creation
#         sample_images = next(iter(train_loader)).to(device)
#         sample_masks = generate_adaptive_mask(sample_images)
#         sample_masked = sample_images * sample_masks
        
#         # Get features from preprocessor
#         features = coarse_generator.preprocessor(sample_masked)
        
#         # Build graph
#         sample_graph = coarse_generator.preprocessor.build_graph(features)
        
#         pruned_graph = prune_graph(coarse_generator, sample_graph)
        
#         # Retrain
#         for _ in range(5):
#             retrain_loss = retrain(coarse_generator, pruned_graph, g_optimizer)
   
#     coarse_generator.eval()
#     fine_generator.eval()
#     discriminator.eval()
#     val_g_loss = val_d_loss = val_batches = 0
#     with torch.no_grad():
#         for images in val_loader:
#             images = images.to(device)
#             masks = generate_adaptive_mask(images)
#             masked_images = images * masks
#             coarse_output, mu, logvar = coarse_generator(masked_images, masks)
#             # Debug: Check shapes
#             if coarse_output.shape != images.shape:
#                 coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
           
#             fine_output = fine_generator(masked_images, coarse_output, masks)
#             # Debug: Check shapes
#             if fine_output.shape != images.shape:
#                 fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
           
#             combined_input = images * masks + fine_output * (1.0 - masks)
           
#             disc_real_logits, disc_features_real = discriminator(images, masks, return_features=True)
#             disc_fake_logits, disc_features_fake = discriminator(combined_input, masks, return_features=True)
           
#             g_loss = generator_loss(disc_fake_logits, images, fine_output, disc_features_real, disc_features_fake, masks, mu, logvar)
#             d_loss = discriminator_loss(disc_real_logits, disc_fake_logits)
           
#             val_g_loss += g_loss.item()
#             val_d_loss += d_loss.item()
#             val_batches += 1
   
#     avg_val_g_loss = val_g_loss / val_batches if val_batches > 0 else 0
#     avg_val_d_loss = val_d_loss / val_batches if val_batches > 0 else 0
#     val_g_losses.append(avg_val_g_loss)
#     val_d_losses.append(avg_val_d_loss)
#     print(f"Epoch {epoch+1} Validation - G_Loss: {avg_val_g_loss:.4f}, D_Loss: {avg_val_d_loss:.4f}")
   
#     if avg_val_g_loss < best_val_loss:
#         best_val_loss = avg_val_g_loss
#         print(f" New best validation loss! Saving best models...")
#         os.makedirs('/kaggle/working/models', exist_ok=True)
#         torch.save(coarse_generator.state_dict(), "/kaggle/working/models/coarse_generator_best.pth")
#         torch.save(fine_generator.state_dict(), "/kaggle/working/models/fine_generator_best.pth")
#         torch.save(discriminator.state_dict(), "/kaggle/working/models/discriminator_best.pth")
   
#     # Save checkpoint every CHECKPOINT_INTERVAL epochs
#     if (epoch + 1) % CHECKPOINT_INTERVAL == 0 or (epoch + 1) == num_epochs:
#         print(f"\n{'='*70}")
#         print(f"💾 SAVING CHECKPOINT - Epoch {epoch+1}/{num_epochs}")
#         print(f"{'='*70}")
        
#         checkpoint_file = os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch+1}.pth')
#         latest_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth')
        
#         checkpoint = {
#             'epoch': epoch,
#             'coarse_generator': coarse_generator.state_dict(),
#             'fine_generator': fine_generator.state_dict(),
#             'discriminator': discriminator.state_dict(),
#             'g_optimizer': g_optimizer.state_dict(),
#             'd_optimizer': d_optimizer.state_dict(),
#             'g_scheduler': g_scheduler.state_dict(),
#             'd_scheduler': d_scheduler.state_dict(),
#             'best_val_loss': best_val_loss,
#             'train_g_losses': train_g_losses,
#             'train_d_losses': train_d_losses,
#             'val_g_losses': val_g_losses,
#             'val_d_losses': val_d_losses,
#         }
        
#         # Save numbered checkpoint
#         torch.save(checkpoint, checkpoint_file)
#         checkpoint_size = os.path.getsize(checkpoint_file) / (1024**2)  # MB
#         print(f"✅ Saved: {checkpoint_file}")
#         print(f"   Size: {checkpoint_size:.2f} MB")
        
#         # Also save as latest checkpoint
#         torch.save(checkpoint, latest_checkpoint_path)
#         print(f"✅ Saved: {latest_checkpoint_path}")
        
#         print(f"\n📊 Checkpoint Info:")
#         print(f"   • Completed epochs: {epoch + 1}")
#         print(f"   • Best val loss: {best_val_loss:.6f}")
#         print(f"   • Current G loss: {avg_g_loss:.6f}")
#         print(f"   • Current D loss: {avg_d_loss:.6f}")
        
#         if (epoch + 1) < num_epochs:
#             print(f"\n🔄 To resume from this checkpoint in next run:")
#             print(f"   1. Download '{CHECKPOINT_DIR}' folder from Kaggle Output")
#             print(f"   2. Upload as Kaggle Dataset (or keep in working directory)")
#             print(f"   3. In code, set: RESUME_TRAINING = True")
#             print(f"   4. If using dataset, set: USE_KAGGLE_INPUT = True")
#             print(f"   5. Update: KAGGLE_INPUT_DATASET = '/kaggle/input/your-dataset-name'")
#         else:
#             print(f"\n🎉 TRAINING COMPLETED!")
#             print(f"   All {num_epochs} epochs finished successfully!")
        
#         print(f"{'='*70}\n")
   
#     # Plot training progress every 5 epochs
#     if (epoch + 1) % 5 == 0:
#         plt.figure(figsize=(15, 5))
       
#         plt.subplot(1, 3, 1)
#         plt.plot(range(1, len(train_g_losses) + 1), train_g_losses, label='Generator Loss', color='blue')
#         plt.plot(range(1, len(val_g_losses) + 1), val_g_losses, label='Val Generator Loss', color='lightblue')
#         plt.title('Generator Loss Progress')
#         plt.xlabel('Epoch')
#         plt.ylabel('Loss')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
       
#         plt.subplot(1, 3, 2)
#         plt.plot(range(1, len(train_d_losses) + 1), train_d_losses, label='Discriminator Loss', color='red')
#         plt.plot(range(1, len(val_d_losses) + 1), val_d_losses, label='Val Discriminator Loss', color='lightcoral')
#         plt.title('Discriminator Loss Progress')
#         plt.xlabel('Epoch')
#         plt.ylabel('Loss')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
       
#         plt.subplot(1, 3, 3)
#         plt.plot(range(1, len(train_g_losses) + 1), train_g_losses, label='G Loss', color='blue')
#         plt.plot(range(1, len(train_d_losses) + 1), train_d_losses, label='D Loss', color='red')
#         plt.title('Training Loss Comparison')
#         plt.xlabel('Epoch')
#         plt.ylabel('Loss')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
       
#         plt.tight_layout()
#         plt.savefig(f'/kaggle/working/training_progress_epoch_{epoch+1}.png', dpi=150, bbox_inches='tight')
#         plt.show()
# # Save models
# print("Saving models...")
# os.makedirs('/kaggle/working/models', exist_ok=True)
# torch.save(coarse_generator.state_dict(), "/kaggle/working/models/coarse_generator_bottom_half.pth")
# torch.save(fine_generator.state_dict(), "/kaggle/working/models/fine_generator_bottom_half.pth")
# torch.save(discriminator.state_dict(), "/kaggle/working/models/discriminator_bottom_half.pth")
# # Enable MTCNN for evaluation
# if 'MTCNN' in globals():
#     USE_MTCNN = True
# # Evaluate on test set
# print("Starting evaluation on test set...")
# os.makedirs('/kaggle/working/results', exist_ok=True)
# coarse_generator.eval()
# fine_generator.eval()
# discriminator.eval()
# psnr_values, ssim_values, mse_values, identity_values, landmark_values = [], [], [], [], []
# num_samples = 10 # Increased for better evaluation
# plt.figure(figsize=(20, 10))
# with torch.no_grad():
#     for i, images in enumerate(test_loader):
#         if i >= num_samples:
#             break
#         images = images.to(device)
#         masks = generate_adaptive_mask(images)
#         masked_images = images * masks
#         coarse_output, mu, logvar = coarse_generator(masked_images, masks)
#         # Debug: Check shapes
#         if coarse_output.shape != images.shape:
#             coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
       
#         fine_output = fine_generator(masked_images, coarse_output, masks)
#         # Debug: Check shapes
#         if fine_output.shape != images.shape:
#             fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
       
#         reconstructed = images * masks + fine_output * (1.0 - masks)
       
#         psnr_val = calculate_psnr_bottom_half(images, reconstructed, masks).item()
#         ssim_val = calculate_ssim_bottom_half(images, reconstructed, masks).item()
#         mse_val = calculate_mse_bottom_half(images, reconstructed, masks).item()
#         identity_val = identity_loss(images, reconstructed, masks).item()
#         landmark_val = landmark_guided_loss(images, reconstructed, masks).item()
       
#         psnr_values.append(psnr_val)
#         ssim_values.append(ssim_val)
#         mse_values.append(mse_val)
#         identity_values.append(identity_val)
#         landmark_values.append(landmark_val)
       
#         plt.subplot(4, num_samples, i + 1)
#         plt.title("Original Image")
#         plt.imshow(images[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#         plt.axis("off")
       
#         plt.subplot(4, num_samples, num_samples + i + 1)
#         plt.title("Top Half (Input)")
#         plt.imshow(masked_images[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#         plt.axis("off")
       
#         plt.subplot(4, num_samples, 2*num_samples + i + 1)
#         plt.title("Reconstructed Bottom Half")
#         plt.imshow(reconstructed[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#         plt.axis("off")
       
#         diff = torch.abs(images - reconstructed) * (1 - masks)
#         plt.subplot(4, num_samples, 3*num_samples + i + 1)
#         plt.title("Difference Map")
#         plt.imshow(diff[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5, cmap='hot')
#         plt.axis("off")
       
#         print(f"Sample {i+1}: PSNR: {psnr_val:.4f}, SSIM: {ssim_val:.4f}, MSE: {mse_val:.4f}, Identity: {identity_val:.4f}, Landmark: {landmark_val:.4f}")
# plt.tight_layout()
# plt.savefig('/kaggle/working/results/bottom_half_reconstruction_results.png', dpi=150, bbox_inches='tight')
# plt.show()
# # Plot metrics
# plt.figure(figsize=(20, 10))
# plt.subplot(2, 3, 1)
# plt.plot(range(1, num_samples + 1), psnr_values, marker='o', linewidth=2, markersize=8)
# plt.title("PSNR (Bottom Half)", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 2)
# plt.plot(range(1, num_samples + 1), ssim_values, marker='s', linewidth=2, markersize=8, color='orange')
# plt.title("SSIM (Bottom Half)", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 3)
# plt.plot(range(1, num_samples + 1), mse_values, marker='^', linewidth=2, markersize=8, color='green')
# plt.title("MSE (Bottom Half)", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 4)
# plt.plot(range(1, num_samples + 1), identity_values, marker='d', linewidth=2, markersize=8, color='purple')
# plt.title("Identity Loss", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 5)
# plt.plot(range(1, num_samples + 1), landmark_values, marker='*', linewidth=2, markersize=8, color='brown')
# plt.title("Landmark Loss", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.savefig('/kaggle/working/results/bottom_half_metrics.png', dpi=150, bbox_inches='tight')
# plt.show()
# # Print and save evaluation results
# print("\n" + "="*60)
# print("Enhanced Evaluation Metrics (Bottom Half) - Test Set:")
# print("="*60)
# print(f"Average PSNR: {np.mean(psnr_values):.4f} dB (±{np.std(psnr_values):.4f})")
# print(f"Average SSIM: {np.mean(ssim_values):.4f} (±{np.std(ssim_values):.4f})")
# print(f"Average MSE: {np.mean(mse_values):.4f} (±{np.std(mse_values):.4f})")
# print(f"Average Identity Loss: {np.mean(identity_values):.4f} (±{np.std(identity_values):.4f})")
# print(f"Average Landmark Loss: {np.mean(landmark_values):.4f} (±{np.std(landmark_values):.4f})")
# print("="*60)
# print(f"Best PSNR: {np.max(psnr_values):.4f} dB")
# print(f"Best SSIM: {np.max(ssim_values):.4f}")
# print(f"Worst PSNR: {np.min(psnr_values):.4f} dB")
# print(f"Worst SSIM: {np.min(ssim_values):.4f}")
# print("="*60)
# with open('/kaggle/working/results/enhanced_bottom_half_metrics.txt', 'w', encoding='utf-8') as f:
#     f.write("Enhanced Bottom Half Face Reconstruction Evaluation Results - Test Set\n")
#     f.write("="*60 + "\n")
#     f.write(f"Average PSNR: {np.mean(psnr_values):.4f} dB (±{np.std(psnr_values):.4f})\n")
#     f.write(f"Average SSIM: {np.mean(ssim_values):.4f} (±{np.std(ssim_values):.4f})\n")
#     f.write(f"Average MSE: {np.mean(mse_values):.4f} (±{np.std(mse_values):.4f})\n")
#     f.write(f"Average Identity Loss: {np.mean(identity_values):.4f} (±{np.std(identity_values):.4f})\n")
#     f.write(f"Average Landmark Loss: {np.mean(landmark_values):.4f} (±{np.std(landmark_values):.4f})\n")
#     f.write("="*60 + "\n")
#     f.write(f"Best PSNR: {np.max(psnr_values):.4f} dB\n")
#     f.write(f"Best SSIM: {np.max(ssim_values):.4f}\n")
#     f.write(f"Worst PSNR: {np.min(psnr_values):.4f} dB\n")
#     f.write(f"Worst SSIM: {np.min(ssim_values):.4f}\n")
#     f.write("="*60 + "\n")
#     for i, (psnr_val, ssim_val, mse_val, id_val, lm_val) in enumerate(zip(psnr_values, ssim_values, mse_values, identity_values, landmark_values)):
#         f.write(f"Sample {i+1}: PSNR={psnr_val:.4f}, SSIM={ssim_val:.4f}, MSE={mse_val:.4f}, Identity={id_val:.4f}, Landmark={lm_val:.4f}\n")
# print(f"\nTraining and evaluation complete!")
# print(f"Models saved in '/kaggle/working/models/' directory.")
# print(f"Evaluation results saved in '/kaggle/working/results/' directory.")

In [ ]:
# import os
# os.environ.setdefault('TORCH_CUDA_ARCH_LIST', '7.5') # For Kaggle GPU compatibility
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torchvision.models as models
# import torchvision.transforms as transforms
# from torch.utils.data import Dataset, DataLoader
# import matplotlib.pyplot as plt
# import numpy as np
# from skimage.metrics import peak_signal_noise_ratio as psnr
# from skimage.metrics import structural_similarity as ssim
# import random
# import cv2
# from PIL import Image
# from torch import amp  # For mixed precision training (new API)
# import networkx as nx  # For graph operations

# # Simplified GNN for graph processing
# class PyramidalGNN(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super(PyramidalGNN, self).__init__()
#         self.lin = nn.Linear(in_channels, out_channels)
#         self.gate = nn.Linear(in_channels * 2, out_channels)

#     def forward(self, x, edge_index):
#         # Simplified version - just apply linear transformation
#         return self.lin(x)



# # /kaggle/input/celeba-resized-6464/img_align_celeba/CelebA_Image_Cropped_64
# # Configuration parameters
# prune_threshold = 0.2  # Threshold for pruning low-value nodes
# lambda_l1 = 100.0  # L1 reconstruction loss weight
# n_critic = 5  # Number of discriminator updates per generator update

# # Global prune_graph function using NetworkX
# def prune_graph(model, graph, threshold=prune_threshold):
#     # NetworkX pruning
#     deg_cent = nx.degree_centrality(graph)
#     low_value_nodes = [n for n, score in deg_cent.items() if score < threshold]
#     graph.remove_nodes_from(low_value_nodes)
#     return graph

# # Try to import MTCNN for landmark detection
# try:
#     from mtcnn import MTCNN
#     USE_MTCNN = False # Disable during training to avoid NumPy conversions
#     print("MTCNN imported successfully for landmark detection (disabled for training).")
# except ImportError:
#     print("MTCNN not found. Landmark loss will be simplified.")
#     USE_MTCNN = False
# # ============================================================================
# # CHECKPOINT CONFIGURATION - Resume Training Setup
# # تنظیمات Checkpoint - برای ادامه آموزش
# # ============================================================================

# # 🔄 RESUME TRAINING: Set to True to continue from previous checkpoint
# # ادامه آموزش: برای ادامه از checkpoint قبلی، True کنید
# RESUME_TRAINING = True  # True: Resume from checkpoint | False: Start from scratch

# # 📂 CHECKPOINT PATHS: Where to load/save checkpoints
# # مسیرهای Checkpoint: محل بارگذاری/ذخیره checkpoint‌ها
# CHECKPOINT_DIR = '/kaggle/working/checkpoints'
# CHECKPOINT_INTERVAL = 1  # Save checkpoint every N epochs (1 = every epoch)

# # 📥 LOAD FROM KAGGLE DATASET: If you uploaded models as Kaggle dataset
# # بارگذاری از Kaggle Dataset: اگر مدل‌ها را به عنوان dataset آپلود کرده‌اید
# USE_KAGGLE_INPUT = True  # Set True if loading from Kaggle dataset input
# KAGGLE_INPUT_DATASET = '/kaggle/input/image-inpainting/pytorch/default/4'  # Update this path (e.g., '/kaggle/input/checkpoints')
# # /kaggle/input/image-inpainting/pytorch/default/2
# # /kaggle/input/image-inpainting/pytorch/default/4
# # /kaggle/input/image-inpainting/pytorch/default/3
# # 🗂️ LOAD FROM MODEL FILES: If you only have model state_dict files (*.pth)
# # بارگذاری از فایل‌های مدل: اگر فقط فایل‌های state_dict مدل دارید
# LOAD_FROM_MODELS = True  # Set True if loading from models folder (coarse_generator_best.pth, etc.)
# MODELS_DIR = '/kaggle/working/models'  # Path to models folder
# USE_BEST_MODELS = True  # True: Load *_best.pth | False: Load *_bottom_half.pth

# # ⚠️ نکته: اگر از LOAD_FROM_MODELS استفاده کنید، فقط وزن‌های مدل بارگذاری می‌شوند
# # optimizer/scheduler/training history بارگذاری نمی‌شوند و از اول مقداردهی می‌شوند

# # ⚙️ TRAINING START POINT: Which epoch to start from
# # نقطه شروع آموزش: از کدام epoch شروع شود
# # Note: This will be automatically set when loading checkpoint
# # توجه: این مقدار به صورت خودکار هنگام بارگذاری checkpoint تنظیم می‌شود

# print("="*70)
# print("🔧 CHECKPOINT CONFIGURATION")
# print("="*70)
# print(f"📌 Resume Training: {RESUME_TRAINING}")
# print(f"📁 Checkpoint Directory: {CHECKPOINT_DIR}")
# print(f"💾 Save Interval: Every {CHECKPOINT_INTERVAL} epoch(s)")
# if USE_KAGGLE_INPUT:
#     print(f"📥 Loading from Kaggle Dataset: {KAGGLE_INPUT_DATASET}")
# if LOAD_FROM_MODELS:
#     print(f"🗂️  Load from Models: {MODELS_DIR}")
#     print(f"   Using: {'*_best.pth' if USE_BEST_MODELS else '*_bottom_half.pth'}")
# print("="*70 + "\n")

# # ============================================================================
# # TRAINING CONFIGURATION - Initial configurations
# # تنظیمات آموزش - پیکربندی اولیه
# # ============================================================================

# image_size = (64, 64)
# batch_size = 128 # Further reduced for stability
# eval_batch_size = 1
# num_epochs = 10 # Increased for better convergence
# learning_rate_g = 0.0001 # 1e-4 for Generator
# learning_rate_d = 0.0004 # 4e-4 for Discriminator (higher to strengthen D)
# lambda_adv = 0.05 # Further reduced adversarial weight
# lambda_style = 0.05 # Reduced style weight
# lambda_fm = 0.05 # Reduced feature matching weight
# lambda_rec = 1.0 # Reduced reconstruction weight
# lambda_identity = 0.005 # Reduced identity weight
# lambda_landmark = 0.01 # Reduced landmark weight
# lambda_perc = 0.1 # Reduced perceptual weight
# lambda_vae = 0.01 # Further reduced VAE weight
# lambda_struct = 0.05 # Reduced structural weight
# lambda_d_reg = 1e-4 # Increased discriminator regularization
# lambda_attention = 0.05 # Reduced attention loss weight
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")
# # Data splitting configuration
# def split_dataset(dataset_path, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     image_files = [f for f in os.listdir(dataset_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
#     random.shuffle(image_files)
#     total_files = len(image_files)
#     train_end = int(total_files * train_ratio)
#     val_end = train_end + int(total_files * val_ratio)
#     train_files, val_files, test_files = image_files[:train_end], image_files[train_end:val_end], image_files[val_end:]
#     print(f"Dataset split: Total={total_files}, Train={len(train_files)} ({len(train_files)/total_files*100:.1f}%), "
#           f"Val={len(val_files)} ({len(val_files)/total_files*100:.1f}%), Test={len(test_files)} ({len(test_files)/total_files*100:.1f}%)")
#     return train_files, val_files, test_files
# # Custom Dataset
# class CelebADataset(Dataset):
#     def __init__(self, dataset_path, file_list, image_size):
#         self.dataset_path = dataset_path
#         self.file_list = file_list
#         self.transform = transforms.Compose([
#             transforms.Resize(image_size),
#             transforms.ToTensor(),
#             transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # -1 to 1
#         ])
   
#     def __len__(self):
#         return len(self.file_list)
   
#     def __getitem__(self, idx):
#         img_path = os.path.join(self.dataset_path, self.file_list[idx])
#         image = Image.open(img_path).convert('RGB')
#         image = self.transform(image)
#         return image
# def create_split_datasets(dataset_path, train_files, val_files, test_files, image_size, batch_size):
#     train_dataset = CelebADataset(dataset_path, train_files, image_size)
#     val_dataset = CelebADataset(dataset_path, val_files, image_size)
#     test_dataset = CelebADataset(dataset_path, test_files, image_size)
   
#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
#     val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
#     test_loader = DataLoader(test_dataset, batch_size=eval_batch_size, shuffle=False, num_workers=2, pin_memory=True)
#     return train_loader, val_loader, test_loader
# # Enhanced ViT-based Preprocessor with Multi-layer Attention and Graph Pruning
# class ViTPreprocessor(nn.Module):
#     def __init__(self, dim=128, num_heads=8, patch_size=8, num_layers=3):
#         super().__init__()
#         self.patch_size = patch_size
#         self.dim = dim
#         self.num_heads = num_heads
#         self.num_patches = (image_size[0] // patch_size) * (image_size[1] // patch_size)
#         self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size, padding=0)
#         self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, dim))
       
#         # Multi-layer attention blocks
#         self.attention_blocks = nn.ModuleList([
#             nn.ModuleDict({
#                 'attn': nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=0.1),
#                 'norm1': nn.LayerNorm(dim, eps=1e-6),
#                 'norm2': nn.LayerNorm(dim, eps=1e-6),
#                 'mlp': nn.Sequential(
#                     nn.Linear(dim, dim * 4),
#                     nn.GELU(),
#                     nn.Dropout(0.1),
#                     nn.Linear(dim * 4, dim),
#                     nn.Dropout(0.1)
#                 )
#             }) for _ in range(num_layers)
#         ])
       
#         self.final_norm = nn.LayerNorm(dim, eps=1e-6)
#         self.gnn = PyramidalGNN(dim, dim)  # Add PyramidalGNN
   
#     def build_graph(self, features):
#         B, C, H, W = features.shape
#         nodes = features.view(B, C, -1).permute(0, 2, 1)  # [B, num_patches, dim]
        
#         # Use NetworkX for graph construction
#         G = nx.Graph()
#         for i in range(nodes.shape[1]):
#             G.add_node(i, feat=nodes[0, i])
#         for i in range(nodes.shape[1]):
#             for j in range(i+1, nodes.shape[1]):
#                 sim = F.cosine_similarity(nodes[0, i:i+1], nodes[0, j:j+1])
#                 if sim > 0.5:
#                     G.add_edge(i, j, weight=sim)
#         return G
   
   
#     def graph_enhanced_forward(self, x, graph):
#         # Check if graph is empty after pruning
#         node_list = list(graph.nodes)
#         if not node_list:
#             return x  # Return input features directly
        
#         # Simple message passing with NetworkX
#         for node in graph.nodes:
#             neighbors = list(graph.neighbors(node))
#             if neighbors:
#                 neighbor_feats = torch.stack([graph.nodes[n]['feat'] for n in neighbors])
#                 aggregated = torch.mean(neighbor_feats, dim=0)
#                 graph.nodes[node]['feat'] = (graph.nodes[node]['feat'] + aggregated) / 2
        
#         # Final check before stacking
#         final_nodes = list(graph.nodes)
#         if not final_nodes:
#             return x
        
#         # Ensure all features have the same dtype
#         node_features = [graph.nodes[n]['feat'] for n in sorted(final_nodes)]
#         return torch.stack(node_features)
   
#     def reconstruct_features(self, pruned_feats, pruned_indices, B, C, H, W):
#         num_patches = H * W
#         # Ensure dtype matches the input features
#         full_feats = torch.zeros(B, num_patches, C, device=pruned_feats.device, dtype=pruned_feats.dtype)
#         full_feats[:, pruned_indices] = pruned_feats
#         return full_feats.reshape(B, H, W, C).permute(0, 3, 1, 2)
   
#     def forward(self, x):
#         embedded = self.patch_embed(x)  # [B, dim, H/p, W/p]
#         B, C, H, W = embedded.shape
#         flat = embedded.permute(0, 2, 3, 1).reshape(B, H * W, C)
        
#         # Adaptive position embedding based on actual patch count
#         num_patches = H * W
#         if num_patches != self.num_patches:
#             # Interpolate position embedding to match current patch count
#             # Reshape to 3D for linear interpolation: [1, dim, num_patches]
#             pos_embed_3d = self.pos_embed.permute(0, 2, 1)  # [1, dim, num_patches]
#             pos_embed_adaptive = F.interpolate(
#                 pos_embed_3d,
#                 size=num_patches,
#                 mode='linear',
#                 align_corners=False
#             ).permute(0, 2, 1)  # [1, num_patches, dim]
#         else:
#             pos_embed_adaptive = self.pos_embed
        
#         flat = flat + pos_embed_adaptive
        
#         # Build and prune graph
#         graph = self.build_graph(embedded)
#         pruned_graph = prune_graph(self, graph)
        
#         # Get pruned features using NetworkX
#         pruned_indices = list(pruned_graph.nodes)
#         if pruned_indices:  # Check if not empty
#             pruned_feats = torch.stack([pruned_graph.nodes[i]['feat'] for i in pruned_indices])
#         else:
#             pruned_feats = flat[0].clone()  # Use original flattened features for batch 0
#             pruned_indices = list(range(flat.shape[1]))  # All indices
        
#         # Enhanced processing on high-value nodes
#         for block in self.attention_blocks:
#             # Self-attention on pruned features
#             attn_out, _ = block['attn'](pruned_feats, pruned_feats, pruned_feats)
#             pruned_feats = block['norm1'](pruned_feats + attn_out)
            
#             # MLP
#             mlp_out = block['mlp'](pruned_feats)
#             pruned_feats = block['norm2'](pruned_feats + mlp_out)
        
#         # Graph-enhanced message passing
#         enhanced_pruned = self.graph_enhanced_forward(pruned_feats, pruned_graph)
        
#         # Reconstruct full features with low-value nodes getting minimal processing
#         x = self.reconstruct_features(enhanced_pruned, pruned_indices, B, C, H, W)
        
#         # Minimal processing for pruned (low-value) regions - e.g., average pooling
#         full_x = self.final_norm(x.view(B, C, -1).permute(0, 2, 1)).permute(0, 2, 1).view(B, C, H, W)
#         return full_x
# # VAE Sampling Layer
# class VAESampling(nn.Module):
#     def __init__(self):
#         super().__init__()
   
#     def forward(self, mu, logvar):
#         std = torch.exp(0.5 * logvar)
#         eps = torch.randn_like(std)
#         return mu + eps * std
# # Enhanced VAE Encoder with Residual Connections and Graph Integration (No BatchNorm, Strided Conv)
# class VAEEncoder(nn.Module):
#     def __init__(self, latent_dim=128):
#         super().__init__()
#         self.latent_dim = latent_dim
       
#         # Enhanced encoder with strided convolutions (no batch norm)
#         self.conv1 = nn.Conv2d(128, 64, kernel_size=4, stride=2, padding=1) # [B, 128, 8, 8] -> [B, 64, 4, 4]
#         self.conv2 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1) # [B, 64, 4, 4] -> [B, 128, 2, 2]
       
#         # Residual block (no batch norm)
#         self.res_conv = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
       
#         self.flatten = nn.Flatten()
#         # Use adaptive pooling to ensure consistent output size
#         self.adaptive_pool = nn.AdaptiveAvgPool2d((2, 2))  # Always output 2x2
#         self.fc_mu = nn.Linear(128 * 2 * 2, latent_dim) # 128 * 2 * 2 = 512
#         self.fc_logvar = nn.Linear(128 * 2 * 2, latent_dim) # 128 * 2 * 2 = 512
   
#     def forward(self, x):
#         # Check input size and use adaptive approach for very small inputs
#         B, C, H, W = x.shape
        
#         # If input is too small for conv layers, use adaptive pooling directly
#         if H < 4 or W < 4:
#             # Use adaptive pooling to get to a reasonable size first
#             x = F.adaptive_avg_pool2d(x, (8, 8))  # Resize to 8x8 minimum
#             x = F.relu(self.conv1(x))  # 8x8 -> 4x4
#             x = F.relu(self.conv2(x))  # 4x4 -> 2x2
#         else:
#             x = F.relu(self.conv1(x))
#             x = F.relu(self.conv2(x))
       
#         # Residual connection
#         residual = x
#         x = F.relu(self.res_conv(x))
#         x = x + residual
       
#         # Use adaptive pooling to ensure consistent size
#         x = self.adaptive_pool(x)  # Always [B, 128, 2, 2]
#         x = self.flatten(x)  # Always [B, 512]
#         mu = self.fc_mu(x)
#         logvar = self.fc_logvar(x)
#         return mu, logvar
# # Enhanced VAE Decoder with Residual Connections and Graph Integration (No BatchNorm)
# class VAEDecoder(nn.Module):
#     def __init__(self, latent_dim=128):
#         super().__init__()
#         self.latent_dim = latent_dim
       
#         # Enhanced decoder with strided transposed convolutions (no batch norm)
#         self.fc = nn.Linear(latent_dim, 8 * 8 * 128)
       
#         # Residual block before upsampling (no batch norm)
#         self.res_conv = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
       
#         self.deconv1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
#         self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
#         self.deconv3 = nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1)
   
#     def forward(self, z):
#         x = F.relu(self.fc(z))
#         x = x.view(-1, 128, 8, 8)
       
#         # Residual connection
#         residual = x
#         x = F.relu(self.res_conv(x))
#         x = x + residual
       
#         x = F.relu(self.deconv1(x)) # 8x8 -> 16x16
#         x = F.relu(self.deconv2(x)) # 16x16 -> 32x32
#         x = torch.tanh(self.deconv3(x)) # 32x32 -> 64x64
#         return x
# # Enhanced CoarseGenerator with Attention, Skip Connections, and Graph Pruning
# class CoarseGenerator(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.preprocessor = ViTPreprocessor(dim=128, num_heads=8, patch_size=8, num_layers=3)
#         self.encoder = VAEEncoder(latent_dim=128) # Increased latent dimension
#         self.sampling = VAESampling()
#         self.decoder = VAEDecoder(latent_dim=128)
       
#         # Attention mechanism for better feature fusion
#         self.attention = nn.MultiheadAttention(embed_dim=128, num_heads=8, dropout=0.1)
#         self.attention_norm = nn.LayerNorm(128, eps=1e-6)
       
#         # Skip connection processing
#         self.skip_conv = nn.Conv2d(128, 128, kernel_size=1, padding=0)
   
#     def forward(self, x, mask):
#         # Preprocess with graph pruning
#         original_features = self.preprocessor(x)
       
#         # Apply attention to enhance features
#         B, C, H, W = original_features.shape
#         attn_input = original_features.permute(0, 2, 3, 1).reshape(B, H * W, C)
#         attn_output, _ = self.attention(attn_input, attn_input, attn_input)
#         enhanced_features = self.attention_norm(attn_input + attn_output)
#         enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
       
#         # Combine with skip connection
#         enhanced_features = enhanced_features + self.skip_conv(original_features)
       
#         mu, logvar = self.encoder(enhanced_features)
#         z = self.sampling(mu, logvar)
#         output = self.decoder(z)
#         return output, mu, logvar
# # Enhanced FineGenerator with U-Net Architecture, Attention, and Graph Cut Integration (No BatchNorm)
# class FineGenerator(nn.Module):
#     def __init__(self):
#         super().__init__()
       
#         # Encoder with skip connections (no batch norm, strided conv)
#         self.enc1 = nn.Sequential(
#             nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
#         self.enc2 = nn.Sequential(
#             nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
#         self.enc3 = nn.Sequential(
#             nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
       
#         # Bottleneck with attention (no batch norm)
#         self.bottleneck = nn.Sequential(
#             nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
#             nn.LeakyReLU(0.2, inplace=True)
#         )
       
#         # Attention mechanism
#         self.attention = nn.MultiheadAttention(embed_dim=512, num_heads=8, dropout=0.1)
#         self.attention_norm = nn.LayerNorm(512, eps=1e-6)
       
#         # Decoder with skip connections (no batch norm, strided transposed conv)
#         self.dec1 = nn.Sequential(
#             nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
#             nn.ReLU(inplace=True)
#         )
#         self.dec2 = nn.Sequential(
#             nn.ConvTranspose2d(512, 128, kernel_size=4, stride=2, padding=1), # 256 + 256 from skip
#             nn.ReLU(inplace=True)
#         )
#         self.dec3 = nn.Sequential(
#             nn.ConvTranspose2d(256, 64, kernel_size=4, stride=2, padding=1), # 128 + 128 from skip
#             nn.ReLU(inplace=True)
#         )
#         self.dec4 = nn.Sequential(
#             nn.ConvTranspose2d(128, 3, kernel_size=4, stride=2, padding=1), # 64 + 64 from skip
#             nn.Tanh()
#         )
   
#     def forward(self, masked_images, coarse_output, masks):
#         # Ensure all inputs have the same spatial dimensions
#         if masked_images.shape[2:] != coarse_output.shape[2:]:
#             coarse_output = F.interpolate(coarse_output, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
       
#         x = torch.cat([masked_images, coarse_output], dim=1)
       
#         # Encoder with skip connections
#         enc1_out = self.enc1(x) # 64x64 -> 32x32
#         enc2_out = self.enc2(enc1_out) # 32x32 -> 16x16
#         enc3_out = self.enc3(enc2_out) # 16x16 -> 8x8
#         bottleneck = self.bottleneck(enc3_out) # 8x8 -> 4x4
       
#         # Apply attention to bottleneck features
#         B, C, H, W = bottleneck.shape
#         attn_input = bottleneck.permute(0, 2, 3, 1).reshape(B, H * W, C)
#         attn_output, _ = self.attention(attn_input, attn_input, attn_input)
#         enhanced_features = self.attention_norm(attn_input + attn_output)
#         enhanced_features = enhanced_features.reshape(B, H, W, C).permute(0, 3, 1, 2)
       
#         # Decoder with skip connections
#         dec1_out = self.dec1(enhanced_features) # 4x4 -> 8x8
#         dec1_out = torch.cat([dec1_out, enc3_out], dim=1) # Skip connection
       
#         dec2_out = self.dec2(dec1_out) # 8x8 -> 16x16
#         dec2_out = torch.cat([dec2_out, enc2_out], dim=1) # Skip connection
       
#         dec3_out = self.dec3(dec2_out) # 16x16 -> 32x32
#         dec3_out = torch.cat([dec3_out, enc1_out], dim=1) # Skip connection
       
#         output = self.dec4(dec3_out) # 32x32 -> 64x64
       
#         # Ensure output matches input size
#         if output.shape[2:] != masked_images.shape[2:]:
#             output = F.interpolate(output, size=masked_images.shape[2:], mode='bilinear', align_corners=False)
       
#         return output
# # Stronger Discriminator with Spectral Normalization (No BatchNorm, More Layers)
# class Discriminator(nn.Module):
#     def __init__(self):
#         super().__init__()
       
#         # Stronger discriminator with more layers and capacity (no batch norm)
#         self.conv1 = nn.utils.spectral_norm(nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1))
#         self.conv2 = nn.utils.spectral_norm(nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1))
#         self.conv3 = nn.utils.spectral_norm(nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1))
#         self.conv4 = nn.utils.spectral_norm(nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1))
#         # Additional layers for stronger discriminator
#         self.conv5 = nn.utils.spectral_norm(nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1))
#         self.conv6 = nn.utils.spectral_norm(nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1))
       
#         # Global average pooling for stability
#         self.global_pool = nn.AdaptiveAvgPool2d(1)
#         # Deeper fully connected layers
#         self.fc1 = nn.utils.spectral_norm(nn.Linear(512, 256))
#         self.fc2 = nn.utils.spectral_norm(nn.Linear(256, 1))
       
#         # Feature layers for feature matching loss
#         self.feature_layers = [self.conv1, self.conv2, self.conv3, self.conv4, self.conv5, self.conv6]
   
#     def forward(self, x, mask, return_features=False):
#         input_concat = torch.cat([x, mask], dim=1)
#         features = []
       
#         # Main discriminator path with more layers
#         x = F.leaky_relu(self.conv1(input_concat), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv2(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv3(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv4(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv5(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         x = F.leaky_relu(self.conv6(x), 0.2, inplace=True)
#         if return_features:
#             features.append(x)
       
#         # Global average pooling for stability
#         x = self.global_pool(x)
#         x = x.view(x.size(0), -1)
        
#         # Deeper FC layers
#         x = F.leaky_relu(self.fc1(x), 0.2, inplace=True)
#         logits = self.fc2(x)
       
#         if return_features:
#             return logits, features
#         return logits
# # Generate adaptive mask
# def generate_adaptive_mask(image):
#     batch_size = image.shape[0]
#     h, w = image_size
#     mask = torch.ones(batch_size, 3, h, w, dtype=torch.float32, device=image.device)
#     half_height = h // 2
#     bottom_mask = torch.zeros(batch_size, 3, h - half_height, w, dtype=torch.float32, device=image.device)
#     top_mask = torch.ones(batch_size, 3, half_height, w, dtype=torch.float32, device=image.device)
#     mask = torch.cat([top_mask, bottom_mask], dim=2)
   
#     if USE_MTCNN:
#         detector = MTCNN()
#         images_np = ((image * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
#         landmarks = []
#         for img in images_np:
#             img_uint8 = img.astype(np.uint8)
#             result = detector.detect_faces(img_uint8)
#             landmarks.append(result[0]['keypoints'] if result else {})
       
#         mask_np = mask.permute(0, 2, 3, 1).cpu().detach().numpy()
#         for i, lm in enumerate(landmarks):
#             if lm:
#                 for key in ['mouth_left', 'mouth_right']:
#                     if key in lm:
#                         x, y = lm[key]
#                         y = min(max(y, half_height), h-1)
#                         x = min(max(x, 0), w-1)
#                         mask_np[i, y-5:y+5, x-5:x+5, :] = 0.0
#         mask = torch.from_numpy(mask_np).permute(0, 3, 1, 2).to(device)
   
#     return mask
# # Loss functions
# # Use new torchvision weights API to avoid deprecation warnings
# try:
#     vgg_weights = models.VGG16_Weights.DEFAULT
# except AttributeError:
#     # Fallback for very old torchvision
#     vgg_weights = None
# vgg = models.vgg16(weights=vgg_weights).features.to(device).eval()
# loss_model = nn.Sequential(*[vgg[i] for i in range(16)]).to(device) # Up to block4_conv3
# for param in loss_model.parameters():
#     param.requires_grad = False
# def perceptual_loss(y_true, y_pred):
#     y_true = F.interpolate(y_true, size=image_size, mode='bilinear', align_corners=False)
#     y_pred = F.interpolate(y_pred, size=image_size, mode='bilinear', align_corners=False)
#     y_true = y_true * 0.5 + 0.5 # Denormalize to [0, 1]
#     y_pred = y_pred * 0.5 + 0.5
#     true_features = loss_model(y_true)
#     pred_features = loss_model(y_pred)
#     return torch.mean((true_features - pred_features) ** 2)
# def style_loss(y_true, y_pred):
#     y_true = F.interpolate(y_true, size=image_size, mode='bilinear', align_corners=False)
#     y_pred = F.interpolate(y_pred, size=image_size, mode='bilinear', align_corners=False)
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     true_features = loss_model(y_true)
#     pred_features = loss_model(y_pred)
   
#     def gram_matrix(feat):
#         B, C, H, W = feat.shape
#         feat = feat.view(B, C, H * W)
#         return torch.bmm(feat, feat.transpose(1, 2)) / (C * H * W)
   
#     style_losses = [torch.mean((gram_matrix(t) - gram_matrix(p)) ** 2) for t, p in zip([true_features], [pred_features])]
#     return torch.mean(torch.stack(style_losses))
# def bottom_half_loss(y_true, y_pred, mask):
#     bottom_region = (mask == 0).float()
#     diff = (y_true - y_pred) ** 2 * bottom_region
#     return diff.sum() / (bottom_region.sum() + 1e-8)
# def feature_matching_loss(real_features, fake_features):
#     losses = [torch.mean((r - f) ** 2) for r, f in zip(real_features, fake_features)]
#     return torch.mean(torch.stack(losses))
# def identity_loss(y_true, y_pred, mask):
#     bottom_region = (mask == 0).float()
#     y_true_face = y_true * bottom_region
#     y_pred_face = y_pred * bottom_region
#     return torch.mean(torch.abs(y_true_face - y_pred_face))
# def landmark_guided_loss(y_true, y_pred, mask):
#     if not USE_MTCNN:
#         return bottom_half_loss(y_true, y_pred, mask)
#     detector = MTCNN()
#     bottom_region = (mask == 0).float()
#     y_true_bottom = y_true * bottom_region
#     y_pred_bottom = y_pred * bottom_region
#     y_true_np = ((y_true_bottom * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
#     y_pred_np = ((y_pred_bottom * 0.5 + 0.5) * 255.0).permute(0, 2, 3, 1).cpu().detach().numpy()
#     loss_tensor = torch.tensor(0.0, device=device, requires_grad=True)
#     for true_img, pred_img in zip(y_true_np, y_pred_np):
#         true_img_uint8 = true_img.astype(np.uint8)
#         pred_img_uint8 = pred_img.astype(np.uint8)
#         true_lm = detector.detect_faces(true_img_uint8)
#         pred_lm = detector.detect_faces(pred_img_uint8)
#         if true_lm and pred_lm:
#             true_points = [true_lm[0]['keypoints'][k] for k in ['mouth_left', 'mouth_right']]
#             pred_points = [pred_lm[0]['keypoints'][k] for k in ['mouth_left', 'mouth_right']]
#             for t, p in zip(true_points, pred_points):
#                 diff_x = torch.tensor(t[0] - p[0], dtype=torch.float32, device=device)
#                 diff_y = torch.tensor(t[1] - p[1], dtype=torch.float32, device=device)
#                 loss_tensor = loss_tensor + (diff_x ** 2 + diff_y ** 2)
#     return loss_tensor / y_true.shape[0]
# def structural_ssim_loss(y_true, y_pred, mask):
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     bottom_mask = (mask == 0).float()
#     y_true_bottom = y_true * bottom_mask
#     y_pred_bottom = y_pred * bottom_mask
#     ssim_vals = []
#     for i in range(y_true.shape[0]):
#         true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         ssim_val = ssim(true_img, pred_img, channel_axis=2, data_range=1.0, win_size=7)
#         ssim_vals.append(ssim_val)
#     return 1.0 - torch.mean(torch.tensor(ssim_vals, device=device))
# # Advanced Loss Functions
# def gradient_penalty_loss(discriminator, real_images, fake_images, masks, device):
#     """Calculate gradient penalty for WGAN-GP"""
#     batch_size = real_images.size(0)
#     alpha = torch.rand(batch_size, 1, 1, 1).to(device)
#     interpolated = alpha * real_images + (1 - alpha) * fake_images
   
#     # Ensure interpolated tensor requires gradients
#     interpolated = interpolated.detach().requires_grad_(True)
   
#     disc_interpolated = discriminator(interpolated, masks, return_features=False)
   
#     # Check if disc_interpolated requires gradients
#     if not disc_interpolated.requires_grad:
#         return torch.tensor(0.0, device=device, requires_grad=True)
   
#     gradients = torch.autograd.grad(
#         outputs=disc_interpolated,
#         inputs=interpolated,
#         grad_outputs=torch.ones_like(disc_interpolated).to(device),
#         create_graph=True,
#         retain_graph=True,
#         only_inputs=True
#     )[0]
   
#     gradients = gradients.view(gradients.size(0), -1)
#     gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
#     return gradient_penalty
# def attention_loss(y_true, y_pred, mask):
#     """Simplified attention-based loss focusing on important regions"""
#     bottom_region = (mask == 0).float()
#     diff = torch.abs(y_true - y_pred) * bottom_region
   
#     # Create simple attention weights based on gradient magnitude
#     true_grad_x = torch.abs(y_true[:, :, :, 1:] - y_true[:, :, :, :-1])
#     true_grad_y = torch.abs(y_true[:, :, 1:, :] - y_true[:, :, :-1, :])
   
#     # Pad gradients to match original size
#     true_grad_x_padded = F.pad(true_grad_x, (0, 1, 0, 0), mode='constant', value=0)
#     true_grad_y_padded = F.pad(true_grad_y, (0, 0, 0, 1), mode='constant', value=0)
   
#     # Combine gradients and normalize
#     attention_weights = (true_grad_x_padded + true_grad_y_padded).mean(dim=1, keepdim=True)
#     attention_weights = attention_weights / (attention_weights.mean() + 1e-8) # Normalize
   
#     # Apply attention weights
#     weighted_loss = diff * attention_weights
#     return weighted_loss.mean()
# def generator_loss(disc_fake_logits, y_true, y_pred, disc_features_real, disc_features_fake, mask, mu, logvar):
#     # Clamp logvar to prevent exp overflow
#     logvar = torch.clamp(logvar, min=-10, max=10)
    
#     # Calculate each loss with NaN protection
#     adv_loss = -torch.mean(disc_fake_logits)
#     adv_loss = torch.nan_to_num(adv_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     # L1 reconstruction loss (high weight for stable inpainting)
#     bottom_region = (mask == 0).float()
#     l1_loss = torch.mean(torch.abs(y_true - y_pred) * bottom_region)
#     l1_loss = torch.nan_to_num(l1_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     rec_loss = bottom_half_loss(y_true, y_pred, mask)
#     rec_loss = torch.nan_to_num(rec_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     perc_loss = perceptual_loss(y_true, y_pred)
#     perc_loss = torch.nan_to_num(perc_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     sty_loss = style_loss(y_true, y_pred)
#     sty_loss = torch.nan_to_num(sty_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     fm_loss = feature_matching_loss(disc_features_real, disc_features_fake)
#     fm_loss = torch.nan_to_num(fm_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     id_loss = identity_loss(y_true, y_pred, mask)
#     id_loss = torch.nan_to_num(id_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     lm_loss = landmark_guided_loss(y_true, y_pred, mask)
#     lm_loss = torch.nan_to_num(lm_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     struct_loss = structural_ssim_loss(y_true, y_pred, mask)
#     struct_loss = torch.nan_to_num(struct_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     att_loss = attention_loss(y_true, y_pred, mask)
#     att_loss = torch.nan_to_num(att_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     # KL loss with clipping
#     kl_loss = -0.5 * torch.mean(1 + logvar - mu**2 - torch.exp(logvar))
#     kl_loss = torch.nan_to_num(kl_loss, nan=0.0, posinf=0.0, neginf=0.0)
#     kl_loss = torch.clamp(kl_loss, min=0.0, max=10.0)  # Prevent extreme values
    
#     # Calculate total loss with L1 loss (high weight for inpainting)
#     total_loss = (lambda_adv * adv_loss + lambda_l1 * l1_loss + lambda_rec * rec_loss + lambda_perc * perc_loss +
#                   lambda_style * sty_loss + lambda_fm * fm_loss + lambda_identity * id_loss +
#                   lambda_landmark * lm_loss + lambda_struct * struct_loss + 
#                   lambda_attention * att_loss + lambda_vae * kl_loss)
    
#     # Final NaN check
#     total_loss = torch.nan_to_num(total_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     return total_loss
# def discriminator_loss(disc_real_logits, disc_fake_logits):
#     # Clamp logits to prevent extreme values
#     disc_real_logits = torch.clamp(disc_real_logits, min=-10, max=10)
#     disc_fake_logits = torch.clamp(disc_fake_logits, min=-10, max=10)
    
#     real_loss = torch.mean(F.relu(1.0 - disc_real_logits))
#     real_loss = torch.nan_to_num(real_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     fake_loss = torch.mean(F.relu(1.0 + disc_fake_logits))
#     fake_loss = torch.nan_to_num(fake_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     reg = lambda_d_reg * (torch.mean(disc_real_logits**2) + torch.mean(disc_fake_logits**2))
#     reg = torch.nan_to_num(reg, nan=0.0, posinf=0.0, neginf=0.0)
    
#     total_loss = real_loss + fake_loss + reg
#     total_loss = torch.nan_to_num(total_loss, nan=0.0, posinf=0.0, neginf=0.0)
    
#     return total_loss
# # Evaluation functions
# def calculate_psnr_bottom_half(y_true, y_pred, mask):
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     bottom_mask = (mask == 0).float()
#     y_true_bottom = y_true * bottom_mask
#     y_pred_bottom = y_pred * bottom_mask
#     psnr_vals = []
#     for i in range(y_true.shape[0]):
#         true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         psnr_val = psnr(true_img, pred_img, data_range=1.0)
#         psnr_vals.append(psnr_val)
#     return torch.mean(torch.tensor(psnr_vals, device=device))
# def calculate_ssim_bottom_half(y_true, y_pred, mask):
#     y_true = y_true * 0.5 + 0.5
#     y_pred = y_pred * 0.5 + 0.5
#     bottom_mask = (mask == 0).float()
#     y_true_bottom = y_true * bottom_mask
#     y_pred_bottom = y_pred * bottom_mask
#     ssim_vals = []
#     for i in range(y_true.shape[0]):
#         true_img = y_true_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         pred_img = y_pred_bottom[i].permute(1, 2, 0).cpu().detach().numpy()
#         ssim_val = ssim(true_img, pred_img, channel_axis=2, data_range=1.0, win_size=7)
#         ssim_vals.append(ssim_val)
#     return torch.mean(torch.tensor(ssim_vals, device=device))
# def calculate_mse_bottom_half(y_true, y_pred, mask):
#     return bottom_half_loss(y_true, y_pred, mask)
# # Training function with graph-aware processing and balanced D/G updates
# def train_step(images, coarse_generator, fine_generator, discriminator, g_optimizer, d_optimizer, scaler, step_counter):
#     images = images.to(device)
#     masks = generate_adaptive_mask(images)
#     masked_images = images * masks
   
#     # Discriminator training (n_critic times more frequent)
#     d_optimizer.zero_grad()
    
#     # Train Discriminator multiple times
#     d_loss_accumulated = 0.0
#     for _ in range(n_critic):
#         with amp.autocast('cuda'):
#             # Generate fake images
#             with torch.no_grad():
#                 coarse_output, mu, logvar = multi_resolution_inpainting(masked_images, masks, coarse_generator)
#                 if coarse_output.shape != images.shape:
#                     coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
                
#                 fine_output = fine_generator(masked_images, coarse_output, masks)
#                 if fine_output.shape != images.shape:
#                     fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
                
#                 combined_input = images * masks + fine_output * (1.0 - masks)
            
#             # Discriminator forward
#             disc_real_logits = discriminator(images, masks, return_features=False)
#             disc_fake_logits = discriminator(combined_input.detach(), masks, return_features=False)
#             d_loss = discriminator_loss(disc_real_logits, disc_fake_logits)
        
#         # Discriminator backward
#         scaler.scale(d_loss).backward()
#         d_loss_accumulated += d_loss.item()
    
#     # Update discriminator after n_critic iterations
#     scaler.unscale_(d_optimizer)
#     torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
#     scaler.step(d_optimizer)
    
#     d_loss_val = d_loss_accumulated / n_critic
   
#     # Generator training (once per n_critic discriminator updates)
#     g_loss_val = 0.0
#     if step_counter % n_critic == 0:
#         g_optimizer.zero_grad()
        
#         with amp.autocast('cuda'):
#             # Generator forward pass and loss
#             coarse_output, mu, logvar = multi_resolution_inpainting(masked_images, masks, coarse_generator)
#             if coarse_output.shape != images.shape:
#                 coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
            
#             fine_output = fine_generator(masked_images, coarse_output, masks)
#             if fine_output.shape != images.shape:
#                 fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
            
#             combined_input = images * masks + fine_output * (1.0 - masks)
#             disc_real_logits, disc_features_real = discriminator(images, masks, return_features=True)
#             disc_fake_logits, disc_features_fake = discriminator(combined_input, masks, return_features=True)
#             g_loss = generator_loss(disc_fake_logits, images, fine_output, disc_features_real, disc_features_fake, masks, mu, logvar)
        
#         # Generator backward pass with gradient clipping
#         scaler.scale(g_loss).backward()
#         scaler.unscale_(g_optimizer)
#         torch.nn.utils.clip_grad_norm_(list(coarse_generator.parameters()) + list(fine_generator.parameters()), max_norm=1.0)
#         scaler.step(g_optimizer)
        
#         g_loss_val = g_loss.item()
   
#     # Update scaler after both steps
#     scaler.update()
    
#     # Check for NaN in losses
#     if np.isnan(g_loss_val) or np.isinf(g_loss_val):
#         print(f"⚠️ WARNING: G_Loss is {g_loss_val}, skipping this batch")
#         g_loss_val = 0.0
        
#     if np.isnan(d_loss_val) or np.isinf(d_loss_val):
#         print(f"⚠️ WARNING: D_Loss is {d_loss_val}, skipping this batch")
#         d_loss_val = 0.0
   
#     return g_loss_val, d_loss_val
# # Add multi_resolution_inpainting function
# def multi_resolution_inpainting(image, mask, model, levels=3):
#     outputs = []
#     mus = []
#     logvars = []
#     for level in range(levels):
#         scaled_image = F.interpolate(image, scale_factor=1 / (2 ** level), mode='bilinear')
#         scaled_mask = F.interpolate(mask, scale_factor=1 / (2 ** level), mode='nearest')
        
#         out, mu, logvar = model(scaled_image, scaled_mask)  # Call full forward
        
#         out_up = F.interpolate(out, size=image.shape[-2:], mode='bilinear')
#         outputs.append(out_up)
#         mus.append(mu)
#         logvars.append(logvar)
    
#     final_out = torch.mean(torch.stack(outputs), dim=0)
#     final_mu = torch.mean(torch.stack(mus), dim=0)
#     final_logvar = torch.mean(torch.stack(logvars), dim=0)
#     return final_out, final_mu, final_logvar

# # Add gradient checking function
# def check_model_gradients(model, model_name="Model"):
#     """Check if model has NaN or Inf gradients"""
#     has_nan = False
#     has_inf = False
#     max_grad = 0.0
    
#     for name, param in model.named_parameters():
#         if param.grad is not None:
#             if torch.isnan(param.grad).any():
#                 has_nan = True
#                 print(f"⚠️ NaN gradient in {model_name}.{name}")
#             if torch.isinf(param.grad).any():
#                 has_inf = True
#                 print(f"⚠️ Inf gradient in {model_name}.{name}")
#             max_grad = max(max_grad, param.grad.abs().max().item())
    
#     return has_nan, has_inf, max_grad

# # Add simple retrain function
# def retrain(model, pruned_graph, optimizer):
#     optimizer.zero_grad()
#     # Dummy forward (adjust to use graph if needed)
#     dummy_input = torch.randn(1, 3, 64, 64).to(device)
#     dummy_mask = torch.randn(1, 3, 64, 64).to(device)
#     output, _, _ = model(dummy_input, dummy_mask)
#     loss = torch.mean(output)
#     loss.backward()
#     optimizer.step()
#     return loss.item()

# # Integrate into training loop
# # In train_step function, after coarse_output:
# # Add pruning and retraining every 10 epochs
# # Load dataset

# # celeba_dir = '/kaggle/input/celeba-resized-6464/img_align_celeba/CelebA_Image_Cropped_64'
# celeba_dir = '/kaggle/input/celeba-6464/selected_celeba'

# print(f"Loading dataset from {celeba_dir}...")
# try:
#     print(f"Dataset directory contents: {os.listdir(celeba_dir)[:5]}")
# except Exception as e:
#     print(f"Error accessing dataset directory: {e}")
#     raise
# # Split dataset
# train_files, val_files, test_files = split_dataset(celeba_dir)
# # Create datasets
# train_loader, val_loader, test_loader = create_split_datasets(celeba_dir, train_files, val_files, test_files, image_size, batch_size)
# # Build models
# print("Building models...")
# coarse_generator = CoarseGenerator().to(device)
# fine_generator = FineGenerator().to(device)
# discriminator = Discriminator().to(device)
# print(f"Coarse Generator: {sum(p.numel() for p in coarse_generator.parameters()):,} parameters")
# print(f"Fine Generator: {sum(p.numel() for p in fine_generator.parameters()):,} parameters")
# print(f"Discriminator: {sum(p.numel() for p in discriminator.parameters()):,} parameters")
# # Test model dimensions
# print("\nTesting model dimensions...")
# test_input = torch.randn(1, 3, 64, 64).to(device)
# test_mask = torch.randn(1, 3, 64, 64).to(device)
# with torch.no_grad():
#     coarse_output, mu, logvar = coarse_generator(test_input, test_mask)
#     print(f"Coarse output shape: {coarse_output.shape}")
#     fine_output = fine_generator(test_input, coarse_output, test_mask)
#     print(f"Fine output shape: {fine_output.shape}")
#     print(f"Expected shape: {test_input.shape}")
   
#     # Test discriminator
#     disc_logits, disc_features = discriminator(test_input, test_mask, return_features=True)
#     print(f"Discriminator logits shape: {disc_logits.shape}")
#     print(f"Discriminator features length: {len(disc_features)}")
   
#     disc_logits_only = discriminator(test_input, test_mask, return_features=False)
#     print(f"Discriminator logits only shape: {disc_logits_only.shape}")
   
#     print("✓ Model dimensions are correct!")
# # Enhanced Optimizers with Learning Rate Scheduling and beta1=0.5 for GAN stability
# g_optimizer = torch.optim.AdamW(
#     list(coarse_generator.parameters()) + list(fine_generator.parameters()),
#     lr=learning_rate_g, betas=(0.5, 0.999), weight_decay=1e-4
# )
# d_optimizer = torch.optim.AdamW(
#     discriminator.parameters(),
#     lr=learning_rate_d, betas=(0.5, 0.999), weight_decay=1e-4
# )
# print(f"Optimizer settings: G_LR={learning_rate_g}, D_LR={learning_rate_d}, beta1=0.5, n_critic={n_critic}")
# # Learning rate schedulers
# g_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     g_optimizer, T_0=10, T_mult=2, eta_min=learning_rate_g * 0.01
# )
# d_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     d_optimizer, T_0=10, T_mult=2, eta_min=learning_rate_d * 0.01
# )
# scaler = amp.GradScaler('cuda')
# # Model health check before training
# print("\n" + "="*60)
# print("🔍 Model Health Check")
# print("="*60)
# test_batch = next(iter(train_loader)).to(device)
# test_masks = generate_adaptive_mask(test_batch)
# test_masked = test_batch * test_masks

# print("Testing forward pass...")
# with torch.no_grad():
#     try:
#         coarse_out, mu, logvar = coarse_generator(test_masked, test_masks)
#         print(f"✓ Coarse Generator: output shape {coarse_out.shape}")
#         print(f"  mu range: [{mu.min().item():.4f}, {mu.max().item():.4f}]")
#         print(f"  logvar range: [{logvar.min().item():.4f}, {logvar.max().item():.4f}]")
        
#         fine_out = fine_generator(test_masked, coarse_out, test_masks)
#         print(f"✓ Fine Generator: output shape {fine_out.shape}")
#         print(f"  output range: [{fine_out.min().item():.4f}, {fine_out.max().item():.4f}]")
        
#         disc_out = discriminator(test_batch, test_masks)
#         print(f"✓ Discriminator: logits shape {disc_out.shape}")
#         print(f"  logits range: [{disc_out.min().item():.4f}, {disc_out.max().item():.4f}]")
        
#         print("✅ All models are healthy!")
#     except Exception as e:
#         print(f"❌ Model health check failed: {e}")
#         raise
# print("="*60 + "\n")

# # ============================================================================
# # CHECKPOINT LOADING & TRAINING SETUP
# # بارگذاری Checkpoint و راه‌اندازی آموزش
# # ============================================================================

# print("\n" + "="*70)
# print("🚀 STARTING TRAINING SETUP")
# print("="*70)

# # Adjust paths if loading from Kaggle input dataset
# if USE_KAGGLE_INPUT:
#     checkpoint_load_dir = os.path.join(KAGGLE_INPUT_DATASET, 'checkpoints')
#     models_load_dir = os.path.join(KAGGLE_INPUT_DATASET, 'models')
#     print(f"📥 Loading from Kaggle dataset: {KAGGLE_INPUT_DATASET}")
# else:
#     checkpoint_load_dir = CHECKPOINT_DIR
#     models_load_dir = MODELS_DIR
#     print(f"📥 Loading from working directory")

# # Create checkpoint save directory (always in /kaggle/working)
# os.makedirs(CHECKPOINT_DIR, exist_ok=True)
# os.makedirs(MODELS_DIR, exist_ok=True)
# print(f"💾 Checkpoints will be saved to: {CHECKPOINT_DIR}")
# print(f"💾 Models will be saved to: {MODELS_DIR}")

# # Initialize training variables
# start_epoch = 0
# best_val_loss = float('inf')
# train_g_losses = []
# train_d_losses = []
# val_g_losses = []
# val_d_losses = []

# # Try to load checkpoint/models if RESUME_TRAINING is True
# checkpoint_loaded = False
# models_loaded = False

# if RESUME_TRAINING:
#     # ========================================================================
#     # Method 1: Try to load from MODEL FILES (state_dict only)
#     # روش 1: بارگذاری از فایل‌های MODEL (فقط state_dict)
#     # ========================================================================
    
#     if LOAD_FROM_MODELS and not checkpoint_loaded:
#         print(f"\n🔍 Method 1: Searching for model files...")
#         print(f"📂 Models directory: {models_load_dir}")
        
#         # Debug: Show what files actually exist
#         if os.path.exists(models_load_dir):
#             print(f"\n📄 Files found in directory:")
#             actual_files = os.listdir(models_load_dir)
#             if actual_files:
#                 for f in sorted(actual_files):
#                     if f.endswith('.pth'):
#                         file_path = os.path.join(models_load_dir, f)
#                         size_mb = os.path.getsize(file_path) / (1024**2)
#                         print(f"   • {f} ({size_mb:.2f} MB)")
#             else:
#                 print(f"   (empty)")
#         else:
#             print(f"   ❌ Directory does not exist!")
        
#         # Determine which models to load
#         suffix = "_best.pth" if USE_BEST_MODELS else "_bottom_half.pth"
        
#         model_files = {
#             'coarse': os.path.join(models_load_dir, f'coarse_generator{suffix}'),
#             'fine': os.path.join(models_load_dir, f'fine_generator{suffix}'),
#             'disc': os.path.join(models_load_dir, f'discriminator{suffix}')
#         }
        
#         print(f"\n🔍 Looking for (USE_BEST_MODELS={USE_BEST_MODELS}):")
#         for name, path in model_files.items():
#             exists = "✅" if os.path.exists(path) else "❌"
#             print(f"   {exists} {os.path.basename(path)}")
        
#         # Check if all model files exist
#         all_exist = all(os.path.exists(path) for path in model_files.values())
        
#         if all_exist:
#             try:
#                 print(f"📂 Found model files in: {models_load_dir}")
#                 print(f"   • coarse_generator{suffix}")
#                 print(f"   • fine_generator{suffix}")
#                 print(f"   • discriminator{suffix}")
#                 print(f"⏳ Loading model weights...")
                
#                 # Load model weights
#                 # Load model weights tolerantly in case of architecture drift
#                 coarse_sd = torch.load(model_files['coarse'], map_location=device)
#                 fine_sd = torch.load(model_files['fine'], map_location=device)
#                 disc_sd = torch.load(model_files['disc'], map_location=device)

#                 missing, unexpected = coarse_generator.load_state_dict(coarse_sd, strict=False)
#                 if missing or unexpected:
#                     print("[coarse_generator] non-strict load:")
#                     if missing:
#                         print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                     if unexpected:
#                         print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                 missing, unexpected = fine_generator.load_state_dict(fine_sd, strict=False)
#                 if missing or unexpected:
#                     print("[fine_generator] non-strict load:")
#                     if missing:
#                         print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                     if unexpected:
#                         print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                 missing, unexpected = discriminator.load_state_dict(disc_sd, strict=False)
#                 if missing or unexpected:
#                     print("[discriminator] non-strict load:")
#                     if missing:
#                         print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                     if unexpected:
#                         print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
                
#                 models_loaded = True
                
#                 print("="*70)
#                 print("✅ MODEL WEIGHTS LOADED SUCCESSFULLY!")
#                 print("="*70)
#                 print(f"📊 Loading Information:")
#                 print(f"   • Model weights: ✅ Loaded")
#                 print(f"   • Optimizer states: ⚠️  Will be initialized (not in model files)")
#                 print(f"   • Scheduler states: ⚠️  Will be initialized (not in model files)")
#                 print(f"   • Training history: ⚠️  Starting fresh (not in model files)")
#                 print(f"   • Starting epoch: 1 (training continues with loaded weights)")
#                 print(f"\n⚠️  NOTE: Model files only contain weights, not optimizer/scheduler.")
#                 print(f"   Training will continue with these weights but fresh optimizer.")
#                 print("="*70)
                
#             except Exception as e:
#                 print(f"⚠️  Failed to load model files")
#                 print(f"   Error: {str(e)}")
#                 models_loaded = False
#         else:
#             print(f"⚠️  Not all model files found in {models_load_dir}")
#             for name, path in model_files.items():
#                 exists = "✓" if os.path.exists(path) else "✗"
#                 print(f"   {exists} {os.path.basename(path)}")
    
#     # ========================================================================
#     # Method 2: Try to load from CHECKPOINT FILES (full checkpoint)
#     # روش 2: بارگذاری از فایل‌های CHECKPOINT (checkpoint کامل)
#     # ========================================================================
    
#     if not checkpoint_loaded and not models_loaded:
#         print(f"\n🔍 Method 2: Searching for checkpoint files...")
        
#         # Try different checkpoint sources
#         checkpoint_paths_to_try = [
#             os.path.join(checkpoint_load_dir, 'latest_checkpoint.pth'),
#             os.path.join(checkpoint_load_dir, f'checkpoint_epoch_{num_epochs}.pth'),
#             os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth'),  # Fallback to working dir
#         ]
        
#         for checkpoint_path in checkpoint_paths_to_try:
#             if os.path.exists(checkpoint_path):
#                 try:
#                     print(f"📂 Found checkpoint: {checkpoint_path}")
#                     print(f"⏳ Loading checkpoint...")
                    
#                     checkpoint = torch.load(checkpoint_path, map_location=device)
                    
#                     # Load model states tolerantly
#                     missing, unexpected = coarse_generator.load_state_dict(checkpoint['coarse_generator'], strict=False)
#                     if missing or unexpected:
#                         print("[coarse_generator ckpt] non-strict load:")
#                         if missing:
#                             print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                         if unexpected:
#                             print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                     missing, unexpected = fine_generator.load_state_dict(checkpoint['fine_generator'], strict=False)
#                     if missing or unexpected:
#                         print("[fine_generator ckpt] non-strict load:")
#                         if missing:
#                             print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                         if unexpected:
#                             print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")

#                     missing, unexpected = discriminator.load_state_dict(checkpoint['discriminator'], strict=False)
#                     if missing or unexpected:
#                         print("[discriminator ckpt] non-strict load:")
#                         if missing:
#                             print(f"  missing keys: {len(missing)} (showing up to 10): {missing[:10]}")
#                         if unexpected:
#                             print(f"  unexpected keys: {len(unexpected)} (showing up to 10): {unexpected[:10]}")
                    
#                     # Load optimizer states
#                     g_optimizer.load_state_dict(checkpoint['g_optimizer'])
#                     d_optimizer.load_state_dict(checkpoint['d_optimizer'])
                    
#                     # Load scheduler states
#                     g_scheduler.load_state_dict(checkpoint['g_scheduler'])
#                     d_scheduler.load_state_dict(checkpoint['d_scheduler'])
                    
#                     # Load training progress
#                     start_epoch = checkpoint['epoch'] + 1
#                     best_val_loss = checkpoint.get('best_val_loss', float('inf'))
#                     train_g_losses = checkpoint.get('train_g_losses', [])
#                     train_d_losses = checkpoint.get('train_d_losses', [])
#                     val_g_losses = checkpoint.get('val_g_losses', [])
#                     val_d_losses = checkpoint.get('val_d_losses', [])
                    
#                     checkpoint_loaded = True
                    
#                     print("="*70)
#                     print("✅ FULL CHECKPOINT LOADED SUCCESSFULLY!")
#                     print("="*70)
#                     print(f"📊 Resume Information:")
#                     print(f"   • Model weights: ✅ Loaded")
#                     print(f"   • Optimizer states: ✅ Loaded")
#                     print(f"   • Scheduler states: ✅ Loaded")
#                     print(f"   • Starting from epoch: {start_epoch + 1}")
#                     print(f"   • Best validation loss: {best_val_loss:.6f}")
#                     print(f"   • Training history: {len(train_g_losses)} epochs")
#                     print(f"   • Remaining epochs: {num_epochs - start_epoch}")
#                     print("="*70)
                    
#                     break  # Successfully loaded, exit loop
                    
#                 except Exception as e:
#                     print(f"⚠️  Failed to load checkpoint from {checkpoint_path}")
#                     print(f"   Error: {str(e)}")
#                     print(f"   Trying next checkpoint source...")
#                     continue

# # ========================================================================
# # If nothing loaded, start fresh
# # اگر چیزی بارگذاری نشد، از اول شروع کن
# # ========================================================================

# if not checkpoint_loaded and not models_loaded:
#     if RESUME_TRAINING:
#         print("\n" + "="*70)
#         print("⚠️  NO CHECKPOINT OR MODELS FOUND - Starting from scratch")
#         print("="*70)
#         print("📝 Locations searched:")
#         print(f"\n   Models folder: {models_load_dir}")
#         if os.path.exists(models_load_dir):
#             files = os.listdir(models_load_dir)
#             if files:
#                 print(f"   Found {len(files)} file(s):")
#                 for f in files[:5]:  # Show first 5
#                     print(f"      • {f}")
#             else:
#                 print(f"      (empty)")
#         else:
#             print(f"      (not found)")
        
#         print(f"\n   Checkpoints folder: {checkpoint_load_dir}")
#         if os.path.exists(checkpoint_load_dir):
#             files = [f for f in os.listdir(checkpoint_load_dir) if f.endswith('.pth')]
#             if files:
#                 print(f"   Found {len(files)} checkpoint(s):")
#                 for f in files[:5]:
#                     print(f"      • {f}")
#             else:
#                 print(f"      (empty)")
#         else:
#             print(f"      (not found)")
        
#         print("\n💡 To resume training in next run:")
#         print("   1. Make sure model/checkpoint files exist")
#         print("   2. Set LOAD_FROM_MODELS = True for model files")
#         print("   3. If using Kaggle dataset, set USE_KAGGLE_INPUT = True")
#         print("   4. Update MODELS_DIR or KAGGLE_INPUT_DATASET path")
#         print("="*70)
#     else:
#         print("\n" + "="*70)
#         print("🆕 STARTING FRESH TRAINING")
#         print("="*70)
#         print("   RESUME_TRAINING is set to False")
#         print("   Training will start from epoch 1")
#         print("="*70)

# print(f"\n🎯 Training will run from epoch {start_epoch + 1} to {num_epochs}")
# print(f"💾 Checkpoints will be saved every {CHECKPOINT_INTERVAL} epoch(s)")
# print("="*70 + "\n")

# for epoch in range(start_epoch, num_epochs):
#     print(f"Epoch {epoch+1}/{num_epochs}")
#     coarse_generator.train()
#     fine_generator.train()
#     discriminator.train()
#     epoch_g_loss = epoch_d_loss = num_batches = 0
   
#     for step, images in enumerate(train_loader):
#         # Pass step counter for balanced D/G updates
#         g_loss, d_loss = train_step(images, coarse_generator, fine_generator, discriminator, g_optimizer, d_optimizer, scaler, step)
#         epoch_g_loss += g_loss
#         epoch_d_loss += d_loss
#         num_batches += 1
        
#         # Enhanced monitoring
#         if step % 50 == 0:
#             current_g_lr = g_optimizer.param_groups[0]['lr']
#             current_d_lr = d_optimizer.param_groups[0]['lr']
#             g_or_d = "G+D" if step % n_critic == 0 else "D only"
#             print(f" Step {step:3d} [{g_or_d}]: G_Loss: {g_loss:.4f}, D_Loss: {d_loss:.4f} | LR: G={current_g_lr:.6f}, D={current_d_lr:.6f}")
            
#         # Emergency stop if losses explode
#         if np.isnan(g_loss) and np.isnan(d_loss):
#             print(f"⚠️ CRITICAL: Both losses are NaN at step {step}. Stopping epoch early.")
#             break
   
#     avg_g_loss = epoch_g_loss / num_batches if num_batches > 0 else 0
#     avg_d_loss = epoch_d_loss / num_batches if num_batches > 0 else 0
#     train_g_losses.append(avg_g_loss)
#     train_d_losses.append(avg_d_loss)
#     print(f"Epoch {epoch+1} Training - G_Loss: {avg_g_loss:.4f}, D_Loss: {avg_d_loss:.4f}")
   
#     # Update learning rates
#     g_scheduler.step()
#     d_scheduler.step()
#     current_g_lr = g_optimizer.param_groups[0]['lr']
#     current_d_lr = d_optimizer.param_groups[0]['lr']
#     print(f" Current LR - G: {current_g_lr:.6f}, D: {current_d_lr:.6f}")
   
#     # Enable MTCNN for validation
#     if 'MTCNN' in globals():
#         USE_MTCNN = True
   
#     if epoch % 10 == 0:
#         # Get sample batch for graph creation
#         sample_images = next(iter(train_loader)).to(device)
#         sample_masks = generate_adaptive_mask(sample_images)
#         sample_masked = sample_images * sample_masks
        
#         # Get features from preprocessor
#         features = coarse_generator.preprocessor(sample_masked)
        
#         # Build graph
#         sample_graph = coarse_generator.preprocessor.build_graph(features)
        
#         pruned_graph = prune_graph(coarse_generator, sample_graph)
        
#         # Retrain
#         for _ in range(5):
#             retrain_loss = retrain(coarse_generator, pruned_graph, g_optimizer)
   
#     coarse_generator.eval()
#     fine_generator.eval()
#     discriminator.eval()
#     val_g_loss = val_d_loss = val_batches = 0
#     with torch.no_grad():
#         for images in val_loader:
#             images = images.to(device)
#             masks = generate_adaptive_mask(images)
#             masked_images = images * masks
#             coarse_output, mu, logvar = coarse_generator(masked_images, masks)
#             # Debug: Check shapes
#             if coarse_output.shape != images.shape:
#                 coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
           
#             fine_output = fine_generator(masked_images, coarse_output, masks)
#             # Debug: Check shapes
#             if fine_output.shape != images.shape:
#                 fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
           
#             combined_input = images * masks + fine_output * (1.0 - masks)
           
#             disc_real_logits, disc_features_real = discriminator(images, masks, return_features=True)
#             disc_fake_logits, disc_features_fake = discriminator(combined_input, masks, return_features=True)
           
#             g_loss = generator_loss(disc_fake_logits, images, fine_output, disc_features_real, disc_features_fake, masks, mu, logvar)
#             d_loss = discriminator_loss(disc_real_logits, disc_fake_logits)
           
#             val_g_loss += g_loss.item()
#             val_d_loss += d_loss.item()
#             val_batches += 1
   
#     avg_val_g_loss = val_g_loss / val_batches if val_batches > 0 else 0
#     avg_val_d_loss = val_d_loss / val_batches if val_batches > 0 else 0
#     val_g_losses.append(avg_val_g_loss)
#     val_d_losses.append(avg_val_d_loss)
#     print(f"Epoch {epoch+1} Validation - G_Loss: {avg_val_g_loss:.4f}, D_Loss: {avg_val_d_loss:.4f}")
   
#     if avg_val_g_loss < best_val_loss:
#         best_val_loss = avg_val_g_loss
#         print(f" New best validation loss! Saving best models...")
#         os.makedirs('/kaggle/working/models', exist_ok=True)
#         torch.save(coarse_generator.state_dict(), "/kaggle/working/models/coarse_generator_best.pth")
#         torch.save(fine_generator.state_dict(), "/kaggle/working/models/fine_generator_best.pth")
#         torch.save(discriminator.state_dict(), "/kaggle/working/models/discriminator_best.pth")
   
#     # Save checkpoint every CHECKPOINT_INTERVAL epochs
#     if (epoch + 1) % CHECKPOINT_INTERVAL == 0 or (epoch + 1) == num_epochs:
#         print(f"\n{'='*70}")
#         print(f"💾 SAVING CHECKPOINT - Epoch {epoch+1}/{num_epochs}")
#         print(f"{'='*70}")
        
#         checkpoint_file = os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch+1}.pth')
#         latest_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pth')
        
#         checkpoint = {
#             'epoch': epoch,
#             'coarse_generator': coarse_generator.state_dict(),
#             'fine_generator': fine_generator.state_dict(),
#             'discriminator': discriminator.state_dict(),
#             'g_optimizer': g_optimizer.state_dict(),
#             'd_optimizer': d_optimizer.state_dict(),
#             'g_scheduler': g_scheduler.state_dict(),
#             'd_scheduler': d_scheduler.state_dict(),
#             'best_val_loss': best_val_loss,
#             'train_g_losses': train_g_losses,
#             'train_d_losses': train_d_losses,
#             'val_g_losses': val_g_losses,
#             'val_d_losses': val_d_losses,
#         }
        
#         # Save numbered checkpoint
#         torch.save(checkpoint, checkpoint_file)
#         checkpoint_size = os.path.getsize(checkpoint_file) / (1024**2)  # MB
#         print(f"✅ Saved: {checkpoint_file}")
#         print(f"   Size: {checkpoint_size:.2f} MB")
        
#         # Also save as latest checkpoint
#         torch.save(checkpoint, latest_checkpoint_path)
#         print(f"✅ Saved: {latest_checkpoint_path}")
        
#         print(f"\n📊 Checkpoint Info:")
#         print(f"   • Completed epochs: {epoch + 1}")
#         print(f"   • Best val loss: {best_val_loss:.6f}")
#         print(f"   • Current G loss: {avg_g_loss:.6f}")
#         print(f"   • Current D loss: {avg_d_loss:.6f}")
        
#         if (epoch + 1) < num_epochs:
#             print(f"\n🔄 To resume from this checkpoint in next run:")
#             print(f"   1. Download '{CHECKPOINT_DIR}' folder from Kaggle Output")
#             print(f"   2. Upload as Kaggle Dataset (or keep in working directory)")
#             print(f"   3. In code, set: RESUME_TRAINING = True")
#             print(f"   4. If using dataset, set: USE_KAGGLE_INPUT = True")
#             print(f"   5. Update: KAGGLE_INPUT_DATASET = '/kaggle/input/your-dataset-name'")
#         else:
#             print(f"\n🎉 TRAINING COMPLETED!")
#             print(f"   All {num_epochs} epochs finished successfully!")
        
#         print(f"{'='*70}\n")
   
#     # Plot training progress every 5 epochs
#     if (epoch + 1) % 5 == 0:
#         plt.figure(figsize=(15, 5))
       
#         plt.subplot(1, 3, 1)
#         plt.plot(range(1, len(train_g_losses) + 1), train_g_losses, label='Generator Loss', color='blue')
#         plt.plot(range(1, len(val_g_losses) + 1), val_g_losses, label='Val Generator Loss', color='lightblue')
#         plt.title('Generator Loss Progress')
#         plt.xlabel('Epoch')
#         plt.ylabel('Loss')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
       
#         plt.subplot(1, 3, 2)
#         plt.plot(range(1, len(train_d_losses) + 1), train_d_losses, label='Discriminator Loss', color='red')
#         plt.plot(range(1, len(val_d_losses) + 1), val_d_losses, label='Val Discriminator Loss', color='lightcoral')
#         plt.title('Discriminator Loss Progress')
#         plt.xlabel('Epoch')
#         plt.ylabel('Loss')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
       
#         plt.subplot(1, 3, 3)
#         plt.plot(range(1, len(train_g_losses) + 1), train_g_losses, label='G Loss', color='blue')
#         plt.plot(range(1, len(train_d_losses) + 1), train_d_losses, label='D Loss', color='red')
#         plt.title('Training Loss Comparison')
#         plt.xlabel('Epoch')
#         plt.ylabel('Loss')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
       
#         plt.tight_layout()
#         plt.savefig(f'/kaggle/working/training_progress_epoch_{epoch+1}.png', dpi=150, bbox_inches='tight')
#         plt.show()
# # Save models
# print("Saving models...")
# os.makedirs('/kaggle/working/models', exist_ok=True)
# torch.save(coarse_generator.state_dict(), "/kaggle/working/models/coarse_generator_bottom_half.pth")
# torch.save(fine_generator.state_dict(), "/kaggle/working/models/fine_generator_bottom_half.pth")
# torch.save(discriminator.state_dict(), "/kaggle/working/models/discriminator_bottom_half.pth")
# # Enable MTCNN for evaluation
# if 'MTCNN' in globals():
#     USE_MTCNN = True
# # Evaluate on test set
# print("Starting evaluation on test set...")
# os.makedirs('/kaggle/working/results', exist_ok=True)
# coarse_generator.eval()
# fine_generator.eval()
# discriminator.eval()
# psnr_values, ssim_values, mse_values, identity_values, landmark_values = [], [], [], [], []
# num_samples = 10 # Increased for better evaluation
# plt.figure(figsize=(20, 10))
# with torch.no_grad():
#     for i, images in enumerate(test_loader):
#         if i >= num_samples:
#             break
#         images = images.to(device)
#         masks = generate_adaptive_mask(images)
#         masked_images = images * masks
#         coarse_output, mu, logvar = coarse_generator(masked_images, masks)
#         # Debug: Check shapes
#         if coarse_output.shape != images.shape:
#             coarse_output = F.interpolate(coarse_output, size=images.shape[2:], mode='bilinear', align_corners=False)
       
#         fine_output = fine_generator(masked_images, coarse_output, masks)
#         # Debug: Check shapes
#         if fine_output.shape != images.shape:
#             fine_output = F.interpolate(fine_output, size=images.shape[2:], mode='bilinear', align_corners=False)
       
#         reconstructed = images * masks + fine_output * (1.0 - masks)
       
#         psnr_val = calculate_psnr_bottom_half(images, reconstructed, masks).item()
#         ssim_val = calculate_ssim_bottom_half(images, reconstructed, masks).item()
#         mse_val = calculate_mse_bottom_half(images, reconstructed, masks).item()
#         identity_val = identity_loss(images, reconstructed, masks).item()
#         landmark_val = landmark_guided_loss(images, reconstructed, masks).item()
       
#         psnr_values.append(psnr_val)
#         ssim_values.append(ssim_val)
#         mse_values.append(mse_val)
#         identity_values.append(identity_val)
#         landmark_values.append(landmark_val)
       
#         plt.subplot(4, num_samples, i + 1)
#         plt.title("Original Image")
#         plt.imshow(images[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#         plt.axis("off")
       
#         plt.subplot(4, num_samples, num_samples + i + 1)
#         plt.title("Top Half (Input)")
#         plt.imshow(masked_images[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#         plt.axis("off")
       
#         plt.subplot(4, num_samples, 2*num_samples + i + 1)
#         plt.title("Reconstructed Bottom Half")
#         plt.imshow(reconstructed[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
#         plt.axis("off")
       
#         diff = torch.abs(images - reconstructed) * (1 - masks)
#         plt.subplot(4, num_samples, 3*num_samples + i + 1)
#         plt.title("Difference Map")
#         plt.imshow(diff[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5, cmap='hot')
#         plt.axis("off")
       
#         print(f"Sample {i+1}: PSNR: {psnr_val:.4f}, SSIM: {ssim_val:.4f}, MSE: {mse_val:.4f}, Identity: {identity_val:.4f}, Landmark: {landmark_val:.4f}")
# plt.tight_layout()
# plt.savefig('/kaggle/working/results/bottom_half_reconstruction_results.png', dpi=150, bbox_inches='tight')
# plt.show()
# # Plot metrics
# plt.figure(figsize=(20, 10))
# plt.subplot(2, 3, 1)
# plt.plot(range(1, num_samples + 1), psnr_values, marker='o', linewidth=2, markersize=8)
# plt.title("PSNR (Bottom Half)", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 2)
# plt.plot(range(1, num_samples + 1), ssim_values, marker='s', linewidth=2, markersize=8, color='orange')
# plt.title("SSIM (Bottom Half)", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 3)
# plt.plot(range(1, num_samples + 1), mse_values, marker='^', linewidth=2, markersize=8, color='green')
# plt.title("MSE (Bottom Half)", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 4)
# plt.plot(range(1, num_samples + 1), identity_values, marker='d', linewidth=2, markersize=8, color='purple')
# plt.title("Identity Loss", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.subplot(2, 3, 5)
# plt.plot(range(1, num_samples + 1), landmark_values, marker='*', linewidth=2, markersize=8, color='brown')
# plt.title("Landmark Loss", fontsize=14)
# plt.xlabel("Sample", fontsize=12)
# plt.ylabel("Value", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.savefig('/kaggle/working/results/bottom_half_metrics.png', dpi=150, bbox_inches='tight')
# plt.show()
# # Print and save evaluation results
# print("\n" + "="*60)
# print("Enhanced Evaluation Metrics (Bottom Half) - Test Set:")
# print("="*60)
# print(f"Average PSNR: {np.mean(psnr_values):.4f} dB (±{np.std(psnr_values):.4f})")
# print(f"Average SSIM: {np.mean(ssim_values):.4f} (±{np.std(ssim_values):.4f})")
# print(f"Average MSE: {np.mean(mse_values):.4f} (±{np.std(mse_values):.4f})")
# print(f"Average Identity Loss: {np.mean(identity_values):.4f} (±{np.std(identity_values):.4f})")
# print(f"Average Landmark Loss: {np.mean(landmark_values):.4f} (±{np.std(landmark_values):.4f})")
# print("="*60)
# print(f"Best PSNR: {np.max(psnr_values):.4f} dB")
# print(f"Best SSIM: {np.max(ssim_values):.4f}")
# print(f"Worst PSNR: {np.min(psnr_values):.4f} dB")
# print(f"Worst SSIM: {np.min(ssim_values):.4f}")
# print("="*60)
# with open('/kaggle/working/results/enhanced_bottom_half_metrics.txt', 'w', encoding='utf-8') as f:
#     f.write("Enhanced Bottom Half Face Reconstruction Evaluation Results - Test Set\n")
#     f.write("="*60 + "\n")
#     f.write(f"Average PSNR: {np.mean(psnr_values):.4f} dB (±{np.std(psnr_values):.4f})\n")
#     f.write(f"Average SSIM: {np.mean(ssim_values):.4f} (±{np.std(ssim_values):.4f})\n")
#     f.write(f"Average MSE: {np.mean(mse_values):.4f} (±{np.std(mse_values):.4f})\n")
#     f.write(f"Average Identity Loss: {np.mean(identity_values):.4f} (±{np.std(identity_values):.4f})\n")
#     f.write(f"Average Landmark Loss: {np.mean(landmark_values):.4f} (±{np.std(landmark_values):.4f})\n")
#     f.write("="*60 + "\n")
#     f.write(f"Best PSNR: {np.max(psnr_values):.4f} dB\n")
#     f.write(f"Best SSIM: {np.max(ssim_values):.4f}\n")
#     f.write(f"Worst PSNR: {np.min(psnr_values):.4f} dB\n")
#     f.write(f"Worst SSIM: {np.min(ssim_values):.4f}\n")
#     f.write("="*60 + "\n")
#     for i, (psnr_val, ssim_val, mse_val, id_val, lm_val) in enumerate(zip(psnr_values, ssim_values, mse_values, identity_values, landmark_values)):
#         f.write(f"Sample {i+1}: PSNR={psnr_val:.4f}, SSIM={ssim_val:.4f}, MSE={mse_val:.4f}, Identity={id_val:.4f}, Landmark={lm_val:.4f}\n")
# print(f"\nTraining and evaluation complete!")
# print(f"Models saved in '/kaggle/working/models/' directory.")
# print(f"Evaluation results saved in '/kaggle/working/results/' directory.")